# 🔬 THỬ NGHIỆM CÁCH LY PHA 4: OCCLUSION-AWARE HEAD (TPSInbetweenOcclusion)
## Warm-Start 100% từ Phase 3 Remake Best | 6 Epochs | 2x GPU Tesla T4 (DDP)
### Tên mã: `Phase4_Occlusion` / `M4_Occ`

---

### 🎯 1. Mục Tiêu & Cải Tiến Kỹ Thuật Độc Lập
1. **Kiến trúc Occlusion Head trong IFNet.block2:**
   - Lớp `lastconv` xuất 6 kênh: $[F_{0\to t}^{(2)}, F_{1\to t}^{(2)}, M_{0\to t}^{(1)}, M_{1\to t}^{(1)}]$.
   - Chuẩn hóa 2 mặt nạ qua hàm $\text{Sigmoid} \in [0.0, 1.0]$.
2. **Bộ Trộn Trọng số Chuẩn hóa (Normalized Visibility Blending):**
   $$W_0 = (1 - t) \cdot M_{0\to t}, \quad W_1 = t \cdot M_{1\to t}$$
   $$I_{\text{fused}} = \frac{W_0 \cdot \text{Warp}(I_0, F_{0\to t}) + W_1 \cdot \text{Warp}(I_1, F_{1\to t})}{W_0 + W_1 + \epsilon}$$
3. **Warm-Start mượt mà tuyệt đối (Zero Loss Shock):**
   - Kế thừa trọn vẹn 158 tensors từ checkpoint `model_tps_finetuned_best.pt` của Pha 3 Remake.
   - Khởi tạo bias 2 kênh mặt nạ che khuất $= +2.0$ ($\text{Sigmoid}(+2.0) \approx 0.8808$, coi ban đầu mọi vùng đều nhìn thấy được).
4. **Hàm Mất Mát Nhận Thức Che Khuất (OcclusionAwareLoss):**
   - $L_{\text{photo}} = M_{0\to t} \cdot \rho(I_{0\to t} - I_t^{\text{GT}}) + M_{1\to t} \cdot \rho(I_{1\to t} - I_t^{\text{GT}})$.
   - $L_{\text{cov}} = \text{mean}\left(\text{ReLU}(0.95 - (M_{0\to t} + M_{1\to t}))^2\right)$ (chống hiện tượng mạng lười triệt tiêu $M \to 0$).
   - $L_{\text{smooth}}$: Làm mượt mặt nạ có hướng dẫn biên nét ảnh GT.

In [ ]:
# ==============================================================================
# 🛠️ [CELL 1]: KHỞI TẠO MÔI TRƯỜNG & NHÚNG MÃ NGUỒN PHA 4 OCCLUSION
# ==============================================================================
import os
import sys
import gc
import base64
import torch
import psutil

os.environ["MALLOC_TRIM_THRESHOLD_"] = "65536"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

try:
    import cv2
    cv2.setNumThreads(0)
    cv2.ocl.setUseOpenCL(False)
except ImportError:
    pass

torch.backends.cudnn.benchmark = True

IS_KAGGLE = os.path.exists("/kaggle")
WORKING_DIR = "/kaggle/working" if IS_KAGGLE else os.path.abspath("./kaggle_local_test_ws")
OUTPUT_CKPT_DIR = os.path.join(WORKING_DIR, "checkpoints_phase4_occlusion")
BENCH_OUT_DIR = os.path.join(WORKING_DIR, "benchmark_outputs_phase4_occlusion")
for _d in (WORKING_DIR, OUTPUT_CKPT_DIR, BENCH_OUT_DIR):
    os.makedirs(_d, exist_ok=True)

if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)

gpu_count = torch.cuda.device_count()
print("=" * 84)
print(f"🖥️ KIỂM TRA PHẦN CỨNG: {gpu_count} GPU khả dụng (Kaggle={IS_KAGGLE})")
for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    print(f"   👉 GPU {i}: {p.name} | VRAM = {p.total_memory / (1024**3):.2f} GB")
mem = psutil.virtual_memory()
print(f"🧠 Host RAM: Tổng = {mem.total / (1024**3):.1f} GB | Trống = {mem.available / (1024**3):.1f} GB")
print("=" * 84)

# Ghi các module cách ly của Phase 4
model_occ_path = os.path.join(WORKING_DIR, "model_occlusion.py")
with open(model_occ_path, "w", encoding="utf-8") as f:
    f.write('#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n"""\nmodel_occlusion.py\n--------------------------------------------------------------------------------\nKIẾN TRÚC MÔ HÌNH PHA 4: COARSE TPS + OCCLUSION-AWARE HEAD (TPSInbetweenOcclusion)\nThử nghiệm Cách ly: Phase 4 Occlusion Head (M4_Occ)\nKhông ảnh hưởng 100% tới Phase 3 Remake (Phase3_Remake / M3_Best).\n\nCác nâng cấp cốt lõi:\n1. Occlusion Head trong IFNet.block2:\n   Lớp ConvTranspose2d cuối cùng xuất 6 kênh:\n     - 4 kênh luồng quang học tinh chỉnh: [F0->t (2), F1->t (2)]\n     - 2 kênh mặt nạ che khuất: [M0->t (1), M1->t (1)] qua Sigmoid in [0.0, 1.0].\n2. Bộ Trộn Trọng số Chuẩn hóa (Normalized Visibility Blending):\n   W0 = (1 - t) * M0->t,  W1 = t * M1->t\n   I_fused = (W0 * Warp(I0, F0->t) + W1 * Warp(I1, F1->t)) / (W0 + W1 + eps)\n   res = unet(...) * 2.0 - 1.0\n   It = clamp(I_fused + res, 0.0, 1.0)\n3. Khởi tạo Trọng số (Warm-Start):\n   Kế thừa toàn bộ 158 tensors từ checkpoint Phase 3 Remake (model_tps_finetuned_best.pt).\n   2 kênh mặt nạ mới được khởi tạo bias = +2.0 (Sigmoid ≈ 0.88), weights = 0.0,\n   đảm bảo tại bước 0 mô hình hội tụ mượt mà, zero loss shock.\n4. OcclusionAwareLoss:\n   - Photometric loss có trọng số hiển thị: L_photo = M0*rho(I0t - GT) + M1*rho(I1t - GT)\n   - Coverage constraint: L_cov = mean(ReLU(0.95 - (M0 + M1))^2)\n   - Edge-aware smoothness: L_smooth giữ biên mặt nạ bám sát nét vẽ GT.\n--------------------------------------------------------------------------------\n"""\n\nimport os\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n_GRID_CACHE = {}\n_PIXEL_GRID_CACHE = {}\n\n\ndef warp_normalized_flow(img, flow):\n    """Backward-warp theo luồng Coarse TPS (tọa độ chuẩn hóa [0, 1] -> [-1, 1])."""\n    _, _, h, w = img.shape\n    key = (h, w, flow.device)\n    base_grid = _GRID_CACHE.get(key)\n    if base_grid is None:\n        gy, gx = torch.meshgrid(\n            torch.linspace(0.0, 1.0, h, device=flow.device, dtype=torch.float32),\n            torch.linspace(0.0, 1.0, w, device=flow.device, dtype=torch.float32),\n            indexing=\'ij\'\n        )\n        base_grid = torch.stack([gx, gy], dim=-1).unsqueeze(0).contiguous()\n        _GRID_CACHE[key] = base_grid\n\n    with torch.amp.autocast(\'cuda\', enabled=False):\n        grid = (base_grid + flow.float().permute(0, 2, 3, 1)) * 2.0 - 1.0\n        out = F.grid_sample(\n            img.float().contiguous(), grid,\n            mode=\'bilinear\', padding_mode=\'border\', align_corners=True\n        )\n    return out.to(img.dtype)\n\n\ndef warp_pixel_flow(img, flow):\n    """Backward-warp theo luồng quang học IFNet (đơn vị pixel)."""\n    _, _, h, w = img.shape\n    key = (h, w, flow.device)\n    base_grid = _PIXEL_GRID_CACHE.get(key)\n    if base_grid is None:\n        gy, gx = torch.meshgrid(\n            torch.linspace(-1.0, 1.0, h, device=flow.device, dtype=torch.float32),\n            torch.linspace(-1.0, 1.0, w, device=flow.device, dtype=torch.float32),\n            indexing=\'ij\'\n        )\n        base_grid = torch.stack([gx, gy], dim=-1).unsqueeze(0).contiguous()\n        _PIXEL_GRID_CACHE[key] = base_grid\n\n    with torch.amp.autocast(\'cuda\', enabled=False):\n        flow_f32 = flow.float()\n        flow_norm = torch.cat([\n            flow_f32[:, 0:1, :, :] / max(1.0, (w - 1.0) / 2.0),\n            flow_f32[:, 1:2, :, :] / max(1.0, (h - 1.0) / 2.0)\n        ], dim=1)\n        grid = (base_grid + flow_norm.permute(0, 2, 3, 1)).contiguous()\n        out = F.grid_sample(\n            img.float().contiguous(), grid,\n            mode=\'bilinear\', padding_mode=\'border\', align_corners=True\n        )\n    return out.to(img.dtype)\n\n\ndef conv(in_p, out_p, kernel_size=3, stride=1, padding=1, dilation=1):\n    return nn.Sequential(\n        nn.Conv2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, bias=True),\n        nn.PReLU(out_p)\n    )\n\n\ndef deconv(in_p, out_p, kernel_size=4, stride=2, padding=1):\n    return nn.Sequential(\n        nn.ConvTranspose2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, bias=True),\n        nn.PReLU(out_p)\n    )\n\n\nclass Conv2(nn.Module):\n    def __init__(self, in_p, out_p, stride=2):\n        super(Conv2, self).__init__()\n        self.conv1 = conv(in_p, out_p, 3, stride, 1)\n        self.conv2 = conv(out_p, out_p, 3, 1, 1)\n\n    def forward(self, x):\n        return self.conv2(self.conv1(x))\n\n\nclass Contextnet(nn.Module):\n    def __init__(self, indim=1, c=16):\n        super(Contextnet, self).__init__()\n        self.conv1 = Conv2(indim, c)\n        self.conv2 = Conv2(c, 2 * c)\n        self.conv3 = Conv2(2 * c, 4 * c)\n        self.conv4 = Conv2(4 * c, 8 * c)\n\n    def forward(self, x, flow):\n        x = self.conv1(x)\n        flow = F.interpolate(flow, size=(x.shape[2], x.shape[3]), mode=\'bilinear\', align_corners=False) * 0.5\n        f1 = warp_normalized_flow(x, flow)\n        x = self.conv2(x)\n        flow = F.interpolate(flow, size=(x.shape[2], x.shape[3]), mode=\'bilinear\', align_corners=False) * 0.5\n        f2 = warp_normalized_flow(x, flow)\n        x = self.conv3(x)\n        flow = F.interpolate(flow, size=(x.shape[2], x.shape[3]), mode=\'bilinear\', align_corners=False) * 0.5\n        f3 = warp_normalized_flow(x, flow)\n        x = self.conv4(x)\n        flow = F.interpolate(flow, size=(x.shape[2], x.shape[3]), mode=\'bilinear\', align_corners=False) * 0.5\n        f4 = warp_normalized_flow(x, flow)\n        return [f1, f2, f3, f4]\n\n\nclass IFBlock(nn.Module):\n    def __init__(self, in_planes, c=64, out_planes=5):\n        super(IFBlock, self).__init__()\n        self.conv0 = nn.Sequential(\n            conv(in_planes, c // 2, 3, 2, 1),\n            conv(c // 2, c, 3, 2, 1),\n        )\n        self.convblock = nn.Sequential(*[conv(c, c) for _ in range(8)])\n        self.lastconv = nn.ConvTranspose2d(c, out_planes, 4, 2, 1)\n\n    def forward(self, x, flow, scale):\n        orig_h, orig_w = x.shape[2], x.shape[3]\n        if scale != 1:\n            x = F.interpolate(x, scale_factor=1.0 / scale, mode=\'bilinear\', align_corners=False)\n        if flow is not None:\n            flow = F.interpolate(flow, size=(x.shape[2], x.shape[3]), mode=\'bilinear\', align_corners=False) * (1.0 / scale)\n            x = torch.cat((x, flow), 1)\n        x = self.conv0(x.contiguous())\n        x = self.convblock(x) + x\n        with torch.amp.autocast(\'cuda\', enabled=False):\n            tmp = self.lastconv(x.float().contiguous())\n            tmp = F.interpolate(tmp, size=(orig_h, orig_w), mode=\'bilinear\', align_corners=False)\n            flow = tmp[:, :4] * scale * 2\n            extra = tmp[:, 4:]\n        return flow, extra\n\n\nclass IFNetOcclusion(nn.Module):\n    """\n    IFNet tích hợp Occlusion Head tại Block 2:\n    - block0: xuất [F0->t (2), F1->t (2), mask (1)] tại scale 4 (5 channels)\n    - block1: xuất [dF0->t (2), dF1->t (2), dmask (1)] tại scale 2 (5 channels)\n    - block2 (Occlusion Head): xuất [dF0->t (2), dF1->t (2), M0->t (1), M1->t (1)] tại scale 1 (6 channels)\n    """\n    def __init__(self):\n        super(IFNetOcclusion, self).__init__()\n        self.block0 = IFBlock(6, c=240, out_planes=5)\n        self.block1 = IFBlock(13 + 4, c=150, out_planes=5)\n        self.block2 = IFBlock(13 + 4, c=90, out_planes=6)\n\n    def forward(self, I0t, I1t, scale=(4, 2, 1)):\n        img0 = I0t.repeat(1, 3, 1, 1)\n        img1 = I1t.repeat(1, 3, 1, 1)\n\n        # Scale 4: Block 0\n        flow, mask = self.block0(torch.cat((img0, img1), 1), None, scale=scale[0])\n        warped_i0 = warp_pixel_flow(I0t, flow[:, :2])\n        warped_i1 = warp_pixel_flow(I1t, flow[:, 2:4])\n\n        # Scale 2: Block 1\n        if flow.shape[2:] != I0t.shape[2:]:\n            flow = F.interpolate(flow, size=I0t.shape[2:], mode=\'bilinear\', align_corners=False)\n        if mask.shape[2:] != I0t.shape[2:]:\n            mask = F.interpolate(mask, size=I0t.shape[2:], mode=\'bilinear\', align_corners=False)\n        inp1 = torch.cat((img0, img1, warped_i0.repeat(1, 3, 1, 1), warped_i1.repeat(1, 3, 1, 1), mask), 1)\n        flow_d1, mask_d1 = self.block1(inp1, flow, scale=scale[1])\n        flow = flow + flow_d1\n        mask = mask + mask_d1\n        warped_i0 = warp_pixel_flow(I0t, flow[:, :2])\n        warped_i1 = warp_pixel_flow(I1t, flow[:, 2:4])\n\n        # Scale 1: Block 2 (Occlusion Head: outputs 4 flow channels + 2 occlusion mask channels)\n        if flow.shape[2:] != I0t.shape[2:]:\n            flow = F.interpolate(flow, size=I0t.shape[2:], mode=\'bilinear\', align_corners=False)\n        if mask.shape[2:] != I0t.shape[2:]:\n            mask = F.interpolate(mask, size=I0t.shape[2:], mode=\'bilinear\', align_corners=False)\n        inp2 = torch.cat((img0, img1, warped_i0.repeat(1, 3, 1, 1), warped_i1.repeat(1, 3, 1, 1), mask), 1)\n        flow_d2, occ_logits = self.block2(inp2, flow, scale=scale[2])\n        flow = flow + flow_d2\n\n        M0 = torch.sigmoid(occ_logits[:, 0:1])\n        M1 = torch.sigmoid(occ_logits[:, 1:2])\n        unet_mask = torch.sigmoid(mask)\n        return flow, M0, M1, unet_mask\n\n\nclass OriginalUnet(nn.Module):\n    """UNet gốc của bài báo (indim=9, outdim=1, đầu ra residual [-1, 1])."""\n    def __init__(self, indim=9, outdim=1, c=16):\n        super(OriginalUnet, self).__init__()\n        self.down0 = Conv2(indim, 2 * c)\n        self.down1 = Conv2(4 * c, 4 * c)\n        self.down2 = Conv2(8 * c, 8 * c)\n        self.down3 = Conv2(16 * c, 16 * c)\n        self.up0 = deconv(32 * c, 8 * c)\n        self.up1 = deconv(16 * c, 4 * c)\n        self.up2 = deconv(8 * c, 2 * c)\n        self.up3 = deconv(4 * c, c)\n        self.conv = nn.Conv2d(c, outdim, 3, 1, 1)\n\n    def forward(self, img0, img1, flow0, flow1, warped0, warped1, mask, c0, c1):\n        s0 = self.down0(torch.cat((img0, img1, warped0, warped1, flow0, flow1, mask), 1))\n        if s0.shape[2:] != c0[0].shape[2:]:\n            s0 = F.interpolate(s0, size=c0[0].shape[2:], mode=\'bilinear\', align_corners=False)\n        s1 = self.down1(torch.cat((s0, c0[0], c1[0]), 1))\n        if s1.shape[2:] != c0[1].shape[2:]:\n            s1 = F.interpolate(s1, size=c0[1].shape[2:], mode=\'bilinear\', align_corners=False)\n        s2 = self.down2(torch.cat((s1, c0[1], c1[1]), 1))\n        if s2.shape[2:] != c0[2].shape[2:]:\n            s2 = F.interpolate(s2, size=c0[2].shape[2:], mode=\'bilinear\', align_corners=False)\n        s3 = self.down3(torch.cat((s2, c0[2], c1[2]), 1))\n        if s3.shape[2:] != c0[3].shape[2:]:\n            s3 = F.interpolate(s3, size=c0[3].shape[2:], mode=\'bilinear\', align_corners=False)\n        x = self.up0(torch.cat((s3, c0[3], c1[3]), 1))\n        if x.shape[2:] != s2.shape[2:]:\n            x = F.interpolate(x, size=s2.shape[2:], mode=\'bilinear\', align_corners=False)\n        x = self.up1(torch.cat((x, s2), 1))\n        if x.shape[2:] != s1.shape[2:]:\n            x = F.interpolate(x, size=s1.shape[2:], mode=\'bilinear\', align_corners=False)\n        x = self.up2(torch.cat((x, s1), 1))\n        if x.shape[2:] != s0.shape[2:]:\n            x = F.interpolate(x, size=s0.shape[2:], mode=\'bilinear\', align_corners=False)\n        x = self.up3(torch.cat((x, s0), 1))\n        if x.shape[2:] != img0.shape[2:]:\n            x = F.interpolate(x, size=img0.shape[2:], mode=\'bilinear\', align_corners=False)\n        x = self.conv(x)\n        return torch.sigmoid(x)\n\n\nclass TPSInbetweenOcclusion(nn.Module):\n    """\n    Kiến trúc TPS-Inbetween nâng cấp Occlusion Head (Phase 4):\n      - Giữ nguyên toàn bộ 156 tensors phụ trợ (encoder + unet + block0 + block1).\n      - Thay đổi duy nhất tại IFNet.block2.lastconv: xuất 6 kênh [F(4), M0(1), M1(1)].\n      - Bộ Trộn Trọng số Chuẩn hóa (Normalized Visibility Blending):\n        W0 = (1 - t) * M0->t\n        W1 = t * M1->t\n        I_fused = (W0 * fine_I0t + W1 * fine_I1t) / (W0 + W1 + eps)\n    """\n    def __init__(self):\n        super(TPSInbetweenOcclusion, self).__init__()\n        self.encoder = Contextnet(indim=1, c=16)\n        self.IFNet = IFNetOcclusion()\n        self.unet = OriginalUnet(indim=9, outdim=1, c=16)\n\n    def forward(self, x0, x1, t_val, coarse_flow01, coarse_flow10, direct_t_flows=False):\n        t = t_val\n        if direct_t_flows:\n            coarse_flow0t = coarse_flow01\n            coarse_flow1t = coarse_flow10\n        else:\n            coarse_flow0t = coarse_flow01 * (t ** 2) - (1.0 - t) * t * coarse_flow10\n            coarse_flow1t = coarse_flow10 * ((1.0 - t) ** 2) - t * (1.0 - t) * coarse_flow01\n\n        coarse_I0t = warp_normalized_flow(x0, coarse_flow0t)\n        coarse_I1t = warp_normalized_flow(x1, coarse_flow1t)\n        coarse_warp = torch.clamp((1.0 - t) * coarse_I0t + t * coarse_I1t, 0.0, 1.0)\n\n        feats0 = self.encoder(x0, coarse_flow0t)\n        feats1 = self.encoder(x1, coarse_flow1t)\n\n        flow_bet_coar, M0, M1, unet_mask = self.IFNet(coarse_I0t, coarse_I1t)\n\n        fine_I0t = warp_pixel_flow(coarse_I0t, flow_bet_coar[:, :2])\n        fine_I1t = warp_pixel_flow(coarse_I1t, flow_bet_coar[:, 2:4])\n\n        # Normalized Visibility Blending\n        w0 = (1.0 - t) * M0\n        w1 = t * M1\n        w_sum = w0 + w1 + 1e-6\n        I_fused = (w0 * fine_I0t + w1 * fine_I1t) / w_sum\n\n        refine_out = self.unet(\n            x0, x1,\n            coarse_flow0t + flow_bet_coar[:, :2],\n            coarse_flow1t + flow_bet_coar[:, 2:4],\n            fine_I0t, fine_I1t, unet_mask, feats0, feats1\n        )\n        res = refine_out[:, 0:1, :, :] * 2.0 - 1.0\n        It = torch.clamp(I_fused + res, 0.0, 1.0)\n        return It, coarse_warp, fine_I0t, fine_I1t, flow_bet_coar, M0, M1\n\n\ndef load_tps_occlusion_weights(model, ckpt_path, device=\'cpu\', prefer_ema=True, is_rank0=True):\n    """\n    Nạp trọng số Warm-Start từ Phase 3 Remake (158 tensors) sang Phase 4 Occlusion Model:\n      - 156 tensors khớp tuyệt đối 100%.\n      - IFNet.block2.lastconv (4 kênh flow + 2 kênh occlusion):\n        + Kế thừa 4 kênh flow từ Phase 3.\n        + Khởi tạo 2 kênh occlusion mới: weights = 0.0, bias = +2.0 (Sigmoid ≈ 0.88).\n    Nếu checkpoint đã là Phase 4 hoàn chỉnh, nạp khớp 100% 158/158 tensors.\n    """\n    if not os.path.exists(ckpt_path):\n        raise FileNotFoundError(f"Không tìm thấy checkpoint tại: {ckpt_path}")\n\n    raw = torch.load(ckpt_path, map_location=device, weights_only=False)\n    if isinstance(raw, dict):\n        if prefer_ema and raw.get(\'ema_state\') is not None:\n            ema_obj = raw[\'ema_state\']\n            sd = ema_obj[\'shadow\'] if isinstance(ema_obj, dict) and \'shadow\' in ema_obj else ema_obj\n        elif \'model_state\' in raw:\n            sd = raw[\'model_state\']\n        elif \'state_dict\' in raw:\n            sd = raw[\'state_dict\']\n        elif \'model\' in raw:\n            sd = raw[\'model\']\n        else:\n            sd = raw\n    else:\n        sd = raw\n\n    model_sd = model.state_dict()\n    matched_sd = {}\n    warm_started_keys = []\n\n    for k, v in sd.items():\n        clean_k = k.replace(\'module.\', \'\').replace(\'base_model.\', \'\')\n        if clean_k in model_sd:\n            tgt_shape = model_sd[clean_k].shape\n            if tgt_shape == v.shape:\n                matched_sd[clean_k] = v\n            elif clean_k == "IFNet.block2.lastconv.weight" and v.shape == torch.Size([90, 5, 4, 4]) and tgt_shape == torch.Size([90, 6, 4, 4]):\n                # Warm-start block2 lastconv weight (copy 4 flow channels, 2 mask channels = 0)\n                new_w = torch.zeros(tgt_shape, dtype=v.dtype, device=v.device)\n                new_w[:, :4, :, :] = v[:, :4, :, :]\n                matched_sd[clean_k] = new_w\n                warm_started_keys.append(clean_k)\n            elif clean_k == "IFNet.block2.lastconv.bias" and v.shape == torch.Size([5]) and tgt_shape == torch.Size([6]):\n                # Warm-start block2 lastconv bias (copy 4 flow biases, 2 mask biases = +2.0)\n                new_b = torch.zeros(tgt_shape, dtype=v.dtype, device=v.device)\n                new_b[:4] = v[:4]\n                new_b[4:] = 2.0  # Sigmoid(+2.0) ≈ 0.8808\n                matched_sd[clean_k] = new_b\n                warm_started_keys.append(clean_k)\n\n    missing, unexpected = model.load_state_dict(matched_sd, strict=False)\n    if len(missing) > 0:\n        raise RuntimeError(f"Thiếu {len(missing)} tensors khi nạp {ckpt_path}: {missing[:5]}")\n\n    if is_rank0:\n        tag = os.path.basename(ckpt_path)\n        if warm_started_keys:\n            print(f"[✓] Warm-Start Phase 4 từ Phase 3 ({tag}): Khớp 158/158 tensors!"\n                  f" (block2.lastconv mở rộng 5->6 kênh với bias=+2.0, Sigmoid≈0.88)", flush=True)\n        else:\n            print(f"[✓] Nạp trọng số Phase 4 ({tag}): Khớp 100% {len(matched_sd)}/{len(model_sd)} active tensors!", flush=True)\n    return model\n\n\nclass OcclusionAwareLoss(nn.Module):\n    """\n    Hàm Mất mát Nhận thức Che khuất (Occlusion-Aware Loss):\n      1. Photometric Loss có trọng số hiển thị:\n         L_photo = M0 * rho(fine_I0t - GT) + M1 * rho(fine_I1t - GT)\n         với rho(x) = sqrt(x^2 + eps^2) (Charbonnier Loss).\n      2. Ràng buộc Nhất quán Bao phủ (Coverage Constraint):\n         L_cov = mean(ReLU(0.95 - (M0 + M1))^2)\n         (Ngăn chặn mạng triệt tiêu M -> 0 để trốn phạt lỗi tái tạo).\n      3. Làm mượt Mặt nạ Che khuất có hướng dẫn nét vẽ (Edge-aware Smoothness):\n         L_smooth = sum_{c} |d_x M_c| * exp(-alpha * |d_x GT|) + |d_y M_c| * exp(-alpha * |d_y GT|)\n         (Giữ biên của M bám sát theo các đường nét viền vật thể, không lốm đốm nhiễu hạt).\n    """\n    def __init__(self, w_photo=0.25, w_cov=0.50, w_smooth=0.10, cov_margin=0.95, smooth_alpha=10.0, eps=1e-3):\n        super(OcclusionAwareLoss, self).__init__()\n        self.w_photo = float(w_photo)\n        self.w_cov = float(w_cov)\n        self.w_smooth = float(w_smooth)\n        self.cov_margin = float(cov_margin)\n        self.smooth_alpha = float(smooth_alpha)\n        self.eps = float(eps)\n\n    def forward(self, fine_I0t, fine_I1t, M0, M1, gt_t):\n        # 1. Photometric Loss có trọng số hiển thị\n        diff0 = fine_I0t - gt_t\n        diff1 = fine_I1t - gt_t\n        rho0 = torch.sqrt(diff0 ** 2 + self.eps ** 2)\n        rho1 = torch.sqrt(diff1 ** 2 + self.eps ** 2)\n        loss_photo = torch.mean(M0 * rho0 + M1 * rho1)\n\n        # 2. Coverage Constraint (Chống lười biếng M -> 0)\n        cov_deficit = F.relu(self.cov_margin - (M0 + M1))\n        loss_cov = torch.mean(cov_deficit ** 2)\n\n        # 3. Edge-aware Smoothness (Bám sát viền nét GT, phẳng mịn trên nền)\n        def _edge_smooth(m_map, img_ref):\n            dx_m = torch.abs(m_map[:, :, :, 1:] - m_map[:, :, :, :-1])\n            dy_m = torch.abs(m_map[:, :, 1:, :] - m_map[:, :, :-1, :])\n            with torch.no_grad():\n                dx_img = torch.abs(img_ref[:, :, :, 1:] - img_ref[:, :, :, :-1])\n                dy_img = torch.abs(img_ref[:, :, 1:, :] - img_ref[:, :, :-1, :])\n                wx = torch.exp(-self.smooth_alpha * dx_img)\n                wy = torch.exp(-self.smooth_alpha * dy_img)\n            return torch.mean(dx_m * wx) + torch.mean(dy_m * wy)\n\n        loss_smooth = _edge_smooth(M0, gt_t) + _edge_smooth(M1, gt_t)\n\n        total_occ_loss = (\n            self.w_photo * loss_photo\n            + self.w_cov * loss_cov\n            + self.w_smooth * loss_smooth\n        )\n        details = {\n            \'loss_occ_photo\': float(loss_photo.detach().item()),\n            \'loss_occ_cov\': float(loss_cov.detach().item()),\n            \'loss_occ_smooth\': float(loss_smooth.detach().item()),\n            \'total_occ_loss\': float(total_occ_loss.detach().item()),\n        }\n        return total_occ_loss, details\n')

worker_occ_path = os.path.join(WORKING_DIR, "ddp_worker_tps_occlusion.py")
with open(worker_occ_path, "w", encoding="utf-8") as f:
    f.write('#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n"""\nddp_worker_tps_occlusion.py\n--------------------------------------------------------------------------------\nĐỘNG CƠ HUẤN LUYỆN DDP 2x GPU PHA 4 — OCCLUSION-AWARE HEAD (TPSInbetweenOcclusion)\nThử nghiệm Cách ly Độc lập: Phase 4 Occlusion Head (M4_Occ)\nGiữ nguyên vẹn 100% Phase 3 Remake (Phase3_Remake / M3_Best).\n\nCác cải tiến cốt lõi:\n1. Warm-Start từ Phase 3 Remake Best (`model_tps_finetuned_best.pt`):\n   Kế thừa 158/158 tensors. 2 kênh mặt nạ che khuất mới khởi tạo bias=+2.0 (Sigmoid ≈ 0.88).\n2. Tối ưu hóa đồng thời 2 mục tiêu:\n   - Lineart Reconstruction Loss (AdaptiveLineartLoss + Exact Density + Cleanliness)\n   - Occlusion-Aware Loss:\n     + L_photo (Photometric hiển thị có trọng số M0, M1)\n     + L_cov (Coverage constraint: ReLU(0.95 - (M0 + M1))^2)\n     + L_smooth (Edge-aware smoothness bám sát viền nét)\n3. Checkpoint Manager độc lập:\n   - model_tps_occlusion_best.pt\n   - model_tps_occlusion_last.pt\n--------------------------------------------------------------------------------\n"""\n\nimport os\nimport sys\nimport gc\nimport time\nimport math\nimport json\nimport glob\nimport random\nimport argparse\nimport contextlib\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import Dataset, DataLoader\nfrom torch.utils.data.distributed import DistributedSampler\n\nos.environ.setdefault("MALLOC_TRIM_THRESHOLD_", "65536")\nos.environ.setdefault("MALLOC_ARENA_MAX", "2")\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("MKL_NUM_THREADS", "1")\n\nif sys.platform.startswith(\'win\'):\n    try:\n        sys.stdout.reconfigure(encoding=\'utf-8\')\n        sys.stderr.reconfigure(encoding=\'utf-8\')\n    except Exception:\n        pass\n\ntry:\n    import cv2\n    cv2.setNumThreads(0)\n    cv2.ocl.setUseOpenCL(False)\nexcept ImportError:\n    cv2 = None\n\ntorch.backends.cudnn.benchmark = True\n\nSCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))\nREPO_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, ".."))\n\nfor _p in (\n    SCRIPT_DIR,\n    os.path.join(REPO_ROOT, "kaggle_experiment_occlusion"),\n    os.path.join(REPO_ROOT, "kaggle_finetune_sequence_phase3"),\n    os.path.join(REPO_ROOT, "kaggle_finetune_sequence_phase2"),\n    os.path.join(REPO_ROOT, "pipelines"),\n):\n    if os.path.isdir(_p) and _p not in sys.path:\n        sys.path.insert(0, _p)\n\n# Import model occlusion & loss\nfrom model_occlusion import (\n    TPSInbetweenOcclusion,\n    load_tps_occlusion_weights,\n    OcclusionAwareLoss,\n    warp_normalized_flow,\n    warp_pixel_flow,\n)\n\n# Import dataset & multi-layer pipeline components\nfrom multi_layer_tps import (\n    FIXED_TPS_ANCHORS,\n    solve_multilayer_cache_entry,\n    reconstruct_multilayer_flow_batch_from_cache,\n)\n\nfrom ddp_worker_tps_sequence_phase2 import (\n    trim_memory,\n    AdaptiveLineartLoss,\n    BidirectionalStrokeWidthLoss,\n    SoftClDiceLoss,\n    VGGPerceptualOrLaplacianLoss,\n    approx_gpu_distance_map,\n    soft_dilate,\n    soft_skeletonize,\n    evaluate_metrics_dual_regime,\n    DDPLengthWidthBatchSampler,\n    AdaptiveEMA,\n    _apply_paired_spatial_aug_gpu,\n)\n\nfrom ddp_worker_tps_sequence_phase3 import (\n    Phase3ExactDensityLoss,\n    Phase3ZoneCleanlinessLoss,\n    SakugaMultiLayerDataset,\n    pad_collate_fn_multilayer,\n)\n\n\n# ==============================================================================\n# CHECKPOINT MANAGER ĐỘC LẬP CHO PHASE 4 OCCLUSION\n# ==============================================================================\nclass OcclusionCheckpointManager:\n    """\n    Chính sách lưu trữ 2 file checkpoint cho Phase 4:\n      1. model_tps_occlusion_best.pt : Trọng số EMA tốt nhất trên tập Val.\n      2. model_tps_occlusion_last.pt : Trọng số Epoch gần nhất kèm trạng thái Resume đầy đủ.\n    """\n    def __init__(self, output_dir, is_rank0=True):\n        self.output_dir = output_dir\n        self.is_rank0 = is_rank0\n        self.best_score = float(\'inf\')\n        self.best_epoch = 0\n        os.makedirs(output_dir, exist_ok=True)\n        self.best_path = os.path.join(output_dir, "model_tps_occlusion_best.pt")\n        self.last_path = os.path.join(output_dir, "model_tps_occlusion_last.pt")\n\n    def _atomic_save(self, obj, dst_path):\n        tmp_path = dst_path + ".tmp"\n        torch.save(obj, tmp_path)\n        if os.path.exists(dst_path):\n            os.remove(dst_path)\n        os.rename(tmp_path, dst_path)\n\n    def save_epoch(self, epoch, base_model, ema, optimizer, scaler, score, history, is_best):\n        if not self.is_rank0:\n            return\n\n        if is_best:\n            self.best_score = float(score)\n            self.best_epoch = int(epoch)\n            best_payload = {\n                \'epoch\': int(epoch),\n                \'best_score\': float(score),\n                \'model_state\': {k: v.detach().cpu() for k, v in ema.shadow.items()},\n                \'raw_model_state\': {k: v.detach().cpu() for k, v in base_model.state_dict().items()},\n                \'history\': history,\n                \'tag\': \'Phase4_Occlusion_Best\',\n            }\n            self._atomic_save(best_payload, self.best_path)\n            print(f"   💾 [BEST SAVED] {os.path.basename(self.best_path)} (Score={score:.4f} @ Ep {epoch:02d})", flush=True)\n\n        last_payload = {\n            \'epoch\': int(epoch),\n            \'best_epoch\': self.best_epoch,\n            \'best_score\': self.best_score,\n            \'model_state\': {k: v.detach().cpu() for k, v in base_model.state_dict().items()},\n            \'ema_state\': {\'shadow\': {k: v.detach().cpu() for k, v in ema.shadow.items()}},\n            \'optimizer_state\': optimizer.state_dict(),\n            \'scaler_state\': scaler.state_dict() if scaler is not None else None,\n            \'history\': history,\n            \'tag\': \'Phase4_Occlusion_Last\',\n        }\n        self._atomic_save(last_payload, self.last_path)\n        print(f"   💾 [LAST SAVED] {os.path.basename(self.last_path)} (Epoch {epoch:02d} complete)", flush=True)\n\n    def try_resume(self, base_model, ema, optimizer, scaler, map_location=\'cpu\'):\n        if not os.path.exists(self.last_path):\n            return 0, []\n        try:\n            ckpt = torch.load(self.last_path, map_location=map_location, weights_only=False)\n            epoch = ckpt.get(\'epoch\', 0)\n            self.best_epoch = ckpt.get(\'best_epoch\', epoch)\n            self.best_score = ckpt.get(\'best_score\', float(\'inf\'))\n            base_model.load_state_dict(ckpt[\'model_state\'])\n            if \'ema_state\' in ckpt and hasattr(ema, \'shadow\'):\n                ema.shadow = {k: v.to(map_location) for k, v in ckpt[\'ema_state\'][\'shadow\'].items()}\n            if optimizer is not None and \'optimizer_state\' in ckpt:\n                optimizer.load_state_dict(ckpt[\'optimizer_state\'])\n            if scaler is not None and ckpt.get(\'scaler_state\') is not None:\n                scaler.load_state_dict(ckpt[\'scaler_state\'])\n            history = ckpt.get(\'history\', [])\n            return int(epoch), history\n        except Exception as e:\n            if self.is_rank0:\n                print(f"   [!] Resume failed ({e}), starting fresh.", flush=True)\n            return 0, []\n\n\n# ==============================================================================\n# HÀM VALIDATION CHO PHASE 4\n# ==============================================================================\ndef validate_epoch_occlusion(val_loader, model, criterion, occ_loss_fn, device, is_rank0=True):\n    model.eval()\n    val_loss_sum = 0.0\n    val_frames = 0\n    all_metrics = []\n\n    with torch.no_grad():\n        for batch in val_loader:\n            h_pad, w_pad = batch[\'h_pad\'], batch[\'w_pad\']\n            orig_hw = batch[\'orig_hw\']\n            x0 = batch[\'x0\'].to(device, non_blocking=True)\n            x1 = batch[\'x1\'].to(device, non_blocking=True)\n            xk = batch[\'xk\'].to(device, non_blocking=True)\n            t_val = batch[\'t_val\'].to(device, non_blocking=True).view(-1, 1, 1, 1)\n\n            cf01, cf10 = reconstruct_multilayer_flow_batch_from_cache(\n                batch[\'c0_dyn\'].to(device, non_blocking=True),\n                batch[\'th01_dyn\'].to(device, non_blocking=True),\n                batch[\'c1_dyn\'].to(device, non_blocking=True),\n                batch[\'th10_dyn\'].to(device, non_blocking=True),\n                batch[\'valid_dyn\'].to(device, non_blocking=True),\n                batch[\'dyn_pts0\'].to(device, non_blocking=True),\n                batch[\'dyn_pts1\'].to(device, non_blocking=True),\n                batch[\'dyn_pts_len\'].to(device, non_blocking=True),\n                batch[\'static_pts0\'].to(device, non_blocking=True),\n                batch[\'static_pts1\'].to(device, non_blocking=True),\n                batch[\'num_static\'].to(device, non_blocking=True),\n                batch[\'num_clusters\'].to(device, non_blocking=True),\n                orig_hw, h_pad, w_pad, device, downscale_factor=2,\n                use_dyn_anchors=True, transition_width_px=22.0, falloff_radius_px=105.0\n            )\n\n            with torch.amp.autocast(\'cuda\', enabled=(device.type == \'cuda\')):\n                pred, cw, f0, f1, fl, M0, M1 = model(x0, x1, t_val, cf01, cf10)\n                l_main, d_main = criterion(pred, xk, valid_hw=orig_hw)\n                l_occ, d_occ = occ_loss_fn(f0, f1, M0, M1, xk)\n                l_total = l_main + l_occ\n\n            b = x0.shape[0]\n            val_loss_sum += float(l_total.item()) * b\n            val_frames += b\n\n            for i in range(b):\n                hi, wi = int(orig_hw[i, 0].item()), int(orig_hw[i, 1].item())\n                gt_i = xk[i:i + 1, :, :hi, :wi]\n                pr_i = pred[i:i + 1, :, :hi, :wi]\n                m = evaluate_metrics_dual_regime(pr_i, gt_i)\n                m_batch = {k: float(m[k].item()) for k in m}\n                m_batch[\'M0_mean\'] = float(M0[i:i + 1, :, :hi, :wi].mean().item())\n                m_batch[\'M1_mean\'] = float(M1[i:i + 1, :, :hi, :wi].mean().item())\n                m_batch[\'loss_occ_photo\'] = d_occ[\'loss_occ_photo\']\n                m_batch[\'loss_occ_cov\'] = d_occ[\'loss_occ_cov\']\n                all_metrics.append(m_batch)\n\n    avg_loss = val_loss_sum / max(1, val_frames)\n    summary = {\'val_loss\': avg_loss}\n    if all_metrics:\n        for k in all_metrics[0]:\n            vals = [m[k] for m in all_metrics if k in m]\n            summary[k] = float(np.mean(vals))\n    return summary\n\n\n# ==============================================================================\n# HÀM MAIN HUẤN LUYỆN DDP CHO PHASE 4\n# ==============================================================================\ndef main():\n    parser = argparse.ArgumentParser(description="Phase 4 Occlusion Head DDP Training Worker")\n    parser.add_argument("--train_csv", type=str, required=True)\n    parser.add_argument("--val_csv", type=str, required=True)\n    parser.add_argument("--train_clips_dir", type=str, required=True)\n    parser.add_argument("--val_clips_dir", type=str, required=True)\n    parser.add_argument("--tps_cache", type=str, required=True)\n    parser.add_argument("--init_checkpoint", type=str, required=True, help="Path to Phase 3 Best checkpoint")\n    parser.add_argument("--output_dir", type=str, default="./checkpoints_phase4_occlusion")\n    parser.add_argument("--epochs", type=int, default=6)\n    parser.add_argument("--grad_accum", type=int, default=2)\n    parser.add_argument("--max_chunk_frames", type=int, default=3)\n    parser.add_argument("--lr_backbone", type=float, default=6.0e-6)\n    parser.add_argument("--lr_unet", type=float, default=1.8e-5)\n    parser.add_argument("--ema_decay", type=float, default=0.996)\n    parser.add_argument("--synth_count", type=int, default=150)\n    parser.add_argument("--static_aug_prob", type=float, default=0.12)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--w_photo", type=float, default=0.25)\n    parser.add_argument("--w_cov", type=float, default=0.50)\n    parser.add_argument("--w_smooth", type=float, default=0.10)\n    parser.add_argument("--w_ghost", type=float, default=0.32)\n    parser.add_argument("--w_temp", type=float, default=0.55)\n    parser.add_argument("--num_workers", type=int, default=2)\n    parser.add_argument("--dry_run", action="store_true")\n    args = parser.parse_args()\n\n    local_rank = int(os.environ.get("LOCAL_RANK", 0))\n    world_size = int(os.environ.get("WORLD_SIZE", 1))\n    is_ddp = world_size > 1\n    is_rank0 = (local_rank == 0)\n\n    if is_ddp:\n        dist.init_process_group("nccl" if torch.cuda.is_available() else "gloo")\n        torch.cuda.set_device(local_rank)\n        device = torch.device(f"cuda:{local_rank}")\n    else:\n        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\n\n    if is_rank0:\n        print("=" * 84)\n        print(f"🚀 [PHASE 4 OCCLUSION HEAD] KHỞI ĐỘNG DDP HUẤN LUYỆN ({world_size} GPUs)")\n        print(f"   Warm-Start Input Checkpoint: {args.init_checkpoint}")\n        print(f"   Output Checkpoint Dir      : {args.output_dir}")\n        print(f"   Epochs: {args.epochs} | LR Backbone: {args.lr_backbone:.2e} | LR UNet: {args.lr_unet:.2e}")\n        print(f"   Aux Loss Weights           : Photo={args.w_photo}, Cov={args.w_cov}, Smooth={args.w_smooth}")\n        print("=" * 84, flush=True)\n\n    # 1. Khởi tạo mô hình & nạp Warm-Start từ Phase 3 Remake\n    base_model = TPSInbetweenOcclusion().to(device)\n    load_tps_occlusion_weights(base_model, args.init_checkpoint, device=device, prefer_ema=True, is_rank0=is_rank0)\n\n    ema = AdaptiveEMA(base_model, max_decay=args.ema_decay)\n    ckpt_mgr = OcclusionCheckpointManager(args.output_dir, is_rank0=is_rank0)\n\n    # 2. Khởi tạo Optimizer & Scaler\n    backbone_params = list(base_model.encoder.parameters()) + list(base_model.IFNet.parameters())\n    unet_params = list(base_model.unet.parameters())\n    optimizer = torch.optim.AdamW([\n        {\'params\': base_model.encoder.parameters(), \'lr\': args.lr_backbone, \'weight_decay\': 1e-4},\n        {\'params\': base_model.IFNet.parameters(), \'lr\': args.lr_backbone, \'weight_decay\': 1e-4},\n        {\'params\': unet_params, \'lr\': args.lr_unet, \'weight_decay\': 1e-4},\n    ])\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=(device.type == \'cuda\'))\n\n    # 3. Hàm mất mát\n    lineart_loss = AdaptiveLineartLoss(train_mode="LONG", is_rank0=is_rank0, w_ghost=args.w_ghost, w_temp=args.w_temp).to(device)\n    occ_loss_fn = OcclusionAwareLoss(\n        w_photo=args.w_photo, w_cov=args.w_cov, w_smooth=args.w_smooth, cov_margin=0.95, smooth_alpha=10.0\n    ).to(device)\n\n    # 4. DataLoaders (kế thừa SakugaMultiLayerDataset & pad_collate_fn_multilayer)\n    train_ds = SakugaMultiLayerDataset(\n        args.train_csv, args.train_clips_dir, args.tps_cache,\n        train_mode="LONG", is_train=True, seed=args.seed,\n        synth_count=args.synth_count, static_aug_prob=args.static_aug_prob\n    )\n    val_ds = SakugaMultiLayerDataset(\n        args.val_csv, args.val_clips_dir, args.tps_cache,\n        train_mode="VAL", is_train=False, seed=args.seed,\n        synth_count=0, static_aug_prob=0.0\n    )\n\n    train_sampler = DDPLengthWidthBatchSampler(\n        train_ds.samples, world_size=world_size, rank=local_rank,\n        batch_n3=4, batch_n5=2, batch_n7=1, seed=args.seed\n    )\n    train_loader = DataLoader(\n        train_ds, batch_sampler=train_sampler, collate_fn=pad_collate_fn_multilayer,\n        num_workers=args.num_workers, pin_memory=(device.type == \'cuda\'),\n        persistent_workers=(args.num_workers > 0)\n    )\n    val_sampler = DistributedSampler(val_ds, shuffle=False) if is_ddp else None\n    val_loader = DataLoader(\n        val_ds, batch_size=4, shuffle=False, sampler=val_sampler,\n        collate_fn=pad_collate_fn_multilayer, num_workers=args.num_workers,\n        pin_memory=(device.type == \'cuda\'), persistent_workers=(args.num_workers > 0), drop_last=False\n    )\n\n    # 5. Vòng lặp huấn luyện\n    start_epoch = 1\n    history = []\n    last_ep, hist = ckpt_mgr.try_resume(base_model, ema, optimizer, scaler, map_location=device)\n    if last_ep > 0:\n        start_epoch = last_ep + 1\n        history = hist\n        if is_rank0:\n            print(f"[♻️ RESUME] Tiếp tục huấn luyện từ Epoch {start_epoch:02d}...", flush=True)\n\n    if is_ddp:\n        ddp_model = DDP(base_model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)\n    else:\n        ddp_model = base_model\n\n    for epoch in range(start_epoch, args.epochs + 1):\n        t0 = time.time()\n        if hasattr(train_sampler, \'set_epoch\'):\n            train_sampler.set_epoch(epoch)\n\n        # Lịch LR: Cosine decay nhẹ\n        lr_scale = 0.5 * (1.0 + math.cos(math.pi * float(epoch - 1) / float(max(1, args.epochs))))\n        lr_scale = max(0.20, lr_scale)\n        optimizer.param_groups[0][\'lr\'] = args.lr_backbone * lr_scale\n        optimizer.param_groups[1][\'lr\'] = args.lr_backbone * lr_scale\n        optimizer.param_groups[2][\'lr\'] = args.lr_unet * lr_scale\n\n        ddp_model.train()\n        optimizer.zero_grad(set_to_none=True)\n        acc_loss = 0.0\n        acc_frames = 0\n        num_batches = len(train_loader)\n        grad_accum = max(1, args.grad_accum)\n\n        for b_idx, batch in enumerate(train_loader):\n            h_pad, w_pad = batch[\'h_pad\'], batch[\'w_pad\']\n            orig_hw = batch[\'orig_hw\'].to(device, non_blocking=True)\n            x0 = batch[\'x0\'].to(device, non_blocking=True)\n            x1 = batch[\'x1\'].to(device, non_blocking=True)\n            xk_all = batch[\'xk_all\'].to(device, non_blocking=True)\n            t_vals = batch[\'t_vals\'].to(device, non_blocking=True)\n            is_direct = bool(batch.get(\'is_direct_t_flows\', False))\n            bsz, m_steps = xk_all.shape[0], xk_all.shape[1]\n\n            if is_direct:\n                cf01_steps, cf10_steps = [], []\n                for m_i in range(m_steps):\n                    cf0_i, cf1_i = reconstruct_multilayer_flow_batch_from_cache(\n                        batch[\'c0_dyn_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'th01_dyn_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'c1_dyn_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'th10_dyn_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'valid_dyn_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'dyn_pts0_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'dyn_pts1_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'dyn_pts_len_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'static_pts0_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'static_pts1_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'num_static_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'num_clusters_all\'][:, m_i].to(device, non_blocking=True),\n                        orig_hw, h_pad, w_pad, device, downscale_factor=2,\n                        use_dyn_anchors=False, transition_width_px=18.0, falloff_radius_px=160.0\n                    )\n                    cf01_steps.append(cf0_i)\n                    cf10_steps.append(cf1_i)\n                cf01_f = torch.stack(cf01_steps, dim=1).reshape(bsz * m_steps, 2, h_pad, w_pad)\n                cf10_f = torch.stack(cf10_steps, dim=1).reshape(bsz * m_steps, 2, h_pad, w_pad)\n            else:\n                cf01, cf10 = reconstruct_multilayer_flow_batch_from_cache(\n                    batch[\'c0_dyn\'].to(device, non_blocking=True),\n                    batch[\'th01_dyn\'].to(device, non_blocking=True),\n                    batch[\'c1_dyn\'].to(device, non_blocking=True),\n                    batch[\'th10_dyn\'].to(device, non_blocking=True),\n                    batch[\'valid_dyn\'].to(device, non_blocking=True),\n                    batch[\'dyn_pts0\'].to(device, non_blocking=True),\n                    batch[\'dyn_pts1\'].to(device, non_blocking=True),\n                    batch[\'dyn_pts_len\'].to(device, non_blocking=True),\n                    batch[\'static_pts0\'].to(device, non_blocking=True),\n                    batch[\'static_pts1\'].to(device, non_blocking=True),\n                    batch[\'num_static\'].to(device, non_blocking=True),\n                    batch[\'num_clusters\'].to(device, non_blocking=True),\n                    orig_hw, h_pad, w_pad, device, downscale_factor=2,\n                    use_dyn_anchors=True, transition_width_px=22.0, falloff_radius_px=105.0\n                )\n                _apply_paired_spatial_aug_gpu(x0, x1, xk_all, cf01, cf10, orig_hw, is_sequence=True)\n                cf01_f = cf01.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 2, h_pad, w_pad)\n                cf10_f = cf10.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 2, h_pad, w_pad)\n\n            x0_f = x0.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 1, h_pad, w_pad)\n            x1_f = x1.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 1, h_pad, w_pad)\n            xk_f = xk_all.reshape(bsz * m_steps, 1, h_pad, w_pad)\n            t_f = t_vals.reshape(bsz * m_steps, 1, 1, 1)\n            hw_f = orig_hw.unsqueeze(1).expand(-1, m_steps, -1).reshape(bsz * m_steps, 2)\n\n            total_f = x0_f.shape[0]\n            do_opt = ((b_idx + 1) % grad_accum == 0) or ((b_idx + 1) == num_batches)\n            chunk_sz = max(1, args.max_chunk_frames)\n            batch_loss_sum = 0.0\n\n            for c_start in range(0, total_f, chunk_sz):\n                c_end = min(c_start + chunk_sz, total_f)\n                need_sync = (do_opt and c_end == total_f)\n                sync_ctx = (ddp_model.no_sync() if (is_ddp and not need_sync) else contextlib.nullcontext())\n\n                with sync_ctx:\n                    with torch.amp.autocast(\'cuda\', enabled=(device.type == \'cuda\')):\n                        pred_c, cw_c, f0_c, f1_c, fl_c, M0_c, M1_c = ddp_model(\n                            x0_f[c_start:c_end], x1_f[c_start:c_end],\n                            t_f[c_start:c_end], cf01_f[c_start:c_end], cf10_f[c_start:c_end],\n                            direct_t_flows=is_direct\n                        )\n                        gt_c = xk_f[c_start:c_end]\n                        hw_c = hw_f[c_start:c_end]\n\n                        l_main, d_main = lineart_loss(pred_c, gt_c, valid_hw=hw_c)\n                        l_occ, d_occ = occ_loss_fn(f0_c, f1_c, M0_c, M1_c, gt_c)\n                        l_step = l_main + l_occ\n                        scaled_l = l_step * (float(c_end - c_start) / float(total_f * grad_accum))\n\n                    scaler.scale(scaled_l).backward()\n                    batch_loss_sum += float(l_step.detach().item()) * (c_end - c_start)\n\n            acc_loss += batch_loss_sum\n            acc_frames += total_f\n\n            if do_opt:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), max_norm=1.0)\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad(set_to_none=True)\n                ema.update(base_model)\n\n            if args.dry_run and b_idx >= 1:\n                break\n\n        train_loss = acc_loss / max(1, acc_frames)\n        t_train = (time.time() - t0) / 60.0\n\n        # Đánh giá trên tập Validation (sử dụng trọng số EMA mượt mà)\n        t_val0 = time.time()\n        ema.apply_shadow(base_model)\n        val_m = validate_epoch_occlusion(val_loader, base_model, lineart_loss, occ_loss_fn, device, is_rank0=is_rank0)\n        ema.restore(base_model)\n        t_val = (time.time() - t_val0) / 60.0\n\n        # Composite score để chọn checkpoint tốt nhất\n        comp_score = val_m.get(\'val_loss\', 0.0) + 0.1 * val_m.get(\'loss_occ_cov\', 0.0)\n        is_best = comp_score < ckpt_mgr.best_score\n\n        row = {\n            \'epoch\': epoch,\n            \'train_loss\': train_loss,\n            \'val_loss\': val_m.get(\'val_loss\', 0.0),\n            \'val_psnr\': val_m.get(\'psnr\', 0.0),\n            \'val_ssim\': val_m.get(\'ssim\', 0.0),\n            \'val_skel_recall\': val_m.get(\'skel_recall\', 0.0),\n            \'val_tip_recall\': val_m.get(\'tip_recall\', 0.0),\n            \'val_M0_mean\': val_m.get(\'M0_mean\', 0.0),\n            \'val_M1_mean\': val_m.get(\'M1_mean\', 0.0),\n            \'comp_score\': comp_score,\n            \'train_time_min\': t_train,\n            \'val_time_min\': t_val,\n        }\n        history.append(row)\n\n        if is_rank0:\n            hist_csv = os.path.join(args.output_dir, "training_history_occlusion.csv")\n            pd.DataFrame(history).to_csv(hist_csv, index=False)\n            ckpt_mgr.save_epoch(epoch, base_model, ema, optimizer, scaler, comp_score, history, is_best=is_best)\n            print(\n                f"📌 EP {epoch:02d}/{args.epochs:02d} [Phase4_Occ] | Train={t_train:.2f}m | Val={t_val:.2f}m | "\n                f"Loss={train_loss:.4f} | ValLoss={row[\'val_loss\']:.4f} | PSNR={row[\'val_psnr\']:.2f}dB | "\n                f"SkelRec={row[\'val_skel_recall\']:.3f} | M0={row[\'val_M0_mean\']:.3f} | Best={\'YES\' if is_best else \'NO\'}",\n                flush=True\n            )\n\n        if args.dry_run:\n            break\n\n    if is_rank0:\n        print("=" * 84)\n        print("🎉 [PHASE 4 OCCLUSION HEAD] HUẤN LUYỆN HOÀN TẤT THÀNH CÔNG!")\n        print(f"   Best Checkpoint: {ckpt_mgr.best_path}")\n        print(f"   Last Checkpoint: {ckpt_mgr.last_path}")\n        print("=" * 84, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')

ml_path = os.path.join(WORKING_DIR, "multi_layer_tps.py")
with open(ml_path, "w", encoding="utf-8") as f:
    f.write('#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n"""\nmulti_layer_tps.py\n--------------------------------------------------------------------------------\nPHA 3 (BƯỚC 1 & BƯỚC 2): MODULE COARSE TPS ĐA LỚP (MULTI-LAYER CLUSTERED TPS)\nĐề tài: Nội suy khung hình hoạt họa Anime Lineart (PBL6)\nKiến trúc đích: TPSInbetweenOriginal (~10.07M active params, 0% thay đổi kiến trúc mạng)\n\nGiải quyết triệt để hạn chế vật lý của Single-Layer Global TPS:\n  1. Bước 1 - Phân cụm điểm chuyển động (Motion Segmentation):\n     - Tính vận tốc tương đối pixel: v_i = (P_{1,i} - P_{0,i}) * [W, H].\n     - Tách Cụm Nền Tĩnh (Static Layer, label = 0) với ||v_i||_2 <= eps_static_px (1.5 px).\n     - Khử nhiễu điểm khớp sai (Local Spatial-Velocity & RANSAC Affine Outlier Rejection, label = -1).\n     - Phân cụm Tiền Cảnh Động (Foreground Layers, label = 1..K) bằng thuật toán\n       Spatial-Velocity DBSCAN kết hợp kiểm tra đồng nhất Affine cục bộ.\n\n  2. Bước 2 - Thuật toán Multi-Layer TPS Flow Solver:\n     - Giải TPS_static trên cụm điểm tĩnh (triệt tiêu trọng số uốn phi tuyến RBF ->\n       đạo hàm bậc 2 bằng 0 tuyệt đối, không làm cong nền tĩnh).\n     - Giải TPS_dynamic độc lập trên từng cụm điểm động (không bị điểm tĩnh kéo ghì).\n     - Tạo mặt nạ mềm (Soft Blending Mask M) bằng khoảng cách Hausdorff / Voronoi\n       với hàm chuyển tiếp Quintic Smoothstep C^2:\n           F_fused = M * F_dynamic + (1 - M) * F_static\n     - Bảo toàn đạo hàm bậc 2 bằng 0 (phẳng tuyệt đối, ||Hessian(F)|| = 0) tại vùng nền tĩnh.\n--------------------------------------------------------------------------------\n"""\n\nimport math\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\ntry:\n    import cv2\nexcept ImportError:\n    cv2 = None\n\n\nFIXED_TPS_ANCHORS = np.array([\n    [0.0, 0.0], [0.5, 0.0], [1.0, 0.0],\n    [0.0, 0.5],             [1.0, 0.5],\n    [0.0, 1.0], [0.5, 1.0], [1.0, 1.0]\n], dtype=np.float32)\n\n_ML_GRID_CACHE = {}\n\n\n# ==============================================================================\n# BƯỚC 1: MODULE PHÂN CỤM ĐIỂM CHUYỂN ĐỘNG (MOTION SEGMENTATION)\n# ==============================================================================\ndef _dbscan_distance_matrix(dist_mat, eps=1.0, min_samples=3):\n    """\n    Thuật toán DBSCAN thuần NumPy trên ma trận khoảng cách chuẩn hóa D_sv (N x N).\n    Trả về mảng nhãn cluster: 0, 1, ..., K-1 cho các cụm hợp lệ, và -1 cho nhiễu (noise).\n    """\n    n = dist_mat.shape[0]\n    if n == 0:\n        return np.empty((0,), dtype=np.int32)\n\n    neighbors = [np.where(dist_mat[i] <= eps)[0] for i in range(n)]\n    is_core = np.array([len(nb) >= min_samples for nb in neighbors], dtype=bool)\n\n    labels = np.full((n,), -1, dtype=np.int32)\n    visited = np.zeros((n,), dtype=bool)\n    cluster_id = 0\n\n    for i in range(n):\n        if visited[i] or not is_core[i]:\n            continue\n        visited[i] = True\n        labels[i] = cluster_id\n        queue = list(neighbors[i])\n        head = 0\n        while head < len(queue):\n            q = queue[head]\n            head += 1\n            if not visited[q]:\n                visited[q] = True\n                if is_core[q]:\n                    for nb_q in neighbors[q]:\n                        if not visited[nb_q] or labels[nb_q] == -1:\n                            queue.append(nb_q)\n            if labels[q] == -1:\n                labels[q] = cluster_id\n        cluster_id += 1\n\n    return labels\n\n\ndef _filter_dynamic_outliers(p0_px, p1_px, vel_px, speed_px, k_nn=5, max_rel_err=0.65, min_abs_tol_px=14.0):\n    """\n    Khử điểm khớp sai (Outliers) từ GlueStick/SIFT dựa trên tính đồng nhất vận tốc - không gian\n    và mô hình Affine cục bộ (bảo toàn 100% biến dạng co giãn hình học như hình tròn thu/phóng).\n    """\n    n = len(p0_px)\n    if n < 6:\n        return np.ones((n,), dtype=bool)\n\n    mid_px = 0.5 * (p0_px + p1_px)\n    dist_mat = np.linalg.norm(mid_px[:, None, :] - mid_px[None, :, :], axis=-1)\n\n    k_use = min(k_nn, n - 1)\n    nn_idx = np.argsort(dist_mat, axis=-1)[:, :k_use + 1]  # bao gồm chính điểm i\n\n    # 1. Kiểm tra sai lệch vận tốc cục bộ so với median của lân cận k_nn\n    local_vels = vel_px[nn_idx]  # [N, k_use+1, 2]\n    med_vel = np.median(local_vels, axis=1)  # [N, 2]\n    med_speed = np.linalg.norm(med_vel, axis=-1)\n    vel_diff = np.linalg.norm(vel_px - med_vel, axis=-1)\n    local_inlier = vel_diff <= np.maximum(min_abs_tol_px, max_rel_err * med_speed)\n\n    # 2. Kiểm tra thêm bằng Affine toàn cục/bán cục bộ (nếu vật thể biến dạng co giãn đều như hình tròn)\n    if cv2 is not None and n >= 6:\n        try:\n            aff_m, aff_mask = cv2.estimateAffine2D(\n                p0_px.astype(np.float32), p1_px.astype(np.float32),\n                method=cv2.RANSAC, ransacReprojThreshold=max(12.0, 0.15 * float(np.median(speed_px)))\n            )\n            if aff_m is not None and aff_mask is not None:\n                aff_inlier = aff_mask.ravel().astype(bool)\n                # Điểm được giữ nếu thỏa mãn lân cận cục bộ HOẶC khớp chính xác phép biến đổi Affine\n                local_inlier = local_inlier | aff_inlier\n        except Exception:\n            pass\n\n    return local_inlier\n\n\ndef segment_motion_keypoints(\n    kps0,\n    kps1,\n    img_hw=(540, 960),\n    eps_static_px=1.5,\n    spatial_radius_ratio=0.18,\n    vel_tol_px=24.0,\n    min_cluster_pts=4,\n    filter_outliers=True,\n):\n    """\n    BƯỚC 1: Phân cụm điểm chuyển động (Motion Segmentation) trên cặp điểm đối sánh P0 <-> P1.\n\n    Tham số:\n      - kps0, kps1: Tọa độ điểm khớp chuẩn hóa [0, 1] kích thước (N, 2).\n      - img_hw: Tuple (H, W) kích thước ảnh gốc tính bằng pixel.\n      - eps_static_px: Ngưỡng vận tốc (px) để phân lập Cụm Nền Tĩnh (mặc định <= 1.5 px).\n      - spatial_radius_ratio: Bán kính không gian chuẩn hóa theo đường chéo ảnh cho DBSCAN.\n      - vel_tol_px: Ngưỡng dung sai vận tốc cơ sở (px) cho Spatial-Velocity DBSCAN.\n      - min_cluster_pts: Số điểm tối thiểu để tạo một cụm TPS động độc lập (>= 4).\n      - filter_outliers: Bật bộ lọc loại bỏ điểm khớp nhiễu cục bộ.\n\n    Trả về dict:\n      - \'labels\': Mảng (N,) với:\n          0     : Cụm Nền Tĩnh (Static Layer)\n          1..K  : Các Cụm Tiền Cảnh Động (Dynamic Foreground Layers)\n          -1    : Điểm nhiễu bị loại bỏ (Outliers)\n      - \'velocities_px\': Vector vận tốc (N, 2) tính theo pixel.\n      - \'speeds_px\': Độ lớn vận tốc ||v_i||_2 (N,) tính theo pixel.\n      - \'static_kps0\', \'static_kps1\': Các điểm thuộc cụm tĩnh.\n      - \'dynamic_clusters\': Danh sách dict mô tả từng cụm động k = 1..K.\n      - \'all_dyn_kps0\', \'all_dyn_kps1\': Toàn bộ điểm động hợp lệ gộp chung.\n    """\n    if kps0 is None or kps1 is None or len(kps0) == 0 or len(kps1) == 0:\n        empty_pts = np.zeros((0, 2), dtype=np.float32)\n        return {\n            \'labels\': np.empty((0,), dtype=np.int32),\n            \'velocities_px\': empty_pts,\n            \'speeds_px\': np.empty((0,), dtype=np.float32),\n            \'static_mask\': np.empty((0,), dtype=bool),\n            \'static_kps0\': empty_pts,\n            \'static_kps1\': empty_pts,\n            \'dynamic_clusters\': [],\n            \'all_dyn_kps0\': empty_pts,\n            \'all_dyn_kps1\': empty_pts,\n            \'num_static\': 0,\n            \'num_dynamic\': 0,\n            \'num_outliers\': 0,\n            \'num_clusters\': 0,\n        }\n\n    p0 = np.asarray(kps0, dtype=np.float32)\n    p1 = np.asarray(kps1, dtype=np.float32)\n    n_pts = len(p0)\n\n    h, w = float(img_hw[0]), float(img_hw[1])\n    scale_wh = np.array([max(1.0, w - 1.0), max(1.0, h - 1.0)], dtype=np.float32)\n    diag_px = math.hypot(w, h)\n\n    p0_px = p0 * scale_wh[None, :]\n    p1_px = p1 * scale_wh[None, :]\n    vel_px = p1_px - p0_px\n    speed_px = np.linalg.norm(vel_px, axis=-1)\n\n    labels = np.full((n_pts,), -1, dtype=np.int32)\n\n    # 1. Phân lập Cụm Tĩnh (Static Layer: ||v_i|| <= eps_static_px)\n    static_mask = speed_px <= float(eps_static_px)\n    labels[static_mask] = 0\n\n    # 2. Xử lý các điểm chuyển động (Dynamic Candidates: ||v_i|| > eps_static_px)\n    dyn_cand_idx = np.where(~static_mask)[0]\n    if len(dyn_cand_idx) > 0:\n        d_p0_px = p0_px[dyn_cand_idx]\n        d_p1_px = p1_px[dyn_cand_idx]\n        d_vel_px = vel_px[dyn_cand_idx]\n        d_speed_px = speed_px[dyn_cand_idx]\n\n        if filter_outliers and len(dyn_cand_idx) >= 6:\n            inlier_mask = _filter_dynamic_outliers(d_p0_px, d_p1_px, d_vel_px, d_speed_px)\n        else:\n            inlier_mask = np.ones((len(dyn_cand_idx),), dtype=bool)\n\n        valid_dyn_idx = dyn_cand_idx[inlier_mask]\n        if len(valid_dyn_idx) > 0:\n            vp0_px = p0_px[valid_dyn_idx]\n            vp1_px = p1_px[valid_dyn_idx]\n            vvel_px = vel_px[valid_dyn_idx]\n            vspeed_px = speed_px[valid_dyn_idx]\n            vmid_px = 0.5 * (vp0_px + vp1_px)\n\n            # Khoảng cách không gian - vận tốc chuẩn hóa cho DBSCAN\n            eps_spatial = max(32.0, float(spatial_radius_ratio) * diag_px)\n            med_v = float(np.median(vspeed_px)) if len(vspeed_px) > 0 else float(vel_tol_px)\n            eps_vel = max(float(vel_tol_px), 0.45 * med_v)\n\n            dist_sp = np.linalg.norm(vmid_px[:, None, :] - vmid_px[None, :, :], axis=-1) / eps_spatial\n            dist_vl = np.linalg.norm(vvel_px[:, None, :] - vvel_px[None, :, :], axis=-1) / eps_vel\n            dist_sv = np.sqrt(dist_sp ** 2 + dist_vl ** 2)\n\n            raw_clusters = _dbscan_distance_matrix(dist_sv, eps=1.0, min_samples=min(3, len(valid_dyn_idx)))\n\n            # Gom các cụm đủ điểm (>= min_cluster_pts) và gán các điểm rời rạc hợp lệ vào cụm gần nhất\n            unique_c = [c for c in sorted(set(raw_clusters)) if c >= 0]\n            major_clusters = [c for c in unique_c if int(np.sum(raw_clusters == c)) >= min_cluster_pts]\n\n            if not major_clusters:\n                # Nếu tất cả cụm đều nhỏ nhưng tổng số điểm động >= min_cluster_pts -> gộp thành 1 cụm động\n                if len(valid_dyn_idx) >= min_cluster_pts:\n                    labels[valid_dyn_idx] = 1\n                elif len(dyn_cand_idx) >= min_cluster_pts:\n                    labels[dyn_cand_idx] = 1\n            else:\n                remap = {c_old: (idx_new + 1) for idx_new, c_old in enumerate(major_clusters)}\n                for local_i, global_i in enumerate(valid_dyn_idx):\n                    c_old = int(raw_clusters[local_i])\n                    if c_old in remap:\n                        labels[global_i] = remap[c_old]\n                    else:\n                        # Gán điểm biên vào cụm chính gần nhất nếu vận tốc tương thích\n                        best_cid = -1\n                        best_d = 1e9\n                        for c_maj in major_clusters:\n                            c_mask = (raw_clusters == c_maj)\n                            d_min = float(np.min(dist_sv[local_i, c_mask]))\n                            if d_min < best_d:\n                                best_d = d_min\n                                best_cid = remap[c_maj]\n                        if best_d <= 2.2:\n                            labels[global_i] = best_cid\n\n    # Tổng hợp danh sách cụm động\n    dynamic_clusters = []\n    max_cid = int(labels.max()) if len(labels) > 0 else 0\n    for cid in range(1, max_cid + 1):\n        c_idx = np.where(labels == cid)[0]\n        if len(c_idx) < min_cluster_pts:\n            labels[c_idx] = -1\n            continue\n        dynamic_clusters.append({\n            \'cluster_id\': len(dynamic_clusters) + 1,\n            \'indices\': c_idx,\n            \'kps0\': p0[c_idx],\n            \'kps1\': p1[c_idx],\n            \'mean_vel_px\': vel_px[c_idx].mean(axis=0),\n            \'mean_speed_px\': float(speed_px[c_idx].mean()),\n        })\n\n    # Chuẩn hóa lại nhãn liên tục 1..K sau khi lọc\n    clean_labels = np.where(labels == 0, 0, -1).astype(np.int32)\n    for dc in dynamic_clusters:\n        clean_labels[dc[\'indices\']] = dc[\'cluster_id\']\n\n    all_dyn_mask = clean_labels >= 1\n    return {\n        \'labels\': clean_labels,\n        \'velocities_px\': vel_px,\n        \'speeds_px\': speed_px,\n        \'static_mask\': (clean_labels == 0),\n        \'static_kps0\': p0[clean_labels == 0],\n        \'static_kps1\': p1[clean_labels == 0],\n        \'dynamic_clusters\': dynamic_clusters,\n        \'all_dyn_kps0\': p0[all_dyn_mask],\n        \'all_dyn_kps1\': p1[all_dyn_mask],\n        \'num_static\': int(np.sum(clean_labels == 0)),\n        \'num_dynamic\': int(np.sum(all_dyn_mask)),\n        \'num_outliers\': int(np.sum(clean_labels == -1)),\n        \'num_clusters\': len(dynamic_clusters),\n    }\n\n\n# ==============================================================================\n# BƯỚC 2: THUẬT TOÁN MULTI-LAYER TPS FLOW SOLVER & SOFT VORONOI MASK\n# ==============================================================================\ndef _stratified_spatial_subsample(c_src, c_dst, n_keep):\n    """\n    Lấy mẫu đại diện phân bố đều theo không gian (Farthest-Point / Spatial Grid Stratification)\n    thay vì cắt chỉ số tuyến tính đơn thuần, giúp giữ trọn vẹn biên hình học của mọi bộ phận.\n    """\n    n = len(c_src)\n    if n <= n_keep:\n        return np.asarray(c_src, dtype=np.float64), np.asarray(c_dst, dtype=np.float64)\n\n    cs = np.asarray(c_src, dtype=np.float64)\n    cd = np.asarray(c_dst, dtype=np.float64)\n\n    # Khởi tạo bằng điểm gần tâm nhất, sau đó chọn lặp điểm xa nhất (Farthest Point Sampling)\n    selected = np.zeros((n_keep,), dtype=np.int64)\n    center = cs.mean(axis=0, keepdims=True)\n    selected[0] = int(np.argmin(np.linalg.norm(cs - center, axis=-1)))\n    min_dists = np.linalg.norm(cs - cs[selected[0]], axis=-1)\n\n    for k in range(1, n_keep):\n        pick = int(np.argmax(min_dists))\n        selected[k] = pick\n        d_new = np.linalg.norm(cs - cs[pick], axis=-1)\n        min_dists = np.minimum(min_dists, d_new)\n\n    selected.sort()\n    return cs[selected], cd[selected]\n\n\ndef _solve_tps_system(cs_all, cd_all, max_pts=128, lambd=0.01):\n    """Giải hệ phương trình tuyến tính TPS (K + lambda*I) w + P a = cd_all."""\n    m = len(cs_all)\n    diff = cs_all[:, None, :] - cs_all[None, :, :]\n    r = np.linalg.norm(diff, axis=-1)\n    K = (r ** 2) * np.log(np.maximum(r, 1e-6))\n    np.fill_diagonal(K, float(lambd))\n\n    P = np.hstack([np.ones((m, 1), dtype=np.float64), cs_all])\n    A = np.zeros((m + 3, m + 3), dtype=np.float64)\n    A[:m, :m] = K\n    A[:m, m:] = P\n    A[m:, :m] = P.T\n\n    Y = np.zeros((m + 3, 2), dtype=np.float64)\n    Y[:m, :] = cd_all\n\n    try:\n        theta = np.linalg.solve(A, Y)\n    except np.linalg.LinAlgError:\n        theta, _, _, _ = np.linalg.lstsq(A, Y, rcond=1e-4)\n\n    valid = 0.0 if (\n        np.isnan(theta).any() or np.isinf(theta).any() or np.max(np.abs(theta)) > 50.0\n    ) else 1.0\n\n    c_pad = np.zeros((max_pts, 2), dtype=np.float32)\n    th_pad = np.zeros((max_pts + 3, 2), dtype=np.float32)\n    if valid > 0.5:\n        c_pad[:m] = cs_all.astype(np.float32)\n        th_pad[:m] = theta[:m].astype(np.float32)\n        th_pad[max_pts:max_pts + 3] = theta[m:m + 3].astype(np.float32)\n    return c_pad, th_pad, valid\n\n\ndef solve_tps_static_layer(c_src_static, c_dst_static, max_pts=128, lambd=0.1, zero_hessian=True):\n    """\n    BƯỚC 2.1 & 2.4: Giải TPS_static cho cụm điểm nền tĩnh.\n    Đảm bảo đạo hàm bậc 2 bằng 0 tuyệt đối (phẳng tuyệt đối, ||Hessian(F_static)|| = 0):\n      - Khi các điểm thuộc Cụm Tĩnh (||v_i|| <= eps_static), nếu zero_hessian=True,\n        ta triệt tiêu hoàn toàn thành phần uốn phi tuyến RBF (th_pad[:m] = 0.0) và\n        nếu độ lệch tĩnh trung vị gần bằng 0, đặt ánh xạ đồng nhất P(q) = q (zero flow tuyệt đối).\n    """\n    c_pad = np.zeros((max_pts, 2), dtype=np.float32)\n    th_pad = np.zeros((max_pts + 3, 2), dtype=np.float32)\n    # Ma trận Affine đồng nhất f(x, y) = [0, 0] + 1*x + 1*y -> disp(q) = f(q) - q = 0\n    th_pad[max_pts + 1, 0] = 1.0\n    th_pad[max_pts + 2, 1] = 1.0\n\n    if c_src_static is None or c_dst_static is None or len(c_src_static) == 0:\n        return c_pad, th_pad, 1.0\n\n    cs = np.asarray(c_src_static, dtype=np.float64)\n    cd = np.asarray(c_dst_static, dtype=np.float64)\n    max_disp = float(np.max(np.linalg.norm(cd - cs, axis=-1))) if len(cs) > 0 else 0.0\n\n    # Nếu cụm tĩnh đứng yên hoặc dịch chuyển vi mô (< 2e-3 trên miền chuẩn hóa [0,1]),\n    # trả về nghiệm đồng nhất phẳng tuyệt đối (Zero Flow, đạo hàm bậc 1 và bậc 2 đều bằng 0)\n    if zero_hessian or max_disp < 2.5e-3 or len(cs) < 4:\n        return c_pad, th_pad, 1.0\n\n    n_keep = max_pts - 8\n    cs_sub, cd_sub = _stratified_spatial_subsample(cs, cd, n_keep)\n    anc = FIXED_TPS_ANCHORS.astype(np.float64)\n    cs_all = np.vstack([cs_sub, anc])\n    cd_all = np.vstack([cd_sub, anc])\n    c_sol, th_sol, val = _solve_tps_system(cs_all, cd_all, max_pts=max_pts, lambd=lambd)\n    if val > 0.5:\n        # Khóa chặt trọng số RBF phi tuyến = 0 để đảm bảo đạo hàm bậc 2 bằng 0 tuyệt đối\n        th_sol[:max_pts, :] = 0.0\n        return c_sol, th_sol, 1.0\n    return c_pad, th_pad, 1.0\n\n\ndef solve_tps_dynamic_layer(c_src_dyn, c_dst_dyn, max_pts=128, lambd=0.01, use_anchors=False):\n    """\n    BƯỚC 2.2: Giải TPS_dynamic độc lập trên cụm điểm động (không bị điểm tĩnh kéo ghì).\n    """\n    if c_src_dyn is None or c_dst_dyn is None or len(c_src_dyn) < 4:\n        return (\n            np.zeros((max_pts, 2), dtype=np.float32),\n            np.zeros((max_pts + 3, 2), dtype=np.float32),\n            0.0,\n        )\n    n_keep = (max_pts - 8) if use_anchors else max_pts\n    cs_sub, cd_sub = _stratified_spatial_subsample(c_src_dyn, c_dst_dyn, n_keep)\n    if use_anchors:\n        anc = FIXED_TPS_ANCHORS.astype(np.float64)\n        cs_all = np.vstack([cs_sub, anc])\n        cd_all = np.vstack([cd_sub, anc])\n    else:\n        cs_all, cd_all = cs_sub, cd_sub\n    return _solve_tps_system(cs_all, cd_all, max_pts=max_pts, lambd=lambd)\n\n\ndef _get_ml_norm_grid(h, w, device):\n    key = (h, w, device, \'ml_tps_grid\')\n    pts = _ML_GRID_CACHE.get(key)\n    if pts is None:\n        gy, gx = torch.meshgrid(\n            torch.linspace(0.0, 1.0, h, device=device, dtype=torch.float32),\n            torch.linspace(0.0, 1.0, w, device=device, dtype=torch.float32),\n            indexing=\'ij\'\n        )\n        pts = torch.stack([gx, gy], dim=-1).view(-1, 2).contiguous()\n        _ML_GRID_CACHE[key] = pts\n    return pts\n\n\n@torch.no_grad()\ndef _eval_tps_flow_single(theta, c_src, valid_flag, orig_hw, h_pad, w_pad, device, downscale_factor=2):\n    """Đánh giá nhanh trường dòng chảy TPS trên GPU từ cặp (theta, c_src) kích thước [1, ...]."""\n    if float(valid_flag) < 0.5:\n        return torch.zeros((1, 2, h_pad, w_pad), dtype=torch.float32, device=device)\n\n    with torch.amp.autocast(\'cuda\', enabled=False):\n        th_t = torch.as_tensor(theta, dtype=torch.float32, device=device)\n        cs_t = torch.as_tensor(c_src, dtype=torch.float32, device=device)\n        if th_t.ndim == 2:\n            th_t = th_t.unsqueeze(0)\n        if cs_t.ndim == 2:\n            cs_t = cs_t.unsqueeze(0)\n\n        b = th_t.shape[0]\n        h_sub = max(16, h_pad // max(1, downscale_factor))\n        w_sub = max(16, w_pad // max(1, downscale_factor))\n\n        pts_pad = _get_ml_norm_grid(h_sub, w_sub, device)\n        total_pts = pts_pad.shape[0]\n        pts_pad_exp = pts_pad.unsqueeze(0).expand(b, -1, -1)\n\n        if isinstance(orig_hw, (tuple, list)):\n            orig_h, orig_w = float(orig_hw[0]), float(orig_hw[1])\n            sx = float(max(1, w_pad - 1)) / max(1.0, orig_w - 1.0)\n            sy = float(max(1, h_pad - 1)) / max(1.0, orig_h - 1.0)\n            scale_xy = torch.tensor([[[sx, sy]]], dtype=torch.float32, device=device)\n        else:\n            orig_hw_f = orig_hw.to(device=device, dtype=torch.float32)\n            sx = (float(max(1, w_pad - 1)) / torch.clamp(orig_hw_f[:, 1] - 1.0, min=1.0)).view(b, 1, 1)\n            sy = (float(max(1, h_pad - 1)) / torch.clamp(orig_hw_f[:, 0] - 1.0, min=1.0)).view(b, 1, 1)\n            scale_xy = torch.cat([sx, sy], dim=-1)\n\n        pts_orig_raw = pts_pad_exp * scale_xy\n        in_bounds = ((pts_orig_raw[..., 0:1] <= 1.001) & (pts_orig_raw[..., 1:2] <= 1.001)).float()\n        pts_orig = torch.clamp(pts_orig_raw, 0.0, 1.0)\n\n        flow_chunks = []\n        chunk_size = 32768\n        for i in range(0, total_pts, chunk_size):\n            end_i = min(i + chunk_size, total_pts)\n            q = pts_orig[:, i:end_i, :]\n            dist = torch.clamp(torch.cdist(q, cs_t, p=2.0), min=1e-6)\n            U = (dist ** 2) * torch.log(dist)\n            ones = torch.ones((b, q.shape[1], 1), device=device, dtype=torch.float32)\n            basis = torch.cat([U, ones, q], dim=-1)\n            mapped = torch.bmm(basis, th_t)\n            disp_orig = mapped - q\n            disp_pad = (disp_orig / scale_xy) * in_bounds[:, i:end_i, :]\n            flow_chunks.append(disp_pad)\n\n        flow = torch.cat(flow_chunks, dim=1).view(b, h_sub, w_sub, 2).permute(0, 3, 1, 2).contiguous()\n        if h_sub != h_pad or w_sub != w_pad:\n            flow = F.interpolate(flow, size=(h_pad, w_pad), mode=\'bilinear\', align_corners=True)\n        return flow\n\n\n@torch.no_grad()\ndef compute_soft_voronoi_blending_mask(\n    dyn_pts_list_norm,\n    static_pts_norm,\n    orig_hw,\n    h_pad,\n    w_pad,\n    device,\n    downscale_factor=2,\n    transition_width_px=18.0,\n    falloff_radius_px=160.0,\n):\n    """\n    BƯỚC 2.3 & 2.4: Tạo mặt nạ hòa trộn mềm (Soft Blending Mask M) dựa trên khoảng cách\n    Hausdorff / Voronoi từ mỗi điểm lưới tới các Cụm Điểm Động và Cụm Điểm Tĩnh.\n\n    Đặc tính toán học:\n      - Với mỗi cụm động k = 1..K, tính khoảng cách Euclidean ngắn nhất (Hausdorff point-to-set)\n        d_dyn^(k)(q) từ điểm lưới q tới quỹ đạo điểm động.\n      - Với cụm tĩnh, tính khoảng cách d_static(q) tới tập điểm tĩnh.\n        (Nếu không có điểm tĩnh tường minh, dùng ngưỡng bán kính ảnh hưởng falloff_radius_px).\n      - Sử dụng hàm Quintic Smoothstep C^2:\n            u(q) = clip((d_static(q) - d_dyn(q) + w_trans) / (2 * w_trans), 0, 1)\n            M(q) = 6*u^5 - 15*u^4 + 10*u^3\n      - Tại vùng nền tĩnh (d_static(q) <= d_dyn(q) - w_trans):\n            M(q) == 0.0 tuyệt đối -> Đạo hàm bậc 1 và bậc 2 bằng 0 hoàn toàn!\n      - Tại lõi vật thể động (d_dyn(q) <= d_static(q) - w_trans):\n            M(q) == 1.0 tuyệt đối -> Giữ nguyên 100% trường biến dạng động!\n\n    Trả về:\n      - weights_dyn: List gồm K tensor mặt nạ [1, 1, h_pad, w_pad] cho từng cụm động.\n      - weight_static: Tensor mặt nạ tĩnh (1 - M_total) kích thước [1, 1, h_pad, w_pad].\n      - mask_total: Tensor M_total = sum_k M_k kích thước [1, 1, h_pad, w_pad].\n    """\n    if not dyn_pts_list_norm:\n        zero_m = torch.zeros((1, 1, h_pad, w_pad), dtype=torch.float32, device=device)\n        one_m = torch.ones((1, 1, h_pad, w_pad), dtype=torch.float32, device=device)\n        return [], one_m, zero_m\n\n    if isinstance(orig_hw, torch.Tensor):\n        orig_h = float(orig_hw[0, 0].item())\n        orig_w = float(orig_hw[0, 1].item())\n    else:\n        orig_h, orig_w = float(orig_hw[0]), float(orig_hw[1])\n\n    h_sub = max(16, h_pad // max(1, downscale_factor))\n    w_sub = max(16, w_pad // max(1, downscale_factor))\n\n    pts_pad = _get_ml_norm_grid(h_sub, w_sub, device)  # [P, 2] trong [0, 1]^2 của canvas pad\n    sx = float(max(1, w_pad - 1)) / max(1.0, orig_w - 1.0)\n    sy = float(max(1, h_pad - 1)) / max(1.0, orig_h - 1.0)\n\n    # Đổi điểm lưới sang đơn vị pixel thực tế của ảnh gốc [0, W] x [0, H]\n    q_px = torch.stack([\n        pts_pad[:, 0] * sx * max(1.0, orig_w - 1.0),\n        pts_pad[:, 1] * sy * max(1.0, orig_h - 1.0),\n    ], dim=-1).unsqueeze(0)  # [1, P, 2]\n\n    scale_wh_t = torch.tensor([[max(1.0, orig_w - 1.0), max(1.0, orig_h - 1.0)]], dtype=torch.float32, device=device)\n\n    # Tính khoảng cách d_dyn^(k)(q) cho từng cụm động\n    d_dyn_list = []\n    for d_pts in dyn_pts_list_norm:\n        dp_t = torch.as_tensor(d_pts, dtype=torch.float32, device=device)\n        if dp_t.ndim == 2:\n            dp_px = (dp_t * scale_wh_t).unsqueeze(0)  # [1, M_k, 2]\n        else:\n            dp_px = dp_t * scale_wh_t.unsqueeze(0)\n        dist_k = torch.cdist(q_px, dp_px, p=2.0).min(dim=-1).values  # [1, P]\n        d_dyn_list.append(dist_k)\n\n    d_dyn_stack = torch.stack(d_dyn_list, dim=0)  # [K, 1, P]\n    d_dyn_min = d_dyn_stack.min(dim=0).values     # [1, P]\n\n    # Tính khoảng cách d_static(q) tới cụm điểm tĩnh\n    if static_pts_norm is not None and len(static_pts_norm) > 0:\n        sp_t = torch.as_tensor(static_pts_norm, dtype=torch.float32, device=device)\n        sp_px = (sp_t * scale_wh_t).unsqueeze(0)  # [1, M_s, 2]\n        d_static = torch.cdist(q_px, sp_px, p=2.0).min(dim=-1).values  # [1, P]\n        if falloff_radius_px is not None and falloff_radius_px > 0:\n            d_static = torch.minimum(d_static, torch.full_like(d_static, float(falloff_radius_px)))\n    else:\n        d_static = torch.full_like(d_dyn_min, float(falloff_radius_px))\n\n    # Mặt nạ tổng M_total bằng hàm Quintic Smoothstep C^2 trên biên Voronoi (d_static - d_dyn_min)\n    w_tr = max(2.0, float(transition_width_px))\n    u = torch.clamp((d_static - d_dyn_min + w_tr) / (2.0 * w_tr), 0.0, 1.0)\n    m_total_flat = (u ** 3) * (u * (u * 6.0 - 15.0) + 10.0)  # [1, P]\n\n    # Bảo vệ tuyệt đối vùng lân cận điểm tĩnh: mọi điểm nằm gần điểm tĩnh hơn điểm động\n    # hoặc cách điểm tĩnh dưới ngưỡng an toàn (22 px) và xa điểm động đều khóa M = 0.0\n    if static_pts_norm is not None and len(static_pts_norm) > 0:\n        strict_static = (d_static < 24.0) & (d_dyn_min > d_static + 6.0)\n        m_total_flat = torch.where(strict_static, torch.zeros_like(m_total_flat), m_total_flat)\n\n    # Khóa cứng vùng biên để đảm bảo đạo hàm bậc 2 bằng 0 tuyệt đối trên vùng tĩnh\n    m_total_flat = torch.where(m_total_flat < 1e-4, torch.zeros_like(m_total_flat), m_total_flat)\n    m_total_flat = torch.where(m_total_flat > 1.0 - 1e-4, torch.ones_like(m_total_flat), m_total_flat)\n\n    m_total = m_total_flat.view(1, 1, h_sub, w_sub)\n    if h_sub != h_pad or w_sub != w_pad:\n        m_total = F.interpolate(m_total, size=(h_pad, w_pad), mode=\'bilinear\', align_corners=True)\n        # Bảo toàn vùng 0 tuyệt đối sau nội suy bilinear\n        m_total = torch.where(m_total < 1e-3, torch.zeros_like(m_total), m_total)\n        m_total = torch.where(m_total > 1.0 - 1e-3, torch.ones_like(m_total), m_total)\n\n    if len(d_dyn_list) == 1:\n        weights_dyn = [m_total]\n    else:\n        # Phân bổ Partition of Unity giữa K cụm động theo Soft Voronoi\n        tau_k = max(4.0, 0.5 * w_tr)\n        logits_k = -d_dyn_stack / tau_k  # [K, 1, P]\n        probs_k = torch.softmax(logits_k, dim=0).view(len(d_dyn_list), 1, h_sub, w_sub)\n        if h_sub != h_pad or w_sub != w_pad:\n            probs_k = F.interpolate(probs_k, size=(h_pad, w_pad), mode=\'bilinear\', align_corners=True)\n            probs_k = probs_k / probs_k.sum(dim=0, keepdim=True).clamp(min=1e-6)\n        weights_dyn = [m_total * probs_k[k:k + 1] for k in range(len(d_dyn_list))]\n\n    weight_static = 1.0 - m_total\n    return weights_dyn, weight_static, m_total\n\n\n@torch.no_grad()\ndef solve_multi_layer_tps(\n    kps0,\n    kps1,\n    orig_hw,\n    h_pad,\n    w_pad,\n    device,\n    t_val=None,\n    direct_t_flows=False,\n    max_pts=128,\n    lambd=0.01,\n    downscale_factor=2,\n    eps_static_px=1.5,\n    use_dyn_anchors=False,\n    transition_width_px=22.0,\n    falloff_radius_px=105.0,\n    filter_outliers=True,\n):\n    """\n    BƯỚC 2: Hàm giải tổng hợp Multi-Layer Clustered TPS Flow Solver (`solve_multi_layer_tps`).\n\n    Hỗ trợ đầy đủ cả 2 chế độ vận hành của kiến trúc `TPSInbetweenOriginal`:\n      1. Chế độ Lagrangian Direct-t (`direct_t_flows=True`, truyền `t_val` thuộc [0, 1]):\n         - Nội suy vị trí trung gian P_t = (1 - t)*P_0 + t*P_1 cho từng lớp.\n         - Giải TPS_static -> F_static_0t, F_static_1t (đạo hàm bậc 2 = 0 tuyệt đối).\n         - Giải TPS_dynamic^(k) -> F_dyn_0t^(k), F_dyn_1t^(k) độc lập với nền tĩnh.\n         - Hòa trộn bằng mặt nạ Voronoi/Hausdorff M:\n             F_fused_0t = sum_k M_k * F_dyn_0t^(k) + (1 - M) * F_static_0t\n             F_fused_1t = sum_k M_k * F_dyn_1t^(k) + (1 - M) * F_static_1t\n      2. Chế độ Backward-Pair (`direct_t_flows=False`):\n         - Giải cặp dòng chảy ngược F_01 (từ P_1 -> P_0) và F_10 (từ P_0 -> P_1) đa lớp.\n         - Kết hợp mặt nạ ống quỹ đạo (Trajectory Tube Voronoi Mask) bảo vệ tuyệt đối\n           các vùng nền tĩnh khỏi năng lượng uốn của vật thể chuyển động.\n\n    Trả về:\n      - flow0_fused: Tensor [1, 2, h_pad, w_pad] (cf0t hoặc cf01)\n      - flow1_fused: Tensor [1, 2, h_pad, w_pad] (cf1t hoặc cf10)\n      - meta_info: Dict chứa kết quả phân cụm Bước 1, mặt nạ M, và kiểm định đạo hàm bậc 2.\n    """\n    if isinstance(orig_hw, torch.Tensor):\n        img_h = int(orig_hw[0, 0].item())\n        img_w = int(orig_hw[0, 1].item())\n    else:\n        img_h, img_w = int(orig_hw[0]), int(orig_hw[1])\n\n    # BƯỚC 1: Phân cụm điểm chuyển động Tĩnh vs Động\n    seg = segment_motion_keypoints(\n        kps0, kps1,\n        img_hw=(img_h, img_w),\n        eps_static_px=eps_static_px,\n        filter_outliers=filter_outliers,\n    )\n\n    zero_flow = torch.zeros((1, 2, h_pad, w_pad), dtype=torch.float32, device=device)\n\n    # Nếu không có cụm động nào hợp lệ -> trả về dòng chảy tĩnh phẳng tuyệt đối (Zero Flow)\n    if seg[\'num_clusters\'] == 0:\n        return zero_flow.clone(), zero_flow.clone(), {\n            \'segmentation\': seg,\n            \'mask_total\': torch.zeros((1, 1, h_pad, w_pad), dtype=torch.float32, device=device),\n            \'static_hessian_max\': 0.0,\n        }\n\n    # BƯỚC 2.1: Giải TPS_static cho Cụm Nền Tĩnh (F_static có đạo hàm bậc 2 = 0 tuyệt đối)\n    f_static_0 = zero_flow.clone()\n    f_static_1 = zero_flow.clone()\n\n    # BƯỚC 2.2: Giải TPS_dynamic cho từng Cụm Tiền Cảnh Động\n    dyn_flows_0 = []\n    dyn_flows_1 = []\n    dyn_corridors = []\n\n    tv = float(t_val) if t_val is not None else 0.5\n\n    for dc in seg[\'dynamic_clusters\']:\n        p0_k = dc[\'kps0\']\n        p1_k = dc[\'kps1\']\n        pt_k = (1.0 - tv) * p0_k + tv * p1_k\n        pm_k = 0.5 * (p0_k + p1_k)\n\n        if direct_t_flows:\n            # Giải trực tiếp tại mốc thời gian t: P_t -> P_0 và P_t -> P_1\n            c_t0, th_t0, v0 = solve_tps_dynamic_layer(pt_k, p0_k, max_pts=max_pts, lambd=lambd, use_anchors=use_dyn_anchors)\n            c_t1, th_t1, v1 = solve_tps_dynamic_layer(pt_k, p1_k, max_pts=max_pts, lambd=lambd, use_anchors=use_dyn_anchors)\n            f0_k = _eval_tps_flow_single(th_t0, c_t0, v0, (img_h, img_w), h_pad, w_pad, device, downscale_factor)\n            f1_k = _eval_tps_flow_single(th_t1, c_t1, v1, (img_h, img_w), h_pad, w_pad, device, downscale_factor)\n            # Tập điểm quỹ đạo để bao phủ cả vị trí đích P_t và vị trí nguồn P_0, P_1 (tránh để lại bóng ma tại nguồn)\n            corridor_k = np.vstack([p0_k, pm_k, pt_k, p1_k])\n        else:\n            # Giải theo chuẩn Backward-Pair: c0 = P_1 -> P_0 (flow01) và c1 = P_0 -> P_1 (flow10)\n            c0_k, th01_k, v01 = solve_tps_dynamic_layer(p1_k, p0_k, max_pts=max_pts, lambd=lambd, use_anchors=use_dyn_anchors)\n            c1_k, th10_k, v10 = solve_tps_dynamic_layer(p0_k, p1_k, max_pts=max_pts, lambd=lambd, use_anchors=use_dyn_anchors)\n            f0_k = _eval_tps_flow_single(th01_k, c0_k, v01, (img_h, img_w), h_pad, w_pad, device, downscale_factor)\n            f1_k = _eval_tps_flow_single(th10_k, c1_k, v10, (img_h, img_w), h_pad, w_pad, device, downscale_factor)\n            corridor_k = np.vstack([p0_k, pm_k, p1_k])\n\n        dyn_flows_0.append(f0_k)\n        dyn_flows_1.append(f1_k)\n        dyn_corridors.append(corridor_k)\n\n    # Tập điểm tĩnh tham chiếu cho biên Voronoi\n    if seg[\'num_static\'] > 0:\n        static_ref_pts = (1.0 - tv) * seg[\'static_kps0\'] + tv * seg[\'static_kps1\']\n        effective_falloff = float(falloff_radius_px)\n    elif use_dyn_anchors:\n        static_ref_pts = FIXED_TPS_ANCHORS\n        effective_falloff = float(falloff_radius_px)\n    else:\n        static_ref_pts = None\n        effective_falloff = max(float(falloff_radius_px), 0.50 * math.hypot(img_w, img_h))\n\n    # BƯỚC 2.3: Tính mặt nạ mềm Soft Blending Mask M dựa trên khoảng cách Hausdorff / Voronoi\n    weights_dyn, weight_static, mask_total = compute_soft_voronoi_blending_mask(\n        dyn_corridors,\n        static_ref_pts,\n        orig_hw=(img_h, img_w),\n        h_pad=h_pad,\n        w_pad=w_pad,\n        device=device,\n        downscale_factor=downscale_factor,\n        transition_width_px=transition_width_px,\n        falloff_radius_px=effective_falloff,\n    )\n\n    # Hòa trộn đa lớp: F_fused = sum_k M_k * F_dynamic^(k) + (1 - M) * F_static\n    flow0_fused = weight_static * f_static_0\n    flow1_fused = weight_static * f_static_1\n    for w_k, f0_k, f1_k in zip(weights_dyn, dyn_flows_0, dyn_flows_1):\n        flow0_fused = flow0_fused + w_k * f0_k\n        flow1_fused = flow1_fused + w_k * f1_k\n\n    # Anti-Ghost Pullback Guard tại ranh giới giao thoa (Occlusion / Transition Boundary):\n    # Ngăn chặn vùng chuyển tiếp (0 < M < 1) vô tình kéo nét vẽ từ Cụm Tĩnh lên vùng trống.\n    # Tính trên lưới giảm mẫu (h_sub, w_sub) để tiết kiệm 4x VRAM và tăng tốc 4x trên GPU.\n    if seg[\'num_static\'] > 0:\n        h_sub_g = max(16, h_pad // max(1, downscale_factor))\n        w_sub_g = max(16, w_pad // max(1, downscale_factor))\n        gy, gx = torch.meshgrid(\n            torch.linspace(0.0, 1.0, h_sub_g, device=device, dtype=torch.float32),\n            torch.linspace(0.0, 1.0, w_sub_g, device=device, dtype=torch.float32),\n            indexing=\'ij\'\n        )\n        base_g = torch.stack([gx, gy], dim=-1).unsqueeze(0)  # [1, h_sub, w_sub, 2]\n        sx = float(max(1, w_pad - 1)) / max(1.0, float(img_w) - 1.0)\n        sy = float(max(1, h_pad - 1)) / max(1.0, float(img_h) - 1.0)\n        scale_wh_px = torch.tensor([[[[float(img_w - 1), float(img_h - 1)]]]], dtype=torch.float32, device=device)\n        scale_pad_to_orig = torch.tensor([[[[sx, sy]]]], dtype=torch.float32, device=device)\n\n        f0_sub = F.interpolate(flow0_fused, size=(h_sub_g, w_sub_g), mode=\'bilinear\', align_corners=True) if (h_sub_g != h_pad or w_sub_g != w_pad) else flow0_fused\n        f1_sub = F.interpolate(flow1_fused, size=(h_sub_g, w_sub_g), mode=\'bilinear\', align_corners=True) if (h_sub_g != h_pad or w_sub_g != w_pad) else flow1_fused\n        m_sub = F.interpolate(mask_total, size=(h_sub_g, w_sub_g), mode=\'bilinear\', align_corners=True) if (h_sub_g != h_pad or w_sub_g != w_pad) else mask_total\n\n        q0_src_px = ((base_g + f0_sub.permute(0, 2, 3, 1)) * scale_pad_to_orig * scale_wh_px).view(1, -1, 2)\n        q1_src_px = ((base_g + f1_sub.permute(0, 2, 3, 1)) * scale_pad_to_orig * scale_wh_px).view(1, -1, 2)\n\n        sp0_px = (torch.as_tensor(seg[\'static_kps0\'], dtype=torch.float32, device=device) * scale_wh_px.view(1, 2)).unsqueeze(0)\n        sp1_px = (torch.as_tensor(seg[\'static_kps1\'], dtype=torch.float32, device=device) * scale_wh_px.view(1, 2)).unsqueeze(0)\n\n        pt_all_px = (torch.as_tensor(\n            np.vstack([(1.0 - tv) * dc[\'kps0\'] + tv * dc[\'kps1\'] for dc in seg[\'dynamic_clusters\']]),\n            dtype=torch.float32, device=device\n        ) * scale_wh_px.view(1, 2)).unsqueeze(0)\n        q_cur_px = (base_g * scale_pad_to_orig * scale_wh_px).view(1, -1, 2)\n        d_cur_dyn = torch.cdist(q_cur_px, pt_all_px, p=2.0).min(dim=-1).values.view(1, 1, h_sub_g, w_sub_g)\n\n        d_src0_stat = torch.cdist(q0_src_px, sp0_px, p=2.0).min(dim=-1).values.view(1, 1, h_sub_g, w_sub_g)\n        d_src1_stat = torch.cdist(q1_src_px, sp1_px, p=2.0).min(dim=-1).values.view(1, 1, h_sub_g, w_sub_g)\n\n        outside_dyn_stroke = (m_sub > 1e-4) & (d_cur_dyn > 10.0)\n        pull_static_0 = (outside_dyn_stroke & (d_src0_stat < 22.0)).float()\n        pull_static_1 = (outside_dyn_stroke & (d_src1_stat < 22.0)).float()\n        if h_sub_g != h_pad or w_sub_g != w_pad:\n            pull_static_0 = F.interpolate(pull_static_0, size=(h_pad, w_pad), mode=\'nearest\')\n            pull_static_1 = F.interpolate(pull_static_1, size=(h_pad, w_pad), mode=\'nearest\')\n        flow0_fused = torch.where(pull_static_0 > 0.5, f_static_0, flow0_fused)\n        flow1_fused = torch.where(pull_static_1 > 0.5, f_static_1, flow1_fused)\n\n    # BƯỚC 2.4: Kiểm định đạo hàm bậc 2 (Hessian) tại vùng nền tĩnh (M == 0)\n    static_zone = (mask_total < 1e-4)\n    hess_max = verify_static_zero_hessian(flow0_fused, static_zone)\n\n    return flow0_fused, flow1_fused, {\n        \'segmentation\': seg,\n        \'mask_total\': mask_total,\n        \'weights_dyn\': weights_dyn,\n        \'weight_static\': weight_static,\n        \'static_hessian_max\': hess_max,\n    }\n\n\n@torch.no_grad()\ndef verify_static_zero_hessian(flow_tensor, static_zone_mask=None):\n    """\n    Tính đạo hàm bậc 2 rời rạc (Discrete Second Derivatives: d2F/dx2, d2F/dy2, d2F/dxdy)\n    của trường dòng chảy tại vùng nền tĩnh để chứng minh toán học ||Hessian(F)|| == 0.\n    """\n    # Sai phân bậc 2 theo trục X và trục Y\n    d2x = flow_tensor[:, :, 1:-1, 2:] - 2.0 * flow_tensor[:, :, 1:-1, 1:-1] + flow_tensor[:, :, 1:-1, :-2]\n    d2y = flow_tensor[:, :, 2:, 1:-1] - 2.0 * flow_tensor[:, :, 1:-1, 1:-1] + flow_tensor[:, :, :-2, 1:-1]\n    hess_mag = torch.sqrt(d2x ** 2 + d2y ** 2)\n\n    if static_zone_mask is not None:\n        inner_mask = static_zone_mask[:, :, 1:-1, 1:-1]\n        # Chỉ xét những điểm mà toàn bộ lân cận 3x3 đều nằm trong vùng tĩnh\n        eroded_static = -F.max_pool2d(-inner_mask.float(), kernel_size=3, stride=1, padding=1) > 0.5\n        if eroded_static.any():\n            return float(hess_mag.masked_select(eroded_static.expand_as(hess_mag)).max().item())\n        return 0.0\n    return float(hess_mag.max().item())\n\n\n# ==============================================================================\n# TIỆN ÍCH PRECOMPUTE & TÁI TẠO NHANH TỪ TPS CACHE ĐA LỚP (.NPZ) CHO BƯỚC 4\n# ==============================================================================\ndef solve_multilayer_cache_entry(\n    kps0,\n    kps1,\n    img_hw,\n    max_pts=128,\n    max_clusters=4,\n    max_mask_pts=64,\n    max_static_pts=96,\n    lambd=0.01,\n    eps_static_px=1.5,\n    use_dyn_anchors=True,\n    filter_outliers=True,\n):\n    """\n    Tiền tính toán toàn bộ tham số Multi-Layer Clustered TPS (Backward-Pair) cho 1 clip\n    để lưu gọn vào `tps_cache_multilayer.npz` (không cần giải np.linalg.solve lúc train).\n    """\n    c0_dyn = np.zeros((max_clusters, max_pts, 2), dtype=np.float32)\n    th01_dyn = np.zeros((max_clusters, max_pts + 3, 2), dtype=np.float32)\n    c1_dyn = np.zeros((max_clusters, max_pts, 2), dtype=np.float32)\n    th10_dyn = np.zeros((max_clusters, max_pts + 3, 2), dtype=np.float32)\n    valid_dyn = np.zeros((max_clusters,), dtype=np.float32)\n\n    dyn_pts0 = np.zeros((max_clusters, max_mask_pts, 2), dtype=np.float32)\n    dyn_pts1 = np.zeros((max_clusters, max_mask_pts, 2), dtype=np.float32)\n    dyn_pts_len = np.zeros((max_clusters,), dtype=np.int32)\n\n    static_pts0 = np.zeros((max_static_pts, 2), dtype=np.float32)\n    static_pts1 = np.zeros((max_static_pts, 2), dtype=np.float32)\n\n    seg = segment_motion_keypoints(\n        kps0, kps1, img_hw=img_hw, eps_static_px=eps_static_px, filter_outliers=filter_outliers\n    )\n\n    n_stat_raw = seg[\'num_static\']\n    if n_stat_raw > 0:\n        if n_stat_raw > max_static_pts:\n            s0_sub, s1_sub = _stratified_spatial_subsample(seg[\'static_kps0\'], seg[\'static_kps1\'], max_static_pts)\n        else:\n            s0_sub, s1_sub = seg[\'static_kps0\'], seg[\'static_kps1\']\n        n_stat = len(s0_sub)\n        static_pts0[:n_stat] = s0_sub.astype(np.float32)\n        static_pts1[:n_stat] = s1_sub.astype(np.float32)\n    else:\n        n_stat = 0\n\n    clusters = seg[\'dynamic_clusters\']\n    if len(clusters) > max_clusters:\n        # Giữ top-(max_clusters-1) cụm lớn nhất và gộp phần còn lại vào cụm cuối\n        clusters_sorted = sorted(clusters, key=lambda c: len(c[\'kps0\']), reverse=True)\n        head_c = clusters_sorted[:max_clusters - 1]\n        tail_k0 = np.vstack([c[\'kps0\'] for c in clusters_sorted[max_clusters - 1:]])\n        tail_k1 = np.vstack([c[\'kps1\'] for c in clusters_sorted[max_clusters - 1:]])\n        clusters = head_c + [{\'kps0\': tail_k0, \'kps1\': tail_k1}]\n\n    n_clusters = len(clusters)\n    for k, dc in enumerate(clusters):\n        p0_k, p1_k = dc[\'kps0\'], dc[\'kps1\']\n        c0_k, th01_k, v01 = solve_tps_dynamic_layer(p1_k, p0_k, max_pts=max_pts, lambd=lambd, use_anchors=use_dyn_anchors)\n        c1_k, th10_k, v10 = solve_tps_dynamic_layer(p0_k, p1_k, max_pts=max_pts, lambd=lambd, use_anchors=use_dyn_anchors)\n        c0_dyn[k], th01_dyn[k] = c0_k, th01_k\n        c1_dyn[k], th10_dyn[k] = c1_k, th10_k\n        valid_dyn[k] = 1.0 if (v01 > 0.5 and v10 > 0.5) else 0.0\n\n        if len(p0_k) > max_mask_pts:\n            m0_sub, m1_sub = _stratified_spatial_subsample(p0_k, p1_k, max_mask_pts)\n        else:\n            m0_sub, m1_sub = p0_k, p1_k\n        n_mk = len(m0_sub)\n        dyn_pts0[k, :n_mk] = m0_sub.astype(np.float32)\n        dyn_pts1[k, :n_mk] = m1_sub.astype(np.float32)\n        dyn_pts_len[k] = n_mk\n\n    return {\n        \'c0_dyn\': c0_dyn,\n        \'th01_dyn\': th01_dyn,\n        \'c1_dyn\': c1_dyn,\n        \'th10_dyn\': th10_dyn,\n        \'valid_dyn\': valid_dyn,\n        \'dyn_pts0\': dyn_pts0,\n        \'dyn_pts1\': dyn_pts1,\n        \'dyn_pts_len\': dyn_pts_len,\n        \'static_pts0\': static_pts0,\n        \'static_pts1\': static_pts1,\n        \'num_static\': int(n_stat),\n        \'num_clusters\': int(n_clusters),\n        \'num_outliers\': int(seg[\'num_outliers\']),\n    }\n\n\n@torch.no_grad()\ndef reconstruct_multilayer_flow_batch_from_cache(\n    c0_dyn_b,\n    th01_dyn_b,\n    c1_dyn_b,\n    th10_dyn_b,\n    valid_dyn_b,\n    dyn_pts0_b,\n    dyn_pts1_b,\n    dyn_pts_len_b,\n    static_pts0_b,\n    static_pts1_b,\n    num_static_b,\n    num_clusters_b,\n    orig_hw_b,\n    h_pad,\n    w_pad,\n    device,\n    downscale_factor=2,\n    use_dyn_anchors=True,\n    transition_width_px=22.0,\n    falloff_radius_px=105.0,\n):\n    """\n    Tái tạo trực tiếp batch flow đa lớp [B, 2, h_pad, w_pad] trên GPU từ `tps_cache_multilayer.npz`\n    mà không cần giải lại bất kỳ hệ phương trình TPS nào.\n    """\n    bsz = c0_dyn_b.shape[0]\n    cf01_list = []\n    cf10_list = []\n\n    for i in range(bsz):\n        nc = int(num_clusters_b[i].item())\n        ns = int(num_static_b[i].item())\n        ih = int(orig_hw_b[i, 0].item())\n        iw = int(orig_hw_b[i, 1].item())\n\n        if nc <= 0:\n            cf01_list.append(torch.zeros((1, 2, h_pad, w_pad), dtype=torch.float32, device=device))\n            cf10_list.append(torch.zeros((1, 2, h_pad, w_pad), dtype=torch.float32, device=device))\n            continue\n\n        dyn_flows_0 = []\n        dyn_flows_1 = []\n        dyn_corridors = []\n        for k in range(nc):\n            vk = float(valid_dyn_b[i, k].item())\n            f0_k = _eval_tps_flow_single(th01_dyn_b[i, k], c0_dyn_b[i, k], vk, (ih, iw), h_pad, w_pad, device, downscale_factor)\n            f1_k = _eval_tps_flow_single(th10_dyn_b[i, k], c1_dyn_b[i, k], vk, (ih, iw), h_pad, w_pad, device, downscale_factor)\n            dyn_flows_0.append(f0_k)\n            dyn_flows_1.append(f1_k)\n\n            n_mk = int(dyn_pts_len_b[i, k].item())\n            if n_mk > 0:\n                p0_k = dyn_pts0_b[i, k, :n_mk]\n                p1_k = dyn_pts1_b[i, k, :n_mk]\n                pm_k = 0.5 * (p0_k + p1_k)\n                dyn_corridors.append(torch.cat([p0_k, pm_k, p1_k], dim=0))\n\n        if not dyn_corridors:\n            cf01_list.append(torch.zeros((1, 2, h_pad, w_pad), dtype=torch.float32, device=device))\n            cf10_list.append(torch.zeros((1, 2, h_pad, w_pad), dtype=torch.float32, device=device))\n            continue\n\n        if ns > 0:\n            sp0 = static_pts0_b[i, :ns]\n            sp1 = static_pts1_b[i, :ns]\n            static_ref = 0.5 * (sp0 + sp1)\n            eff_falloff = float(falloff_radius_px)\n        elif use_dyn_anchors:\n            static_ref = torch.as_tensor(FIXED_TPS_ANCHORS, dtype=torch.float32, device=device)\n            eff_falloff = float(falloff_radius_px)\n        else:\n            static_ref = None\n            eff_falloff = max(float(falloff_radius_px), 0.50 * math.hypot(float(iw), float(ih)))\n\n        weights_dyn, _, mask_total = compute_soft_voronoi_blending_mask(\n            dyn_corridors,\n            static_ref,\n            orig_hw=(ih, iw),\n            h_pad=h_pad,\n            w_pad=w_pad,\n            device=device,\n            downscale_factor=downscale_factor,\n            transition_width_px=transition_width_px,\n            falloff_radius_px=eff_falloff,\n        )\n\n        f0_fused = torch.zeros((1, 2, h_pad, w_pad), dtype=torch.float32, device=device)\n        f1_fused = torch.zeros((1, 2, h_pad, w_pad), dtype=torch.float32, device=device)\n        for w_k, f0_k, f1_k in zip(weights_dyn, dyn_flows_0, dyn_flows_1):\n            f0_fused = f0_fused + w_k * f0_k\n            f1_fused = f1_fused + w_k * f1_k\n\n        if ns > 0:\n            h_sub_g = max(16, h_pad // max(1, downscale_factor))\n            w_sub_g = max(16, w_pad // max(1, downscale_factor))\n            gy, gx = torch.meshgrid(\n                torch.linspace(0.0, 1.0, h_sub_g, device=device, dtype=torch.float32),\n                torch.linspace(0.0, 1.0, w_sub_g, device=device, dtype=torch.float32),\n                indexing=\'ij\'\n            )\n            base_g = torch.stack([gx, gy], dim=-1).unsqueeze(0)\n            sx = float(max(1, w_pad - 1)) / max(1.0, float(iw) - 1.0)\n            sy = float(max(1, h_pad - 1)) / max(1.0, float(ih) - 1.0)\n            scale_wh_px = torch.tensor([[[[float(iw - 1), float(ih - 1)]]]], dtype=torch.float32, device=device)\n            scale_pad_to_orig = torch.tensor([[[[sx, sy]]]], dtype=torch.float32, device=device)\n\n            f0_sub = F.interpolate(f0_fused, size=(h_sub_g, w_sub_g), mode=\'bilinear\', align_corners=True) if (h_sub_g != h_pad or w_sub_g != w_pad) else f0_fused\n            f1_sub = F.interpolate(f1_fused, size=(h_sub_g, w_sub_g), mode=\'bilinear\', align_corners=True) if (h_sub_g != h_pad or w_sub_g != w_pad) else f1_fused\n            m_sub = F.interpolate(mask_total, size=(h_sub_g, w_sub_g), mode=\'bilinear\', align_corners=True) if (h_sub_g != h_pad or w_sub_g != w_pad) else mask_total\n\n            q0_src_px = ((base_g + f0_sub.permute(0, 2, 3, 1)) * scale_pad_to_orig * scale_wh_px).view(1, -1, 2)\n            q1_src_px = ((base_g + f1_sub.permute(0, 2, 3, 1)) * scale_pad_to_orig * scale_wh_px).view(1, -1, 2)\n            sp0_px = (static_pts0_b[i, :ns] * scale_wh_px.view(1, 2)).unsqueeze(0)\n            sp1_px = (static_pts1_b[i, :ns] * scale_wh_px.view(1, 2)).unsqueeze(0)\n\n            pt_all_px = (torch.cat([\n                0.5 * (dyn_pts0_b[i, k, :int(dyn_pts_len_b[i, k].item())] + dyn_pts1_b[i, k, :int(dyn_pts_len_b[i, k].item())])\n                for k in range(nc) if int(dyn_pts_len_b[i, k].item()) > 0\n            ], dim=0) * scale_wh_px.view(1, 2)).unsqueeze(0)\n            q_cur_px = (base_g * scale_pad_to_orig * scale_wh_px).view(1, -1, 2)\n            d_cur_dyn = torch.cdist(q_cur_px, pt_all_px, p=2.0).min(dim=-1).values.view(1, 1, h_sub_g, w_sub_g)\n\n            d_src0_stat = torch.cdist(q0_src_px, sp0_px, p=2.0).min(dim=-1).values.view(1, 1, h_sub_g, w_sub_g)\n            d_src1_stat = torch.cdist(q1_src_px, sp1_px, p=2.0).min(dim=-1).values.view(1, 1, h_sub_g, w_sub_g)\n            outside_dyn_stroke = (m_sub > 1e-4) & (d_cur_dyn > 10.0)\n            pull0 = (outside_dyn_stroke & (d_src0_stat < 22.0)).float()\n            pull1 = (outside_dyn_stroke & (d_src1_stat < 22.0)).float()\n            if h_sub_g != h_pad or w_sub_g != w_pad:\n                pull0 = F.interpolate(pull0, size=(h_pad, w_pad), mode=\'nearest\')\n                pull1 = F.interpolate(pull1, size=(h_pad, w_pad), mode=\'nearest\')\n            f0_fused = torch.where(pull0 > 0.5, torch.zeros_like(f0_fused), f0_fused)\n            f1_fused = torch.where(pull1 > 0.5, torch.zeros_like(f1_fused), f1_fused)\n\n        cf01_list.append(f0_fused)\n        cf10_list.append(f1_fused)\n\n    return torch.cat(cf01_list, dim=0), torch.cat(cf10_list, dim=0)\n')

build_cache_path = os.path.join(WORKING_DIR, "build_tps_cache_multilayer.py")
with open(build_cache_path, "w", encoding="utf-8") as f:
    f.write('#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n"""\nbuild_tps_cache_multilayer.py\n--------------------------------------------------------------------------------\nPHA 3 - BƯỚC 4: TẠO BỘ CACHE `tps_cache_multilayer.npz` TRÊN SAKUGA GOLDEN DATASET\nVÀ KIỂM ĐỊNH ĐỐI CHỨNG TRÊN TẬP VAL (190 CLIPS / 420 FRAMES).\n\nTạo ra:\n  1. `tps_cache_multilayer.npz` (~12-15 MB nén, 100% tương thích DDP training).\n  2. `train_sequences_indexed.csv` & `val_sequences_indexed.csv` (có cột `cache_idx`).\n  3. Thống kê phân cụm toàn tập Train/Val & Đánh giá nhanh M0/Blended trên tập Val.\n--------------------------------------------------------------------------------\n"""\n\nimport os\nimport sys\nimport glob\nimport time\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nif sys.platform.startswith(\'win\'):\n    try:\n        sys.stdout.reconfigure(encoding=\'utf-8\')\n        sys.stderr.reconfigure(encoding=\'utf-8\')\n    except Exception:\n        pass\n\nSCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))\nREPO_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, ".."))\nDEFAULT_DATASET_DIR = os.path.join(\n    REPO_ROOT, "Dataset", "sakuga_pipeline_sequence", "final_dataset",\n    "sakuga_golden_dataset_sequences_tps_v2"\n)\n\nif SCRIPT_DIR not in sys.path:\n    sys.path.insert(0, SCRIPT_DIR)\n\nfrom multi_layer_tps import solve_multilayer_cache_entry\n\n\ndef build_multilayer_cache(\n    dataset_dir,\n    out_npz_path=None,\n    max_pts=128,\n    max_clusters=4,\n    max_mask_pts=64,\n    max_static_pts=96,\n    lambd=0.01,\n    eps_static_px=1.5,\n):\n    if out_npz_path is None:\n        out_npz_path = os.path.join(dataset_dir, "tps_cache_multilayer.npz")\n\n    train_csv = os.path.join(dataset_dir, "train_sequences.csv")\n    val_csv = os.path.join(dataset_dir, "val_sequences.csv")\n    train_clips_dir = os.path.join(dataset_dir, "train")\n    val_clips_dir = os.path.join(dataset_dir, "val")\n    matches_dir = os.path.join(dataset_dir, "matches")\n\n    df_tr = pd.read_csv(train_csv)\n    df_va = pd.read_csv(val_csv)\n    df_tr[\'_clip_root\'] = train_clips_dir\n    df_va[\'_clip_root\'] = val_clips_dir\n    df_all = pd.concat([df_tr, df_va], ignore_index=True)\n    n_total = len(df_all)\n    df_all[\'cache_idx\'] = np.arange(n_total, dtype=np.int32)\n\n    df_tr_out = df_all.iloc[:len(df_tr)].drop(columns=[\'_clip_root\']).copy()\n    df_va_out = df_all.iloc[len(df_tr):].drop(columns=[\'_clip_root\']).copy()\n    df_tr_out.to_csv(os.path.join(dataset_dir, "train_sequences_indexed.csv"), index=False)\n    df_va_out.to_csv(os.path.join(dataset_dir, "val_sequences_indexed.csv"), index=False)\n\n    c0_dyn = np.zeros((n_total, max_clusters, max_pts, 2), dtype=np.float32)\n    th01_dyn = np.zeros((n_total, max_clusters, max_pts + 3, 2), dtype=np.float32)\n    c1_dyn = np.zeros((n_total, max_clusters, max_pts, 2), dtype=np.float32)\n    th10_dyn = np.zeros((n_total, max_clusters, max_pts + 3, 2), dtype=np.float32)\n    valid_dyn = np.zeros((n_total, max_clusters), dtype=np.float32)\n\n    dyn_pts0 = np.zeros((n_total, max_clusters, max_mask_pts, 2), dtype=np.float32)\n    dyn_pts1 = np.zeros((n_total, max_clusters, max_mask_pts, 2), dtype=np.float32)\n    dyn_pts_len = np.zeros((n_total, max_clusters), dtype=np.int32)\n\n    static_pts0 = np.zeros((n_total, max_static_pts, 2), dtype=np.float32)\n    static_pts1 = np.zeros((n_total, max_static_pts, 2), dtype=np.float32)\n    num_static = np.zeros((n_total,), dtype=np.int32)\n    num_clusters = np.zeros((n_total,), dtype=np.int32)\n    num_outliers = np.zeros((n_total,), dtype=np.int32)\n    orig_hw = np.zeros((n_total, 2), dtype=np.int32)\n\n    def _task(item):\n        idx, row = item\n        cid = str(row[\'clip_id\'])\n        sl = int(row[\'seq_len\'])\n        c_root = row[\'_clip_root\']\n        img0_p = os.path.join(c_root, cid, "000001.png")\n        with Image.open(img0_p) as im:\n            w, h = im.size\n\n        m_p = os.path.join(matches_dir, cid, f"match_000001_{sl:06d}.npy")\n        if not os.path.exists(m_p):\n            cands = sorted(glob.glob(os.path.join(matches_dir, cid, "*.npy")))\n            m_p = cands[0] if cands else ""\n\n        if m_p and os.path.exists(m_p):\n            kps = np.load(m_p).astype(np.float32)\n            entry = solve_multilayer_cache_entry(\n                kps[0], kps[1], img_hw=(h, w),\n                max_pts=max_pts, max_clusters=max_clusters,\n                max_mask_pts=max_mask_pts, max_static_pts=max_static_pts,\n                lambd=lambd, eps_static_px=eps_static_px,\n                use_dyn_anchors=True, filter_outliers=True,\n            )\n        else:\n            entry = solve_multilayer_cache_entry(\n                np.zeros((0, 2), dtype=np.float32), np.zeros((0, 2), dtype=np.float32),\n                img_hw=(h, w), max_pts=max_pts, max_clusters=max_clusters,\n                max_mask_pts=max_mask_pts, max_static_pts=max_static_pts,\n            )\n        return idx, h, w, entry\n\n    t0 = time.time()\n    print(f"⚙️ Đang tạo `tps_cache_multilayer.npz` cho {n_total:,} sequences "\n          f"(MAX_PTS={max_pts}, MAX_CLUSTERS={max_clusters}, eps_static={eps_static_px}px)...", flush=True)\n\n    with ThreadPoolExecutor(max_workers=min(16, (os.cpu_count() or 4) * 2)) as pool:\n        for idx, h, w, entry in pool.map(_task, list(df_all.iterrows())):\n            c0_dyn[idx] = entry[\'c0_dyn\']\n            th01_dyn[idx] = entry[\'th01_dyn\']\n            c1_dyn[idx] = entry[\'c1_dyn\']\n            th10_dyn[idx] = entry[\'th10_dyn\']\n            valid_dyn[idx] = entry[\'valid_dyn\']\n            dyn_pts0[idx] = entry[\'dyn_pts0\']\n            dyn_pts1[idx] = entry[\'dyn_pts1\']\n            dyn_pts_len[idx] = entry[\'dyn_pts_len\']\n            static_pts0[idx] = entry[\'static_pts0\']\n            static_pts1[idx] = entry[\'static_pts1\']\n            num_static[idx] = entry[\'num_static\']\n            num_clusters[idx] = entry[\'num_clusters\']\n            num_outliers[idx] = entry[\'num_outliers\']\n            orig_hw[idx] = [h, w]\n\n    np.savez_compressed(\n        out_npz_path,\n        c0_dyn=c0_dyn,\n        th01_dyn=th01_dyn,\n        c1_dyn=c1_dyn,\n        th10_dyn=th10_dyn,\n        valid_dyn=valid_dyn,\n        dyn_pts0=dyn_pts0,\n        dyn_pts1=dyn_pts1,\n        dyn_pts_len=dyn_pts_len,\n        static_pts0=static_pts0,\n        static_pts1=static_pts1,\n        num_static=num_static,\n        num_clusters=num_clusters,\n        num_outliers=num_outliers,\n        orig_hw=orig_hw,\n    )\n    elapsed = time.time() - t0\n    size_mb = os.path.getsize(out_npz_path) / (1024 ** 2)\n    print(f"[✓] Đã lưu thành công: {out_npz_path} ({size_mb:.2f} MB) trong {elapsed:.1f}s!")\n    print(f"    - Số clips có >= 1 cụm động hợp lệ: {int((num_clusters >= 1).sum()):,}/{n_total:,}")\n    print(f"    - Số clips có >= 2 cụm động: {int((num_clusters >= 2).sum()):,}/{n_total:,}")\n    print(f"    - Số clips có điểm nền tĩnh (>0): {int((num_static > 0).sum()):,}/{n_total:,}")\n    print(f"    - Tổng số điểm khớp nhiễu (outliers) đã lọc: {int(num_outliers.sum()):,}")\n    return out_npz_path\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Tạo tps_cache_multilayer.npz cho Bước 4 Pha 3")\n    parser.add_argument("--dataset_dir", type=str, default=DEFAULT_DATASET_DIR)\n    parser.add_argument("--out_npz", type=str, default=None)\n    args = parser.parse_args()\n    build_multilayer_cache(args.dataset_dir, args.out_npz)\n\n\nif __name__ == "__main__":\n    main()\n')

worker_p2_path = os.path.join(WORKING_DIR, "ddp_worker_tps_sequence_phase2.py")
with open(worker_p2_path, "w", encoding="utf-8") as f:
    f.write('#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n"""\nddp_worker_tps_sequence_final.py\n--------------------------------------------------------------------------------\nĐộng cơ Huấn luyện & Kiểm định DDP 2x GPU Tesla T4 (Kaggle Dual-GPU) cho:\nFine-tune Mô hình TPS-Inbetween Gốc (model_latest.pt) trên Sakuga Sequence Final Dataset\nđể kiểm chứng khách quan (Zero-Bias Hypothesis Testing):\n  - M0: Coarse TPS Warp (Toán tử uốn lưới hình học thuần túy, không mạng nơ-ron)\n  - M1: Original TPS-Inbetween (model_latest.pt — Baseline gốc ML240)\n  - M3: Finetuned TPS-Inbetween (100% cùng kiến trúc TPSInbetweenOriginal với M1)\n\nCác chuẩn mực kỹ thuật cốt lõi được đảm bảo trong file này:\n1. 100% ZERO ARCHITECTURAL CHANGE:\n   - M1 và M3 dùng chung duy nhất class TPSInbetweenOriginal (10,068,668 active params).\n2. SỬA TRIỆT ĐỂ LỖI NGƯỢC DẤU BACKWARD-WARP TPS & DYNAMIC MULT32 PADDING:\n   - Khôi phục đúng quy ước tpsinbet.py: flow01 (kps1 -> kps0), flow10 (kps0 -> kps1).\n   - Xử lý chính xác mọi chiều rộng khung hình W in [720, 1480] khi đệm mult-32.\n3. HỖ TRỢ 3 TÙY CHỌN TỐC ĐỘ HUẤN LUYỆN (SHORT / MEDIUM / LONG):\n   - Mặc định theo duyệt Plan: TRAIN_MODE = "LONG" (Option 3 — Full-Trajectory\n     540p, 8-term Anti-Thickening Loss: 1.0*Charb + 0.35*Sym-DT(8) + 0.20*Sobel +\n     0.20*Soft-clDice + 0.15*VGG16 + 0.50*Exact-STE Density + 0.25*Stroke-Width +\n     0.20*Ghost-Cleanliness, EMA với adaptive warmup, Cosine LR Warmup).\n   - DDP Length & Width-Bucketed Batch Sampler giữ cả 2 GPU ở ~98-100% tải.\n   - Chunked Multi-frame Gradient Accumulation giữ VRAM an toàn (~5-6 GB / 15 GB T4).\n4. CHÍNH SÁCH CHECKPOINT TINH GỌN TUYỆT ĐỐI (CHỈ 2 FILE):\n   - model_tps_finetuned_best.pt : Checkpoint tốt nhất theo Two-Tier Gated Normalized Score.\n   - model_tps_finetuned_last.pt : Checkpoint epoch gần nhất + trạng thái Auto-Resume.\n   - Không lưu rời từng epoch! Tích hợp RAM Watchdog + glibc malloc_trim().\n--------------------------------------------------------------------------------\n"""\n\nimport os\nimport sys\nimport gc\nimport time\nimport math\nimport json\nimport glob\nimport random\nimport argparse\nimport contextlib\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import Dataset, DataLoader, Sampler\nfrom torch.utils.data.distributed import DistributedSampler\n\n# ==============================================================================\n# 0. TỐI ƯU HỆ THỐNG, KHÓA PHÂN MẢNH RAM C & CẤU HÌNH MÔI TRƯỜNG\n# ==============================================================================\nos.environ.setdefault("MALLOC_TRIM_THRESHOLD_", "65536")\nos.environ.setdefault("MALLOC_ARENA_MAX", "2")\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("MKL_NUM_THREADS", "1")\n\nif sys.platform.startswith(\'win\'):\n    try:\n        sys.stdout.reconfigure(encoding=\'utf-8\')\n        sys.stderr.reconfigure(encoding=\'utf-8\')\n    except Exception:\n        pass\n\ntry:\n    import cv2\n    cv2.setNumThreads(0)\n    cv2.ocl.setUseOpenCL(False)\nexcept ImportError:\n    cv2 = None\n\ntorch.backends.cudnn.benchmark = True\n\n\ndef trim_memory():\n    """Giải phóng bộ nhớ Python và ép glibc trả lại vùng nhớ trống cho OS (Linux/Kaggle)."""\n    gc.collect()\n    if sys.platform.startswith(\'linux\'):\n        try:\n            import ctypes\n            ctypes.CDLL(\'libc.so.6\').malloc_trim(0)\n        except Exception:\n            pass\n\n\n# ==============================================================================\n# 1. TOÁN TỬ BIẾN DẠNG LƯỚI & GIẢI TPS CHUẨN BACKWARD-WARP (KHỚP 100% TPSINBET.PY)\n# ==============================================================================\nFIXED_TPS_ANCHORS = np.array([\n    [0.0, 0.0], [0.5, 0.0], [1.0, 0.0],\n    [0.0, 0.5],             [1.0, 0.5],\n    [0.0, 1.0], [0.5, 1.0], [1.0, 1.0]\n], dtype=np.float32)\n\n_GRID_CACHE = {}\n_PIXEL_GRID_CACHE = {}\n\n\ndef warp_normalized_flow(img, flow):\n    """Backward-warp theo luồng Coarse TPS (tọa độ chuẩn hóa [0, 1] -> [-1, 1])."""\n    _, _, h, w = img.shape\n    key = (h, w, flow.device)\n    base_grid = _GRID_CACHE.get(key)\n    if base_grid is None:\n        gy, gx = torch.meshgrid(\n            torch.linspace(0.0, 1.0, h, device=flow.device, dtype=torch.float32),\n            torch.linspace(0.0, 1.0, w, device=flow.device, dtype=torch.float32),\n            indexing=\'ij\'\n        )\n        base_grid = torch.stack([gx, gy], dim=-1).unsqueeze(0).contiguous()\n        _GRID_CACHE[key] = base_grid\n\n    with torch.amp.autocast(\'cuda\', enabled=False):\n        grid = (base_grid + flow.float().permute(0, 2, 3, 1)) * 2.0 - 1.0\n        out = F.grid_sample(\n            img.float().contiguous(), grid,\n            mode=\'bilinear\', padding_mode=\'border\', align_corners=True\n        )\n    return out.to(img.dtype)\n\n\ndef warp_pixel_flow(img, flow):\n    """Backward-warp theo luồng quang học IFNet (đơn vị pixel)."""\n    _, _, h, w = img.shape\n    key = (h, w, flow.device)\n    base_grid = _PIXEL_GRID_CACHE.get(key)\n    if base_grid is None:\n        gy, gx = torch.meshgrid(\n            torch.linspace(-1.0, 1.0, h, device=flow.device, dtype=torch.float32),\n            torch.linspace(-1.0, 1.0, w, device=flow.device, dtype=torch.float32),\n            indexing=\'ij\'\n        )\n        base_grid = torch.stack([gx, gy], dim=-1).unsqueeze(0).contiguous()\n        _PIXEL_GRID_CACHE[key] = base_grid\n\n    with torch.amp.autocast(\'cuda\', enabled=False):\n        flow_f32 = flow.float()\n        flow_norm = torch.cat([\n            flow_f32[:, 0:1, :, :] / max(1.0, (w - 1.0) / 2.0),\n            flow_f32[:, 1:2, :, :] / max(1.0, (h - 1.0) / 2.0)\n        ], dim=1)\n        grid = (base_grid + flow_norm.permute(0, 2, 3, 1)).contiguous()\n        out = F.grid_sample(\n            img.float().contiguous(), grid,\n            mode=\'bilinear\', padding_mode=\'border\', align_corners=True\n        )\n    return out.to(img.dtype)\n\n\ndef tps_rbf_torch(r):\n    r = torch.clamp(r, min=1e-6)\n    return (r ** 2) * torch.log(r)\n\n\ndef _solve_tps_system(cs_all, cd_all, max_pts=128, lambd=0.01):\n    """Giải hệ phương trình tuyến tính TPS chung (dùng cho cả Backward-Pair và Lagrangian Direct)."""\n    m = len(cs_all)\n    diff = cs_all[:, None, :] - cs_all[None, :, :]\n    r = np.linalg.norm(diff, axis=-1)\n    K = (r ** 2) * np.log(np.maximum(r, 1e-6))\n    np.fill_diagonal(K, float(lambd))\n\n    P = np.hstack([np.ones((m, 1), dtype=np.float64), cs_all])\n    A = np.zeros((m + 3, m + 3), dtype=np.float64)\n    A[:m, :m] = K\n    A[:m, m:] = P\n    A[m:, :m] = P.T\n\n    Y = np.zeros((m + 3, 2), dtype=np.float64)\n    Y[:m, :] = cd_all\n\n    try:\n        theta = np.linalg.solve(A, Y)\n    except np.linalg.LinAlgError:\n        theta, _, _, _ = np.linalg.lstsq(A, Y, rcond=1e-4)\n\n    valid = 0.0 if (\n        np.isnan(theta).any() or np.isinf(theta).any() or np.max(np.abs(theta)) > 50.0\n    ) else 1.0\n\n    c_pad = np.zeros((max_pts, 2), dtype=np.float32)\n    th_pad = np.zeros((max_pts + 3, 2), dtype=np.float32)\n    if valid > 0.5:\n        c_pad[:m] = cs_all.astype(np.float32)\n        th_pad[:m] = theta[:m].astype(np.float32)\n        th_pad[max_pts:max_pts + 3] = theta[m:m + 3].astype(np.float32)\n    return c_pad, th_pad, valid\n\n\ndef solve_tps_direct(c_src, c_dst, max_pts=128, lambd=0.01, use_anchors=False):\n    """Giải TPS trực tiếp từ c_src -> c_dst (dùng cho Lagrangian Direct TPS tại mốc t)."""\n    if c_src is None or c_dst is None or len(c_src) < 4:\n        return (\n            np.zeros((max_pts, 2), dtype=np.float32),\n            np.zeros((max_pts + 3, 2), dtype=np.float32),\n            0.0,\n        )\n    n_keep = (max_pts - 8) if use_anchors else max_pts\n    if len(c_src) > n_keep:\n        idx = np.linspace(0, len(c_src) - 1, n_keep, dtype=int)\n        cs_sub = np.asarray(c_src[idx], dtype=np.float64)\n        cd_sub = np.asarray(c_dst[idx], dtype=np.float64)\n    else:\n        cs_sub = np.asarray(c_src, dtype=np.float64)\n        cd_sub = np.asarray(c_dst, dtype=np.float64)\n    if use_anchors:\n        anchors64 = FIXED_TPS_ANCHORS.astype(np.float64)\n        cs_all = np.vstack([cs_sub, anchors64])\n        cd_all = np.vstack([cd_sub, anchors64])\n    else:\n        cs_all, cd_all = cs_sub, cd_sub\n    return _solve_tps_system(cs_all, cd_all, max_pts=max_pts, lambd=lambd)\n\n\ndef solve_tps_backward_pair(kps0, kps1, max_pts=128, lambd=0.01):\n    """\n    Giải hệ phương trình TPS theo đúng chiều Backward-Warp của tpsinbet.py:\n      - flow01 được nội suy từ các điểm điều khiển tại kps1 nhắm tới kps0 (c0=kps1 -> kps0)\n      - flow10 được nội suy từ các điểm điều khiển tại kps0 nhắm tới kps1 (c1=kps0 -> kps1)\n    Tự động giảm mẫu đều xuống (max_pts - 8) điểm + 8 điểm neo biên cố định [0, 1]^2.\n    Trả về: c0, th01, c1, th10, valid_flag.\n    """\n    c0, th01, v01 = solve_tps_direct(kps1, kps0, max_pts=max_pts, lambd=lambd, use_anchors=True)\n    c1, th10, v10 = solve_tps_direct(kps0, kps1, max_pts=max_pts, lambd=lambd, use_anchors=True)\n    v = 1.0 if (v01 > 0.5 and v10 > 0.5) else 0.0\n    return c0, th01, c1, th10, v\n\n\ndef _get_norm_grid_pts(h, w, device):\n    key = (h, w, device, \'tps_pts_2d\')\n    pts = _GRID_CACHE.get(key)\n    if pts is None:\n        gy, gx = torch.meshgrid(\n            torch.linspace(0.0, 1.0, h, device=device, dtype=torch.float32),\n            torch.linspace(0.0, 1.0, w, device=device, dtype=torch.float32),\n            indexing=\'ij\'\n        )\n        pts = torch.stack([gx, gy], dim=-1).view(-1, 2).contiguous()\n        _GRID_CACHE[key] = pts\n    return pts\n\n\n@torch.no_grad()\ndef tps_flow_from_theta_batch(theta_batch, c_src_batch, valid_mask, orig_hw, h_pad, w_pad, device, downscale_factor=2):\n    """\n    Nội suy chính xác trường dòng chảy Coarse TPS trên GPU từ nghiệm đã tiền tính toán,\n    tự động co giãn chính xác theo tỷ lệ khung ảnh gốc (orig_h, orig_w) bên trong canvas đệm (h_pad, w_pad).\n    Dùng chunking (chunk_size=32768 điểm) để giữ mức đỉnh VRAM thấp (<35 MB).\n    """\n    with torch.amp.autocast(\'cuda\', enabled=False):\n        theta_f32 = theta_batch.float()\n        c_src_f32 = c_src_batch.float()\n        valid_f32 = valid_mask.float()\n        b = theta_f32.shape[0]\n\n        h_sub = max(16, h_pad // max(1, downscale_factor))\n        w_sub = max(16, w_pad // max(1, downscale_factor))\n\n        pts_pad = _get_norm_grid_pts(h_sub, w_sub, device)  # [P, 2]\n        total_pts = pts_pad.shape[0]\n        pts_pad_exp = pts_pad.unsqueeze(0).expand(b, -1, -1)  # [B, P, 2]\n\n        # Tỷ lệ quy đổi giữa tọa độ chuẩn hóa của canvas (h_pad, w_pad) và ảnh gốc (orig_h, orig_w)\n        orig_hw_f = orig_hw.to(device=device, dtype=torch.float32)  # [B, 2] -> (H_orig, W_orig)\n        sx = (float(max(1, w_pad - 1)) / torch.clamp(orig_hw_f[:, 1] - 1.0, min=1.0)).view(b, 1, 1)\n        sy = (float(max(1, h_pad - 1)) / torch.clamp(orig_hw_f[:, 0] - 1.0, min=1.0)).view(b, 1, 1)\n        scale_xy = torch.cat([sx, sy], dim=-1)  # [B, 1, 2]\n\n        # Quy về tọa độ gốc [0, 1] và giữ dấu vùng biên hợp lệ\n        pts_orig_raw = pts_pad_exp * scale_xy\n        in_bounds = ((pts_orig_raw[..., 0:1] <= 1.001) & (pts_orig_raw[..., 1:2] <= 1.001)).float()\n        pts_orig = torch.clamp(pts_orig_raw, 0.0, 1.0)\n\n        flow_chunks = []\n        chunk_size = 32768\n        for i in range(0, total_pts, chunk_size):\n            end_i = min(i + chunk_size, total_pts)\n            q = pts_orig[:, i:end_i, :]  # [B, C, 2]\n            dist = torch.cdist(q, c_src_f32, p=2.0)  # [B, C, M]\n            U = tps_rbf_torch(dist)\n            ones = torch.ones((b, q.shape[1], 1), device=device, dtype=torch.float32)\n            basis = torch.cat([U, ones, q], dim=-1)  # [B, C, M+3]\n            mapped = torch.bmm(basis, theta_f32)  # [B, C, 2]\n            disp_orig = mapped - q\n            disp_pad = (disp_orig / scale_xy) * in_bounds[:, i:end_i, :]\n            flow_chunks.append(disp_pad)\n\n        flow = torch.cat(flow_chunks, dim=1).view(b, h_sub, w_sub, 2).permute(0, 3, 1, 2).contiguous()\n        if h_sub != h_pad or w_sub != w_pad:\n            flow = F.interpolate(flow, size=(h_pad, w_pad), mode=\'bilinear\', align_corners=True)\n\n        flow.mul_(valid_f32.view(b, 1, 1, 1))\n        return flow\n\n\n# ==============================================================================\n# 2. KIẾN TRÚC MÔ HÌNH NGUYÊN BẢN 100% (TPSInbetweenOriginal: ZHU ET AL., 2024)\n# ==============================================================================\ndef conv(in_p, out_p, kernel_size=3, stride=1, padding=1, dilation=1):\n    return nn.Sequential(\n        nn.Conv2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, bias=True),\n        nn.PReLU(out_p)\n    )\n\n\ndef deconv(in_p, out_p, kernel_size=4, stride=2, padding=1):\n    return nn.Sequential(\n        nn.ConvTranspose2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, bias=True),\n        nn.PReLU(out_p)\n    )\n\n\nclass Conv2(nn.Module):\n    def __init__(self, in_p, out_p, stride=2):\n        super(Conv2, self).__init__()\n        self.conv1 = conv(in_p, out_p, 3, stride, 1)\n        self.conv2 = conv(out_p, out_p, 3, 1, 1)\n\n    def forward(self, x):\n        return self.conv2(self.conv1(x))\n\n\nclass Contextnet(nn.Module):\n    def __init__(self, indim=1, c=16):\n        super(Contextnet, self).__init__()\n        self.conv1 = Conv2(indim, c)\n        self.conv2 = Conv2(c, 2 * c)\n        self.conv3 = Conv2(2 * c, 4 * c)\n        self.conv4 = Conv2(4 * c, 8 * c)\n\n    def forward(self, x, flow):\n        x = self.conv1(x)\n        flow = F.interpolate(flow, size=(x.shape[2], x.shape[3]), mode=\'bilinear\', align_corners=False) * 0.5\n        f1 = warp_normalized_flow(x, flow)\n        x = self.conv2(x)\n        flow = F.interpolate(flow, size=(x.shape[2], x.shape[3]), mode=\'bilinear\', align_corners=False) * 0.5\n        f2 = warp_normalized_flow(x, flow)\n        x = self.conv3(x)\n        flow = F.interpolate(flow, size=(x.shape[2], x.shape[3]), mode=\'bilinear\', align_corners=False) * 0.5\n        f3 = warp_normalized_flow(x, flow)\n        x = self.conv4(x)\n        flow = F.interpolate(flow, size=(x.shape[2], x.shape[3]), mode=\'bilinear\', align_corners=False) * 0.5\n        f4 = warp_normalized_flow(x, flow)\n        return [f1, f2, f3, f4]\n\n\nclass IFBlock(nn.Module):\n    def __init__(self, in_planes, c=64):\n        super(IFBlock, self).__init__()\n        self.conv0 = nn.Sequential(\n            conv(in_planes, c // 2, 3, 2, 1),\n            conv(c // 2, c, 3, 2, 1),\n        )\n        self.convblock = nn.Sequential(*[conv(c, c) for _ in range(8)])\n        self.lastconv = nn.ConvTranspose2d(c, 5, 4, 2, 1)\n\n    def forward(self, x, flow, scale):\n        orig_h, orig_w = x.shape[2], x.shape[3]\n        if scale != 1:\n            x = F.interpolate(x, scale_factor=1.0 / scale, mode=\'bilinear\', align_corners=False)\n        if flow is not None:\n            flow = F.interpolate(flow, size=(x.shape[2], x.shape[3]), mode=\'bilinear\', align_corners=False) * (1.0 / scale)\n            x = torch.cat((x, flow), 1)\n        x = self.conv0(x.contiguous())\n        x = self.convblock(x) + x\n        with torch.amp.autocast(\'cuda\', enabled=False):\n            tmp = self.lastconv(x.float().contiguous())\n            tmp = F.interpolate(tmp, size=(orig_h, orig_w), mode=\'bilinear\', align_corners=False)\n            flow = tmp[:, :4] * scale * 2\n            mask = tmp[:, 4:5]\n        return flow, mask\n\n\nclass IFNet(nn.Module):\n    def __init__(self):\n        super(IFNet, self).__init__()\n        self.block0 = IFBlock(6, c=240)\n        self.block1 = IFBlock(13 + 4, c=150)\n        self.block2 = IFBlock(13 + 4, c=90)\n\n    def forward(self, I0t, I1t, scale=(4, 2, 1)):\n        img0 = I0t.repeat(1, 3, 1, 1)\n        img1 = I1t.repeat(1, 3, 1, 1)\n        warped_i0 = I0t\n        warped_i1 = I1t\n        flow = None\n        mask = None\n        stu = [self.block0, self.block1, self.block2]\n        for i in range(3):\n            if flow is not None:\n                if flow.shape[2:] != I0t.shape[2:]:\n                    flow = F.interpolate(flow, size=I0t.shape[2:], mode=\'bilinear\', align_corners=False)\n                if mask.shape[2:] != I0t.shape[2:]:\n                    mask = F.interpolate(mask, size=I0t.shape[2:], mode=\'bilinear\', align_corners=False)\n                inp = torch.cat((img0, img1, warped_i0.repeat(1, 3, 1, 1), warped_i1.repeat(1, 3, 1, 1), mask), 1)\n                flow_d, mask_d = stu[i](inp, flow, scale=scale[i])\n                flow = flow + flow_d\n                mask = mask + mask_d\n            else:\n                flow, mask = stu[i](torch.cat((img0, img1), 1), None, scale=scale[i])\n            if i < 2:\n                warped_i0 = warp_pixel_flow(I0t, flow[:, :2])\n                warped_i1 = warp_pixel_flow(I1t, flow[:, 2:4])\n        return flow, torch.sigmoid(mask)\n\n\nclass OriginalUnet(nn.Module):\n    """UNet gốc của bài báo (indim=9, outdim=1, đầu ra residual [-1, 1])."""\n    def __init__(self, indim=9, outdim=1, c=16):\n        super(OriginalUnet, self).__init__()\n        self.down0 = Conv2(indim, 2 * c)\n        self.down1 = Conv2(4 * c, 4 * c)\n        self.down2 = Conv2(8 * c, 8 * c)\n        self.down3 = Conv2(16 * c, 16 * c)\n        self.up0 = deconv(32 * c, 8 * c)\n        self.up1 = deconv(16 * c, 4 * c)\n        self.up2 = deconv(8 * c, 2 * c)\n        self.up3 = deconv(4 * c, c)\n        self.conv = nn.Conv2d(c, outdim, 3, 1, 1)\n\n    def forward(self, img0, img1, flow0, flow1, warped0, warped1, mask, c0, c1):\n        s0 = self.down0(torch.cat((img0, img1, warped0, warped1, flow0, flow1, mask), 1))\n        if s0.shape[2:] != c0[0].shape[2:]:\n            s0 = F.interpolate(s0, size=c0[0].shape[2:], mode=\'bilinear\', align_corners=False)\n        s1 = self.down1(torch.cat((s0, c0[0], c1[0]), 1))\n        if s1.shape[2:] != c0[1].shape[2:]:\n            s1 = F.interpolate(s1, size=c0[1].shape[2:], mode=\'bilinear\', align_corners=False)\n        s2 = self.down2(torch.cat((s1, c0[1], c1[1]), 1))\n        if s2.shape[2:] != c0[2].shape[2:]:\n            s2 = F.interpolate(s2, size=c0[2].shape[2:], mode=\'bilinear\', align_corners=False)\n        s3 = self.down3(torch.cat((s2, c0[2], c1[2]), 1))\n        if s3.shape[2:] != c0[3].shape[2:]:\n            s3 = F.interpolate(s3, size=c0[3].shape[2:], mode=\'bilinear\', align_corners=False)\n        x = self.up0(torch.cat((s3, c0[3], c1[3]), 1))\n        if x.shape[2:] != s2.shape[2:]:\n            x = F.interpolate(x, size=s2.shape[2:], mode=\'bilinear\', align_corners=False)\n        x = self.up1(torch.cat((x, s2), 1))\n        if x.shape[2:] != s1.shape[2:]:\n            x = F.interpolate(x, size=s1.shape[2:], mode=\'bilinear\', align_corners=False)\n        x = self.up2(torch.cat((x, s1), 1))\n        if x.shape[2:] != s0.shape[2:]:\n            x = F.interpolate(x, size=s0.shape[2:], mode=\'bilinear\', align_corners=False)\n        x = self.up3(torch.cat((x, s0), 1))\n        if x.shape[2:] != img0.shape[2:]:\n            x = F.interpolate(x, size=img0.shape[2:], mode=\'bilinear\', align_corners=False)\n        x = self.conv(x)\n        return torch.sigmoid(x)\n\n\nclass TPSInbetweenOriginal(nn.Module):\n    """\n    Kiến trúc nguyên bản TPS-Inbetween (Zhu et al., HIT 2024).\n    Dùng chung DUY NHẤT 1 class này cho M1 (model_latest.pt) và M3 (Finetuned):\n    13,026,543 active parameters — CAM KẾT 0% THAY ĐỔI KIẾN TRÚC.\n    """\n    def __init__(self):\n        super(TPSInbetweenOriginal, self).__init__()\n        self.encoder = Contextnet(indim=1, c=16)\n        self.IFNet = IFNet()\n        self.unet = OriginalUnet(indim=9, outdim=1, c=16)\n\n    def forward(self, x0, x1, t_val, coarse_flow01, coarse_flow10, direct_t_flows=False):\n        t = t_val\n        if direct_t_flows:\n            coarse_flow0t = coarse_flow01\n            coarse_flow1t = coarse_flow10\n        else:\n            coarse_flow0t = coarse_flow01 * (t ** 2) - (1.0 - t) * t * coarse_flow10\n            coarse_flow1t = coarse_flow10 * ((1.0 - t) ** 2) - t * (1.0 - t) * coarse_flow01\n\n        coarse_I0t = warp_normalized_flow(x0, coarse_flow0t)\n        coarse_I1t = warp_normalized_flow(x1, coarse_flow1t)\n        coarse_warp = torch.clamp((1.0 - t) * coarse_I0t + t * coarse_I1t, 0.0, 1.0)\n\n        feats0 = self.encoder(x0, coarse_flow0t)\n        feats1 = self.encoder(x1, coarse_flow1t)\n        flow_bet_coar, m = self.IFNet(coarse_I0t, coarse_I1t)\n\n        fine_I0t = warp_pixel_flow(coarse_I0t, flow_bet_coar[:, :2])\n        fine_I1t = warp_pixel_flow(coarse_I1t, flow_bet_coar[:, 2:4])\n\n        refine_out = self.unet(\n            x0, x1,\n            coarse_flow0t + flow_bet_coar[:, :2],\n            coarse_flow1t + flow_bet_coar[:, 2:4],\n            fine_I0t, fine_I1t, m, feats0, feats1\n        )\n        res = refine_out[:, 0:1, :, :] * 2.0 - 1.0\n        It = torch.clamp((1.0 - t) * fine_I0t + t * fine_I1t + res, 0.0, 1.0)\n        return It, coarse_warp, fine_I0t, fine_I1t, flow_bet_coar, m\n\n\ndef load_tps_weights(model, ckpt_path, device=\'cpu\', prefer_ema=False, is_rank0=True):\n    """\n    Nạp trọng số từ model_latest.pt gốc (252 keys gồm 94 dead keys) hoặc checkpoint finetune.\n    """\n    if not os.path.exists(ckpt_path):\n        raise FileNotFoundError(f"Không tìm thấy file checkpoint tại: {ckpt_path}")\n\n    raw = torch.load(ckpt_path, map_location=device, weights_only=False)\n    if isinstance(raw, dict):\n        if prefer_ema and raw.get(\'ema_state\') is not None:\n            ema_obj = raw[\'ema_state\']\n            sd = ema_obj[\'shadow\'] if isinstance(ema_obj, dict) and \'shadow\' in ema_obj else ema_obj\n        elif \'model_state\' in raw:\n            sd = raw[\'model_state\']\n        elif \'state_dict\' in raw:\n            sd = raw[\'state_dict\']\n        elif \'model\' in raw:\n            sd = raw[\'model\']\n        else:\n            sd = raw\n    else:\n        sd = raw\n\n    model_sd = model.state_dict()\n    matched_sd = {}\n    skipped_keys = []\n    for k, v in sd.items():\n        clean_k = k.replace(\'module.\', \'\').replace(\'base_model.\', \'\')\n        if clean_k in model_sd and model_sd[clean_k].shape == v.shape:\n            matched_sd[clean_k] = v\n        else:\n            skipped_keys.append(clean_k)\n\n    missing, _ = model.load_state_dict(matched_sd, strict=False)\n    if len(missing) > 0:\n        raise RuntimeError(f"Thiếu {len(missing)} active tensors khi nạp {ckpt_path}: {missing[:5]}")\n\n    if is_rank0:\n        tag = os.path.basename(ckpt_path)\n        print(f"[✓] Nạp trọng số ({tag}): Khớp {len(matched_sd)}/{len(model_sd)} active tensors 100%!"\n              + (f" (Bỏ qua {len(skipped_keys)} unused/dead keys)" if skipped_keys else ""), flush=True)\n    return model\n\n\n# ==============================================================================\n# 3. HÀM MẤT MÁT CHUYÊN DỤNG CHO LINEART (8-TERM ANTI-THICKENING LOSS)\n# ==============================================================================\ndef soft_erode(img):\n    p1 = -F.max_pool2d(-img, (3, 1), (1, 1), (1, 0))\n    p2 = -F.max_pool2d(-img, (1, 3), (1, 1), (0, 1))\n    return torch.min(p1, p2)\n\n\ndef soft_dilate(img):\n    return F.max_pool2d(img, (3, 3), (1, 1), (1, 1))\n\n\ndef soft_open(img):\n    return soft_dilate(soft_erode(img))\n\n\ndef soft_skeletonize(img, iters=2):\n    img = torch.clamp(img, 0.0, 1.0)\n    skel = F.relu(img - soft_open(img))\n    for _ in range(iters - 1):\n        img = soft_erode(img)\n        skel = skel + F.relu(img - soft_open(img))\n    return torch.clamp(skel, 0.0, 1.0)\n\n\ndef approx_gpu_distance_map(bin_mask, steps=8):\n    """\n    Bản đồ khoảng cách Euclidean xấp xỉ siêu nhanh bằng phép giãn 3x3 lặp trên GPU [0, 1].\n    Giá trị 0.0 tại chính pixel nét và tăng tuyến tính tới 1.0 ngoài bán kính `steps` pixel.\n    """\n    curr = bin_mask\n    d_sum = curr\n    for _ in range(steps):\n        curr = F.max_pool2d(curr, kernel_size=3, stride=1, padding=1)\n        d_sum = d_sum + curr\n    dt = 1.0 - (d_sum / float(steps + 1))\n    return torch.clamp(dt, 0.0, 1.0)\n\n\nclass ExactSTEDensityLoss(nn.Module):\n    """\n    Hàm phạt độ dày nét vẽ (Density Loss) với Straight-Through Estimator (STE)\n    và Leaky Surrogate Ramp tại ngưỡng nhị phân tau=0.15.\n    Dead-zone [0.96, 1.04], phạt bất đối xứng cực mạnh khi dày nét (dev_high).\n    """\n    def __init__(self, tau=0.15, dead_zone_low=0.96, dead_zone_high=1.04):\n        super().__init__()\n        self.tau = tau\n        self.low = dead_zone_low\n        self.high = dead_zone_high\n\n    def forward(self, pred, gt):\n        s_p = torch.clamp((pred - 0.06) / 0.18, 0.0, 1.0) + 0.25 * F.relu(pred - 0.24)\n        h_p = (pred >= self.tau).float()\n        m_p = h_p - s_p.detach() + s_p  # STE\n\n        h_g = (gt >= self.tau).float()\n        area_p = torch.sum(m_p, dim=(-2, -1))\n        area_g = torch.sum(h_g, dim=(-2, -1)).detach()\n\n        r = area_p / torch.clamp(area_g, min=1.0)\n\n        dev_low = F.relu(self.low - r)\n        dev_high = F.relu(r - self.high)\n        dev_center = r - 1.0\n\n        loss = 1.8 * dev_low + 2.8 * dev_high + 1.5 * (dev_high ** 2) + 0.06 * (dev_center ** 2)\n        return loss.mean()\n\n\nclass BidirectionalStrokeWidthLoss(nn.Module):\n    """\n    Hàm kiểm soát bề rộng nét vẽ hai chiều:\n      - Nhánh 1: Bảo vệ nét mảnh & ngọn tóc bên trong lõi nét GT (anti-erosion).\n      - Nhánh 2: Phạt overspill tràn mực trên vành 1-2px bên ngoài nét GT (anti-dilation).\n    """\n    def __init__(self):\n        super().__init__()\n\n    def forward(self, pred, gt):\n        with torch.no_grad():\n            gt_core = soft_skeletonize(gt, iters=2)\n            gt_stroke = (gt > 0.08).float()\n            gt_dilated = soft_dilate(soft_dilate(gt_stroke))\n            gt_inner = (gt > 0.12).float()\n            edge_ring = torch.clamp(gt_dilated - gt_inner, 0.0, 1.0)\n\n        # Nhánh 1: Bảo vệ lõi skeleton không bị mờ/đứt nét\n        loss_core = torch.sum(gt_core * F.relu(gt_core - pred), dim=(-3, -2, -1)) / torch.clamp(torch.sum(gt_core, dim=(-3, -2, -1)), min=1.0)\n\n        # Nhánh 2: Trừng phạt tràn mực (overspill) trên vành edge_ring quanh nét\n        over_spill = F.relu(pred - torch.maximum(gt, torch.tensor(0.06, device=pred.device))) + 2.0 * F.relu(pred - 0.14)\n        loss_ring = torch.sum(edge_ring * over_spill, dim=(-3, -2, -1)) / torch.clamp(torch.sum(edge_ring, dim=(-3, -2, -1)), min=1.0)\n\n        return (loss_core + loss_ring).mean()\n\n\nclass GhostAndBackgroundCleanlinessLoss(nn.Module):\n    """\n    Hàm triệt tiêu sương mù (haze), bóng ma xa (far ghost) và quầng mờ sát biên (halo) - Pha 2 Cleanliness Purge:\n      - Zone 1: Nền xa (cách nét GT > 5px): chuẩn hóa theo diện tích nét GT, bổ sung thành phần phạt bậc hai (quad-penalty) cực mạnh khi pred >= 0.035 để dọn sạch lượng sương mờ tồn đọng từ Pha 1.\n      - Zone 2: Vùng hào quang sát nét: phạt mực lơ lửng >= 0.06.\n    """\n    def __init__(self):\n        super().__init__()\n\n    def forward(self, pred, gt):\n        with torch.no_grad():\n            gt_stroke_wide = soft_dilate(soft_dilate((gt > 0.04).float()))\n            far_bg = torch.clamp(1.0 - gt_stroke_wide, 0.0, 1.0)\n            near_halo = torch.clamp(gt_stroke_wide - (gt > 0.15).float(), 0.0, 1.0)\n            gt_area = torch.clamp(torch.sum((gt > 0.15).float(), dim=(-3, -2, -1)), min=500.0)\n\n        near_count = torch.clamp(torch.sum(near_halo, dim=(-3, -2, -1)), min=200.0)\n\n        # Phạt mạnh lũy tiến cho các giá trị pred lơ lửng trên nền xa:\n        # Số hạng bậc hai (pen_far_p035 ** 2) tạo gradient rất mạnh đối với các vết mực mờ >= 0.035\n        pen_far_p035 = F.relu(pred - 0.035)\n        pen_far = (\n            1.5 * F.relu(pred - 0.012)\n            + 4.0 * pen_far_p035\n            + 6.0 * (pen_far_p035 ** 2)\n            + 8.0 * F.relu(pred - 0.20)\n        )\n        loss_far = torch.sum(far_bg * pen_far, dim=(-3, -2, -1)) / gt_area\n\n        pen_halo_p06 = F.relu(pred - 0.06)\n        pen_halo = 0.8 * pen_halo_p06 + 1.2 * (pen_halo_p06 ** 2)\n        loss_halo = torch.sum(near_halo * pen_halo, dim=(-3, -2, -1)) / near_count\n\n        return (loss_far + loss_halo).mean()\n\n\nclass SoftClDiceLoss(nn.Module):\n    def __init__(self, iters=2, smooth=1.0):\n        super().__init__()\n        self.iters = iters\n        self.smooth = smooth\n\n    def forward(self, pred, target, t_skel=None):\n        p_skel = soft_skeletonize(pred, iters=self.iters)\n        if t_skel is None:\n            t_skel = soft_skeletonize(target, iters=self.iters)\n        tprec = (torch.sum(p_skel * target) + self.smooth) / (torch.sum(p_skel) + self.smooth)\n        tsens = (torch.sum(t_skel * pred) + self.smooth) / (torch.sum(t_skel) + self.smooth)\n        cldice = 2.0 * (tprec * tsens) / (tprec + tsens + 1e-6)\n        return 1.0 - cldice\n\n\nclass VGGPerceptualOrLaplacianLoss(nn.Module):\n    """\n    Perceptual Loss dựa trên VGG16 đông băng (khớp tinh thần bài báo gốc TPS-Inbetween).\n    Tự động dự phòng sang Multi-Scale Laplacian Pyramid Loss nếu chạy ở môi trường offline\n    chưa cache sẵn weights VGG16 (không bao giờ gây crash).\n    """\n    def __init__(self, is_rank0=True):\n        super().__init__()\n        self.use_vgg = False\n        self.blocks = None\n        try:\n            import torchvision.models as tv_models\n            weights_enum = getattr(tv_models, \'VGG16_Weights\', None)\n            if weights_enum is not None:\n                vgg = tv_models.vgg16(weights=weights_enum.DEFAULT).features\n            else:\n                vgg = tv_models.vgg16(pretrained=True).features\n            vgg.eval()\n            for p in vgg.parameters():\n                p.requires_grad = False\n            # Lấy 4 tầng relu1_2(4), relu2_2(9), relu3_3(16), relu4_3(23)\n            self.slice1 = nn.Sequential(*list(vgg[:4]))\n            self.slice2 = nn.Sequential(*list(vgg[4:9]))\n            self.slice3 = nn.Sequential(*list(vgg[9:16]))\n            self.slice4 = nn.Sequential(*list(vgg[16:23]))\n            self.use_vgg = True\n            self.register_buffer(\'mean\', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))\n            self.register_buffer(\'std\', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))\n            if is_rank0:\n                print("[✓] VGG16 Perceptual Loss đã kích hoạt (Option 3 LONG Mode).", flush=True)\n        except Exception as e:\n            if is_rank0:\n                print(f"[*] VGG16 chưa tải được ({e}) -> Fallback sang 4-Scale Laplacian Perceptual Loss.", flush=True)\n\n    def _lap_pyr_loss(self, pred, gt):\n        loss = 0.0\n        cur_p, cur_g = pred, gt\n        for w in [0.4, 0.3, 0.2, 0.1]:\n            blur_p = F.avg_pool2d(F.pad(cur_p, (1, 1, 1, 1), mode=\'replicate\'), 3, stride=1)\n            blur_g = F.avg_pool2d(F.pad(cur_g, (1, 1, 1, 1), mode=\'replicate\'), 3, stride=1)\n            loss = loss + w * F.l1_loss(cur_p - blur_p, cur_g - blur_g)\n            cur_p = F.avg_pool2d(cur_p, 2, stride=2)\n            cur_g = F.avg_pool2d(cur_g, 2, stride=2)\n        return loss\n\n    def forward(self, pred_ink, gt_ink):\n        if not self.use_vgg:\n            return self._lap_pyr_loss(pred_ink, gt_ink)\n\n        p_rgb = (1.0 - pred_ink).repeat(1, 3, 1, 1)\n        g_rgb = (1.0 - gt_ink).repeat(1, 3, 1, 1)\n        p_norm = (p_rgb - self.mean) / self.std\n        g_norm = (g_rgb - self.mean) / self.std\n\n        weights = [0.20, 0.30, 0.30, 0.20]\n        total = 0.0\n        h_p, h_g = p_norm, g_norm\n        for blk, w in zip([self.slice1, self.slice2, self.slice3, self.slice4], weights):\n            h_p = blk(h_p)\n            with torch.no_grad():\n                h_g = blk(h_g)\n            total = total + w * F.l1_loss(h_p, h_g)\n        return total\n\n\nclass AdaptiveLineartLoss(nn.Module):\n    """\n    Hàm mục tiêu 8-term chống dày nét và bảo toàn độ mảnh:\n      - 1.00*Charbonnier\n      - 0.35*Symmetric-DT (steps=8, khả vi 2 chiều)\n      - 0.20*Sobel Edge Gradient\n      - 0.50*Exact-STE Density Ratio (deadzone [0.96, 1.04])\n      - 0.25*Bidirectional Stroke Width (core protection + ring overspill penalty)\n      - 0.20*Ghost & Cleanliness (far background + near halo)\n      - 0.20*Soft-clDice (cân bằng alpha=0.5, beta=0.5)\n      - 0.15*VGG16 Perceptual (chỉ ở LONG mode)\n    Hỗ trợ cắt biên chính xác (valid_hw) để không tính loss lên phần pad biên phải/dưới!\n    """\n    def __init__(self, train_mode="LONG", is_rank0=True, w_ghost=0.45, w_temp=0.50):\n        super().__init__()\n        self.train_mode = train_mode.upper()\n        self.dt_steps = 8\n        self.w_charb = 1.0\n        self.w_dt = 0.35\n        self.w_sobel = 0.20\n        self.w_dens = 0.50\n        self.w_width = 0.25\n        self.w_ghost = float(w_ghost)\n\n        if self.train_mode == "SHORT":\n            self.w_cldice = 0.0\n            self.w_perc = 0.0\n            self.w_temp = 0.0\n        elif self.train_mode == "MEDIUM":\n            self.w_cldice = 0.20\n            self.w_perc = 0.0\n            self.w_temp = float(w_temp) * 0.7\n        else:\n            self.w_cldice = 0.20\n            self.w_perc = 0.15\n            self.w_temp = float(w_temp)\n\n        self.density_loss = ExactSTEDensityLoss(tau=0.15, dead_zone_low=0.96, dead_zone_high=1.04)\n        self.stroke_width_loss = BidirectionalStrokeWidthLoss()\n        self.ghost_loss = GhostAndBackgroundCleanlinessLoss()\n        self.cldice = SoftClDiceLoss(iters=2) if self.w_cldice > 0 else None\n        self.perceptual = VGGPerceptualOrLaplacianLoss(is_rank0=is_rank0) if self.w_perc > 0 else None\n\n        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3) / 8.0\n        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3) / 8.0\n        self.register_buffer(\'sobel_x\', sobel_x)\n        self.register_buffer(\'sobel_y\', sobel_y)\n\n    def forward(self, pred, gt, valid_hw=None):\n        if valid_hw is not None:\n            if torch.all(valid_hw == valid_hw[0:1]):\n                hi, wi = int(valid_hw[0, 0].item()), int(valid_hw[0, 1].item())\n                return self._compute_single(pred[:, :, :hi, :wi], gt[:, :, :hi, :wi])\n            total_l = 0.0\n            details_acc = {\n                \'loss_charb\': 0.0, \'loss_dt\': 0.0, \'loss_sobel\': 0.0,\n                \'loss_dens\': 0.0, \'loss_width\': 0.0, \'loss_ghost\': 0.0,\n                \'loss_cldice\': 0.0, \'loss_perc\': 0.0, \'total_loss\': 0.0\n            }\n            b = pred.shape[0]\n            for i in range(b):\n                hi, wi = int(valid_hw[i, 0].item()), int(valid_hw[i, 1].item())\n                p_i = pred[i:i + 1, :, :hi, :wi]\n                g_i = gt[i:i + 1, :, :hi, :wi]\n                l_i, d_i = self._compute_single(p_i, g_i)\n                total_l = total_l + l_i / float(b)\n                for k in details_acc:\n                    if k in d_i:\n                        details_acc[k] += d_i[k] / float(b)\n            details_acc[\'total_loss\'] = float(total_l.detach().item())\n            return total_l, details_acc\n        return self._compute_single(pred, gt)\n\n    def _compute_single(self, pred, gt):\n        # 1. Charbonnier\n        loss_charb = torch.mean(torch.sqrt((pred - gt) ** 2 + 1e-6))\n\n        # 2. Symmetric Differentiable Distance Transform Loss (khả vi 2 chiều)\n        pred_soft = torch.clamp((pred - 0.06) / 0.18, 0.0, 1.0)\n        dt_p = approx_gpu_distance_map(pred_soft, steps=self.dt_steps)\n        with torch.no_grad():\n            gt_bin = (gt > 0.15).float()\n            dt_g = approx_gpu_distance_map(gt_bin, steps=self.dt_steps)\n            if self.cldice is not None:\n                skel_gt = soft_skeletonize(gt, iters=2)\n\n        loss_dt = F.l1_loss(dt_p, dt_g) + 0.5 * torch.mean(pred * dt_g)\n\n        # 3. Sobel Edge Gradient\n        gx_p = F.conv2d(pred, self.sobel_x, padding=1)\n        gy_p = F.conv2d(pred, self.sobel_y, padding=1)\n        gx_g = F.conv2d(gt, self.sobel_x, padding=1)\n        gy_g = F.conv2d(gt, self.sobel_y, padding=1)\n        loss_sobel = F.l1_loss(gx_p, gx_g) + F.l1_loss(gy_p, gy_g)\n\n        # 4. Anti-Thickening Terms\n        loss_dens = self.density_loss(pred, gt)\n        loss_width = self.stroke_width_loss(pred, gt)\n        loss_ghost = self.ghost_loss(pred, gt)\n\n        # 5. Soft-clDice\n        loss_cldice = self.cldice(pred, gt, t_skel=skel_gt) if self.cldice is not None else pred.new_zeros(())\n\n        # 6. Perceptual / VGG\n        loss_perc = self.perceptual(pred, gt) if self.perceptual is not None else pred.new_zeros(())\n\n        total_loss = (\n            self.w_charb * loss_charb\n            + self.w_dt * loss_dt\n            + self.w_sobel * loss_sobel\n            + self.w_dens * loss_dens\n            + self.w_width * loss_width\n            + self.w_ghost * loss_ghost\n            + self.w_cldice * loss_cldice\n            + self.w_perc * loss_perc\n        )\n        return total_loss, {\n            \'loss_charb\': float(loss_charb.detach().item()),\n            \'loss_dt\': float(loss_dt.detach().item()),\n            \'loss_sobel\': float(loss_sobel.detach().item()),\n            \'loss_dens\': float(loss_dens.detach().item()),\n            \'loss_width\': float(loss_width.detach().item()),\n            \'loss_ghost\': float(loss_ghost.detach().item()),\n            \'loss_cldice\': float(loss_cldice.detach().item()),\n            \'loss_perc\': float(loss_perc.detach().item()),\n            \'total_loss\': float(total_loss.detach().item()),\n        }\n\n    def compute_temporal_pair_loss(self, pred_a, pred_b, gt_a, gt_b, valid_hw=None):\n        """\n        Ràng buộc nhất quán thời gian (2C) giữa 2 khung hình liên tiếp (k, k+1) trong cùng chuỗi:\n          - Phạt sai phân chuyển động (pred_b - pred_a) lệch khỏi (gt_b - gt_a).\n          - Phạt nhấp nháy/rung sương trên vùng cùng là nền trắng của 2 frame liên tiếp.\n        """\n        if self.w_temp <= 0.0 or pred_a.shape[0] == 0:\n            return pred_a.new_zeros(())\n\n        total = pred_a.new_zeros(())\n        n = pred_a.shape[0]\n        for i in range(n):\n            if valid_hw is not None:\n                hi, wi = int(valid_hw[i, 0].item()), int(valid_hw[i, 1].item())\n                pa, pb = pred_a[i:i + 1, :, :hi, :wi], pred_b[i:i + 1, :, :hi, :wi]\n                ga, gb = gt_a[i:i + 1, :, :hi, :wi], gt_b[i:i + 1, :, :hi, :wi]\n            else:\n                pa, pb = pred_a[i:i + 1], pred_b[i:i + 1]\n                ga, gb = gt_a[i:i + 1], gt_b[i:i + 1]\n\n            dp = pb - pa\n            dg = gb - ga\n            loss_tdiff = torch.mean(torch.sqrt((dp - dg) ** 2 + 1e-6))\n\n            with torch.no_grad():\n                both_bg = ((ga < 0.04) & (gb < 0.04)).float()\n                gt_area = torch.clamp(\n                    0.5 * (torch.sum((ga > 0.15).float()) + torch.sum((gb > 0.15).float())),\n                    min=500.0\n                )\n            loss_tflicker = torch.sum(torch.abs(dp) * both_bg) / gt_area\n            total = total + (loss_tdiff + 0.5 * loss_tflicker)\n        return self.w_temp * total\n\n\n# ==============================================================================\n# 4. HỆ THỐNG ĐO LƯỜNG CHỈ SỐ KHÁCH QUAN KÉP (DUAL-REGIME ZERO-BIAS METRICS)\n# ==============================================================================\ndef compute_wcd_batch_gpu(p_bin, g_bin):\n    """\n    Weighted Chamfer Distance trên mặt nạ nhị phân [B, 1, H, W].\n    Công thức chuẩn hóa Paper Eq. (15):\n      h_weight = sigmoid(|area_p - area_g| / min(area_p, area_g))\n    Phạt nặng khi mất cân đối diện tích nét vẽ (chống dày nét).\n    Đồng thời tính `wcd_legacy` (công thức cũ) để đối chiếu lịch sử.\n    """\n    dist_g = approx_gpu_distance_map(g_bin, steps=10)\n    dist_p = approx_gpu_distance_map(p_bin, steps=10)\n    d_p_to_g = torch.sum(p_bin * F.softplus(dist_g * 10.0), dim=(-2, -1)) / torch.clamp(torch.sum(p_bin, dim=(-2, -1)), min=1.0)\n    d_g_to_p = torch.sum(g_bin * F.softplus(dist_p * 10.0), dim=(-2, -1)) / torch.clamp(torch.sum(g_bin, dim=(-2, -1)), min=1.0)\n    area_p = torch.sum(p_bin, dim=(-2, -1))\n    area_g = torch.sum(g_bin, dim=(-2, -1))\n\n    # Chuẩn Paper Eq. (15): sigmoid(|Ap - Ag| / min(Ap, Ag))\n    diff_rel = torch.abs(area_p - area_g) / torch.clamp(torch.minimum(area_p, area_g), min=1.0)\n    h_weight = torch.sigmoid(diff_rel)\n    wcd = h_weight * (d_p_to_g + d_g_to_p)\n\n    # Legacy: 0.5 * (1 + min/max)\n    h_weight_legacy = 0.5 * (1.0 + torch.minimum(area_p, area_g) / torch.clamp(torch.maximum(area_p, area_g), min=1.0))\n    wcd_legacy = h_weight_legacy * (d_p_to_g + d_g_to_p)\n\n    return wcd, wcd_legacy\n\n\ndef compute_semd_batch_gpu(p_ink, g_ink):\n    """1D-Sliced Wasserstein EMD trên phân bố nét vẽ dọc hai trục X/Y."""\n    p_norm = p_ink / torch.clamp(torch.sum(p_ink, dim=(-2, -1), keepdim=True), min=1e-6)\n    g_norm = g_ink / torch.clamp(torch.sum(g_ink, dim=(-2, -1), keepdim=True), min=1e-6)\n    emd_h = torch.mean(torch.abs(torch.cumsum(torch.sum(p_norm, dim=-1), dim=-1) - torch.cumsum(torch.sum(g_norm, dim=-1), dim=-1)), dim=(-2, -1))\n    emd_w = torch.mean(torch.abs(torch.cumsum(torch.sum(p_norm, dim=-2), dim=-1) - torch.cumsum(torch.sum(g_norm, dim=-2), dim=-1)), dim=(-2, -1))\n    return 0.5 * (emd_h + emd_w)\n\n\n_DOG_KERNEL_CACHE = {}\n\n\ndef _gaussian_blur_2d(img, sigma):\n    key = (round(float(sigma), 4), img.device, img.dtype)\n    k2d = _DOG_KERNEL_CACHE.get(key)\n    if k2d is None:\n        rad = max(1, int(round(3.0 * sigma)))\n        coords = torch.arange(-rad, rad + 1, device=img.device, dtype=torch.float32)\n        g1 = torch.exp(-(coords ** 2) / (2.0 * sigma * sigma))\n        g1 = g1 / g1.sum()\n        k2d = torch.outer(g1, g1).view(1, 1, 2 * rad + 1, 2 * rad + 1).to(img.dtype)\n        _DOG_KERNEL_CACHE[key] = k2d\n    rad = k2d.shape[-1] // 2\n    return F.conv2d(F.pad(img, (rad, rad, rad, rad), mode=\'replicate\'), k2d)\n\n\ndef compute_dog_bin_mask(ink_img, sigma=1.0, k=1.6, tau=0.02):\n    """\n    Bộ lọc Difference-of-Gaussians (DoG) trích xuất nét thuần túy,\n    triệt tiêu hoàn toàn độ xám nền (nền xám nhạt vs nền trắng tinh) để đánh giá công bằng M1 vs M3.\n    """\n    gray = 1.0 - torch.clamp(ink_img, 0.0, 1.0)\n    g1 = _gaussian_blur_2d(gray, sigma)\n    g2 = _gaussian_blur_2d(gray, sigma * k)\n    dog = g1 - g2\n    return (dog < -tau).float()\n\n\n@torch.no_grad()\ndef evaluate_metrics_dual_regime(pred_ink, gt_ink):\n    """\n    Đo lường 10+ chỉ số khách quan trên cùng 1 cặp ảnh (pred_ink, gt_ink) shape [B, 1, H, W]:\n      - Regime A (Raw Continuous): PSNR, SSIM, IE_MAE (0..255), PWCR, RGD, Ghost_Px, Dark_Ghost_Px, Val_L1\n      - Regime B (Thresholded tau=0.15 & DoG): WCD (Eq.15), WCD_DoG, WCD_Legacy, WCD_DoG_Legacy,\n                                               S-EMD, Skeleton Recall, Hair-Tip Recall,\n                                               Stroke Recall, Stroke Precision, Stroke F1, Density Ratio\n    """\n    p = torch.clamp(pred_ink.float(), 0.0, 1.0)\n    g = torch.clamp(gt_ink.float(), 0.0, 1.0)\n\n    # ---- REGIME A: RAW CONTINUOUS ----\n    mse = torch.mean((p - g) ** 2, dim=(-3, -2, -1))\n    psnr = 10.0 * torch.log10(1.0 / torch.clamp(mse, min=1e-8))\n\n    mu_p = F.avg_pool2d(p, 11, stride=1, padding=5)\n    mu_g = F.avg_pool2d(g, 11, stride=1, padding=5)\n    sig_p = F.avg_pool2d(p * p, 11, stride=1, padding=5) - mu_p ** 2\n    sig_g = F.avg_pool2d(g * g, 11, stride=1, padding=5) - mu_g ** 2\n    sig_pg = F.avg_pool2d(p * g, 11, stride=1, padding=5) - mu_p * mu_g\n    ssim_map = ((2 * mu_p * mu_g + 1e-4) * (2 * sig_pg + 9e-4)) / ((mu_p ** 2 + mu_g ** 2 + 1e-4) * (sig_p + sig_g + 9e-4))\n    ssim = torch.mean(ssim_map, dim=(-3, -2, -1))\n\n    val_l1 = torch.mean(torch.abs(p - g), dim=(-3, -2, -1))\n    ie_mae = val_l1 * 255.0\n\n    # Pixel brightness domain [0, 255]\n    p_255 = (1.0 - p) * 255.0\n    g_255 = (1.0 - g) * 255.0\n    pwcr = torch.mean((p_255 >= 250.0).float(), dim=(-3, -2, -1))\n    rgd = torch.mean(((p_255 >= 180.0) & (p_255 < 250.0)).float(), dim=(-3, -2, -1))\n\n    gt_white_safe = (g_255 > 250.0).float()\n    gt_stroke_safe = soft_dilate(soft_dilate((g > 0.08).float()))\n    outside_gt = (1.0 - gt_stroke_safe) * gt_white_safe\n    ghost_px = torch.sum(((p_255 >= 180.0) & (p_255 <= 250.0)).float() * gt_white_safe, dim=(-3, -2, -1))\n    dark_ghost_px = torch.sum((p > 0.35).float() * (1.0 - gt_stroke_safe), dim=(-3, -2, -1))\n\n    # ---- REGIME B: BINARIZED TOPOLOGY (TH=0.15 + DoG) ----\n    p_bin = (p > 0.15).float()\n    g_bin = (g > 0.15).float()\n    p_dog = compute_dog_bin_mask(p)\n    g_dog = compute_dog_bin_mask(g)\n\n    wcd, wcd_legacy = compute_wcd_batch_gpu(p_bin, g_bin)\n    wcd_dog, wcd_dog_legacy = compute_wcd_batch_gpu(p_dog, g_dog)\n    semd = compute_semd_batch_gpu(p, g)\n\n    # Stroke Recall / Precision / F1 (dung sai 2px dilation cho nét mảnh 1px)\n    g_dil = soft_dilate(soft_dilate(g_bin))\n    p_dil = soft_dilate(soft_dilate(p_bin))\n    tp_r = torch.sum(g_bin * p_dil, dim=(-3, -2, -1))\n    tp_p = torch.sum(p_bin * g_dil, dim=(-3, -2, -1))\n    cnt_p = torch.sum(p_bin, dim=(-3, -2, -1))\n    cnt_g = torch.sum(g_bin, dim=(-3, -2, -1))\n    recall = tp_r / torch.clamp(cnt_g, min=1.0)\n    precision = tp_p / torch.clamp(cnt_p, min=1.0)\n    f1 = 2.0 * recall * precision / torch.clamp(recall + precision, min=1e-6)\n    density_ratio = cnt_p / torch.clamp(cnt_g, min=1.0)\n\n    # Skeleton Recall & Hair-Tip Recall\n    skel_g = soft_skeletonize(g, iters=2)\n    skel_mask = (skel_g > 0.20).float()\n    p_dil1 = soft_dilate(p_bin)\n    skel_recall = torch.sum(p_dil1 * skel_mask, dim=(-3, -2, -1)) / torch.clamp(torch.sum(skel_mask, dim=(-3, -2, -1)), min=1.0)\n\n    nbrs = F.avg_pool2d(skel_g, kernel_size=3, stride=1, padding=1) * 9.0 - skel_g\n    endpoints = ((nbrs <= 1.8) & (skel_g > 0.20)).float()\n    tip_recall = torch.sum(p_dil1 * endpoints, dim=(-3, -2, -1)) / torch.clamp(torch.sum(endpoints, dim=(-3, -2, -1)), min=1.0)\n\n    return {\n        \'psnr\': psnr.squeeze(-1) if psnr.ndim > 1 else psnr,\n        \'ssim\': ssim.squeeze(-1) if ssim.ndim > 1 else ssim,\n        \'val_l1\': val_l1.squeeze(-1) if val_l1.ndim > 1 else val_l1,\n        \'ie_mae\': ie_mae.squeeze(-1) if ie_mae.ndim > 1 else ie_mae,\n        \'pwcr\': pwcr.squeeze(-1) if pwcr.ndim > 1 else pwcr,\n        \'rgd\': rgd.squeeze(-1) if rgd.ndim > 1 else rgd,\n        \'ghost\': ghost_px.squeeze(-1) if ghost_px.ndim > 1 else ghost_px,\n        \'dark_ghost\': dark_ghost_px.squeeze(-1) if dark_ghost_px.ndim > 1 else dark_ghost_px,\n        \'wcd\': wcd.squeeze(-1) if wcd.ndim > 1 else wcd,\n        \'wcd_legacy\': wcd_legacy.squeeze(-1) if wcd_legacy.ndim > 1 else wcd_legacy,\n        \'wcd_dog\': wcd_dog.squeeze(-1) if wcd_dog.ndim > 1 else wcd_dog,\n        \'wcd_dog_legacy\': wcd_dog_legacy.squeeze(-1) if wcd_dog_legacy.ndim > 1 else wcd_dog_legacy,\n        \'semd\': semd.squeeze(-1) if semd.ndim > 1 else semd,\n        \'recall\': recall.squeeze(-1) if recall.ndim > 1 else recall,\n        \'precision\': precision.squeeze(-1) if precision.ndim > 1 else precision,\n        \'f1\': f1.squeeze(-1) if f1.ndim > 1 else f1,\n        \'density_ratio\': density_ratio.squeeze(-1) if density_ratio.ndim > 1 else density_ratio,\n        \'skel_recall\': skel_recall.squeeze(-1) if skel_recall.ndim > 1 else skel_recall,\n        \'tip_recall\': tip_recall.squeeze(-1) if tip_recall.ndim > 1 else tip_recall,\n    }\n\n\n# ==============================================================================\n# 5. DATASET CHUỖI SEQUENCE ĐA KHUNG HÌNH N={3,5,7} & DDP BUCKET SAMPLER\n# ==============================================================================\ndef pad_to_canvas(arr_2d, target_h, target_w):\n    """Đệm ảnh ink [H, W] bằng nền trắng (0.0) ở đáy/phải tới (target_h, target_w)."""\n    h, w = arr_2d.shape\n    if h == target_h and w == target_w:\n        return arr_2d\n    out = np.zeros((target_h, target_w), dtype=arr_2d.dtype)\n    out[:h, :w] = arr_2d\n    return out\n\n\ndef _generate_synthetic_geometric_sequence(seq_len, orig_h=540, orig_w=960, max_pts=128, lambd=0.01):\n    """\n    Sinh động chuỗi hình học nét mảnh 540p kèm điểm điều khiển Lagrangian Direct TPS:\n      - Hình tròn / elip dịch chuyển xa (60..420 px) & co giãn kích thước (0.40x..2.25x)\n      - Hình tròn bay + đường nền / hộp tĩnh\n      - Cung cong xoay & tịnh tiến\n      - Cặp 2 hình tròn chuyển động độc lập\n    """\n    def _ellipse_pts(cx, cy, rx, ry, rot_rad, n_pts=32):\n        ang = np.linspace(0.0, 2.0 * np.pi, n_pts, endpoint=False)\n        lx = rx * np.cos(ang)\n        ly = ry * np.sin(ang)\n        ca, sa = math.cos(rot_rad), math.sin(rot_rad)\n        px = cx + lx * ca - ly * sa\n        py = cy + lx * sa + ly * ca\n        pts = np.stack([px / float(orig_w - 1), py / float(orig_h - 1)], axis=1)\n        return pts.astype(np.float32)\n\n    mode_roll = random.random()\n    frames = []\n    kps_per_t = []\n    t_list = [float(k) / float(seq_len - 1) for k in range(seq_len)]\n\n    if mode_roll < 0.65:\n        # Mode 1 (40%) & Mode 2 (25%): Shrinking/expanding circle/oval (+ optional static ground/box)\n        has_static = (mode_roll >= 0.40)\n        y_max_c = 340 if has_static else 420\n        rx0 = random.uniform(26.0, 92.0)\n        ry0 = rx0 * random.uniform(0.75, 1.25)\n        scale_r = random.uniform(0.42, 2.20)\n        rx1 = float(np.clip(rx0 * scale_r, 18.0, 96.0))\n        ry1 = float(np.clip(ry0 * scale_r * random.uniform(0.85, 1.15), 18.0, 96.0))\n        cx0 = random.uniform(110.0, 420.0)\n        cx1 = float(np.clip(cx0 + random.choice([-1.0, 1.0]) * random.uniform(80.0, 420.0), 100.0, orig_w - 100.0))\n        if random.random() < 0.5:\n            cx0, cx1 = cx1, cx0\n        cy0 = random.uniform(115.0, y_max_c - 60.0)\n        cy1 = float(np.clip(cy0 + random.uniform(-90.0, 90.0), 105.0, y_max_c - 40.0))\n        rot0 = random.uniform(-0.35, 0.35)\n        rot1 = rot0 + random.uniform(-0.35, 0.35)\n        th0 = random.uniform(2.0, 5.0)\n        th1 = random.uniform(2.0, 5.0)\n\n        # Static element (ground line or static rectangle)\n        st_type = random.choice(["line", "rect"])\n        st_y = random.uniform(410.0, 490.0)\n        st_x0, st_x1 = random.uniform(80.0, 200.0), random.uniform(orig_w - 200.0, orig_w - 80.0)\n        st_th = int(round(random.uniform(2.5, 4.5)))\n        if st_type == "line":\n            xs_st = np.linspace(st_x0, st_x1, 24)\n            pts_st = np.stack([xs_st / float(orig_w - 1), np.full_like(xs_st, st_y) / float(orig_h - 1)], axis=1).astype(np.float32)\n        else:\n            bx0, by0 = random.uniform(120.0, 740.0), st_y - 45.0\n            bw, bh = random.uniform(70.0, 140.0), random.uniform(40.0, 65.0)\n            pts_st = np.vstack([\n                np.stack([np.linspace(bx0, bx0 + bw, 7)[:-1], np.full(6, by0)], axis=1),\n                np.stack([np.full(6, bx0 + bw), np.linspace(by0, by0 + bh, 7)[:-1]], axis=1),\n                np.stack([np.linspace(bx0 + bw, bx0, 7)[:-1], np.full(6, by0 + bh)], axis=1),\n                np.stack([np.full(6, bx0), np.linspace(by0 + bh, by0, 7)[:-1]], axis=1),\n            ])\n            pts_st[:, 0] /= float(orig_w - 1)\n            pts_st[:, 1] /= float(orig_h - 1)\n            pts_st = pts_st.astype(np.float32)\n\n        for t in t_list:\n            im = np.zeros((orig_h, orig_w), dtype=np.float32)\n            cx = (1.0 - t) * cx0 + t * cx1\n            cy = (1.0 - t) * cy0 + t * cy1\n            rx = (1.0 - t) * rx0 + t * rx1\n            ry = (1.0 - t) * ry0 + t * ry1\n            rot = (1.0 - t) * rot0 + t * rot1\n            th = max(2, int(round((1.0 - t) * th0 + t * th1)))\n            if cv2 is not None:\n                cv2.ellipse(im, (int(round(cx)), int(round(cy))), (int(round(rx)), int(round(ry))), math.degrees(rot), 0, 360, 1.0, th, lineType=cv2.LINE_AA)\n                if has_static:\n                    if st_type == "line":\n                        cv2.line(im, (int(st_x0), int(st_y)), (int(st_x1), int(st_y)), 1.0, st_th, lineType=cv2.LINE_AA)\n                    else:\n                        cv2.rectangle(im, (int(bx0), int(by0)), (int(bx0 + bw), int(by0 + bh)), 1.0, st_th, lineType=cv2.LINE_AA)\n            pts_mv = _ellipse_pts(cx, cy, rx, ry, rot, n_pts=32)\n            pts_all = np.vstack([pts_mv, pts_st]) if has_static else pts_mv\n            frames.append(im)\n            kps_per_t.append(pts_all)\n    elif mode_roll < 0.85:\n        # Mode 3 (20%): Smooth curved arc / open stroke translating + rotating + scaling\n        n_ctrl = 24\n        s_u = np.linspace(-1.0, 1.0, n_ctrl)\n        base_len = random.uniform(60.0, 140.0)\n        bend0 = random.uniform(-55.0, 55.0)\n        bend1 = random.uniform(-55.0, 55.0)\n        cx0, cy0 = random.uniform(160.0, 440.0), random.uniform(140.0, 400.0)\n        cx1 = float(np.clip(cx0 + random.choice([-1.0, 1.0]) * random.uniform(60.0, 320.0), 140.0, orig_w - 140.0))\n        cy1 = float(np.clip(cy0 + random.uniform(-100.0, 100.0), 120.0, orig_h - 120.0))\n        sc0, sc1 = random.uniform(0.75, 1.25), random.uniform(0.65, 1.45)\n        ang0 = random.uniform(-0.7, 0.7)\n        ang1 = ang0 + random.uniform(-0.6, 0.6)\n        th = int(round(random.uniform(2.0, 4.5)))\n\n        def _arc_px(t):\n            cx = (1.0 - t) * cx0 + t * cx1\n            cy = (1.0 - t) * cy0 + t * cy1\n            sc = (1.0 - t) * sc0 + t * sc1\n            ang = (1.0 - t) * ang0 + t * ang1\n            bend = (1.0 - t) * bend0 + t * bend1\n            lx = s_u * base_len * sc\n            ly = bend * sc * (1.0 - s_u ** 2)\n            ca, sa = math.cos(ang), math.sin(ang)\n            return np.stack([cx + lx * ca - ly * sa, cy + lx * sa + ly * ca], axis=1)\n\n        for t in t_list:\n            im = np.zeros((orig_h, orig_w), dtype=np.float32)\n            p_px = _arc_px(t)\n            if cv2 is not None:\n                cv2.polylines(im, [np.round(p_px).astype(np.int32)], False, 1.0, th, lineType=cv2.LINE_AA)\n            p_norm = np.stack([p_px[:, 0] / float(orig_w - 1), p_px[:, 1] / float(orig_h - 1)], axis=1).astype(np.float32)\n            frames.append(im)\n            kps_per_t.append(p_norm)\n    else:\n        # Mode 4 (15%): Two independent moving circles/ovals in upper & lower lanes\n        lanes = [(145.0, 40.0), (385.0, 40.0)]\n        params = []\n        for cy_base, cy_jit in lanes:\n            rx0 = random.uniform(24.0, 62.0)\n            ry0 = rx0 * random.uniform(0.8, 1.2)\n            sc = random.uniform(0.55, 1.80)\n            rx1 = float(np.clip(rx0 * sc, 18.0, 72.0))\n            ry1 = float(np.clip(ry0 * sc, 18.0, 72.0))\n            cx0 = random.uniform(120.0, 450.0)\n            cx1 = float(np.clip(cx0 + random.choice([-1.0, 1.0]) * random.uniform(80.0, 350.0), 100.0, orig_w - 100.0))\n            cy0 = cy_base + random.uniform(-cy_jit, cy_jit)\n            cy1 = cy_base + random.uniform(-cy_jit, cy_jit)\n            params.append((cx0, cy0, rx0, ry0, cx1, cy1, rx1, ry1))\n        th = int(round(random.uniform(2.5, 4.5)))\n        for t in t_list:\n            im = np.zeros((orig_h, orig_w), dtype=np.float32)\n            pts_two = []\n            for (cx0, cy0, rx0, ry0, cx1, cy1, rx1, ry1) in params:\n                cx = (1.0 - t) * cx0 + t * cx1\n                cy = (1.0 - t) * cy0 + t * cy1\n                rx = (1.0 - t) * rx0 + t * rx1\n                ry = (1.0 - t) * ry0 + t * ry1\n                if cv2 is not None:\n                    cv2.ellipse(im, (int(round(cx)), int(round(cy))), (int(round(rx)), int(round(ry))), 0, 0, 360, 1.0, th, lineType=cv2.LINE_AA)\n                pts_two.append(_ellipse_pts(cx, cy, rx, ry, 0.0, n_pts=24))\n            frames.append(im)\n            kps_per_t.append(np.vstack(pts_two))\n\n    x0 = frames[0]\n    x1 = frames[-1]\n    xk_all = np.stack(frames[1:-1], axis=0)\n    kps0 = kps_per_t[0]\n    kps1 = kps_per_t[-1]\n    t_mid = np.array(t_list[1:-1], dtype=np.float32)\n\n    m_steps = seq_len - 2\n    c_t0_all = np.zeros((m_steps, max_pts, 2), dtype=np.float32)\n    th_t0_all = np.zeros((m_steps, max_pts + 3, 2), dtype=np.float32)\n    c_t1_all = np.zeros((m_steps, max_pts, 2), dtype=np.float32)\n    th_t1_all = np.zeros((m_steps, max_pts + 3, 2), dtype=np.float32)\n    valid_all = np.zeros((m_steps,), dtype=np.float32)\n\n    for m_idx in range(m_steps):\n        kpt = kps_per_t[m_idx + 1]\n        c_t0, th_t0, v0 = solve_tps_direct(kpt, kps0, max_pts=max_pts, lambd=lambd, use_anchors=False)\n        c_t1, th_t1, v1 = solve_tps_direct(kpt, kps1, max_pts=max_pts, lambd=lambd, use_anchors=False)\n        c_t0_all[m_idx] = c_t0\n        th_t0_all[m_idx] = th_t0\n        c_t1_all[m_idx] = c_t1\n        th_t1_all[m_idx] = th_t1\n        valid_all[m_idx] = 1.0 if (v0 > 0.5 and v1 > 0.5) else 0.0\n\n    return {\n        \'is_sequence\': True,\n        \'is_direct_t_flows\': True,\n        \'x0\': torch.from_numpy(x0).unsqueeze(0),\n        \'x1\': torch.from_numpy(x1).unsqueeze(0),\n        \'xk_all\': torch.from_numpy(xk_all).unsqueeze(1),\n        \'t_vals\': torch.from_numpy(t_mid),\n        \'c_t0_all\': torch.from_numpy(c_t0_all),\n        \'th_t0_all\': torch.from_numpy(th_t0_all),\n        \'c_t1_all\': torch.from_numpy(c_t1_all),\n        \'th_t1_all\': torch.from_numpy(th_t1_all),\n        \'valid_all\': torch.from_numpy(valid_all),\n        \'orig_hw\': torch.tensor([orig_h, orig_w], dtype=torch.int32),\n        \'seq_len\': seq_len,\n        \'clip_id\': f"synth_N{seq_len}",\n        \'scene_id\': "synth_geom",\n        \'mean_disp\': 120.0,\n    }\n\n\nclass SakugaSequenceDataset(Dataset):\n    """\n    Dataset Sequence cho cả 3 chế độ (SHORT / MEDIUM / LONG) và Validation:\n      - `mode == \'LONG\'`: trả về NGUYÊN CHUỖI (x0, x1, xk_all[M], t_vals[M]) ở độ phân giải gốc 540p,\n        kèm tùy chọn trộn các chuỗi hình học tổng hợp (`synth_count`) dùng Lagrangian Direct TPS.\n      - `mode == \'SHORT\'`: mỗi epoch lấy 1 frame trung gian ngẫu nhiên/chuỗi + Ink Crop 384x384.\n      - `mode == \'MEDIUM\'`: mở rộng thành toàn bộ 3,169 mẫu đơn + Motion-BBox Crop 512x512.\n    """\n    def __init__(self, metadata_csv, clips_dir, tps_cache_path, train_mode="LONG", is_train=True, seed=42, synth_count=0):\n        super().__init__()\n        self.clips_dir = clips_dir\n        self.train_mode = train_mode.upper()\n        self.is_train = is_train\n        self.df = pd.read_csv(metadata_csv).reset_index(drop=True)\n\n        # Nạp nhẹ toàn bộ mảng TPS (~7 MB) vào RAM và đóng ngay file .npz để an toàn tuyệt đối khi fork num_workers>0\n        self.tps_cache_path = tps_cache_path\n        with np.load(tps_cache_path) as cache_data:\n            self.c0_arr = np.array(cache_data[\'c0\'])\n            self.th01_arr = np.array(cache_data[\'th01\'])\n            self.c1_arr = np.array(cache_data[\'c1\'])\n            self.th10_arr = np.array(cache_data[\'th10\'])\n            self.valid_arr = np.array(cache_data[\'valid\'])\n            self.orig_hw_arr = np.array(cache_data[\'orig_hw\'])\n\n        self.samples = []\n        if self.is_train and self.train_mode == "LONG":\n            # 1 item = toàn bộ 1 sequence (x0, x1, toàn bộ N-2 frames trung gian)\n            for idx, row in self.df.iterrows():\n                cid = str(row[\'clip_id\'])\n                seq_len = int(row[\'seq_len\'])\n                cache_idx = int(row[\'cache_idx\']) if \'cache_idx\' in row else idx\n                hw = self.orig_hw_arr[cache_idx]\n                t_list = [float(k) / float(seq_len - 1) for k in range(1, seq_len - 1)]\n                self.samples.append({\n                    \'is_synth\': False,\n                    \'clip_id\': cid,\n                    \'scene_id\': str(row.get(\'scene_id\', \'\')),\n                    \'seq_len\': seq_len,\n                    \'cache_idx\': cache_idx,\n                    \'orig_h\': int(hw[0]),\n                    \'orig_w\': int(hw[1]),\n                    \'t_list\': t_list,\n                    \'mean_disp\': float(row.get(\'mean_disp\', 0.0)),\n                })\n            # Bổ sung chuỗi hình học tổng hợp phân bổ đều cho N={3, 5, 7}\n            n_synth = int(synth_count) if (cv2 is not None and synth_count > 0) else 0\n            for s_i in range(n_synth):\n                sl = (3, 5, 7)[s_i % 3]\n                self.samples.append({\n                    \'is_synth\': True,\n                    \'clip_id\': f"synth_{s_i:04d}_N{sl}",\n                    \'scene_id\': "synth_geom",\n                    \'seq_len\': sl,\n                    \'cache_idx\': -1,\n                    \'orig_h\': 540,\n                    \'orig_w\': 960,\n                    \'t_list\': [float(k) / float(sl - 1) for k in range(1, sl - 1)],\n                    \'mean_disp\': 120.0,\n                })\n        elif self.is_train and self.train_mode == "SHORT":\n            # 1 item = 1 chuỗi (tự chọn 1 frame ngẫu nhiên mỗi epoch trong __getitem__)\n            for idx, row in self.df.iterrows():\n                cid = str(row[\'clip_id\'])\n                seq_len = int(row[\'seq_len\'])\n                cache_idx = int(row[\'cache_idx\']) if \'cache_idx\' in row else idx\n                hw = self.orig_hw_arr[cache_idx]\n                self.samples.append({\n                    \'is_synth\': False,\n                    \'clip_id\': cid,\n                    \'scene_id\': str(row.get(\'scene_id\', \'\')),\n                    \'seq_len\': seq_len,\n                    \'cache_idx\': cache_idx,\n                    \'orig_h\': int(hw[0]),\n                    \'orig_w\': int(hw[1]),\n                    \'frame_k\': None,\n                    \'mean_disp\': float(row.get(\'mean_disp\', 0.0)),\n                })\n        else:\n            # Validation hoặc MEDIUM Train: mở rộng thành mọi frame trung gian (t_k, I_k)\n            for idx, row in self.df.iterrows():\n                cid = str(row[\'clip_id\'])\n                seq_len = int(row[\'seq_len\'])\n                cache_idx = int(row[\'cache_idx\']) if \'cache_idx\' in row else idx\n                hw = self.orig_hw_arr[cache_idx]\n                for k in range(1, seq_len - 1):\n                    self.samples.append({\n                        \'is_synth\': False,\n                        \'clip_id\': cid,\n                        \'scene_id\': str(row.get(\'scene_id\', \'\')),\n                        \'seq_len\': seq_len,\n                        \'cache_idx\': cache_idx,\n                        \'orig_h\': int(hw[0]),\n                        \'orig_w\': int(hw[1]),\n                        \'frame_k\': k + 1,  # 1-based file index: 000002..00000(N-1)\n                        \'t_val\': float(k) / float(seq_len - 1),\n                        \'mean_disp\': float(row.get(\'mean_disp\', 0.0)),\n                    })\n\n    def __len__(self):\n        return len(self.samples)\n\n    def _read_ink(self, clip_id, frame_idx_1b):\n        p = os.path.join(self.clips_dir, clip_id, f"{frame_idx_1b:06d}.png")\n        with Image.open(p) as im:\n            arr = np.asarray(im.convert(\'L\'), dtype=np.float32)\n        return (255.0 - arr) / 255.0\n\n    def __getitem__(self, idx):\n        s = self.samples[idx]\n        if s.get(\'is_synth\', False):\n            return _generate_synthetic_geometric_sequence(s[\'seq_len\'], orig_h=s[\'orig_h\'], orig_w=s[\'orig_w\'])\n\n        cid = s[\'clip_id\']\n        seq_len = s[\'seq_len\']\n        cache_idx = s[\'cache_idx\']\n\n        x0 = self._read_ink(cid, 1)\n        x1 = self._read_ink(cid, seq_len)\n\n        c0 = np.array(self.c0_arr[cache_idx], copy=True)\n        th01 = np.array(self.th01_arr[cache_idx], copy=True)\n        c1 = np.array(self.c1_arr[cache_idx], copy=True)\n        th10 = np.array(self.th10_arr[cache_idx], copy=True)\n        valid = float(self.valid_arr[cache_idx])\n        orig_h, orig_w = x0.shape\n\n        if self.is_train and self.train_mode == "LONG":\n            # Tải đồng thời toàn bộ N-2 khung hình trung gian của chuỗi\n            xk_list = [self._read_ink(cid, f_idx) for f_idx in range(2, seq_len)]\n            xk_all = np.stack(xk_list, axis=0)  # [M, H, W]\n            t_arr = np.array(s[\'t_list\'], dtype=np.float32)\n\n            # Data Augmentation: Temporal Reversal (đảo chiều I0 <-> I1, xk[::-1], t <-> 1-t)\n            if random.random() < 0.5:\n                x0, x1 = x1, x0\n                c0, c1 = c1, c0\n                th01, th10 = th10, th01\n                xk_all = xk_all[::-1].copy()\n                t_arr = (1.0 - t_arr[::-1]).copy()\n\n            return {\n                \'is_sequence\': True,\n                \'is_direct_t_flows\': False,\n                \'x0\': torch.from_numpy(x0).unsqueeze(0),               # [1, H, W]\n                \'x1\': torch.from_numpy(x1).unsqueeze(0),               # [1, H, W]\n                \'xk_all\': torch.from_numpy(xk_all).unsqueeze(1),       # [M, 1, H, W]\n                \'t_vals\': torch.from_numpy(t_arr),                     # [M]\n                \'c0\': torch.from_numpy(c0),\n                \'th01\': torch.from_numpy(th01),\n                \'c1\': torch.from_numpy(c1),\n                \'th10\': torch.from_numpy(th10),\n                \'valid\': torch.tensor(valid, dtype=torch.float32),\n                \'orig_hw\': torch.tensor([orig_h, orig_w], dtype=torch.int32),\n                \'seq_len\': seq_len,\n                \'clip_id\': cid,\n                \'scene_id\': s[\'scene_id\'],\n                \'mean_disp\': s[\'mean_disp\'],\n            }\n\n        # SINGLE FRAME (SHORT / MEDIUM / VAL)\n        if s[\'frame_k\'] is None:\n            k_rel = random.randint(1, seq_len - 2)\n            fk = k_rel + 1\n            t_val = float(k_rel) / float(seq_len - 1)\n        else:\n            fk = s[\'frame_k\']\n            t_val = s[\'t_val\']\n\n        xk = self._read_ink(cid, fk)\n        if self.is_train and random.random() < 0.5:\n            x0, x1 = x1, x0\n            c0, c1 = c1, c0\n            th01, th10 = th10, th01\n            t_val = 1.0 - t_val\n\n        return {\n            \'is_sequence\': False,\n            \'is_direct_t_flows\': False,\n            \'x0\': torch.from_numpy(x0).unsqueeze(0),\n            \'x1\': torch.from_numpy(x1).unsqueeze(0),\n            \'xk\': torch.from_numpy(xk).unsqueeze(0),\n            \'t_val\': torch.tensor([t_val], dtype=torch.float32),\n            \'c0\': torch.from_numpy(c0),\n            \'th01\': torch.from_numpy(th01),\n            \'c1\': torch.from_numpy(c1),\n            \'th10\': torch.from_numpy(th10),\n            \'valid\': torch.tensor(valid, dtype=torch.float32),\n            \'orig_hw\': torch.tensor([orig_h, orig_w], dtype=torch.int32),\n            \'seq_len\': seq_len,\n            \'frame_k\': fk,\n            \'clip_id\': cid,\n            \'scene_id\': s[\'scene_id\'],\n            \'mean_disp\': s[\'mean_disp\'],\n        }\n\n\ndef pad_collate_fn(batch):\n    """\n    Collate tự động đệm các ảnh trong cùng micro-batch lên kích thước chia hết cho 32\n    mà KHÔNG làm biến dạng tỷ lệ nét vẽ. Giữ nguyên `orig_hw` để TPS tính chuẩn xác.\n    """\n    max_h = max(item[\'orig_hw\'][0].item() for item in batch)\n    max_w = max(item[\'orig_hw\'][1].item() for item in batch)\n    h_pad = ((max_h + 31) // 32) * 32\n    w_pad = ((max_w + 31) // 32) * 32\n\n    is_seq = batch[0][\'is_sequence\']\n    is_direct = bool(batch[0].get(\'is_direct_t_flows\', False))\n\n    def _pad_t(t, th, tw):\n        dh = th - t.shape[-2]\n        dw = tw - t.shape[-1]\n        if dh == 0 and dw == 0:\n            return t\n        return F.pad(t, (0, dw, 0, dh), mode=\'constant\', value=0.0)\n\n    out = {\n        \'is_sequence\': is_seq,\n        \'is_direct_t_flows\': is_direct,\n        \'h_pad\': h_pad,\n        \'w_pad\': w_pad,\n        \'x0\': torch.stack([_pad_t(b[\'x0\'], h_pad, w_pad) for b in batch], dim=0),\n        \'x1\': torch.stack([_pad_t(b[\'x1\'], h_pad, w_pad) for b in batch], dim=0),\n        \'orig_hw\': torch.stack([b[\'orig_hw\'] for b in batch], dim=0),\n        \'seq_len\': torch.tensor([b[\'seq_len\'] for b in batch], dtype=torch.int32),\n        \'clip_id\': [b[\'clip_id\'] for b in batch],\n        \'scene_id\': [b[\'scene_id\'] for b in batch],\n        \'mean_disp\': [b[\'mean_disp\'] for b in batch],\n    }\n    if is_direct:\n        out[\'c_t0_all\'] = torch.stack([b[\'c_t0_all\'] for b in batch], dim=0)\n        out[\'th_t0_all\'] = torch.stack([b[\'th_t0_all\'] for b in batch], dim=0)\n        out[\'c_t1_all\'] = torch.stack([b[\'c_t1_all\'] for b in batch], dim=0)\n        out[\'th_t1_all\'] = torch.stack([b[\'th_t1_all\'] for b in batch], dim=0)\n        out[\'valid_all\'] = torch.stack([b[\'valid_all\'] for b in batch], dim=0)\n    else:\n        out[\'c0\'] = torch.stack([b[\'c0\'] for b in batch], dim=0)\n        out[\'th01\'] = torch.stack([b[\'th01\'] for b in batch], dim=0)\n        out[\'c1\'] = torch.stack([b[\'c1\'] for b in batch], dim=0)\n        out[\'th10\'] = torch.stack([b[\'th10\'] for b in batch], dim=0)\n        out[\'valid\'] = torch.stack([b[\'valid\'] for b in batch], dim=0)\n\n    if is_seq:\n        # Lưu ý: Sampler đảm bảo tất cả item trong cùng sequence batch có cùng seq_len -> cùng M!\n        out[\'xk_all\'] = torch.stack([_pad_t(b[\'xk_all\'], h_pad, w_pad) for b in batch], dim=0)\n        out[\'t_vals\'] = torch.stack([b[\'t_vals\'] for b in batch], dim=0)\n    else:\n        out[\'xk\'] = torch.stack([_pad_t(b[\'xk\'], h_pad, w_pad) for b in batch], dim=0)\n        out[\'t_val\'] = torch.stack([b[\'t_val\'] for b in batch], dim=0)\n        out[\'frame_k\'] = [b[\'frame_k\'] for b in batch]\n    return out\n\n\nclass DDPLengthWidthBatchSampler(Sampler):\n    """\n    Sampler cân bằng tải tuyệt đối giữa 2 GPU (97-100% GPU Utilization):\n      1. Nhóm các sequence theo `(is_synth, seq_len)` và sắp xếp theo chiều rộng `orig_w`\n         để không lãng phí padding và tách riêng batch thực vs tổng hợp.\n      2. Tạo micro-batch kích thước chuẩn cho từng N:\n         - N=3 (1 frame giữa): batch = 4 seqs (4 frames / GPU forward)\n         - N=5 (3 frames giữa): batch = 2 seqs (6 frames / GPU) [hoặc 1 nếu W > 1120]\n         - N=7 (5 frames giữa): batch = 1 seq  (5 frames / GPU)\n      3. Ghép từng cặp batch có cùng `(is_synth, seq_len)` cho (Rank 0, Rank 1) để hai GPU không bao giờ\n         lệch tải khi đồng bộ NCCL all_reduce!\n    """\n    def __init__(self, samples, world_size=1, rank=0, batch_n3=4, batch_n5=2, batch_n7=1, seed=42):\n        self.samples = samples\n        self.world_size = max(1, world_size)\n        self.rank = rank\n        self.bs_map = {3: batch_n3, 5: batch_n5, 7: batch_n7}\n        self.seed = seed\n        self.epoch = 0\n        self._rank_batches = []\n        self.set_epoch(0)\n\n    def set_epoch(self, epoch):\n        self.epoch = epoch\n        rng = random.Random(self.seed + epoch * 997)\n\n        # Phân loại theo (is_synth, seq_len)\n        by_len = {}\n        for idx, s in enumerate(self.samples):\n            grp = (1 if s.get(\'is_synth\', False) else 0, s[\'seq_len\'])\n            by_len.setdefault(grp, []).append((s[\'orig_w\'], rng.random(), idx))\n\n        # Gom thành các micro-batch gần tương đồng chiều rộng cho từng nhóm\n        global_micros = {}\n        for grp, items in sorted(by_len.items()):\n            sl = grp[1]\n            items_sorted = sorted(items, key=lambda x: (((x[0] + 31) // 32) * 32, x[1]))\n            indices = [x[2] for x in items_sorted]\n            bs_base = self.bs_map.get(sl, 1)\n\n            batches_for_len = []\n            cur = []\n            for idx in indices:\n                w = self.samples[idx][\'orig_w\']\n                bs_eff = 1 if (sl >= 5 and w > 1120 and bs_base > 1) else bs_base\n                cur.append(idx)\n                if len(cur) >= bs_eff:\n                    batches_for_len.append((grp, self.samples[idx][\'orig_w\'], cur))\n                    cur = []\n            if cur:\n                if batches_for_len and len(cur) < max(1, bs_base // 2):\n                    pass\n                else:\n                    batches_for_len.append((grp, self.samples[cur[-1]][\'orig_w\'], cur))\n            global_micros[grp] = batches_for_len\n\n        # Ghép cặp micro-batches theo world_size để mọi rank có cùng (is_synth, seq_len) ở mỗi step\n        sync_steps = []\n        for grp, b_list in global_micros.items():\n            rem = len(b_list) % self.world_size\n            if rem != 0 and len(b_list) >= self.world_size:\n                b_list = b_list[:-rem]\n            for j in range(0, len(b_list), self.world_size):\n                tuple_batches = [b_list[j + r][2] for r in range(self.world_size)]\n                sync_steps.append(tuple_batches)\n\n        rng.shuffle(sync_steps)\n        self._rank_batches = [step[self.rank] for step in sync_steps]\n\n    def __iter__(self):\n        return iter(self._rank_batches)\n\n    def __len__(self):\n        return len(self._rank_batches)\n\n\n# ==============================================================================\n# 6. EMA SHADOW & CHECKPOINT MANAGER (TUYỆT ĐỐI CHỈ LƯU 2 FILE: BEST VÀ LAST)\n# ==============================================================================\nclass AdaptiveEMA:\n    """\n    Exponential Moving Average nhẹ trên GPU với Adaptive Warmup Decay:\n      decay(step) = min(max_decay, (1 + step) / (10 + step))\n    Giúp trọng số thích ứng nhanh ở 3 epoch đầu rồi làm mượt cực tốt ở cuối kỳ.\n    """\n    def __init__(self, model, max_decay=0.995):\n        self.max_decay = max_decay\n        self.num_updates = 0\n        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        self._backup = None\n\n    @torch.no_grad()\n    def update(self, model):\n        self.num_updates += 1\n        d = min(self.max_decay, (1.0 + self.num_updates) / (10.0 + self.num_updates))\n        for k, v in model.state_dict().items():\n            if v.dtype.is_floating_point:\n                self.shadow[k].mul_(d).add_(v.detach(), alpha=(1.0 - d))\n            else:\n                self.shadow[k].copy_(v)\n\n    def apply_shadow(self, model):\n        self._backup = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        model.load_state_dict(self.shadow, strict=True)\n\n    def restore(self, model):\n        if self._backup is not None:\n            model.load_state_dict(self._backup, strict=True)\n            self._backup = None\n\n    def state_dict(self):\n        return {\'num_updates\': self.num_updates, \'shadow\': self.shadow}\n\n    def load_state_dict(self, state, device):\n        if not state:\n            return\n        self.num_updates = int(state.get(\'num_updates\', 0))\n        loaded_sh = state.get(\'shadow\', {})\n        for k in self.shadow:\n            if k in loaded_sh and self.shadow[k].shape == loaded_sh[k].shape:\n                self.shadow[k].copy_(loaded_sh[k].to(device))\n\n\nclass TwoFileCheckpointManager:\n    """\n    Chính sách lưu trữ tối giản tuyệt đối theo yêu cầu đồ án:\n    CHỈ DUY TRÌ ĐÚNG 2 FILE TRONG `output_dir`:\n      1. model_tps_finetuned_best.pt : Lưu trọng số tốt nhất trên tập Val (EMA weights).\n      2. model_tps_finetuned_last.pt : Lưu trọng số Epoch gần nhất KÈM trạng thái Resume đầy đủ.\n    Tuyệt đối KHÔNG sinh thêm file thứ 3 hoặc các file rời từng epoch!\n    """\n    def __init__(self, output_dir, is_rank0=True):\n        self.output_dir = output_dir\n        self.is_rank0 = is_rank0\n        self.best_score = float(\'inf\')\n        self.best_epoch = 0\n        os.makedirs(output_dir, exist_ok=True)\n        self.best_path = os.path.join(output_dir, "model_tps_finetuned_best.pt")\n        self.last_path = os.path.join(output_dir, "model_tps_finetuned_last.pt")\n\n    def _atomic_save(self, obj, dst_path):\n        tmp_path = dst_path + ".tmp"\n        torch.save(obj, tmp_path)\n        if os.path.exists(dst_path):\n            os.remove(dst_path)\n        os.rename(tmp_path, dst_path)\n\n    def save_epoch(self, epoch, base_model, ema, optimizer, scaler, score, history, is_best):\n        if not self.is_rank0:\n            return\n\n        if is_best:\n            self.best_score = float(score)\n            self.best_epoch = int(epoch)\n            best_payload = {\n                \'epoch\': int(epoch),\n                \'best_score\': float(score),\n                \'model_state\': {k: v.detach().cpu() for k, v in ema.shadow.items()},\n                \'raw_model_state\': {k: v.detach().cpu() for k, v in base_model.state_dict().items()},\n                \'history\': history,\n            }\n            self._atomic_save(best_payload, self.best_path)\n            print(f"🌟 [KỶ LỤC MỚI]: Cập nhật {os.path.basename(self.best_path)} tại Epoch {epoch:02d} (Composite Score = {score:.4f})!", flush=True)\n\n        # Lưu gộp Epoch gần nhất + trạng thái Resume vào duy nhất file last.pt\n        last_payload = {\n            \'epoch\': int(epoch),\n            \'model_state\': {k: v.detach().cpu() for k, v in base_model.state_dict().items()},\n            \'ema_state\': {\n                \'num_updates\': ema.num_updates,\n                \'shadow\': {k: v.detach().cpu() for k, v in ema.shadow.items()},\n            },\n            \'optimizer_state\': optimizer.state_dict() if optimizer is not None else None,\n            \'scaler_state\': scaler.state_dict() if scaler is not None else None,\n            \'best_score\': float(self.best_score),\n            \'best_epoch\': int(self.best_epoch),\n            \'history\': history,\n        }\n        self._atomic_save(last_payload, self.last_path)\n\n    def try_resume(self, base_model, ema, optimizer, scaler, map_location):\n        if not os.path.exists(self.last_path):\n            return 0, []\n        ckpt = torch.load(self.last_path, map_location=map_location, weights_only=False)\n        last_ep = int(ckpt.get(\'epoch\', 0))\n        self.best_score = float(ckpt.get(\'best_score\', float(\'inf\')))\n        self.best_epoch = int(ckpt.get(\'best_epoch\', 0))\n        history = ckpt.get(\'history\', [])\n\n        if \'model_state\' in ckpt:\n            base_model.load_state_dict(ckpt[\'model_state\'], strict=True)\n        if \'ema_state\' in ckpt and ckpt[\'ema_state\'] is not None:\n            ema.load_state_dict(ckpt[\'ema_state\'], map_location)\n        if optimizer is not None and ckpt.get(\'optimizer_state\') is not None:\n            optimizer.load_state_dict(ckpt[\'optimizer_state\'])\n        if scaler is not None and ckpt.get(\'scaler_state\') is not None:\n            scaler.load_state_dict(ckpt[\'scaler_state\'])\n        return last_ep, history\n\n\n# ==============================================================================\n# 7. VALIDATION ĐỐI CHỨNG & CHỌN CHECKPOINT (KHÔNG BIAS, THUẦN FP32)\n# ==============================================================================\n@torch.no_grad()\ndef run_validation_suite(model, val_loader, device, downscale_factor=2, is_ddp=False, evaluate_m0=False):\n    model.eval()\n    keys_sum = [\'psnr\', \'ssim\', \'val_l1\', \'ie_mae\', \'pwcr\', \'rgd\', \'ghost\', \'dark_ghost\',\n                \'wcd\', \'wcd_legacy\', \'wcd_dog\', \'wcd_dog_legacy\', \'semd\',\n                \'recall\', \'precision\', \'f1\', \'density_ratio\', \'skel_recall\', \'tip_recall\']\n    acc_main = {k: 0.0 for k in keys_sum}\n    acc_main[\'count\'] = 0.0\n    acc_m0 = {k: 0.0 for k in keys_sum}\n    acc_m0[\'count\'] = 0.0\n\n    brk_keys = [\'count\', \'psnr\', \'ssim\', \'wcd\', \'wcd_legacy\', \'wcd_dog\', \'wcd_dog_legacy\',\n                \'semd\', \'skel_recall\', \'tip_recall\', \'f1\', \'ghost\', \'density_ratio\']\n    breakdown = {n: {k: 0.0 for k in brk_keys} for n in (3, 5, 7)}\n\n    for batch in val_loader:\n        h_pad, w_pad = batch[\'h_pad\'], batch[\'w_pad\']\n        x0 = batch[\'x0\'].to(device, non_blocking=True)\n        x1 = batch[\'x1\'].to(device, non_blocking=True)\n        xk = batch[\'xk\'].to(device, non_blocking=True)\n        t_val = batch[\'t_val\'].to(device, non_blocking=True).view(-1, 1, 1, 1)\n        c0 = batch[\'c0\'].to(device, non_blocking=True)\n        th01 = batch[\'th01\'].to(device, non_blocking=True)\n        c1 = batch[\'c1\'].to(device, non_blocking=True)\n        th10 = batch[\'th10\'].to(device, non_blocking=True)\n        valid = batch[\'valid\'].to(device, non_blocking=True)\n        orig_hw = batch[\'orig_hw\']\n        seq_lens = batch[\'seq_len\']\n\n        cf01 = tps_flow_from_theta_batch(th01, c0, valid, orig_hw, h_pad, w_pad, device, downscale_factor=downscale_factor)\n        cf10 = tps_flow_from_theta_batch(th10, c1, valid, orig_hw, h_pad, w_pad, device, downscale_factor=downscale_factor)\n\n        # Validation chạy hoàn toàn trên FP32 thuần túy (loại bỏ nhiễu lượng tử hóa nửa độ chính xác)\n        with torch.amp.autocast(\'cuda\', enabled=False):\n            pred_it, coarse_warp, *_ = model(x0.float(), x1.float(), t_val.float(), cf01.float(), cf10.float())\n\n        b = x0.shape[0]\n        for i in range(b):\n            hi, wi = int(orig_hw[i, 0].item()), int(orig_hw[i, 1].item())\n            sl = int(seq_lens[i].item())\n            gt_i = xk[i:i + 1, :, :hi, :wi]\n            pr_i = pred_it[i:i + 1, :, :hi, :wi]\n\n            m = evaluate_metrics_dual_regime(pr_i, gt_i)\n            acc_main[\'count\'] += 1.0\n            for k in keys_sum:\n                acc_main[k] += float(m[k].item())\n\n            if sl in breakdown:\n                breakdown[sl][\'count\'] += 1.0\n                for bk in brk_keys:\n                    if bk != \'count\':\n                        breakdown[sl][bk] += float(m[bk].item())\n\n            if evaluate_m0:\n                m0_pr = coarse_warp[i:i + 1, :, :hi, :wi]\n                mm0 = evaluate_metrics_dual_regime(m0_pr, gt_i)\n                acc_m0[\'count\'] += 1.0\n                for k in keys_sum:\n                    acc_m0[k] += float(mm0[k].item())\n\n    def _finalize(acc, brk=None):\n        if is_ddp:\n            vec = torch.tensor([acc[\'count\']] + [acc[k] for k in keys_sum], dtype=torch.float64, device=device)\n            dist.all_reduce(vec, op=dist.ReduceOp.SUM)\n            cnt = max(1.0, float(vec[0].item()))\n            out = {k: float(vec[idx + 1].item()) / cnt for idx, k in enumerate(keys_sum)}\n            out[\'total_samples\'] = int(cnt)\n        else:\n            cnt = max(1.0, acc[\'count\'])\n            out = {k: acc[k] / cnt for k in keys_sum}\n            out[\'total_samples\'] = int(cnt)\n\n        if brk is not None:\n            brk_out = {}\n            for sl in (3, 5, 7):\n                if is_ddp:\n                    bvec = torch.tensor([brk[sl][k] for k in brk_keys], dtype=torch.float64, device=device)\n                    dist.all_reduce(bvec, op=dist.ReduceOp.SUM)\n                    bcnt = max(1.0, float(bvec[0].item()))\n                    brk_out[sl] = {k: float(bvec[idx].item()) / bcnt for idx, k in enumerate(brk_keys) if k != \'count\'}\n                    brk_out[sl][\'count\'] = int(bvec[0].item())\n                else:\n                    bcnt = max(1.0, brk[sl][\'count\'])\n                    brk_out[sl] = {k: brk[sl][k] / bcnt for k in brk_keys if k != \'count\'}\n                    brk_out[sl][\'count\'] = int(brk[sl][\'count\'])\n            out[\'breakdown\'] = brk_out\n        return out\n\n    res_main = _finalize(acc_main, breakdown)\n    if evaluate_m0:\n        return res_main, _finalize(acc_m0)\n    return res_main\n\n\n# ==============================================================================\n# 8. ĐỌC TẬP BENCHMARK ĐỘC LẬP SONG SONG TRÊN 2 GPU\n# ==============================================================================\n@torch.no_grad()\ndef evaluate_per_sample_benchmark(model, bench_loader, device, downscale_factor=2):\n    """Chạy mô hình trên dataloader Benchmark và trả về dict keyed by (clip_id, frame_k)."""\n    model.eval()\n    results = {}\n    for batch in bench_loader:\n        h_pad, w_pad = batch[\'h_pad\'], batch[\'w_pad\']\n        x0 = batch[\'x0\'].to(device, non_blocking=True)\n        x1 = batch[\'x1\'].to(device, non_blocking=True)\n        xk = batch[\'xk\'].to(device, non_blocking=True)\n        t_val = batch[\'t_val\'].to(device, non_blocking=True).view(-1, 1, 1, 1)\n        c0 = batch[\'c0\'].to(device, non_blocking=True)\n        th01 = batch[\'th01\'].to(device, non_blocking=True)\n        c1 = batch[\'c1\'].to(device, non_blocking=True)\n        th10 = batch[\'th10\'].to(device, non_blocking=True)\n        valid = batch[\'valid\'].to(device, non_blocking=True)\n        orig_hw = batch[\'orig_hw\']\n\n        cf01 = tps_flow_from_theta_batch(th01, c0, valid, orig_hw, h_pad, w_pad, device, downscale_factor=downscale_factor)\n        cf10 = tps_flow_from_theta_batch(th10, c1, valid, orig_hw, h_pad, w_pad, device, downscale_factor=downscale_factor)\n\n        if device.type == \'cuda\':\n            torch.cuda.synchronize(device)\n        t_start = time.time()\n        with torch.amp.autocast(\'cuda\', enabled=False):\n            pred_it, coarse_warp, *_ = model(x0.float(), x1.float(), t_val.float(), cf01.float(), cf10.float())\n        if device.type == \'cuda\':\n            torch.cuda.synchronize(device)\n        lat_ms = ((time.time() - t_start) * 1000.0) / max(1, x0.shape[0])\n\n        for i in range(x0.shape[0]):\n            hi, wi = int(orig_hw[i, 0].item()), int(orig_hw[i, 1].item())\n            gt_i = xk[i:i + 1, :, :hi, :wi]\n            pr_i = pred_it[i:i + 1, :, :hi, :wi]\n            m0_i = coarse_warp[i:i + 1, :, :hi, :wi]\n\n            m_pred = evaluate_metrics_dual_regime(pr_i, gt_i)\n            m_coarse = evaluate_metrics_dual_regime(m0_i, gt_i)\n\n            rec = {\n                \'clip_id\': batch[\'clip_id\'][i],\n                \'scene_id\': batch[\'scene_id\'][i],\n                \'seq_len\': int(batch[\'seq_len\'][i].item()),\n                \'frame_k\': int(batch[\'frame_k\'][i]),\n                \'t_val\': float(batch[\'t_val\'][i].item()),\n                \'mean_disp\': float(batch[\'mean_disp\'][i]),\n                \'latency_ms\': float(lat_ms),\n            }\n            for k, v in m_pred.items():\n                rec[k] = float(v.item())\n            for k, v in m_coarse.items():\n                rec[f"m0_{k}"] = float(v.item())\n            results[(rec[\'clip_id\'], rec[\'frame_k\'])] = rec\n    return results\n\n\n# ==============================================================================\n# 9. CHƯƠNG TRÌNH HUẤN LUYỆN CHÍNH (DDP WORKER)\n# ==============================================================================\ndef _apply_paired_spatial_aug_gpu(x0, x1, xk_tensor, cf01, cf10, orig_hw, is_sequence):\n    """Áp dụng Paired H-Flip (p=0.50) và V-Flip (p=0.25) đồng bộ ảnh + TPS flow bên trong orig_hw."""\n    b = x0.shape[0]\n    for i in range(b):\n        hi, wi = int(orig_hw[i, 0].item()), int(orig_hw[i, 1].item())\n        if random.random() < 0.50:\n            x0[i:i + 1, :, :hi, :wi] = torch.flip(x0[i:i + 1, :, :hi, :wi], dims=[-1])\n            x1[i:i + 1, :, :hi, :wi] = torch.flip(x1[i:i + 1, :, :hi, :wi], dims=[-1])\n            if is_sequence:\n                xk_tensor[i:i + 1, :, :, :hi, :wi] = torch.flip(xk_tensor[i:i + 1, :, :, :hi, :wi], dims=[-1])\n            else:\n                xk_tensor[i:i + 1, :, :hi, :wi] = torch.flip(xk_tensor[i:i + 1, :, :hi, :wi], dims=[-1])\n            cf01[i:i + 1, :, :hi, :wi] = torch.flip(cf01[i:i + 1, :, :hi, :wi], dims=[-1])\n            cf01[i, 0, :hi, :wi].neg_()\n            cf10[i:i + 1, :, :hi, :wi] = torch.flip(cf10[i:i + 1, :, :hi, :wi], dims=[-1])\n            cf10[i, 0, :hi, :wi].neg_()\n\n        if random.random() < 0.25:\n            x0[i:i + 1, :, :hi, :wi] = torch.flip(x0[i:i + 1, :, :hi, :wi], dims=[-2])\n            x1[i:i + 1, :, :hi, :wi] = torch.flip(x1[i:i + 1, :, :hi, :wi], dims=[-2])\n            if is_sequence:\n                xk_tensor[i:i + 1, :, :, :hi, :wi] = torch.flip(xk_tensor[i:i + 1, :, :, :hi, :wi], dims=[-2])\n            else:\n                xk_tensor[i:i + 1, :, :hi, :wi] = torch.flip(xk_tensor[i:i + 1, :, :hi, :wi], dims=[-2])\n            cf01[i:i + 1, :, :hi, :wi] = torch.flip(cf01[i:i + 1, :, :hi, :wi], dims=[-2])\n            cf01[i, 1, :hi, :wi].neg_()\n            cf10[i:i + 1, :, :hi, :wi] = torch.flip(cf10[i:i + 1, :, :hi, :wi], dims=[-2])\n            cf10[i, 1, :hi, :wi].neg_()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="DDP Worker Fine-tune TPS-Inbetween (Pha 1: Temporal + Ghost + Scale)")\n    parser.add_argument(\'--train_csv\', type=str, required=True)\n    parser.add_argument(\'--val_csv\', type=str, required=True)\n    parser.add_argument(\'--train_clips_dir\', type=str, required=True, help="Thư mục train/")\n    parser.add_argument(\'--val_clips_dir\', type=str, required=True, help="Thư mục val/")\n    parser.add_argument(\'--tps_cache\', type=str, required=True)\n    parser.add_argument(\'--m1_checkpoint\', type=str, required=True, help="Checkpoint gốc M1 dùng đo baseline đối chuẩn")\n    parser.add_argument(\'--init_checkpoint\', type=str, default="", help="Checkpoint warm-start tùy chọn (vd: Seq2 Ep15)")\n    parser.add_argument(\'--output_dir\', type=str, default="/kaggle/working/checkpoints_tps_finetune")\n    parser.add_argument(\'--train_mode\', type=str, default="LONG", choices=["SHORT", "MEDIUM", "LONG"])\n    parser.add_argument(\'--epochs\', type=int, default=5, help="Số epoch cho Pha 2 Cleanliness Polish (mặc định: 5)")\n    parser.add_argument(\'--grad_accum\', type=int, default=2)\n    parser.add_argument(\'--max_chunk_frames\', type=int, default=3, help="Số frame tối đa mỗi forward micro-chunk")\n    parser.add_argument(\'--lr_backbone\', type=float, default=1.5e-5, help="LR Backbone thấp để bảo toàn nét (Pha 2: 1.5e-5)")\n    parser.add_argument(\'--lr_unet\', type=float, default=3.5e-5, help="LR Unet thấp để tinh chỉnh dọn nền (Pha 2: 3.5e-5)")\n    parser.add_argument(\'--ema_decay\', type=float, default=0.995)\n    parser.add_argument(\'--synth_count\', type=int, default=120, help="Số chuỗi hình học tổng hợp mỗi epoch ở LONG mode")\n    parser.add_argument(\'--w_ghost\', type=float, default=1.10, help="Trọng số quét sạch nền Pha 2 (mặc định: 1.10)")\n    parser.add_argument(\'--w_temp\', type=float, default=0.60, help="Trọng số nhất quán thời gian (mặc định: 0.60)")\n    parser.add_argument(\'--num_workers\', type=int, default=2)\n    parser.add_argument(\'--ram_limit_gb\', type=float, default=23.5)\n    parser.add_argument(\'--seed\', type=int, default=42)\n    parser.add_argument(\'--dry_run\', action=\'store_true\', help="Chạy kiểm tra 2 bước khởi động nhanh rồi thoát")\n    args = parser.parse_args()\n\n    local_rank = int(os.environ.get("LOCAL_RANK", 0))\n    world_size = int(os.environ.get("WORLD_SIZE", 1))\n    is_ddp = world_size > 1\n\n    if is_ddp:\n        if torch.cuda.is_available():\n            torch.cuda.set_device(local_rank)\n            device = torch.device(f\'cuda:{local_rank}\')\n            backend = \'nccl\' if dist.is_nccl_available() else \'gloo\'\n        else:\n            device = torch.device(\'cpu\')\n            backend = \'gloo\'\n        dist.init_process_group(backend=backend, init_method=os.environ.get(\'DDP_INIT_METHOD\', \'env://\'))\n    else:\n        device = torch.device(\'cuda:0\' if torch.cuda.is_available() else \'cpu\')\n\n    is_rank0 = (local_rank == 0)\n    random.seed(args.seed + local_rank)\n    np.random.seed(args.seed + local_rank)\n    torch.manual_seed(args.seed + local_rank)\n\n    mode = args.train_mode.upper()\n    tps_downscale = 2 if mode == "LONG" else 4\n\n    if is_rank0:\n        init_tag = os.path.basename(args.init_checkpoint) if (args.init_checkpoint and os.path.exists(args.init_checkpoint)) else os.path.basename(args.m1_checkpoint)\n        print("=" * 84)\n        print(f"🚀 KHỞI ĐỘNG HUẤN LUYỆN PHA 1 DDP TPS-INBETWEEN (MODE={mode} | INIT={init_tag})")\n        print(f"   World Size = {world_size} GPU(s) | Device = {device} | TPS Downscale = 1/{tps_downscale}")\n        print(f"   Epochs = {args.epochs} | Synth = {args.synth_count} | w_ghost={args.w_ghost} | w_temp={args.w_temp}")\n        print(f"   Checkpoint Output = {args.output_dir} (Chỉ lưu 2 file: best.pt & last.pt)")\n        print("=" * 84, flush=True)\n\n    ckpt_mgr = TwoFileCheckpointManager(args.output_dir, is_rank0=is_rank0)\n    base_model = TPSInbetweenOriginal().to(device)\n    load_tps_weights(base_model, args.m1_checkpoint, device=device, prefer_ema=False, is_rank0=is_rank0)\n\n    # Dataset & Loader\n    train_ds = SakugaSequenceDataset(\n        args.train_csv, args.train_clips_dir, args.tps_cache,\n        train_mode=mode, is_train=True, seed=args.seed, synth_count=args.synth_count\n    )\n    val_ds = SakugaSequenceDataset(\n        args.val_csv, args.val_clips_dir, args.tps_cache,\n        train_mode="VAL", is_train=False, seed=args.seed, synth_count=0\n    )\n\n    if mode == "LONG":\n        train_sampler = DDPLengthWidthBatchSampler(\n            train_ds.samples, world_size=world_size, rank=local_rank,\n            batch_n3=4, batch_n5=2, batch_n7=1, seed=args.seed\n        )\n        train_loader = DataLoader(\n            train_ds, batch_sampler=train_sampler, collate_fn=pad_collate_fn,\n            num_workers=args.num_workers, pin_memory=(device.type == \'cuda\'),\n            persistent_workers=(args.num_workers > 0)\n        )\n    else:\n        bs_single = 12 if mode == "SHORT" else 8\n        train_sampler = DistributedSampler(train_ds, shuffle=True) if is_ddp else None\n        train_loader = DataLoader(\n            train_ds, batch_size=bs_single, shuffle=(train_sampler is None), sampler=train_sampler,\n            collate_fn=pad_collate_fn, num_workers=args.num_workers,\n            pin_memory=(device.type == \'cuda\'), persistent_workers=(args.num_workers > 0), drop_last=True\n        )\n\n    val_sampler = DistributedSampler(val_ds, shuffle=False) if is_ddp else None\n    val_loader = DataLoader(\n        val_ds, batch_size=4, shuffle=False, sampler=val_sampler,\n        collate_fn=pad_collate_fn, num_workers=args.num_workers,\n        pin_memory=(device.type == \'cuda\'), persistent_workers=(args.num_workers > 0), drop_last=False\n    )\n\n    if is_rank0:\n        print(f"[✓] Dữ liệu sẵn sàng: Train Items = {len(train_ds):,} ({len(train_loader):,} batches/GPU) | Val Samples = {len(val_ds):,}", flush=True)\n\n    # Đo Baseline M0 & M1 tại Epoch 0 trên đúng model_latest.pt trước khi nạp warm-start\n    baseline_json_path = os.path.join(os.path.dirname(args.output_dir), "val_baseline_m0_m1.json")\n    has_resume = os.path.exists(ckpt_mgr.last_path)\n    need_baseline = (not has_resume and not args.dry_run and not os.path.exists(baseline_json_path))\n    if is_ddp:\n        base_flag = torch.tensor([1 if need_baseline else 0], dtype=torch.int32, device=device)\n        dist.broadcast(base_flag, src=0)\n        need_baseline = bool(base_flag.item())\n\n    m0_val_cache, m1_val_cache = None, None\n    if need_baseline:\n        if is_rank0:\n            print("\\n🔬 ĐANG ĐO BASELINE EPOCH 0 TRÊN TẬP VAL (M0 COARSE TPS & M1 ORIGINAL)...", flush=True)\n        m1_val, m0_val = run_validation_suite(base_model, val_loader, device, downscale_factor=tps_downscale, is_ddp=is_ddp, evaluate_m0=True)\n        m0_val_cache, m1_val_cache = m0_val, m1_val\n        if is_rank0:\n            with open(baseline_json_path, \'w\', encoding=\'utf-8\') as f:\n                json.dump({\'M0_Coarse\': m0_val, \'M1_Original\': m1_val}, f, indent=2, ensure_ascii=False)\n            print(f"📊 M0 (Coarse TPS): PSNR={m0_val[\'psnr\']:.2f}dB | SSIM={m0_val[\'ssim\']:.4f} | WCD_DoG={m0_val[\'wcd_dog\']:.4f} | Ghost={m0_val[\'ghost\']:.0f}", flush=True)\n            print(f"📊 M1 (Orig ML240): PSNR={m1_val[\'psnr\']:.2f}dB | SSIM={m1_val[\'ssim\']:.4f} | WCD_DoG={m1_val[\'wcd_dog\']:.4f} | Ghost={m1_val[\'ghost\']:.0f}\\n", flush=True)\n        if is_ddp:\n            dist.barrier()\n    elif os.path.exists(baseline_json_path):\n        try:\n            with open(baseline_json_path, \'r\', encoding=\'utf-8\') as f:\n                b_all = json.load(f)\n                m0_val_cache = b_all.get(\'M0_Coarse\', {})\n                m1_val_cache = b_all.get(\'M1_Original\', {})\n        except Exception:\n            m0_val_cache, m1_val_cache = None, None\n\n    # Nếu có --init_checkpoint (vd: Seq2 Ep15), nạp sau khi đã đo xong M1 Baseline Epoch 0\n    if args.init_checkpoint and os.path.exists(args.init_checkpoint):\n        if os.path.abspath(args.init_checkpoint) != os.path.abspath(args.m1_checkpoint):\n            load_tps_weights(base_model, args.init_checkpoint, device=device, prefer_ema=True, is_rank0=is_rank0)\n\n    if mode == "SHORT":\n        for p in base_model.IFNet.block0.parameters():\n            p.requires_grad = False\n\n    # Optimizer, EMA, Loss & Scaler\n    ema = AdaptiveEMA(base_model, max_decay=args.ema_decay)\n    optimizer = torch.optim.AdamW([\n        {\'params\': [p for p in base_model.encoder.parameters() if p.requires_grad], \'lr\': args.lr_backbone, \'weight_decay\': 1e-4},\n        {\'params\': [p for p in base_model.IFNet.parameters() if p.requires_grad], \'lr\': args.lr_backbone, \'weight_decay\': 1e-4},\n        {\'params\': [p for p in base_model.unet.parameters() if p.requires_grad], \'lr\': args.lr_unet, \'weight_decay\': 1e-4},\n    ])\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=(device.type == \'cuda\'))\n    if is_ddp:\n        if is_rank0:\n            criterion = AdaptiveLineartLoss(train_mode=mode, is_rank0=True, w_ghost=args.w_ghost, w_temp=args.w_temp).to(device)\n        dist.barrier()\n        if not is_rank0:\n            criterion = AdaptiveLineartLoss(train_mode=mode, is_rank0=False, w_ghost=args.w_ghost, w_temp=args.w_temp).to(device)\n        dist.barrier()\n    else:\n        criterion = AdaptiveLineartLoss(train_mode=mode, is_rank0=True, w_ghost=args.w_ghost, w_temp=args.w_temp).to(device)\n\n    start_epoch = 1\n    history = []\n    if not args.dry_run:\n        last_ep, hist = ckpt_mgr.try_resume(base_model, ema, optimizer, scaler, map_location=device)\n        if last_ep > 0:\n            start_epoch = last_ep + 1\n            history = hist\n            if is_rank0:\n                print(f"[♻️ AUTO-RESUME] Khôi phục thành công Epoch {last_ep:02d}. Tiếp tục từ Epoch {start_epoch:02d} (Best Ep={ckpt_mgr.best_epoch:02d}).", flush=True)\n\n    # Mốc chuẩn M1 Epoch 0 để chuẩn hóa Composite Score\n    m1_ref = m1_val_cache or {}\n    m0_ref = m0_val_cache or {}\n    base_wcd_dog = float(m1_ref.get(\'wcd_dog\', 1.4620))\n    base_semd = float(m1_ref.get(\'semd\', 0.0131))\n    base_skel = float(m1_ref.get(\'skel_recall\', 0.8164))\n    base_tip = float(m1_ref.get(\'tip_recall\', 0.8204))\n    base_ghost = float(m1_ref.get(\'ghost\', 1409.0))\n    m0_ghost_ref = float(m0_ref.get(\'ghost\', 17975.0))\n    best_ghost_gate = 9500.0  # Mốc khởi điểm Pha 2 (kế thừa Pha 1 Ep14 ~9185 px), Gate sẽ tự động siết chặt dần khi Ghost giảm\n    if is_rank0:\n        print(f"⚖️ MỐC ĐỐI CHUẨN EPOCH 0 (M1): WCD_DoG={base_wcd_dog:.4f} | SkelRec={base_skel*100:.1f}% | Ghost={base_ghost:.0f} | GhostGateStart={best_ghost_gate:.0f}", flush=True)\n\n    if is_ddp:\n        ddp_kwargs = {\'device_ids\': [local_rank], \'output_device\': local_rank} if device.type == \'cuda\' else {}\n        ddp_model = DDP(base_model, find_unused_parameters=(mode == "SHORT"), **ddp_kwargs)\n    else:\n        ddp_model = base_model\n\n    history_csv_path = os.path.join(os.path.dirname(args.output_dir), "training_history_finetune.csv")\n    watchdog_paused = False\n    t_total_start = time.time()\n\n    for epoch in range(start_epoch, args.epochs + 1):\n        t_ep0 = time.time()\n        if hasattr(train_sampler, \'set_epoch\'):\n            train_sampler.set_epoch(epoch)\n\n        # Lịch Cosine Warmup 2 epochs\n        if epoch == 1:\n            lr_scale = 0.35\n        elif epoch == 2:\n            lr_scale = 0.70\n        elif epoch == 3:\n            lr_scale = 1.00\n        else:\n            prog = float(epoch - 3) / float(max(1, args.epochs - 3))\n            lr_scale = max(0.10, 0.5 * (1.0 + math.cos(math.pi * prog)))\n\n        optimizer.param_groups[0][\'lr\'] = args.lr_backbone * lr_scale\n        optimizer.param_groups[1][\'lr\'] = args.lr_backbone * lr_scale\n        optimizer.param_groups[2][\'lr\'] = args.lr_unet * lr_scale\n\n        ddp_model.train()\n        optimizer.zero_grad(set_to_none=True)\n        acc_loss = 0.0\n        acc_frames = 0\n        num_batches = len(train_loader)\n        grad_accum = max(1, args.grad_accum)\n\n        for b_idx, batch in enumerate(train_loader):\n            h_pad, w_pad = batch[\'h_pad\'], batch[\'w_pad\']\n            is_direct = bool(batch.get(\'is_direct_t_flows\', False))\n            x0 = batch[\'x0\'].to(device, non_blocking=True)\n            x1 = batch[\'x1\'].to(device, non_blocking=True)\n            orig_hw = batch[\'orig_hw\'].to(device, non_blocking=True)\n\n            if is_direct:\n                # Chuỗi hình học tổng hợp dùng trực tiếp Lagrangian Direct TPS (cf0t, cf1t)\n                xk_all = batch[\'xk_all\'].to(device, non_blocking=True)\n                t_vals = batch[\'t_vals\'].to(device, non_blocking=True)\n                bsz, m_steps = xk_all.shape[0], xk_all.shape[1]\n                c_t0_f = batch[\'c_t0_all\'].to(device, non_blocking=True).reshape(bsz * m_steps, -1, 2)\n                th_t0_f = batch[\'th_t0_all\'].to(device, non_blocking=True).reshape(bsz * m_steps, -1, 2)\n                c_t1_f = batch[\'c_t1_all\'].to(device, non_blocking=True).reshape(bsz * m_steps, -1, 2)\n                th_t1_f = batch[\'th_t1_all\'].to(device, non_blocking=True).reshape(bsz * m_steps, -1, 2)\n                val_f = batch[\'valid_all\'].to(device, non_blocking=True).reshape(bsz * m_steps)\n                hw_f = orig_hw.unsqueeze(1).expand(-1, m_steps, -1).reshape(bsz * m_steps, 2)\n\n                cf01_f = tps_flow_from_theta_batch(th_t0_f, c_t0_f, val_f, hw_f, h_pad, w_pad, device, downscale_factor=tps_downscale)\n                cf10_f = tps_flow_from_theta_batch(th_t1_f, c_t1_f, val_f, hw_f, h_pad, w_pad, device, downscale_factor=tps_downscale)\n\n                x0_f = x0.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 1, h_pad, w_pad)\n                x1_f = x1.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 1, h_pad, w_pad)\n                xk_f = xk_all.reshape(bsz * m_steps, 1, h_pad, w_pad)\n                t_f = t_vals.reshape(bsz * m_steps, 1, 1, 1)\n                seq_ids_f = torch.arange(bsz, device=device).unsqueeze(1).expand(-1, m_steps).reshape(-1)\n            else:\n                c0 = batch[\'c0\'].to(device, non_blocking=True)\n                th01 = batch[\'th01\'].to(device, non_blocking=True)\n                c1 = batch[\'c1\'].to(device, non_blocking=True)\n                th10 = batch[\'th10\'].to(device, non_blocking=True)\n                valid = batch[\'valid\'].to(device, non_blocking=True)\n\n                cf01 = tps_flow_from_theta_batch(th01, c0, valid, orig_hw, h_pad, w_pad, device, downscale_factor=tps_downscale)\n                cf10 = tps_flow_from_theta_batch(th10, c1, valid, orig_hw, h_pad, w_pad, device, downscale_factor=tps_downscale)\n\n                if batch[\'is_sequence\']:\n                    xk_all = batch[\'xk_all\'].to(device, non_blocking=True)\n                    t_vals = batch[\'t_vals\'].to(device, non_blocking=True)\n                    _apply_paired_spatial_aug_gpu(x0, x1, xk_all, cf01, cf10, orig_hw, is_sequence=True)\n                    bsz, m_steps = xk_all.shape[0], xk_all.shape[1]\n\n                    x0_f = x0.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 1, h_pad, w_pad)\n                    x1_f = x1.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 1, h_pad, w_pad)\n                    cf01_f = cf01.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 2, h_pad, w_pad)\n                    cf10_f = cf10.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 2, h_pad, w_pad)\n                    xk_f = xk_all.reshape(bsz * m_steps, 1, h_pad, w_pad)\n                    t_f = t_vals.reshape(bsz * m_steps, 1, 1, 1)\n                    hw_f = orig_hw.unsqueeze(1).expand(-1, m_steps, -1).reshape(bsz * m_steps, 2)\n                    seq_ids_f = torch.arange(bsz, device=device).unsqueeze(1).expand(-1, m_steps).reshape(-1)\n                else:\n                    xk_f = batch[\'xk\'].to(device, non_blocking=True)\n                    _apply_paired_spatial_aug_gpu(x0, x1, xk_f, cf01, cf10, orig_hw, is_sequence=False)\n                    t_f = batch[\'t_val\'].to(device, non_blocking=True).view(-1, 1, 1, 1)\n                    x0_f, x1_f, cf01_f, cf10_f, hw_f = x0, x1, cf01, cf10, orig_hw\n                    seq_ids_f = torch.arange(x0.shape[0], device=device)\n\n            total_f = x0_f.shape[0]\n            total_pairs = int((seq_ids_f[1:] == seq_ids_f[:-1]).sum().item()) if total_f >= 2 else 0\n            do_opt_step = ((b_idx + 1) % grad_accum == 0) or ((b_idx + 1) == num_batches)\n            chunk_sz = max(1, args.max_chunk_frames)\n\n            batch_loss_sum = 0.0\n            prev_pred_last = None\n            prev_gt_last = None\n            prev_hw_last = None\n            prev_seq_id = None\n\n            for c_start in range(0, total_f, chunk_sz):\n                c_end = min(c_start + chunk_sz, total_f)\n                is_last_chunk = (c_end == total_f)\n                need_sync = (do_opt_step and is_last_chunk)\n\n                sync_ctx = (ddp_model.no_sync() if (is_ddp and not need_sync) else contextlib.nullcontext())\n                with sync_ctx:\n                    with torch.amp.autocast(\'cuda\', enabled=(device.type == \'cuda\')):\n                        pred_c, *_ = ddp_model(\n                            x0_f[c_start:c_end], x1_f[c_start:c_end],\n                            t_f[c_start:c_end], cf01_f[c_start:c_end], cf10_f[c_start:c_end],\n                            direct_t_flows=is_direct\n                        )\n                        gt_c = xk_f[c_start:c_end]\n                        hw_c = hw_f[c_start:c_end]\n                        seq_c = seq_ids_f[c_start:c_end]\n                        l_c, d_c = criterion(pred_c, gt_c, valid_hw=hw_c)\n                        weight_scale = float(c_end - c_start) / float(total_f * grad_accum)\n                        scaled_l = l_c * weight_scale\n\n                        # Temporal Consistency Loss (2C) cho các cặp kề nhau cùng sequence\n                        if total_pairs > 0 and criterion.w_temp > 0.0:\n                            l_temp_sum = pred_c.new_zeros(())\n                            if pred_c.shape[0] >= 2:\n                                same_m = (seq_c[1:] == seq_c[:-1])\n                                if same_m.any():\n                                    l_temp_sum = l_temp_sum + criterion.compute_temporal_pair_loss(\n                                        pred_c[:-1][same_m], pred_c[1:][same_m],\n                                        gt_c[:-1][same_m], gt_c[1:][same_m],\n                                        valid_hw=hw_c[1:][same_m]\n                                    )\n                            if prev_pred_last is not None and int(seq_c[0].item()) == prev_seq_id:\n                                l_temp_sum = l_temp_sum + criterion.compute_temporal_pair_loss(\n                                    prev_pred_last, pred_c[0:1], prev_gt_last, gt_c[0:1], valid_hw=hw_c[0:1]\n                                )\n                            scaled_l = scaled_l + (l_temp_sum / float(total_pairs * grad_accum))\n\n                    scaler.scale(scaled_l).backward()\n                    batch_loss_sum += d_c[\'total_loss\'] * float(c_end - c_start)\n                    prev_pred_last = pred_c[-1:].detach()\n                    prev_gt_last = gt_c[-1:].detach()\n                    prev_hw_last = hw_c[-1:].detach()\n                    prev_seq_id = int(seq_c[-1].item())\n\n            acc_loss += batch_loss_sum\n            acc_frames += total_f\n\n            if do_opt_step:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), max_norm=1.0)\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad(set_to_none=True)\n                ema.update(base_model)\n\n            if args.dry_run and b_idx >= 1:\n                break\n\n        train_min = (time.time() - t_ep0) / 60.0\n        if args.dry_run:\n            if is_rank0:\n                print(f"[✓ DRY-RUN THÀNH CÔNG] Đã kiểm chứng {acc_frames} frames qua DDP forward/backward + L_temp + SpatialAug + EMA! Avg Loss={acc_loss/max(1,acc_frames):.4f}", flush=True)\n            break\n\n        # Kiểm định bằng trọng số EMA trên toàn bộ tập Val (tạm tráo EMA -> restore, 0 tốn VRAM)\n        t_val0 = time.time()\n        ema.apply_shadow(base_model)\n        val_m = run_validation_suite(base_model, val_loader, device, downscale_factor=tps_downscale, is_ddp=is_ddp, evaluate_m0=False)\n        ema.restore(base_model)\n        val_min = (time.time() - t_val0) / 60.0\n\n        # Thước đo Epoch-0 Normalized + Two-Tier Gating cân bằng Ghost & Density\n        dens = float(val_m[\'density_ratio\'])\n        skel = float(val_m[\'skel_recall\'])\n        tip = float(val_m[\'tip_recall\'])\n        wcd_dog = float(val_m[\'wcd_dog\'])\n        semd = float(val_m[\'semd\'])\n        ghost = float(val_m[\'ghost\'])\n\n        # Pha 2: Dynamic Monotonic Ghost Gate\n        gate_density = (0.93 <= dens <= 1.07)\n        gate_skel = (skel >= max(0.840, base_skel + 0.015))\n        gate_wcd = (wcd_dog <= base_wcd_dog * 1.02)\n        gate_ghost = (ghost < best_ghost_gate)\n        tier1_passed = bool(gate_density and gate_skel and gate_wcd and gate_ghost)\n\n        term_wcd = 0.25 * (wcd_dog / max(1e-4, base_wcd_dog))\n        term_semd = 0.08 * (semd / max(1e-4, base_semd))\n        term_skel = 0.18 * ((1.0 - skel) / max(1e-4, 1.0 - base_skel))\n        term_tip = 0.08 * ((1.0 - tip) / max(1e-4, 1.0 - base_tip))\n        term_dens = 0.16 * (abs(dens - 1.00) / 0.10)\n        term_ghost = 0.25 * (ghost / max(1500.0, base_ghost * 2.0))\n        s_val = term_wcd + term_semd + term_skel + term_tip + term_dens + term_ghost\n\n        if tier1_passed:\n            comp_score = s_val\n            best_ghost_gate = ghost  # Cập nhật mốc Ghost kỷ lục mới để siết chặt các epoch sau!\n        else:\n            over_dens = max(0.0, dens - 1.07)\n            under_dens = max(0.0, 0.93 - dens)\n            over_ghost = max(0.0, (ghost - best_ghost_gate) / 1000.0)\n            comp_score = s_val + 10.0 + 5.0 * over_dens + 3.0 * under_dens + 1.0 * over_ghost\n\n        is_best = comp_score < ckpt_mgr.best_score\n\n        if is_rank0:\n            row = {\n                \'epoch\': epoch,\n                \'lr_scale\': round(lr_scale, 4),\n                \'train_loss\': acc_loss / max(1, acc_frames),\n                \'val_l1\': val_m[\'val_l1\'],\n                \'psnr\': val_m[\'psnr\'],\n                \'ssim\': val_m[\'ssim\'],\n                \'wcd\': val_m[\'wcd\'],\n                \'wcd_legacy\': val_m[\'wcd_legacy\'],\n                \'wcd_dog\': val_m[\'wcd_dog\'],\n                \'wcd_dog_legacy\': val_m[\'wcd_dog_legacy\'],\n                \'semd\': val_m[\'semd\'],\n                \'skel_recall\': val_m[\'skel_recall\'],\n                \'tip_recall\': val_m[\'tip_recall\'],\n                \'recall\': val_m[\'recall\'],\n                \'precision\': val_m[\'precision\'],\n                \'f1\': val_m[\'f1\'],\n                \'density_ratio\': val_m[\'density_ratio\'],\n                \'pwcr\': val_m[\'pwcr\'],\n                \'rgd\': val_m[\'rgd\'],\n                \'ghost\': val_m[\'ghost\'],\n                \'s_val_norm\': s_val,\n                \'score\': comp_score,\n                \'gate_passed\': tier1_passed,\n                \'is_best\': is_best,\n                \'train_time_min\': train_min,\n                \'val_time_min\': val_min,\n            }\n            for sl in (3, 5, 7):\n                if sl in val_m.get(\'breakdown\', {}):\n                    row[f\'psnr_N{sl}\'] = val_m[\'breakdown\'][sl][\'psnr\']\n                    row[f\'wcd_N{sl}\'] = val_m[\'breakdown\'][sl][\'wcd\']\n                    row[f\'wcd_dog_N{sl}\'] = val_m[\'breakdown\'][sl][\'wcd_dog\']\n                    row[f\'skel_recall_N{sl}\'] = val_m[\'breakdown\'][sl][\'skel_recall\']\n                    row[f\'tip_recall_N{sl}\'] = val_m[\'breakdown\'][sl][\'tip_recall\']\n                    row[f\'ghost_N{sl}\'] = val_m[\'breakdown\'][sl][\'ghost\']\n                    row[f\'density_N{sl}\'] = val_m[\'breakdown\'][sl][\'density_ratio\']\n\n            history.append(row)\n            pd.DataFrame(history).to_csv(history_csv_path, index=False)\n            ckpt_mgr.save_epoch(epoch, base_model, ema, optimizer, scaler, comp_score, history, is_best=is_best)\n\n            gate_badge = "✅ PASS" if tier1_passed else "⚠️ GATE_BREACH"\n            print(\n                f"📌 EP {epoch:02d}/{args.epochs:02d} [{mode}] | Train={train_min:.2f}m | Val={val_min:.2f}m | "\n                f"Loss={row[\'train_loss\']:.4f} | PSNR={val_m[\'psnr\']:.2f}dB | "\n                f"WCD_DoG={val_m[\'wcd_dog\']:.4f} | SkelRec={val_m[\'skel_recall\']*100:.1f}% | "\n                f"Dens={dens:.3f} | Ghost={ghost:.0f} | Gate={gate_badge} | Score={comp_score:.4f}",\n                flush=True\n            )\n\n        trim_memory()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n\n        stop_flag = torch.tensor([0], dtype=torch.int32, device=device)\n        if is_rank0:\n            try:\n                import psutil\n                mem_gb = psutil.virtual_memory().used / (1024 ** 3)\n                if mem_gb >= args.ram_limit_gb:\n                    print(f"\\n⚠️ [RAM WATCHDOG]: Host RAM đạt {mem_gb:.2f} GB >= {args.ram_limit_gb} GB. Thoát an toàn mã 42!", flush=True)\n                    stop_flag.fill_(1)\n            except Exception:\n                pass\n\n        if is_ddp:\n            dist.broadcast(stop_flag, src=0)\n            dist.barrier()\n\n        if int(stop_flag.item()) == 1:\n            watchdog_paused = True\n            break\n\n    if is_ddp:\n        dist.destroy_process_group()\n\n    if watchdog_paused:\n        sys.exit(42)\n    if is_rank0 and not args.dry_run:\n        print(f"\\n🎉 HOÀN TẤT HUẤN LUYỆN TRONG {(time.time() - t_total_start)/60.0:.1f} PHÚT!", flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')

worker_p3_path = os.path.join(WORKING_DIR, "ddp_worker_tps_sequence_phase3.py")
with open(worker_p3_path, "w", encoding="utf-8") as f:
    f.write('#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n"""\nddp_worker_tps_sequence_phase3.py\n--------------------------------------------------------------------------------\nĐỘNG CƠ HUẤN LUYỆN DDP 2x GPU PHA 3 — COARSE TPS ĐA LỚP (MULTI-LAYER CLUSTERED TPS)\nKế thừa 100% kiến trúc `TPSInbetweenOriginal` (10,068,668 active params — 0% sửa kiến trúc)\nvà tinh chỉnh nhẹ (2-3 Epochs) từ `model_tps_blended.pt` để mạng neural thích nghi hoàn toàn\nvới đặc tính trường dòng chảy đa lớp (Static Layer + Clustered Dynamic Layers + Voronoi Mask).\n\nCác nâng cấp cốt lõi của Pha 3 (Bước 4):\n  1. Nạp trực tiếp bộ cache `tps_cache_multilayer.npz` (hoặc tương thích fallback single-layer)\n     và tái tạo batch flow đa lớp siêu tốc trên GPU (`reconstruct_multilayer_flow_batch_from_cache`).\n  2. Làm giàu dữ liệu có chủ đích tại chỗ (Zero-cost Compositional Multi-Layer Enrichment):\n     - Static-Background Compositing (`--static_aug_prob 0.25`) trên chuỗi Sakuga thực tế:\n       trộn nền kiến trúc/đường chân trời tĩnh vào (x0, x1, xk_all) kèm static keypoints.\n     - Nâng cấp 100% chuỗi hình học tổng hợp (`--synth_count 180`) sang Multi-Layer TPS với 6 Mode,\n       bổ sung Mode Occlusion Crossing (vật thể động lướt qua nét tĩnh) & Multi-Cluster Crossing.\n  3. Cân bằng trọng số Loss Pha 3 (`w_ghost=0.65, w_temp=0.55`) + LR phẫu thuật (`1.0e-5 / 2.5e-5`)\n     để vừa làm sạch ranh giới giao thoa, vừa giữ vững mật độ nét chuẩn ~1.00 của Blended Model.\n--------------------------------------------------------------------------------\n"""\n\nimport os\nimport sys\nimport gc\nimport time\nimport math\nimport json\nimport glob\nimport random\nimport argparse\nimport contextlib\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import Dataset, DataLoader, Sampler\nfrom torch.utils.data.distributed import DistributedSampler\n\nos.environ.setdefault("MALLOC_TRIM_THRESHOLD_", "65536")\nos.environ.setdefault("MALLOC_ARENA_MAX", "2")\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("MKL_NUM_THREADS", "1")\n\nif sys.platform.startswith(\'win\'):\n    try:\n        sys.stdout.reconfigure(encoding=\'utf-8\')\n        sys.stderr.reconfigure(encoding=\'utf-8\')\n    except Exception:\n        pass\n\ntry:\n    import cv2\n    cv2.setNumThreads(0)\n    cv2.ocl.setUseOpenCL(False)\nexcept ImportError:\n    cv2 = None\n\ntorch.backends.cudnn.benchmark = True\n\nSCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))\nREPO_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, ".."))\nfor _p in (SCRIPT_DIR, os.path.join(REPO_ROOT, "pipelines")):\n    if os.path.isdir(_p) and _p not in sys.path:\n        sys.path.insert(0, _p)\n\nfrom multi_layer_tps import (\n    FIXED_TPS_ANCHORS,\n    solve_multilayer_cache_entry,\n    reconstruct_multilayer_flow_batch_from_cache,\n)\n\n# Import các thành phần lõi không thay đổi (TPSInbetweenOriginal, Loss, Metrics, Sampler, EMA, Manager)\n_P2_DIR = os.path.join(REPO_ROOT, "kaggle_finetune_sequence_phase2")\nif os.path.isdir(_P2_DIR) and _P2_DIR not in sys.path:\n    sys.path.insert(0, _P2_DIR)\n\ntry:\n    from ddp_worker_tps_sequence_phase2 import (\n        trim_memory,\n        TPSInbetweenOriginal,\n        load_tps_weights,\n        solve_tps_backward_pair,\n        tps_flow_from_theta_batch,\n        AdaptiveLineartLoss,\n        BidirectionalStrokeWidthLoss,\n        SoftClDiceLoss,\n        VGGPerceptualOrLaplacianLoss,\n        approx_gpu_distance_map,\n        soft_dilate,\n        soft_skeletonize,\n        evaluate_metrics_dual_regime,\n        DDPLengthWidthBatchSampler,\n        AdaptiveEMA,\n        TwoFileCheckpointManager,\n        _apply_paired_spatial_aug_gpu,\n    )\n    _IMPORTED_P2 = True\nexcept ImportError:\n    _IMPORTED_P2 = False\n\nif not _IMPORTED_P2:\n    # Khi chạy độc lập trên Kaggle, ddp_worker_tps_sequence_phase2.py được Cell 1 ghi chung cùng thư mục\n    raise ImportError("Không tìm thấy module nền tảng `ddp_worker_tps_sequence_phase2.py`!")\n\n\n# ==============================================================================\n# 0b. HÀM MẤT MÁT PHA 3 CÂN BẰNG: BẢO TOÀN SKEL / DENSITY / WCD_DOG & SẠCH NỀN XA\n# ==============================================================================\nclass Phase3ExactDensityLoss(nn.Module):\n    """\n    Siết chặt sàn mật độ nét (dead_zone_low=0.99) và tăng phạt làm nhạt/mảnh nét (3.2*dev_low + 1.5*dev_low^2)\n    để ngăn chặn hoàn toàn hiện tượng trôi mảnh nét khi fine-tune từ `model_tps_blended.pt`.\n    """\n    def __init__(self, tau=0.15, dead_zone_low=0.99, dead_zone_high=1.04):\n        super().__init__()\n        self.tau = tau\n        self.low = dead_zone_low\n        self.high = dead_zone_high\n\n    def forward(self, pred, gt):\n        s_p = torch.clamp((pred - 0.06) / 0.18, 0.0, 1.0) + 0.25 * F.relu(pred - 0.24)\n        h_p = (pred >= self.tau).float()\n        m_p = h_p - s_p.detach() + s_p\n\n        h_g = (gt >= self.tau).float()\n        area_p = torch.sum(m_p, dim=(-2, -1))\n        area_g = torch.sum(h_g, dim=(-2, -1)).detach()\n        r = area_p / torch.clamp(area_g, min=1.0)\n\n        dev_low = F.relu(self.low - r)\n        dev_high = F.relu(r - self.high)\n        dev_center = r - 1.0\n        loss = (\n            3.2 * dev_low + 1.5 * (dev_low ** 2)\n            + 2.8 * dev_high + 1.5 * (dev_high ** 2)\n            + 0.12 * (dev_center ** 2)\n        )\n        return loss.mean()\n\n\nclass Phase3ZoneCleanlinessLoss(nn.Module):\n    """\n    Làm sạch có chọn lọc (Surgical Background Cleanliness):\n      - Nền xa (> 2px khỏi nét GT): dọn sương mờ và đặc biệt phạt mạnh mực đậm sai vị trí (Dark Ghost >= 0.20).\n      - Vành sát nét (near_halo): chỉ phạt mực >= 0.12 (nới từ 0.06 -> 0.12) để bảo toàn 100% tóc mảnh anti-aliased!\n    """\n    def __init__(self):\n        super().__init__()\n\n    def forward(self, pred, gt):\n        with torch.no_grad():\n            gt_stroke_wide = soft_dilate(soft_dilate((gt > 0.04).float()))\n            far_bg = torch.clamp(1.0 - gt_stroke_wide, 0.0, 1.0)\n            near_halo = torch.clamp(gt_stroke_wide - (gt > 0.12).float(), 0.0, 1.0)\n            gt_area = torch.clamp(torch.sum((gt > 0.15).float(), dim=(-3, -2, -1)), min=500.0)\n\n        near_count = torch.clamp(torch.sum(near_halo, dim=(-3, -2, -1)), min=200.0)\n        pen_far_p045 = F.relu(pred - 0.045)\n        pen_far = (\n            1.0 * F.relu(pred - 0.018)\n            + 2.5 * pen_far_p045\n            + 3.0 * (pen_far_p045 ** 2)\n            + 9.0 * F.relu(pred - 0.20)\n        )\n        loss_far = torch.sum(far_bg * pen_far, dim=(-3, -2, -1)) / gt_area\n\n        pen_halo = F.relu(pred - 0.12)\n        loss_halo = 0.35 * torch.sum(near_halo * pen_halo, dim=(-3, -2, -1)) / near_count\n        return (loss_far + loss_halo).mean()\n\n\nclass Phase3BalancedLineartLoss(AdaptiveLineartLoss):\n    """\n    Kế thừa `AdaptiveLineartLoss` nhưng thay thế các thành phần gây tẩy mảnh nét bằng cấu hình cân bằng Pha 3 v2:\n      - w_dt = 0.40 (tối ưu WCD_DoG)\n      - w_dens = 0.70 (giữ chặt Density Ratio ~ 0.98 - 1.02)\n      - w_width = 0.35 (bảo vệ lõi skeleton)\n      - w_cldice = 0.28 (bảo vệ ngọn tóc mảnh & Skeleton Recall)\n      - w_ghost = 0.32 (vừa dọn Dark Ghost, không làm mờ nét)\n    """\n    def __init__(self, train_mode="LONG", is_rank0=True, w_ghost=0.32, w_temp=0.55):\n        super().__init__(train_mode=train_mode, is_rank0=is_rank0, w_ghost=w_ghost, w_temp=w_temp)\n        self.w_dt = 0.40\n        self.w_dens = 0.70\n        self.w_width = 0.35\n        self.w_cldice = 0.28 if self.train_mode == "LONG" else self.w_cldice\n        self.density_loss = Phase3ExactDensityLoss(tau=0.15, dead_zone_low=0.99, dead_zone_high=1.04)\n        self.ghost_loss = Phase3ZoneCleanlinessLoss()\n        self.stroke_width_loss = BidirectionalStrokeWidthLoss()\n\n\n# ==============================================================================\n# 1. LÀM GIÀU DỮ LIỆU ĐA LỚP TẠI CHỖ (ON-THE-FLY MULTI-LAYER ENRICHMENT)\n# ==============================================================================\ndef _inject_static_background_on_sequence(x0, x1, xk_all, orig_h, orig_w, cache_entry):\n    """\n    Trộn lớp nền đường thẳng tĩnh mảnh/thưa (sparse static background lineart, 1.5-2.5px)\n    vào chuỗi Sakuga thực tế để không làm loãng mật độ mực nhân vật mà vẫn rèn luyện\n    xử lý ranh giới giao thoa che khuất (Occlusion Crossing).\n    """\n    if cv2 is None or cache_entry[\'num_clusters\'] <= 0:\n        return x0, x1, xk_all, cache_entry\n\n    bg = np.zeros((orig_h, orig_w), dtype=np.float32)\n    pts_st = []\n    style = random.choice(["ground_horizon", "corner_frame", "single_pillar"])\n    th = int(round(random.uniform(1.5, 2.5)))\n    intensity = random.uniform(0.80, 1.0)\n\n    if style == "ground_horizon":\n        y_l = random.uniform(0.68 * orig_h, 0.88 * orig_h)\n        x_a, x_b = random.uniform(0.08 * orig_w, 0.20 * orig_w), random.uniform(0.80 * orig_w, 0.92 * orig_w)\n        cv2.line(bg, (int(x_a), int(y_l)), (int(x_b), int(y_l)), intensity, th, lineType=cv2.LINE_AA)\n        for xv in np.linspace(x_a, x_b, 20):\n            pts_st.append([xv / max(1.0, orig_w - 1), y_l / max(1.0, orig_h - 1)])\n    elif style == "corner_frame":\n        bx0 = random.uniform(0.10 * orig_w, 0.25 * orig_w)\n        by0 = random.uniform(0.60 * orig_h, 0.78 * orig_h)\n        bx1 = random.uniform(0.75 * orig_w, 0.90 * orig_w)\n        by1 = random.uniform(0.14 * orig_h, 0.35 * orig_h)\n        cv2.line(bg, (int(bx0), int(by0)), (int(bx1), int(by0)), intensity, th, lineType=cv2.LINE_AA)\n        cv2.line(bg, (int(bx0), int(by0)), (int(bx0), int(by1)), intensity, th, lineType=cv2.LINE_AA)\n        for xv in np.linspace(bx0, bx1, 12):\n            pts_st.append([xv / max(1.0, orig_w - 1), by0 / max(1.0, orig_h - 1)])\n        for yv in np.linspace(by1, by0, 10):\n            pts_st.append([bx0 / max(1.0, orig_w - 1), yv / max(1.0, orig_h - 1)])\n    else:\n        xv = random.uniform(0.18 * orig_w, 0.82 * orig_w)\n        ya, yb = random.uniform(0.12 * orig_h, 0.24 * orig_h), random.uniform(0.76 * orig_h, 0.88 * orig_h)\n        cv2.line(bg, (int(xv), int(ya)), (int(xv), int(yb)), intensity, th, lineType=cv2.LINE_AA)\n        for yv in np.linspace(ya, yb, 16):\n            pts_st.append([xv / max(1.0, orig_w - 1), yv / max(1.0, orig_h - 1)])\n\n    x0_aug = np.maximum(x0, bg)\n    x1_aug = np.maximum(x1, bg)\n    xk_aug = np.maximum(xk_all, bg[None, :, :])\n\n    pts_st_np = np.asarray(pts_st, dtype=np.float32)\n    max_s = cache_entry[\'static_pts0\'].shape[0]\n    cur_ns = int(cache_entry[\'num_static\'])\n    sp0 = cache_entry[\'static_pts0\'].copy()\n    sp1 = cache_entry[\'static_pts1\'].copy()\n    avail = max(0, max_s - cur_ns)\n    n_add = min(avail, len(pts_st_np))\n    if n_add > 0:\n        sp0[cur_ns:cur_ns + n_add] = pts_st_np[:n_add]\n        sp1[cur_ns:cur_ns + n_add] = pts_st_np[:n_add]\n        cache_entry = dict(cache_entry)\n        cache_entry[\'static_pts0\'] = sp0\n        cache_entry[\'static_pts1\'] = sp1\n        cache_entry[\'num_static\'] = cur_ns + n_add\n\n    return x0_aug, x1_aug, xk_aug, cache_entry\n\n\ndef _generate_multilayer_synthetic_sequence(seq_len, orig_h=540, orig_w=960, max_pts=128, max_clusters=4, lambd=0.01):\n    """\n    Sinh chuỗi hình học tổng hợp 540p sử dụng 100% Multi-Layer Clustered TPS (Direct-t) với 6 kịch bản:\n      - Mode 1 (25%): Hình tròn/elip thu phóng & dịch xa\n      - Mode 2 (25%): Hình tròn/elip bay + đường nền / khung phòng tĩnh (Case 3)\n      - Mode 3 (15%): Cung cong mở xoay & biến dạng\n      - Mode 4 (15%): Hai hình tròn chuyển động song song độc lập\n      - Mode 5 (10%): Occlusion Crossing — vật thể chuyển động cắt ngang trực diện qua đường tĩnh\n      - Mode 6 (10%): Multi-Cluster Crossing — hai vật thể chuyển động ngược/chéo chiều nhau\n    """\n    def _ellipse_pts(cx, cy, rx, ry, rot_rad, n_pts=32):\n        ang = np.linspace(0.0, 2.0 * np.pi, n_pts, endpoint=False)\n        lx, ly = rx * np.cos(ang), ry * np.sin(ang)\n        ca, sa = math.cos(rot_rad), math.sin(rot_rad)\n        px = cx + lx * ca - ly * sa\n        py = cy + lx * sa + ly * ca\n        return np.stack([px / float(orig_w - 1), py / float(orig_h - 1)], axis=1).astype(np.float32)\n\n    mode_roll = random.random()\n    frames = []\n    kps_per_t = []\n    t_list = [float(k) / float(seq_len - 1) for k in range(seq_len)]\n\n    if mode_roll < 0.50:\n        # Mode 1 (25%) & Mode 2 (25%): Circle/oval + optional static ground/box\n        has_static = (mode_roll >= 0.25)\n        y_max_c = 340 if has_static else 420\n        rx0 = random.uniform(26.0, 92.0)\n        ry0 = rx0 * random.uniform(0.75, 1.25)\n        scale_r = random.uniform(0.42, 2.20)\n        rx1 = float(np.clip(rx0 * scale_r, 18.0, 96.0))\n        ry1 = float(np.clip(ry0 * scale_r * random.uniform(0.85, 1.15), 18.0, 96.0))\n        cx0 = random.uniform(110.0, 420.0)\n        cx1 = float(np.clip(cx0 + random.choice([-1.0, 1.0]) * random.uniform(80.0, 420.0), 100.0, orig_w - 100.0))\n        if random.random() < 0.5:\n            cx0, cx1 = cx1, cx0\n        cy0 = random.uniform(115.0, y_max_c - 60.0)\n        cy1 = float(np.clip(cy0 + random.uniform(-90.0, 90.0), 105.0, y_max_c - 40.0))\n        rot0 = random.uniform(-0.35, 0.35)\n        rot1 = rot0 + random.uniform(-0.35, 0.35)\n        th0, th1 = random.uniform(2.0, 5.0), random.uniform(2.0, 5.0)\n\n        st_type = random.choice(["line", "rect"])\n        st_y = random.uniform(410.0, 490.0)\n        st_x0, st_x1 = random.uniform(80.0, 200.0), random.uniform(orig_w - 200.0, orig_w - 80.0)\n        st_th = int(round(random.uniform(2.5, 4.5)))\n        if st_type == "line":\n            xs_st = np.linspace(st_x0, st_x1, 24)\n            pts_st = np.stack([xs_st / float(orig_w - 1), np.full_like(xs_st, st_y) / float(orig_h - 1)], axis=1).astype(np.float32)\n        else:\n            bx0, by0 = random.uniform(120.0, 740.0), st_y - 45.0\n            bw, bh = random.uniform(70.0, 140.0), random.uniform(40.0, 65.0)\n            pts_st = np.vstack([\n                np.stack([np.linspace(bx0, bx0 + bw, 7)[:-1], np.full(6, by0)], axis=1),\n                np.stack([np.full(6, bx0 + bw), np.linspace(by0, by0 + bh, 7)[:-1]], axis=1),\n                np.stack([np.linspace(bx0 + bw, bx0, 7)[:-1], np.full(6, by0 + bh)], axis=1),\n                np.stack([np.full(6, bx0), np.linspace(by0 + bh, by0, 7)[:-1]], axis=1),\n            ])\n            pts_st[:, 0] /= float(orig_w - 1)\n            pts_st[:, 1] /= float(orig_h - 1)\n            pts_st = pts_st.astype(np.float32)\n\n        for t in t_list:\n            im = np.zeros((orig_h, orig_w), dtype=np.float32)\n            cx, cy = (1.0 - t) * cx0 + t * cx1, (1.0 - t) * cy0 + t * cy1\n            rx, ry = (1.0 - t) * rx0 + t * rx1, (1.0 - t) * ry0 + t * ry1\n            rot = (1.0 - t) * rot0 + t * rot1\n            th = max(2, int(round((1.0 - t) * th0 + t * th1)))\n            if cv2 is not None:\n                cv2.ellipse(im, (int(round(cx)), int(round(cy))), (int(round(rx)), int(round(ry))), math.degrees(rot), 0, 360, 1.0, th, lineType=cv2.LINE_AA)\n                if has_static:\n                    if st_type == "line":\n                        cv2.line(im, (int(st_x0), int(st_y)), (int(st_x1), int(st_y)), 1.0, st_th, lineType=cv2.LINE_AA)\n                    else:\n                        cv2.rectangle(im, (int(bx0), int(by0)), (int(bx0 + bw), int(by0 + bh)), 1.0, st_th, lineType=cv2.LINE_AA)\n            pts_mv = _ellipse_pts(cx, cy, rx, ry, rot, n_pts=32)\n            frames.append(im)\n            kps_per_t.append(np.vstack([pts_mv, pts_st]) if has_static else pts_mv)\n    elif mode_roll < 0.65:\n        # Mode 3 (15%): Smooth curved arc\n        n_ctrl = 24\n        s_u = np.linspace(-1.0, 1.0, n_ctrl)\n        base_len = random.uniform(60.0, 140.0)\n        bend0, bend1 = random.uniform(-55.0, 55.0), random.uniform(-55.0, 55.0)\n        cx0, cy0 = random.uniform(160.0, 440.0), random.uniform(140.0, 400.0)\n        cx1 = float(np.clip(cx0 + random.choice([-1.0, 1.0]) * random.uniform(60.0, 320.0), 140.0, orig_w - 140.0))\n        cy1 = float(np.clip(cy0 + random.uniform(-100.0, 100.0), 120.0, orig_h - 120.0))\n        sc0, sc1 = random.uniform(0.75, 1.25), random.uniform(0.65, 1.45)\n        ang0 = random.uniform(-0.7, 0.7)\n        ang1 = ang0 + random.uniform(-0.6, 0.6)\n        th = int(round(random.uniform(2.0, 4.5)))\n\n        def _arc_px(t):\n            cx, cy = (1.0 - t) * cx0 + t * cx1, (1.0 - t) * cy0 + t * cy1\n            sc, ang, bend = (1.0 - t) * sc0 + t * sc1, (1.0 - t) * ang0 + t * ang1, (1.0 - t) * bend0 + t * bend1\n            lx, ly = s_u * base_len * sc, bend * sc * (1.0 - s_u ** 2)\n            ca, sa = math.cos(ang), math.sin(ang)\n            return np.stack([cx + lx * ca - ly * sa, cy + lx * sa + ly * ca], axis=1)\n\n        for t in t_list:\n            im = np.zeros((orig_h, orig_w), dtype=np.float32)\n            p_px = _arc_px(t)\n            if cv2 is not None:\n                cv2.polylines(im, [np.round(p_px).astype(np.int32)], False, 1.0, th, lineType=cv2.LINE_AA)\n            frames.append(im)\n            kps_per_t.append(np.stack([p_px[:, 0] / float(orig_w - 1), p_px[:, 1] / float(orig_h - 1)], axis=1).astype(np.float32))\n    elif mode_roll < 0.80:\n        # Mode 4 (15%): Two independent moving circles/ovals\n        lanes = [(145.0, 40.0), (385.0, 40.0)]\n        params = []\n        for cy_base, cy_jit in lanes:\n            rx0 = random.uniform(24.0, 62.0)\n            ry0 = rx0 * random.uniform(0.8, 1.2)\n            sc = random.uniform(0.55, 1.80)\n            rx1, ry1 = float(np.clip(rx0 * sc, 18.0, 72.0)), float(np.clip(ry0 * sc, 18.0, 72.0))\n            cx0 = random.uniform(120.0, 450.0)\n            cx1 = float(np.clip(cx0 + random.choice([-1.0, 1.0]) * random.uniform(80.0, 350.0), 100.0, orig_w - 100.0))\n            params.append((cx0, cy_base + random.uniform(-cy_jit, cy_jit), rx0, ry0, cx1, cy_base + random.uniform(-cy_jit, cy_jit), rx1, ry1))\n        th = int(round(random.uniform(2.5, 4.5)))\n        for t in t_list:\n            im = np.zeros((orig_h, orig_w), dtype=np.float32)\n            pts_two = []\n            for (cx0, cy0, rx0, ry0, cx1, cy1, rx1, ry1) in params:\n                cx, cy = (1.0 - t) * cx0 + t * cx1, (1.0 - t) * cy0 + t * cy1\n                rx, ry = (1.0 - t) * rx0 + t * rx1, (1.0 - t) * ry0 + t * ry1\n                if cv2 is not None:\n                    cv2.ellipse(im, (int(round(cx)), int(round(cy))), (int(round(rx)), int(round(ry))), 0, 0, 360, 1.0, th, lineType=cv2.LINE_AA)\n                pts_two.append(_ellipse_pts(cx, cy, rx, ry, 0.0, n_pts=24))\n            frames.append(im)\n            kps_per_t.append(np.vstack(pts_two))\n    elif mode_roll < 0.90:\n        # Mode 5 (10%): Occlusion Crossing — hình tròn/elip chuyển động cắt qua đường kẻ tĩnh\n        rx0 = random.uniform(35.0, 70.0)\n        ry0 = rx0 * random.uniform(0.85, 1.15)\n        rx1, ry1 = rx0 * random.uniform(0.65, 1.35), ry0 * random.uniform(0.65, 1.35)\n        cx0, cx1 = random.uniform(140.0, 260.0), random.uniform(680.0, 820.0)\n        if random.random() < 0.5:\n            cx0, cx1 = cx1, cx0\n        st_y = random.uniform(210.0, 330.0)\n        cy0 = st_y + random.choice([-1.0, 1.0]) * random.uniform(60.0, 120.0)\n        cy1 = st_y - (cy0 - st_y) * random.uniform(0.6, 1.1)\n        xs_st = np.linspace(100.0, orig_w - 100.0, 28)\n        pts_st = np.stack([xs_st / float(orig_w - 1), np.full_like(xs_st, st_y) / float(orig_h - 1)], axis=1).astype(np.float32)\n        th = int(round(random.uniform(2.5, 4.0)))\n        for t in t_list:\n            im = np.zeros((orig_h, orig_w), dtype=np.float32)\n            cx, cy = (1.0 - t) * cx0 + t * cx1, (1.0 - t) * cy0 + t * cy1\n            rx, ry = (1.0 - t) * rx0 + t * rx1, (1.0 - t) * ry0 + t * ry1\n            if cv2 is not None:\n                cv2.line(im, (100, int(round(st_y))), (orig_w - 100, int(round(st_y))), 1.0, th, lineType=cv2.LINE_AA)\n                cv2.ellipse(im, (int(round(cx)), int(round(cy))), (int(round(rx)), int(round(ry))), 0, 0, 360, 1.0, th, lineType=cv2.LINE_AA)\n            frames.append(im)\n            kps_per_t.append(np.vstack([_ellipse_pts(cx, cy, rx, ry, 0.0, n_pts=32), pts_st]))\n    else:\n        # Mode 6 (10%): Multi-Cluster Crossing — hai hình tròn chạy ngược/chéo chiều\n        rx_a, ry_a = random.uniform(30.0, 55.0), random.uniform(30.0, 55.0)\n        rx_b, ry_b = random.uniform(28.0, 50.0), random.uniform(28.0, 50.0)\n        p_a = (150.0, random.uniform(130.0, 220.0), 800.0, random.uniform(310.0, 410.0))\n        p_b = (800.0, random.uniform(130.0, 220.0), 150.0, random.uniform(310.0, 410.0))\n        th = int(round(random.uniform(2.5, 4.0)))\n        for t in t_list:\n            im = np.zeros((orig_h, orig_w), dtype=np.float32)\n            cxa, cya = (1.0 - t) * p_a[0] + t * p_a[2], (1.0 - t) * p_a[1] + t * p_a[3]\n            cxb, cyb = (1.0 - t) * p_b[0] + t * p_b[2], (1.0 - t) * p_b[1] + t * p_b[3]\n            if cv2 is not None:\n                cv2.ellipse(im, (int(round(cxa)), int(round(cya))), (int(round(rx_a)), int(round(ry_a))), 0, 0, 360, 1.0, th, lineType=cv2.LINE_AA)\n                cv2.ellipse(im, (int(round(cxb)), int(round(cyb))), (int(round(rx_b)), int(round(ry_b))), 0, 0, 360, 1.0, th, lineType=cv2.LINE_AA)\n            pts_ab = np.vstack([\n                _ellipse_pts(cxa, cya, rx_a, ry_a, 0.0, n_pts=24),\n                _ellipse_pts(cxb, cyb, rx_b, ry_b, 0.0, n_pts=24),\n            ])\n            frames.append(im)\n            kps_per_t.append(pts_ab)\n\n    x0, x1 = frames[0], frames[-1]\n    xk_all = np.stack(frames[1:-1], axis=0)\n    kps0, kps1 = kps_per_t[0], kps_per_t[-1]\n    t_mid = np.array(t_list[1:-1], dtype=np.float32)\n\n    # Giải Multi-Layer TPS Cache Entry (Direct-t) cho từng khung hình trung gian m_idx\n    m_steps = seq_len - 2\n    max_mask_pts, max_static_pts = 64, 96\n    c0_m = np.zeros((m_steps, max_clusters, max_pts, 2), dtype=np.float32)\n    th01_m = np.zeros((m_steps, max_clusters, max_pts + 3, 2), dtype=np.float32)\n    c1_m = np.zeros((m_steps, max_clusters, max_pts, 2), dtype=np.float32)\n    th10_m = np.zeros((m_steps, max_clusters, max_pts + 3, 2), dtype=np.float32)\n    val_m = np.zeros((m_steps, max_clusters), dtype=np.float32)\n    dpts0_m = np.zeros((m_steps, max_clusters, max_mask_pts, 2), dtype=np.float32)\n    dpts1_m = np.zeros((m_steps, max_clusters, max_mask_pts, 2), dtype=np.float32)\n    dlen_m = np.zeros((m_steps, max_clusters), dtype=np.int32)\n    spts0_m = np.zeros((m_steps, max_static_pts, 2), dtype=np.float32)\n    spts1_m = np.zeros((m_steps, max_static_pts, 2), dtype=np.float32)\n    nstat_m = np.zeros((m_steps,), dtype=np.int32)\n    nclus_m = np.zeros((m_steps,), dtype=np.int32)\n\n    from multi_layer_tps import segment_motion_keypoints, solve_tps_dynamic_layer, _stratified_spatial_subsample\n    seg = segment_motion_keypoints(kps0, kps1, img_hw=(orig_h, orig_w), eps_static_px=1.5, filter_outliers=False)\n    clusters = seg[\'dynamic_clusters\'][:max_clusters]\n    if seg[\'num_static\'] > 0:\n        if seg[\'num_static\'] > max_static_pts:\n            s0_sub, s1_sub = _stratified_spatial_subsample(seg[\'static_kps0\'], seg[\'static_kps1\'], max_static_pts)\n        else:\n            s0_sub, s1_sub = seg[\'static_kps0\'], seg[\'static_kps1\']\n        n_stat = len(s0_sub)\n    else:\n        s0_sub = s1_sub = np.zeros((0, 2), dtype=np.float32)\n        n_stat = 0\n\n    for m_idx in range(m_steps):\n        tv = float(t_mid[m_idx])\n        if n_stat > 0:\n            spts0_m[m_idx, :n_stat] = s0_sub.astype(np.float32)\n            spts1_m[m_idx, :n_stat] = s1_sub.astype(np.float32)\n        nstat_m[m_idx] = n_stat\n        nclus_m[m_idx] = len(clusters)\n\n        for k, dc in enumerate(clusters):\n            p0_k, p1_k = dc[\'kps0\'], dc[\'kps1\']\n            pt_k = (1.0 - tv) * p0_k + tv * p1_k\n            ct0, tht0, v0 = solve_tps_dynamic_layer(pt_k, p0_k, max_pts=max_pts, lambd=lambd, use_anchors=False)\n            ct1, tht1, v1 = solve_tps_dynamic_layer(pt_k, p1_k, max_pts=max_pts, lambd=lambd, use_anchors=False)\n            c0_m[m_idx, k], th01_m[m_idx, k] = ct0, tht0\n            c1_m[m_idx, k], th10_m[m_idx, k] = ct1, tht1\n            val_m[m_idx, k] = 1.0 if (v0 > 0.5 and v1 > 0.5) else 0.0\n\n            # Đưa pt_k vào dyn_pts0/dyn_pts1 để mặt nạ Voronoi đặt tâm tại đúng quỹ đạo t\n            if len(pt_k) > max_mask_pts:\n                pt_sub, _ = _stratified_spatial_subsample(pt_k, pt_k, max_mask_pts)\n            else:\n                pt_sub = pt_k\n            n_mk = len(pt_sub)\n            dpts0_m[m_idx, k, :n_mk] = pt_sub.astype(np.float32)\n            dpts1_m[m_idx, k, :n_mk] = pt_sub.astype(np.float32)\n            dlen_m[m_idx, k] = n_mk\n\n    return {\n        \'is_sequence\': True,\n        \'is_direct_t_flows\': True,\n        \'is_multilayer_cache\': True,\n        \'x0\': torch.from_numpy(x0).unsqueeze(0),\n        \'x1\': torch.from_numpy(x1).unsqueeze(0),\n        \'xk_all\': torch.from_numpy(xk_all).unsqueeze(1),\n        \'t_vals\': torch.from_numpy(t_mid),\n        \'c0_dyn_all\': torch.from_numpy(c0_m),\n        \'th01_dyn_all\': torch.from_numpy(th01_m),\n        \'c1_dyn_all\': torch.from_numpy(c1_m),\n        \'th10_dyn_all\': torch.from_numpy(th10_m),\n        \'valid_dyn_all\': torch.from_numpy(val_m),\n        \'dyn_pts0_all\': torch.from_numpy(dpts0_m),\n        \'dyn_pts1_all\': torch.from_numpy(dpts1_m),\n        \'dyn_pts_len_all\': torch.from_numpy(dlen_m),\n        \'static_pts0_all\': torch.from_numpy(spts0_m),\n        \'static_pts1_all\': torch.from_numpy(spts1_m),\n        \'num_static_all\': torch.from_numpy(nstat_m),\n        \'num_clusters_all\': torch.from_numpy(nclus_m),\n        \'orig_hw\': torch.tensor([orig_h, orig_w], dtype=torch.int32),\n        \'seq_len\': seq_len,\n        \'clip_id\': f"synth_ml_N{seq_len}",\n        \'scene_id\': "synth_geom_ml",\n        \'mean_disp\': 120.0,\n    }\n\n\n# ==============================================================================\n# 2. DATASET ĐA LỚP (ĐỌC TPS_CACHE_MULTILAYER.NPZ + STATIC BG AUGMENTATION)\n# ==============================================================================\nclass SakugaMultiLayerDataset(Dataset):\n    def __init__(self, metadata_csv, clips_dir, tps_cache_path, train_mode="LONG", is_train=True, seed=42, synth_count=0, static_aug_prob=0.25):\n        super().__init__()\n        self.clips_dir = clips_dir\n        self.train_mode = train_mode.upper()\n        self.is_train = is_train\n        self.static_aug_prob = float(static_aug_prob) if is_train else 0.0\n        self.df = pd.read_csv(metadata_csv).reset_index(drop=True)\n\n        self.tps_cache_path = tps_cache_path\n        with np.load(tps_cache_path) as cache_data:\n            self.is_multilayer = (\'c0_dyn\' in cache_data.files)\n            self.orig_hw_arr = np.array(cache_data[\'orig_hw\'])\n            if self.is_multilayer:\n                self.c0_dyn = np.array(cache_data[\'c0_dyn\'])\n                self.th01_dyn = np.array(cache_data[\'th01_dyn\'])\n                self.c1_dyn = np.array(cache_data[\'c1_dyn\'])\n                self.th10_dyn = np.array(cache_data[\'th10_dyn\'])\n                self.valid_dyn = np.array(cache_data[\'valid_dyn\'])\n                self.dyn_pts0 = np.array(cache_data[\'dyn_pts0\'])\n                self.dyn_pts1 = np.array(cache_data[\'dyn_pts1\'])\n                self.dyn_pts_len = np.array(cache_data[\'dyn_pts_len\'])\n                self.static_pts0 = np.array(cache_data[\'static_pts0\'])\n                self.static_pts1 = np.array(cache_data[\'static_pts1\'])\n                self.num_static = np.array(cache_data[\'num_static\'])\n                self.num_clusters = np.array(cache_data[\'num_clusters\'])\n            else:\n                self.c0_arr = np.array(cache_data[\'c0\'])\n                self.th01_arr = np.array(cache_data[\'th01\'])\n                self.c1_arr = np.array(cache_data[\'c1\'])\n                self.th10_arr = np.array(cache_data[\'th10\'])\n                self.valid_arr = np.array(cache_data[\'valid\'])\n\n        self.samples = []\n        if self.is_train and self.train_mode == "LONG":\n            for idx, row in self.df.iterrows():\n                cid = str(row[\'clip_id\'])\n                seq_len = int(row[\'seq_len\'])\n                cache_idx = int(row[\'cache_idx\']) if \'cache_idx\' in row else idx\n                hw = self.orig_hw_arr[cache_idx]\n                t_list = [float(k) / float(seq_len - 1) for k in range(1, seq_len - 1)]\n                self.samples.append({\n                    \'is_synth\': False,\n                    \'clip_id\': cid,\n                    \'scene_id\': str(row.get(\'scene_id\', \'\')),\n                    \'seq_len\': seq_len,\n                    \'cache_idx\': cache_idx,\n                    \'orig_h\': int(hw[0]),\n                    \'orig_w\': int(hw[1]),\n                    \'t_list\': t_list,\n                    \'mean_disp\': float(row.get(\'mean_disp\', 0.0)),\n                })\n            n_synth = int(synth_count) if (cv2 is not None and synth_count > 0) else 0\n            for s_i in range(n_synth):\n                sl = (3, 5, 7)[s_i % 3]\n                self.samples.append({\n                    \'is_synth\': True,\n                    \'clip_id\': f"synth_ml_{s_i:04d}_N{sl}",\n                    \'scene_id\': "synth_geom_ml",\n                    \'seq_len\': sl,\n                    \'cache_idx\': -1,\n                    \'orig_h\': 540,\n                    \'orig_w\': 960,\n                    \'t_list\': [float(k) / float(sl - 1) for k in range(1, sl - 1)],\n                    \'mean_disp\': 120.0,\n                })\n        else:\n            for idx, row in self.df.iterrows():\n                cid = str(row[\'clip_id\'])\n                seq_len = int(row[\'seq_len\'])\n                cache_idx = int(row[\'cache_idx\']) if \'cache_idx\' in row else idx\n                hw = self.orig_hw_arr[cache_idx]\n                for k in range(1, seq_len - 1):\n                    self.samples.append({\n                        \'is_synth\': False,\n                        \'clip_id\': cid,\n                        \'scene_id\': str(row.get(\'scene_id\', \'\')),\n                        \'seq_len\': seq_len,\n                        \'cache_idx\': cache_idx,\n                        \'orig_h\': int(hw[0]),\n                        \'orig_w\': int(hw[1]),\n                        \'frame_k\': k + 1,\n                        \'t_val\': float(k) / float(seq_len - 1),\n                        \'mean_disp\': float(row.get(\'mean_disp\', 0.0)),\n                    })\n\n    def __len__(self):\n        return len(self.samples)\n\n    def _read_ink(self, clip_id, frame_idx_1b):\n        p = os.path.join(self.clips_dir, clip_id, f"{frame_idx_1b:06d}.png")\n        with Image.open(p) as im:\n            arr = np.asarray(im.convert(\'L\'), dtype=np.float32)\n        return (255.0 - arr) / 255.0\n\n    def __getitem__(self, idx):\n        s = self.samples[idx]\n        if s.get(\'is_synth\', False):\n            return _generate_multilayer_synthetic_sequence(s[\'seq_len\'], orig_h=s[\'orig_h\'], orig_w=s[\'orig_w\'])\n\n        cid = s[\'clip_id\']\n        seq_len = s[\'seq_len\']\n        cache_idx = s[\'cache_idx\']\n\n        x0 = self._read_ink(cid, 1)\n        x1 = self._read_ink(cid, seq_len)\n        orig_h, orig_w = x0.shape\n\n        if self.is_multilayer:\n            c_entry = {\n                \'c0_dyn\': np.array(self.c0_dyn[cache_idx], copy=True),\n                \'th01_dyn\': np.array(self.th01_dyn[cache_idx], copy=True),\n                \'c1_dyn\': np.array(self.c1_dyn[cache_idx], copy=True),\n                \'th10_dyn\': np.array(self.th10_dyn[cache_idx], copy=True),\n                \'valid_dyn\': np.array(self.valid_dyn[cache_idx], copy=True),\n                \'dyn_pts0\': np.array(self.dyn_pts0[cache_idx], copy=True),\n                \'dyn_pts1\': np.array(self.dyn_pts1[cache_idx], copy=True),\n                \'dyn_pts_len\': np.array(self.dyn_pts_len[cache_idx], copy=True),\n                \'static_pts0\': np.array(self.static_pts0[cache_idx], copy=True),\n                \'static_pts1\': np.array(self.static_pts1[cache_idx], copy=True),\n                \'num_static\': int(self.num_static[cache_idx]),\n                \'num_clusters\': int(self.num_clusters[cache_idx]),\n            }\n\n            if self.is_train and self.train_mode == "LONG":\n                xk_list = [self._read_ink(cid, f_idx) for f_idx in range(2, seq_len)]\n                xk_all = np.stack(xk_list, axis=0)\n                t_arr = np.array(s[\'t_list\'], dtype=np.float32)\n\n                # Làm giàu có chủ đích: trộn nền tĩnh nhân tạo vào chuỗi Sakuga thực tế\n                if self.static_aug_prob > 0.0 and random.random() < self.static_aug_prob:\n                    x0, x1, xk_all, c_entry = _inject_static_background_on_sequence(\n                        x0, x1, xk_all, orig_h, orig_w, c_entry\n                    )\n\n                # Temporal Reversal (đảo chiều I0 <-> I1)\n                if random.random() < 0.5:\n                    x0, x1 = x1, x0\n                    c_entry[\'c0_dyn\'], c_entry[\'c1_dyn\'] = c_entry[\'c1_dyn\'], c_entry[\'c0_dyn\']\n                    c_entry[\'th01_dyn\'], c_entry[\'th10_dyn\'] = c_entry[\'th10_dyn\'], c_entry[\'th01_dyn\']\n                    c_entry[\'dyn_pts0\'], c_entry[\'dyn_pts1\'] = c_entry[\'dyn_pts1\'], c_entry[\'dyn_pts0\']\n                    c_entry[\'static_pts0\'], c_entry[\'static_pts1\'] = c_entry[\'static_pts1\'], c_entry[\'static_pts0\']\n                    xk_all = xk_all[::-1].copy()\n                    t_arr = (1.0 - t_arr[::-1]).copy()\n\n                return {\n                    \'is_sequence\': True,\n                    \'is_direct_t_flows\': False,\n                    \'is_multilayer_cache\': True,\n                    \'x0\': torch.from_numpy(x0).unsqueeze(0),\n                    \'x1\': torch.from_numpy(x1).unsqueeze(0),\n                    \'xk_all\': torch.from_numpy(xk_all).unsqueeze(1),\n                    \'t_vals\': torch.from_numpy(t_arr),\n                    \'c0_dyn\': torch.from_numpy(c_entry[\'c0_dyn\']),\n                    \'th01_dyn\': torch.from_numpy(c_entry[\'th01_dyn\']),\n                    \'c1_dyn\': torch.from_numpy(c_entry[\'c1_dyn\']),\n                    \'th10_dyn\': torch.from_numpy(c_entry[\'th10_dyn\']),\n                    \'valid_dyn\': torch.from_numpy(c_entry[\'valid_dyn\']),\n                    \'dyn_pts0\': torch.from_numpy(c_entry[\'dyn_pts0\']),\n                    \'dyn_pts1\': torch.from_numpy(c_entry[\'dyn_pts1\']),\n                    \'dyn_pts_len\': torch.from_numpy(c_entry[\'dyn_pts_len\']),\n                    \'static_pts0\': torch.from_numpy(c_entry[\'static_pts0\']),\n                    \'static_pts1\': torch.from_numpy(c_entry[\'static_pts1\']),\n                    \'num_static\': torch.tensor(c_entry[\'num_static\'], dtype=torch.int32),\n                    \'num_clusters\': torch.tensor(c_entry[\'num_clusters\'], dtype=torch.int32),\n                    \'orig_hw\': torch.tensor([orig_h, orig_w], dtype=torch.int32),\n                    \'seq_len\': seq_len,\n                    \'clip_id\': cid,\n                    \'scene_id\': s[\'scene_id\'],\n                    \'mean_disp\': s[\'mean_disp\'],\n                }\n\n            fk = s[\'frame_k\']\n            t_val = s[\'t_val\']\n            xk = self._read_ink(cid, fk)\n            return {\n                \'is_sequence\': False,\n                \'is_direct_t_flows\': False,\n                \'is_multilayer_cache\': True,\n                \'x0\': torch.from_numpy(x0).unsqueeze(0),\n                \'x1\': torch.from_numpy(x1).unsqueeze(0),\n                \'xk\': torch.from_numpy(xk).unsqueeze(0),\n                \'t_val\': torch.tensor([t_val], dtype=torch.float32),\n                \'c0_dyn\': torch.from_numpy(c_entry[\'c0_dyn\']),\n                \'th01_dyn\': torch.from_numpy(c_entry[\'th01_dyn\']),\n                \'c1_dyn\': torch.from_numpy(c_entry[\'c1_dyn\']),\n                \'th10_dyn\': torch.from_numpy(c_entry[\'th10_dyn\']),\n                \'valid_dyn\': torch.from_numpy(c_entry[\'valid_dyn\']),\n                \'dyn_pts0\': torch.from_numpy(c_entry[\'dyn_pts0\']),\n                \'dyn_pts1\': torch.from_numpy(c_entry[\'dyn_pts1\']),\n                \'dyn_pts_len\': torch.from_numpy(c_entry[\'dyn_pts_len\']),\n                \'static_pts0\': torch.from_numpy(c_entry[\'static_pts0\']),\n                \'static_pts1\': torch.from_numpy(c_entry[\'static_pts1\']),\n                \'num_static\': torch.tensor(c_entry[\'num_static\'], dtype=torch.int32),\n                \'num_clusters\': torch.tensor(c_entry[\'num_clusters\'], dtype=torch.int32),\n                \'orig_hw\': torch.tensor([orig_h, orig_w], dtype=torch.int32),\n                \'seq_len\': seq_len,\n                \'frame_k\': fk,\n                \'clip_id\': cid,\n                \'scene_id\': s[\'scene_id\'],\n                \'mean_disp\': s[\'mean_disp\'],\n            }\n\n        raise RuntimeError("Dataset Pha 3 yêu cầu bộ cache đa lớp `tps_cache_multilayer.npz`!")\n\n\ndef pad_collate_fn_multilayer(batch):\n    max_h = max(item[\'orig_hw\'][0].item() for item in batch)\n    max_w = max(item[\'orig_hw\'][1].item() for item in batch)\n    h_pad = ((max_h + 31) // 32) * 32\n    w_pad = ((max_w + 31) // 32) * 32\n\n    is_seq = batch[0][\'is_sequence\']\n    is_direct = bool(batch[0].get(\'is_direct_t_flows\', False))\n\n    def _pad_t(t, th, tw):\n        dh = th - t.shape[-2]\n        dw = tw - t.shape[-1]\n        if dh == 0 and dw == 0:\n            return t\n        return F.pad(t, (0, dw, 0, dh), mode=\'constant\', value=0.0)\n\n    out = {\n        \'is_sequence\': is_seq,\n        \'is_direct_t_flows\': is_direct,\n        \'is_multilayer_cache\': True,\n        \'h_pad\': h_pad,\n        \'w_pad\': w_pad,\n        \'x0\': torch.stack([_pad_t(b[\'x0\'], h_pad, w_pad) for b in batch], dim=0),\n        \'x1\': torch.stack([_pad_t(b[\'x1\'], h_pad, w_pad) for b in batch], dim=0),\n        \'orig_hw\': torch.stack([b[\'orig_hw\'] for b in batch], dim=0),\n        \'seq_len\': torch.tensor([b[\'seq_len\'] for b in batch], dtype=torch.int32),\n        \'clip_id\': [b[\'clip_id\'] for b in batch],\n        \'scene_id\': [b[\'scene_id\'] for b in batch],\n        \'mean_disp\': [b[\'mean_disp\'] for b in batch],\n    }\n\n    if is_direct:\n        for k in [\'c0_dyn_all\', \'th01_dyn_all\', \'c1_dyn_all\', \'th10_dyn_all\', \'valid_dyn_all\',\n                  \'dyn_pts0_all\', \'dyn_pts1_all\', \'dyn_pts_len_all\',\n                  \'static_pts0_all\', \'static_pts1_all\', \'num_static_all\', \'num_clusters_all\']:\n            out[k] = torch.stack([b[k] for b in batch], dim=0)\n    else:\n        for k in [\'c0_dyn\', \'th01_dyn\', \'c1_dyn\', \'th10_dyn\', \'valid_dyn\',\n                  \'dyn_pts0\', \'dyn_pts1\', \'dyn_pts_len\',\n                  \'static_pts0\', \'static_pts1\', \'num_static\', \'num_clusters\']:\n            out[k] = torch.stack([b[k] for b in batch], dim=0)\n\n    if is_seq:\n        out[\'xk_all\'] = torch.stack([_pad_t(b[\'xk_all\'], h_pad, w_pad) for b in batch], dim=0)\n        out[\'t_vals\'] = torch.stack([b[\'t_vals\'] for b in batch], dim=0)\n    else:\n        out[\'xk\'] = torch.stack([_pad_t(b[\'xk\'], h_pad, w_pad) for b in batch], dim=0)\n        out[\'t_val\'] = torch.stack([b[\'t_val\'] for b in batch], dim=0)\n        out[\'frame_k\'] = [b[\'frame_k\'] for b in batch]\n    return out\n\n\n# ==============================================================================\n# 3. VALIDATION ĐA LỚP (THUẦN FP32)\n# ==============================================================================\n@torch.no_grad()\ndef run_validation_suite_multilayer(model, val_loader, device, downscale_factor=2, is_ddp=False, evaluate_m0=False):\n    model.eval()\n    keys_sum = [\'psnr\', \'ssim\', \'val_l1\', \'ie_mae\', \'pwcr\', \'rgd\', \'ghost\', \'dark_ghost\',\n                \'wcd\', \'wcd_legacy\', \'wcd_dog\', \'wcd_dog_legacy\', \'semd\',\n                \'recall\', \'precision\', \'f1\', \'density_ratio\', \'skel_recall\', \'tip_recall\']\n    acc_main = {k: 0.0 for k in keys_sum}\n    acc_main[\'count\'] = 0.0\n    acc_m0 = {k: 0.0 for k in keys_sum}\n    acc_m0[\'count\'] = 0.0\n\n    brk_keys = [\'count\', \'psnr\', \'ssim\', \'wcd\', \'wcd_legacy\', \'wcd_dog\', \'wcd_dog_legacy\',\n                \'semd\', \'skel_recall\', \'tip_recall\', \'f1\', \'ghost\', \'density_ratio\']\n    breakdown = {n: {k: 0.0 for k in brk_keys} for n in (3, 5, 7)}\n\n    for batch in val_loader:\n        h_pad, w_pad = batch[\'h_pad\'], batch[\'w_pad\']\n        x0 = batch[\'x0\'].to(device, non_blocking=True)\n        x1 = batch[\'x1\'].to(device, non_blocking=True)\n        xk = batch[\'xk\'].to(device, non_blocking=True)\n        t_val = batch[\'t_val\'].to(device, non_blocking=True).view(-1, 1, 1, 1)\n        orig_hw = batch[\'orig_hw\']\n        seq_lens = batch[\'seq_len\']\n\n        cf01, cf10 = reconstruct_multilayer_flow_batch_from_cache(\n            batch[\'c0_dyn\'].to(device, non_blocking=True),\n            batch[\'th01_dyn\'].to(device, non_blocking=True),\n            batch[\'c1_dyn\'].to(device, non_blocking=True),\n            batch[\'th10_dyn\'].to(device, non_blocking=True),\n            batch[\'valid_dyn\'].to(device, non_blocking=True),\n            batch[\'dyn_pts0\'].to(device, non_blocking=True),\n            batch[\'dyn_pts1\'].to(device, non_blocking=True),\n            batch[\'dyn_pts_len\'].to(device, non_blocking=True),\n            batch[\'static_pts0\'].to(device, non_blocking=True),\n            batch[\'static_pts1\'].to(device, non_blocking=True),\n            batch[\'num_static\'].to(device, non_blocking=True),\n            batch[\'num_clusters\'].to(device, non_blocking=True),\n            orig_hw, h_pad, w_pad, device, downscale_factor=downscale_factor,\n        )\n\n        with torch.amp.autocast(\'cuda\', enabled=False):\n            pred_it, coarse_warp, *_ = model(x0.float(), x1.float(), t_val.float(), cf01.float(), cf10.float())\n\n        b = x0.shape[0]\n        for i in range(b):\n            hi, wi = int(orig_hw[i, 0].item()), int(orig_hw[i, 1].item())\n            sl = int(seq_lens[i].item())\n            gt_i = xk[i:i + 1, :, :hi, :wi]\n            pr_i = pred_it[i:i + 1, :, :hi, :wi]\n\n            m = evaluate_metrics_dual_regime(pr_i, gt_i)\n            acc_main[\'count\'] += 1.0\n            for k in keys_sum:\n                acc_main[k] += float(m[k].item())\n\n            if sl in breakdown:\n                breakdown[sl][\'count\'] += 1.0\n                for bk in brk_keys:\n                    if bk != \'count\':\n                        breakdown[sl][bk] += float(m[bk].item())\n\n            if evaluate_m0:\n                m0_pr = coarse_warp[i:i + 1, :, :hi, :wi]\n                mm0 = evaluate_metrics_dual_regime(m0_pr, gt_i)\n                acc_m0[\'count\'] += 1.0\n                for k in keys_sum:\n                    acc_m0[k] += float(mm0[k].item())\n\n    def _finalize(acc, brk=None):\n        if is_ddp:\n            vec = torch.tensor([acc[\'count\']] + [acc[k] for k in keys_sum], dtype=torch.float64, device=device)\n            dist.all_reduce(vec, op=dist.ReduceOp.SUM)\n            cnt = max(1.0, float(vec[0].item()))\n            out = {k: float(vec[idx + 1].item()) / cnt for idx, k in enumerate(keys_sum)}\n            out[\'total_samples\'] = int(cnt)\n        else:\n            cnt = max(1.0, acc[\'count\'])\n            out = {k: acc[k] / cnt for k in keys_sum}\n            out[\'total_samples\'] = int(cnt)\n\n        if brk is not None:\n            brk_out = {}\n            for sl in (3, 5, 7):\n                if is_ddp:\n                    bvec = torch.tensor([brk[sl][k] for k in brk_keys], dtype=torch.float64, device=device)\n                    dist.all_reduce(bvec, op=dist.ReduceOp.SUM)\n                    bcnt = max(1.0, float(bvec[0].item()))\n                    brk_out[sl] = {k: float(bvec[idx].item()) / bcnt for idx, k in enumerate(brk_keys) if k != \'count\'}\n                    brk_out[sl][\'count\'] = int(bvec[0].item())\n                else:\n                    bcnt = max(1.0, brk[sl][\'count\'])\n                    brk_out[sl] = {k: brk[sl][k] / bcnt for k in brk_keys if k != \'count\'}\n                    brk_out[sl][\'count\'] = int(brk[sl][\'count\'])\n            out[\'breakdown\'] = brk_out\n        return out\n\n    res_main = _finalize(acc_main, breakdown)\n    if evaluate_m0:\n        return res_main, _finalize(acc_m0)\n    return res_main\n\n\n# ==============================================================================\n# 4. CHƯƠNG TRÌNH HUẤN LUYỆN CHÍNH PHA 3 (DDP WORKER)\n# ==============================================================================\ndef main():\n    parser = argparse.ArgumentParser(description="DDP Worker Pha 3: Fine-tune nhẹ trên Coarse TPS Đa lớp")\n    parser.add_argument(\'--train_csv\', type=str, required=True)\n    parser.add_argument(\'--val_csv\', type=str, required=True)\n    parser.add_argument(\'--train_clips_dir\', type=str, required=True)\n    parser.add_argument(\'--val_clips_dir\', type=str, required=True)\n    parser.add_argument(\'--tps_cache\', type=str, required=True, help="Đường dẫn tới tps_cache_multilayer.npz")\n    parser.add_argument(\'--m1_checkpoint\', type=str, required=True)\n    parser.add_argument(\'--init_checkpoint\', type=str, default="", help="Ưu tiên model_tps_blended.pt")\n    parser.add_argument(\'--output_dir\', type=str, default="/kaggle/working/checkpoints_tps_finetune")\n    parser.add_argument(\'--train_mode\', type=str, default="LONG", choices=["LONG"])\n    parser.add_argument(\'--epochs\', type=int, default=4, help="Số epoch nhẹ cho Pha 3 v2 (mặc định: 4)")\n    parser.add_argument(\'--grad_accum\', type=int, default=2)\n    parser.add_argument(\'--max_chunk_frames\', type=int, default=3)\n    parser.add_argument(\'--lr_backbone\', type=float, default=6.0e-6)\n    parser.add_argument(\'--lr_unet\', type=float, default=1.8e-5)\n    parser.add_argument(\'--ema_decay\', type=float, default=0.996)\n    parser.add_argument(\'--synth_count\', type=int, default=150)\n    parser.add_argument(\'--static_aug_prob\', type=float, default=0.12)\n    parser.add_argument(\'--w_ghost\', type=float, default=0.32)\n    parser.add_argument(\'--w_temp\', type=float, default=0.55)\n    parser.add_argument(\'--num_workers\', type=int, default=2)\n    parser.add_argument(\'--ram_limit_gb\', type=float, default=23.5)\n    parser.add_argument(\'--seed\', type=int, default=42)\n    parser.add_argument(\'--dry_run\', action=\'store_true\')\n    args = parser.parse_args()\n\n    local_rank = int(os.environ.get("LOCAL_RANK", 0))\n    world_size = int(os.environ.get("WORLD_SIZE", 1))\n    is_ddp = world_size > 1\n\n    if is_ddp:\n        if torch.cuda.is_available():\n            torch.cuda.set_device(local_rank)\n            device = torch.device(f\'cuda:{local_rank}\')\n            backend = \'nccl\' if dist.is_nccl_available() else \'gloo\'\n        else:\n            device = torch.device(\'cpu\')\n            backend = \'gloo\'\n        dist.init_process_group(backend=backend, init_method=os.environ.get(\'DDP_INIT_METHOD\', \'env://\'))\n    else:\n        device = torch.device(\'cuda:0\' if torch.cuda.is_available() else \'cpu\')\n\n    is_rank0 = (local_rank == 0)\n    random.seed(args.seed + local_rank)\n    np.random.seed(args.seed + local_rank)\n    torch.manual_seed(args.seed + local_rank)\n\n    tps_downscale = 2\n    if is_rank0:\n        init_tag = os.path.basename(args.init_checkpoint) if (args.init_checkpoint and os.path.exists(args.init_checkpoint)) else os.path.basename(args.m1_checkpoint)\n        print("=" * 84)\n        print(f"🚀 KHỞI ĐỘNG HUẤN LUYỆN PHA 3 DDP MULTI-LAYER TPS (INIT={init_tag})")\n        print(f"   World Size = {world_size} GPU(s) | Device = {device} | TPS Downscale = 1/{tps_downscale}")\n        print(f"   Epochs = {args.epochs} | Synth_ML = {args.synth_count} | StaticBG_Aug = {args.static_aug_prob:.2f}")\n        print(f"   w_ghost={args.w_ghost} | w_temp={args.w_temp} | LR=({args.lr_backbone:.1e}, {args.lr_unet:.1e})")\n        print("=" * 84, flush=True)\n\n    ckpt_mgr = TwoFileCheckpointManager(args.output_dir, is_rank0=is_rank0)\n    base_model = TPSInbetweenOriginal().to(device)\n    load_tps_weights(base_model, args.m1_checkpoint, device=device, prefer_ema=False, is_rank0=is_rank0)\n\n    train_ds = SakugaMultiLayerDataset(\n        args.train_csv, args.train_clips_dir, args.tps_cache,\n        train_mode="LONG", is_train=True, seed=args.seed,\n        synth_count=args.synth_count, static_aug_prob=args.static_aug_prob\n    )\n    val_ds = SakugaMultiLayerDataset(\n        args.val_csv, args.val_clips_dir, args.tps_cache,\n        train_mode="VAL", is_train=False, seed=args.seed, synth_count=0, static_aug_prob=0.0\n    )\n\n    train_sampler = DDPLengthWidthBatchSampler(\n        train_ds.samples, world_size=world_size, rank=local_rank,\n        batch_n3=4, batch_n5=2, batch_n7=1, seed=args.seed\n    )\n    train_loader = DataLoader(\n        train_ds, batch_sampler=train_sampler, collate_fn=pad_collate_fn_multilayer,\n        num_workers=args.num_workers, pin_memory=(device.type == \'cuda\'),\n        persistent_workers=(args.num_workers > 0)\n    )\n    val_sampler = DistributedSampler(val_ds, shuffle=False) if is_ddp else None\n    val_loader = DataLoader(\n        val_ds, batch_size=4, shuffle=False, sampler=val_sampler,\n        collate_fn=pad_collate_fn_multilayer, num_workers=args.num_workers,\n        pin_memory=(device.type == \'cuda\'), persistent_workers=(args.num_workers > 0), drop_last=False\n    )\n\n    if is_rank0:\n        print(f"[✓] Dữ liệu Pha 3 sẵn sàng: Train={len(train_ds):,} ({len(train_loader):,} batches/GPU) | Val={len(val_ds):,}", flush=True)\n\n    baseline_json_path = os.path.join(os.path.dirname(args.output_dir), "val_baseline_m0_m1.json")\n    has_resume = os.path.exists(ckpt_mgr.last_path)\n    need_baseline = (not has_resume and not args.dry_run and not os.path.exists(baseline_json_path))\n    if is_ddp:\n        base_flag = torch.tensor([1 if need_baseline else 0], dtype=torch.int32, device=device)\n        dist.broadcast(base_flag, src=0)\n        need_baseline = bool(base_flag.item())\n\n    m0_val_cache, m1_val_cache = None, None\n    if need_baseline:\n        if is_rank0:\n            print("\\n🔬 ĐANG ĐO BASELINE EPOCH 0 TRÊN TẬP VAL VỚI MULTI-LAYER TPS...", flush=True)\n        m1_val, m0_val = run_validation_suite_multilayer(base_model, val_loader, device, downscale_factor=tps_downscale, is_ddp=is_ddp, evaluate_m0=True)\n        m0_val_cache, m1_val_cache = m0_val, m1_val\n        if is_rank0:\n            with open(baseline_json_path, \'w\', encoding=\'utf-8\') as f:\n                json.dump({\'M0_MultiLayer\': m0_val, \'M1_Original\': m1_val}, f, indent=2, ensure_ascii=False)\n            print(f"📊 M0_Multi (Coarse): PSNR={m0_val[\'psnr\']:.2f}dB | WCD_DoG={m0_val[\'wcd_dog\']:.4f} | Ghost={m0_val[\'ghost\']:.0f}", flush=True)\n            print(f"📊 M1_Multi (Orig)  : PSNR={m1_val[\'psnr\']:.2f}dB | WCD_DoG={m1_val[\'wcd_dog\']:.4f} | Ghost={m1_val[\'ghost\']:.0f}\\n", flush=True)\n        if is_ddp:\n            dist.barrier()\n    elif os.path.exists(baseline_json_path):\n        try:\n            with open(baseline_json_path, \'r\', encoding=\'utf-8\') as f:\n                b_all = json.load(f)\n                m0_val_cache = b_all.get(\'M0_MultiLayer\', b_all.get(\'M0_Coarse\', {}))\n                m1_val_cache = b_all.get(\'M1_Original\', {})\n        except Exception:\n            m0_val_cache, m1_val_cache = None, None\n\n    # Nạp Warm-Start (model_tps_blended.pt) và đo mốc Init chuẩn để chống tụt nét\n    init_val_ghost = 9200.0\n    init_val_skel = 0.850\n    init_val_wcd_dog = 1.395\n    init_val_dens = 0.960\n    if args.init_checkpoint and os.path.exists(args.init_checkpoint):\n        if os.path.abspath(args.init_checkpoint) != os.path.abspath(args.m1_checkpoint):\n            load_tps_weights(base_model, args.init_checkpoint, device=device, prefer_ema=True, is_rank0=is_rank0)\n            if not has_resume and not args.dry_run:\n                init_val = run_validation_suite_multilayer(base_model, val_loader, device, downscale_factor=tps_downscale, is_ddp=is_ddp, evaluate_m0=False)\n                init_val_ghost = float(init_val[\'ghost\'])\n                init_val_skel = float(init_val[\'skel_recall\'])\n                init_val_wcd_dog = float(init_val[\'wcd_dog\'])\n                init_val_dens = float(init_val[\'density_ratio\'])\n                if is_rank0:\n                    print(f"📊 Init Warm-Start (Blended + Multi-Layer): PSNR={init_val[\'psnr\']:.2f}dB | WCD_DoG={init_val_wcd_dog:.4f} | Skel={init_val_skel*100:.1f}% | Dens={init_val_dens:.3f} | Ghost={init_val_ghost:.0f}\\n", flush=True)\n\n    ema = AdaptiveEMA(base_model, max_decay=args.ema_decay)\n    optimizer = torch.optim.AdamW([\n        {\'params\': [p for p in base_model.encoder.parameters() if p.requires_grad], \'lr\': args.lr_backbone, \'weight_decay\': 1e-4},\n        {\'params\': [p for p in base_model.IFNet.parameters() if p.requires_grad], \'lr\': args.lr_backbone, \'weight_decay\': 1e-4},\n        {\'params\': [p for p in base_model.unet.parameters() if p.requires_grad], \'lr\': args.lr_unet, \'weight_decay\': 1e-4},\n    ])\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=(device.type == \'cuda\'))\n    if is_ddp:\n        if is_rank0:\n            criterion = Phase3BalancedLineartLoss(train_mode="LONG", is_rank0=True, w_ghost=args.w_ghost, w_temp=args.w_temp).to(device)\n        dist.barrier()\n        if not is_rank0:\n            criterion = Phase3BalancedLineartLoss(train_mode="LONG", is_rank0=False, w_ghost=args.w_ghost, w_temp=args.w_temp).to(device)\n        dist.barrier()\n    else:\n        criterion = Phase3BalancedLineartLoss(train_mode="LONG", is_rank0=True, w_ghost=args.w_ghost, w_temp=args.w_temp).to(device)\n\n    start_epoch = 1\n    history = []\n    if not args.dry_run:\n        last_ep, hist = ckpt_mgr.try_resume(base_model, ema, optimizer, scaler, map_location=device)\n        if last_ep > 0:\n            start_epoch = last_ep + 1\n            history = hist\n            if is_rank0:\n                print(f"[♻️ AUTO-RESUME] Khôi phục thành công Epoch {last_ep:02d}. Tiếp tục Epoch {start_epoch:02d}.", flush=True)\n\n    m1_ref = m1_val_cache or {}\n    base_wcd_dog = float(m1_ref.get(\'wcd_dog\', 1.460))\n    base_semd = float(m1_ref.get(\'semd\', 0.0131))\n    base_skel = float(m1_ref.get(\'skel_recall\', 0.816))\n    base_tip = float(m1_ref.get(\'tip_recall\', 0.820))\n    base_ghost = float(m1_ref.get(\'ghost\', 1409.0))\n    best_ghost_gate = init_val_ghost * 1.02\n\n    if is_ddp:\n        ddp_kwargs = {\'device_ids\': [local_rank], \'output_device\': local_rank} if device.type == \'cuda\' else {}\n        ddp_model = DDP(base_model, find_unused_parameters=False, **ddp_kwargs)\n    else:\n        ddp_model = base_model\n\n    history_csv_path = os.path.join(os.path.dirname(args.output_dir), "training_history_finetune.csv")\n    watchdog_paused = False\n    t_total_start = time.time()\n\n    for epoch in range(start_epoch, args.epochs + 1):\n        t_ep0 = time.time()\n        if hasattr(train_sampler, \'set_epoch\'):\n            train_sampler.set_epoch(epoch)\n\n        # Lịch LR phẫu thuật 4 Epochs: Warmup -> Peak -> Cosine Decay\n        if epoch == 1:\n            lr_scale = 0.45\n        elif epoch == 2:\n            lr_scale = 1.00\n        else:\n            prog = float(epoch - 2) / float(max(1, args.epochs - 2))\n            lr_scale = max(0.15, 0.5 * (1.0 + math.cos(math.pi * prog)))\n\n        optimizer.param_groups[0][\'lr\'] = args.lr_backbone * lr_scale\n        optimizer.param_groups[1][\'lr\'] = args.lr_backbone * lr_scale\n        optimizer.param_groups[2][\'lr\'] = args.lr_unet * lr_scale\n\n        ddp_model.train()\n        optimizer.zero_grad(set_to_none=True)\n        acc_loss = 0.0\n        acc_frames = 0\n        num_batches = len(train_loader)\n        grad_accum = max(1, args.grad_accum)\n\n        for b_idx, batch in enumerate(train_loader):\n            h_pad, w_pad = batch[\'h_pad\'], batch[\'w_pad\']\n            is_direct = bool(batch.get(\'is_direct_t_flows\', False))\n            x0 = batch[\'x0\'].to(device, non_blocking=True)\n            x1 = batch[\'x1\'].to(device, non_blocking=True)\n            xk_all = batch[\'xk_all\'].to(device, non_blocking=True)\n            t_vals = batch[\'t_vals\'].to(device, non_blocking=True)\n            orig_hw = batch[\'orig_hw\'].to(device, non_blocking=True)\n            bsz, m_steps = xk_all.shape[0], xk_all.shape[1]\n\n            if is_direct:\n                # Tổng hợp hình học đa lớp (Direct-t per step)\n                cf01_steps, cf10_steps = [], []\n                for m_i in range(m_steps):\n                    cf0_i, cf1_i = reconstruct_multilayer_flow_batch_from_cache(\n                        batch[\'c0_dyn_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'th01_dyn_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'c1_dyn_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'th10_dyn_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'valid_dyn_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'dyn_pts0_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'dyn_pts1_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'dyn_pts_len_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'static_pts0_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'static_pts1_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'num_static_all\'][:, m_i].to(device, non_blocking=True),\n                        batch[\'num_clusters_all\'][:, m_i].to(device, non_blocking=True),\n                        orig_hw, h_pad, w_pad, device, downscale_factor=tps_downscale,\n                        use_dyn_anchors=False, transition_width_px=18.0, falloff_radius_px=160.0\n                    )\n                    cf01_steps.append(cf0_i)\n                    cf10_steps.append(cf1_i)\n                cf01_f = torch.stack(cf01_steps, dim=1).reshape(bsz * m_steps, 2, h_pad, w_pad)\n                cf10_f = torch.stack(cf10_steps, dim=1).reshape(bsz * m_steps, 2, h_pad, w_pad)\n            else:\n                cf01, cf10 = reconstruct_multilayer_flow_batch_from_cache(\n                    batch[\'c0_dyn\'].to(device, non_blocking=True),\n                    batch[\'th01_dyn\'].to(device, non_blocking=True),\n                    batch[\'c1_dyn\'].to(device, non_blocking=True),\n                    batch[\'th10_dyn\'].to(device, non_blocking=True),\n                    batch[\'valid_dyn\'].to(device, non_blocking=True),\n                    batch[\'dyn_pts0\'].to(device, non_blocking=True),\n                    batch[\'dyn_pts1\'].to(device, non_blocking=True),\n                    batch[\'dyn_pts_len\'].to(device, non_blocking=True),\n                    batch[\'static_pts0\'].to(device, non_blocking=True),\n                    batch[\'static_pts1\'].to(device, non_blocking=True),\n                    batch[\'num_static\'].to(device, non_blocking=True),\n                    batch[\'num_clusters\'].to(device, non_blocking=True),\n                    orig_hw, h_pad, w_pad, device, downscale_factor=tps_downscale,\n                    use_dyn_anchors=True, transition_width_px=22.0, falloff_radius_px=105.0\n                )\n                _apply_paired_spatial_aug_gpu(x0, x1, xk_all, cf01, cf10, orig_hw, is_sequence=True)\n                cf01_f = cf01.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 2, h_pad, w_pad)\n                cf10_f = cf10.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 2, h_pad, w_pad)\n\n            x0_f = x0.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 1, h_pad, w_pad)\n            x1_f = x1.unsqueeze(1).expand(-1, m_steps, -1, -1, -1).reshape(bsz * m_steps, 1, h_pad, w_pad)\n            xk_f = xk_all.reshape(bsz * m_steps, 1, h_pad, w_pad)\n            t_f = t_vals.reshape(bsz * m_steps, 1, 1, 1)\n            hw_f = orig_hw.unsqueeze(1).expand(-1, m_steps, -1).reshape(bsz * m_steps, 2)\n            seq_ids_f = torch.arange(bsz, device=device).unsqueeze(1).expand(-1, m_steps).reshape(-1)\n\n            total_f = x0_f.shape[0]\n            total_pairs = int((seq_ids_f[1:] == seq_ids_f[:-1]).sum().item()) if total_f >= 2 else 0\n            do_opt_step = ((b_idx + 1) % grad_accum == 0) or ((b_idx + 1) == num_batches)\n            chunk_sz = max(1, args.max_chunk_frames)\n\n            batch_loss_sum = 0.0\n            prev_pred_last, prev_gt_last, prev_seq_id = None, None, None\n\n            for c_start in range(0, total_f, chunk_sz):\n                c_end = min(c_start + chunk_sz, total_f)\n                need_sync = (do_opt_step and c_end == total_f)\n                sync_ctx = (ddp_model.no_sync() if (is_ddp and not need_sync) else contextlib.nullcontext())\n\n                with sync_ctx:\n                    with torch.amp.autocast(\'cuda\', enabled=(device.type == \'cuda\')):\n                        pred_c, *_ = ddp_model(\n                            x0_f[c_start:c_end], x1_f[c_start:c_end],\n                            t_f[c_start:c_end], cf01_f[c_start:c_end], cf10_f[c_start:c_end],\n                            direct_t_flows=is_direct\n                        )\n                        gt_c = xk_f[c_start:c_end]\n                        hw_c = hw_f[c_start:c_end]\n                        seq_c = seq_ids_f[c_start:c_end]\n                        l_c, d_c = criterion(pred_c, gt_c, valid_hw=hw_c)\n                        scaled_l = l_c * (float(c_end - c_start) / float(total_f * grad_accum))\n\n                        if total_pairs > 0 and criterion.w_temp > 0.0:\n                            l_temp_sum = pred_c.new_zeros(())\n                            if pred_c.shape[0] >= 2:\n                                same_m = (seq_c[1:] == seq_c[:-1])\n                                if same_m.any():\n                                    l_temp_sum = l_temp_sum + criterion.compute_temporal_pair_loss(\n                                        pred_c[:-1][same_m], pred_c[1:][same_m],\n                                        gt_c[:-1][same_m], gt_c[1:][same_m],\n                                        valid_hw=hw_c[1:][same_m]\n                                    )\n                            if prev_pred_last is not None and int(seq_c[0].item()) == prev_seq_id:\n                                l_temp_sum = l_temp_sum + criterion.compute_temporal_pair_loss(\n                                    prev_pred_last, pred_c[0:1], prev_gt_last, gt_c[0:1], valid_hw=hw_c[0:1]\n                                )\n                            scaled_l = scaled_l + (l_temp_sum / float(total_pairs * grad_accum))\n\n                    scaler.scale(scaled_l).backward()\n                    batch_loss_sum += d_c[\'total_loss\'] * float(c_end - c_start)\n                    prev_pred_last = pred_c[-1:].detach()\n                    prev_gt_last = gt_c[-1:].detach()\n                    prev_seq_id = int(seq_c[-1].item())\n\n            acc_loss += batch_loss_sum\n            acc_frames += total_f\n\n            if do_opt_step:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), max_norm=1.0)\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad(set_to_none=True)\n                ema.update(base_model)\n\n            if args.dry_run and b_idx >= 1:\n                break\n\n        train_min = (time.time() - t_ep0) / 60.0\n        if args.dry_run:\n            if is_rank0:\n                print(f"[✓ DRY-RUN PHA 3 THÀNH CÔNG] {acc_frames} frames qua Multi-Layer DDP + StaticBG Aug + EMA! Avg Loss={acc_loss/max(1,acc_frames):.4f}", flush=True)\n            break\n\n        t_val0 = time.time()\n        ema.apply_shadow(base_model)\n        val_m = run_validation_suite_multilayer(base_model, val_loader, device, downscale_factor=tps_downscale, is_ddp=is_ddp, evaluate_m0=False)\n        ema.restore(base_model)\n        val_min = (time.time() - t_val0) / 60.0\n\n        dens = float(val_m[\'density_ratio\'])\n        skel = float(val_m[\'skel_recall\'])\n        tip = float(val_m[\'tip_recall\'])\n        wcd_dog = float(val_m[\'wcd_dog\'])\n        semd = float(val_m[\'semd\'])\n        ghost = float(val_m[\'ghost\'])\n\n        # Cổng kiểm duyệt Strict Non-Regression so với Init Warm-Start (Blended):\n        # Không cho phép tụt Density, Skel Recall hay phình WCD_DoG!\n        gate_density = (0.955 <= dens <= 1.055)\n        gate_skel = (skel >= init_val_skel - 0.002)\n        gate_wcd = (wcd_dog <= init_val_wcd_dog * 1.005)\n        gate_ghost = (ghost <= best_ghost_gate)\n        tier1_passed = bool(gate_density and gate_skel and gate_wcd and gate_ghost)\n\n        term_wcd = 0.25 * (wcd_dog / max(1e-4, base_wcd_dog))\n        term_semd = 0.08 * (semd / max(1e-4, base_semd))\n        term_skel = 0.22 * ((1.0 - skel) / max(1e-4, 1.0 - base_skel))\n        term_tip = 0.09 * ((1.0 - tip) / max(1e-4, 1.0 - base_tip))\n        term_dens = 0.16 * (abs(dens - 1.00) / 0.10)\n        term_ghost = 0.20 * (ghost / max(1500.0, base_ghost * 2.0))\n        s_val = term_wcd + term_semd + term_skel + term_tip + term_dens + term_ghost\n\n        if tier1_passed:\n            comp_score = s_val\n            best_ghost_gate = min(best_ghost_gate, ghost)\n        else:\n            over_dens = max(0.0, dens - 1.055)\n            under_dens = max(0.0, 0.955 - dens)\n            under_skel = max(0.0, (init_val_skel - 0.002 - skel) * 50.0)\n            over_wcd = max(0.0, (wcd_dog - init_val_wcd_dog * 1.005) * 15.0)\n            over_ghost = max(0.0, (ghost - best_ghost_gate) / 1000.0)\n            comp_score = s_val + 10.0 + 5.0 * over_dens + 4.0 * under_dens + under_skel + over_wcd + 1.0 * over_ghost\n\n        is_best = comp_score < ckpt_mgr.best_score\n\n        if is_rank0:\n            row = {\n                \'epoch\': epoch, \'lr_scale\': round(lr_scale, 4),\n                \'train_loss\': acc_loss / max(1, acc_frames),\n                \'val_l1\': val_m[\'val_l1\'], \'psnr\': val_m[\'psnr\'], \'ssim\': val_m[\'ssim\'],\n                \'wcd\': val_m[\'wcd\'], \'wcd_legacy\': val_m[\'wcd_legacy\'],\n                \'wcd_dog\': val_m[\'wcd_dog\'], \'wcd_dog_legacy\': val_m[\'wcd_dog_legacy\'],\n                \'semd\': val_m[\'semd\'], \'skel_recall\': val_m[\'skel_recall\'],\n                \'tip_recall\': val_m[\'tip_recall\'], \'recall\': val_m[\'recall\'],\n                \'precision\': val_m[\'precision\'], \'f1\': val_m[\'f1\'],\n                \'density_ratio\': val_m[\'density_ratio\'], \'pwcr\': val_m[\'pwcr\'],\n                \'rgd\': val_m[\'rgd\'], \'ghost\': val_m[\'ghost\'],\n                \'s_val_norm\': s_val, \'score\': comp_score,\n                \'gate_passed\': tier1_passed, \'is_best\': is_best,\n                \'train_time_min\': train_min, \'val_time_min\': val_min,\n            }\n            for sl in (3, 5, 7):\n                if sl in val_m.get(\'breakdown\', {}):\n                    row[f\'psnr_N{sl}\'] = val_m[\'breakdown\'][sl][\'psnr\']\n                    row[f\'wcd_N{sl}\'] = val_m[\'breakdown\'][sl][\'wcd\']\n                    row[f\'wcd_dog_N{sl}\'] = val_m[\'breakdown\'][sl][\'wcd_dog\']\n                    row[f\'skel_recall_N{sl}\'] = val_m[\'breakdown\'][sl][\'skel_recall\']\n                    row[f\'tip_recall_N{sl}\'] = val_m[\'breakdown\'][sl][\'tip_recall\']\n                    row[f\'ghost_N{sl}\'] = val_m[\'breakdown\'][sl][\'ghost\']\n                    row[f\'density_N{sl}\'] = val_m[\'breakdown\'][sl][\'density_ratio\']\n\n            history.append(row)\n            pd.DataFrame(history).to_csv(history_csv_path, index=False)\n            ckpt_mgr.save_epoch(epoch, base_model, ema, optimizer, scaler, comp_score, history, is_best=is_best)\n\n            gate_badge = "✅ PASS" if tier1_passed else "⚠️ GATE_BREACH"\n            print(\n                f"📌 EP {epoch:02d}/{args.epochs:02d} [PHA3_ML] | Train={train_min:.2f}m | Val={val_min:.2f}m | "\n                f"Loss={row[\'train_loss\']:.4f} | PSNR={val_m[\'psnr\']:.2f}dB | "\n                f"WCD_DoG={val_m[\'wcd_dog\']:.4f} | SkelRec={val_m[\'skel_recall\']*100:.1f}% | "\n                f"Dens={dens:.3f} | Ghost={ghost:.0f} | Gate={gate_badge} | Score={comp_score:.4f}",\n                flush=True\n            )\n\n        trim_memory()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n\n        stop_flag = torch.tensor([0], dtype=torch.int32, device=device)\n        if is_rank0:\n            try:\n                import psutil\n                mem_gb = psutil.virtual_memory().used / (1024 ** 3)\n                if mem_gb >= args.ram_limit_gb:\n                    print(f"\\n⚠️ [RAM WATCHDOG]: Host RAM đạt {mem_gb:.2f} GB >= {args.ram_limit_gb} GB. Thoát an toàn mã 42!", flush=True)\n                    stop_flag.fill_(1)\n            except Exception:\n                pass\n\n        if is_ddp:\n            dist.broadcast(stop_flag, src=0)\n            dist.barrier()\n        if int(stop_flag.item()) == 1:\n            watchdog_paused = True\n            break\n\n    if is_ddp:\n        dist.destroy_process_group()\n    if watchdog_paused:\n        sys.exit(42)\n    if is_rank0 and not args.dry_run:\n        print(f"\\n🎉 HOÀN TẤT HUẤN LUYỆN PHA 3 TRONG {(time.time() - t_total_start)/60.0:.1f} PHÚT!", flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')

bench_script_path = os.path.join(WORKING_DIR, "run_benchmark_suite.py")
with open(bench_script_path, "w", encoding="utf-8") as f:
    f.write('#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n"""\nrun_benchmark_suite.py\n--------------------------------------------------------------------------------\nBỘ CÔNG CỤ BENCHMARK ĐỐI CHỨNG KHÁCH QUAN KÉP (ZERO-BIAS DUAL-BENCHMARK SUITE)\nKiểm chứng đóng góp thực tế của Dataset Custom đối với TPS-Inbetween (M1 gốc vs M3 finetune)\nvới CAM KẾT 100% ZERO ARCHITECTURAL MODIFICATION.\n\nHai bài kiểm định chuẩn hóa:\n  [Benchmark #1] Tập Chuỗi Độc Lập `sakuga_golden_benchmark_sequences_v2`\n                 (198 sequences | 149 scenes không trùng lặp | 412 frames N=3,5,7)\n    - So sánh: M0 (Coarse TPS Warp) | M1 (Original model_latest.pt) | M3_best | M3_last\n    - Dual-Regime Metrics:\n        + Regime A (Raw Continuous): PSNR, SSIM, IE-MAE, PWCR, RGD, Ghost Pixels\n        + Regime B (Thresholded tau=0.15 & DoG): WCD, WCD_DoG, S-EMD,\n          Skeleton Recall, Hair-Tip Recall, Stroke Recall/Precision/F1, Density Ratio\n    - Phân tầng theo N={3, 5, 7} & Biên độ chuyển động (mean_disp <6, 6..14, >=14 px)\n    - Thống kê chống thiên vị: Head-to-Head Win Rate (%) & Wilcoxon Signed-Rank Test\n      gom nhóm theo 149 scenes độc lập (tránh pseudoreplication).\n    - Tự động xuất Top-3 Mẫu Cải Thiện Mạnh Nhất & Top-3 Failure Cases (5 cột HUD).\n\n  [Benchmark #2] Tập Thực Tế Liên Tục `data/lineart_experiments/test`\n                 (24 frames 540p | 7 Keyframes -> 17 Inbetween frames | delta_f = 3,4,5)\n    - Dùng đúng điểm neo GlueStick chính thống (hoặc precomputed cache .npz) + pad 540->544.\n    - Hỗ trợ đối chiếu SIFT Reference tương thích bảng lịch sử evolution_timeline_master.csv.\n    - Xuất bảng CSV từng frame, từng chặng delta_f, ảnh Side-by-Side 4 cột,\n      Error Heatmap và Video đối chiếu MP4/GIF 8 FPS.\n--------------------------------------------------------------------------------\n"""\n\nimport os\nimport sys\nimport time\nimport json\nimport glob\nimport math\nimport types\nimport argparse\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nif sys.platform.startswith(\'win\'):\n    try:\n        sys.stdout.reconfigure(encoding=\'utf-8\')\n        sys.stderr.reconfigure(encoding=\'utf-8\')\n    except Exception:\n        pass\n\ntry:\n    import cv2\n    cv2.setNumThreads(0)\nexcept ImportError:\n    cv2 = None\n\ntry:\n    from scipy.stats import wilcoxon\nexcept ImportError:\n    wilcoxon = None\n\ntry:\n    import imageio\nexcept ImportError:\n    imageio = None\n\n\nSCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))\n\n\ndef _find_repo_root(start_dir):\n    cur = start_dir\n    for _ in range(7):\n        if os.path.isdir(os.path.join(cur, "checkpoints")) and os.path.isdir(os.path.join(cur, "models")):\n            return cur\n        parent = os.path.dirname(cur)\n        if parent == cur:\n            break\n        cur = parent\n    return os.path.abspath(os.path.join(start_dir, "..", "..", "..", ".."))\n\n\nREPO_ROOT = _find_repo_root(SCRIPT_DIR)\nDEFAULT_ORIG_CKPT = os.path.join(REPO_ROOT, "checkpoints", "pretrained_baselines", "tps-original", "model_latest.pt")\nDEFAULT_LINEART_TEST = os.path.join(REPO_ROOT, "data", "lineart_experiments", "test")\nDEFAULT_GLUESTICK_CACHE = os.path.join(REPO_ROOT, "kaggle_finetune_sequence_final", "lineart_test_gluestick_matches.npz")\n\n# Tái sử dụng trực tiếp toán tử từ ddp_worker nếu nằm trong cùng repo để tránh trùng lặp,\n# kèm fallback độc lập khi script chạy tách rời trên Kaggle/server khác.\n_WORKER_DIR = os.path.join(REPO_ROOT, "kaggle_finetune_sequence_final")\n_PIPELINES_DIR = os.path.join(REPO_ROOT, "pipelines")\nfor _d in (_PIPELINES_DIR, _WORKER_DIR):\n    if os.path.isdir(_d) and _d not in sys.path:\n        sys.path.insert(0, _d)\n\ntry:\n    from multi_layer_tps import solve_multi_layer_tps, segment_motion_keypoints\nexcept ImportError:\n    solve_multi_layer_tps = None\n    segment_motion_keypoints = None\n\ntry:\n    from ddp_worker_tps_sequence_final import (\n        TPSInbetweenOriginal,\n        load_tps_weights,\n        solve_tps_backward_pair,\n        solve_tps_direct,\n        tps_flow_from_theta_batch,\n        evaluate_metrics_dual_regime,\n    )\n    _IMPORTED_CORE = True\nexcept ImportError:\n    _IMPORTED_CORE = False\n\n\nif not _IMPORTED_CORE:\n    FIXED_TPS_ANCHORS = np.array([\n        [0.0, 0.0], [0.5, 0.0], [1.0, 0.0],\n        [0.0, 0.5],             [1.0, 0.5],\n        [0.0, 1.0], [0.5, 1.0], [1.0, 1.0]\n    ], dtype=np.float32)\n    _GRID_CACHE, _PIXEL_GRID_CACHE, _DOG_KERNEL_CACHE = {}, {}, {}\n\n    def warp_normalized_flow(img, flow):\n        _, _, h, w = img.shape\n        key = (h, w, flow.device)\n        base_grid = _GRID_CACHE.get(key)\n        if base_grid is None:\n            gy, gx = torch.meshgrid(\n                torch.linspace(0.0, 1.0, h, device=flow.device),\n                torch.linspace(0.0, 1.0, w, device=flow.device), indexing=\'ij\'\n            )\n            base_grid = torch.stack([gx, gy], dim=-1).unsqueeze(0).contiguous()\n            _GRID_CACHE[key] = base_grid\n        grid = (base_grid + flow.float().permute(0, 2, 3, 1)) * 2.0 - 1.0\n        return F.grid_sample(img.float().contiguous(), grid, mode=\'bilinear\', padding_mode=\'border\', align_corners=True).to(img.dtype)\n\n    def warp_pixel_flow(img, flow):\n        _, _, h, w = img.shape\n        key = (h, w, flow.device)\n        base_grid = _PIXEL_GRID_CACHE.get(key)\n        if base_grid is None:\n            gy, gx = torch.meshgrid(\n                torch.linspace(-1.0, 1.0, h, device=flow.device),\n                torch.linspace(-1.0, 1.0, w, device=flow.device), indexing=\'ij\'\n            )\n            base_grid = torch.stack([gx, gy], dim=-1).unsqueeze(0).contiguous()\n            _PIXEL_GRID_CACHE[key] = base_grid\n        flow_f = flow.float()\n        fn = torch.cat([flow_f[:, 0:1] / max(1.0, (w - 1.0) / 2.0), flow_f[:, 1:2] / max(1.0, (h - 1.0) / 2.0)], dim=1)\n        grid = (base_grid + fn.permute(0, 2, 3, 1)).contiguous()\n        return F.grid_sample(img.float().contiguous(), grid, mode=\'bilinear\', padding_mode=\'border\', align_corners=True).to(img.dtype)\n\n    def solve_tps_direct(c_src, c_dst, max_pts=128, lambd=0.01, use_anchors=False):\n        if c_src is None or c_dst is None or len(c_src) < 4:\n            return np.zeros((max_pts, 2), np.float32), np.zeros((max_pts + 3, 2), np.float32), 0.0\n        n_keep = (max_pts - 8) if use_anchors else max_pts\n        if len(c_src) > n_keep:\n            idx = np.linspace(0, len(c_src) - 1, n_keep, dtype=int)\n            cs, cd = np.asarray(c_src[idx], np.float64), np.asarray(c_dst[idx], np.float64)\n        else:\n            cs, cd = np.asarray(c_src, np.float64), np.asarray(c_dst, np.float64)\n        if use_anchors:\n            anc = FIXED_TPS_ANCHORS.astype(np.float64)\n            cs_all, cd_all = np.vstack([cs, anc]), np.vstack([cd, anc])\n        else:\n            cs_all, cd_all = cs, cd\n        m = len(cs_all)\n        r = np.linalg.norm(cs_all[:, None, :] - cs_all[None, :, :], axis=-1)\n        K = (r ** 2) * np.log(np.maximum(r, 1e-6))\n        np.fill_diagonal(K, float(lambd))\n        P = np.hstack([np.ones((m, 1), np.float64), cs_all])\n        A = np.zeros((m + 3, m + 3), np.float64)\n        A[:m, :m], A[:m, m:], A[m:, :m] = K, P, P.T\n        Y = np.zeros((m + 3, 2), np.float64)\n        Y[:m, :] = cd_all\n        try:\n            th = np.linalg.solve(A, Y)\n        except np.linalg.LinAlgError:\n            th = np.linalg.lstsq(A, Y, rcond=1e-4)[0]\n        valid = 0.0 if (np.isnan(th).any() or np.isinf(th).any() or np.max(np.abs(th)) > 50.0) else 1.0\n        cp, tp = np.zeros((max_pts, 2), np.float32), np.zeros((max_pts + 3, 2), np.float32)\n        if valid > 0.5:\n            cp[:m], tp[:m], tp[max_pts:] = cs_all.astype(np.float32), th[:m].astype(np.float32), th[m:].astype(np.float32)\n        return cp, tp, valid\n\n    def solve_tps_backward_pair(kps0, kps1, max_pts=128, lambd=0.01):\n        c0, th01, v01 = solve_tps_direct(kps1, kps0, max_pts=max_pts, lambd=lambd, use_anchors=True)\n        c1, th10, v10 = solve_tps_direct(kps0, kps1, max_pts=max_pts, lambd=lambd, use_anchors=True)\n        return c0, th01, c1, th10, (1.0 if (v01 > 0.5 and v10 > 0.5) else 0.0)\n\n    @torch.no_grad()\n    def tps_flow_from_theta_batch(theta_batch, c_src_batch, valid_mask, orig_hw, h_pad, w_pad, device, downscale_factor=2):\n        th_f, cs_f, v_f = theta_batch.float(), c_src_batch.float(), valid_mask.float()\n        b = th_f.shape[0]\n        h_sub, w_sub = max(16, h_pad // max(1, downscale_factor)), max(16, w_pad // max(1, downscale_factor))\n        gy, gx = torch.meshgrid(torch.linspace(0, 1, h_sub, device=device), torch.linspace(0, 1, w_sub, device=device), indexing=\'ij\')\n        pts_pad = torch.stack([gx, gy], dim=-1).view(1, -1, 2).expand(b, -1, -1)\n        hw_f = orig_hw.to(device=device, dtype=torch.float32)\n        sx = (float(max(1, w_pad - 1)) / torch.clamp(hw_f[:, 1] - 1.0, min=1.0)).view(b, 1, 1)\n        sy = (float(max(1, h_pad - 1)) / torch.clamp(hw_f[:, 0] - 1.0, min=1.0)).view(b, 1, 1)\n        scale_xy = torch.cat([sx, sy], dim=-1)\n        pts_raw = pts_pad * scale_xy\n        in_b = ((pts_raw[..., 0:1] <= 1.001) & (pts_raw[..., 1:2] <= 1.001)).float()\n        pts_q = torch.clamp(pts_raw, 0.0, 1.0)\n        chunks = []\n        for i in range(0, pts_q.shape[1], 32768):\n            q = pts_q[:, i:i + 32768]\n            r = torch.clamp(torch.cdist(q, cs_f, p=2.0), min=1e-6)\n            U = (r ** 2) * torch.log(r)\n            basis = torch.cat([U, torch.ones((b, q.shape[1], 1), device=device), q], dim=-1)\n            chunks.append(((torch.bmm(basis, th_f) - q) / scale_xy) * in_b[:, i:i + 32768])\n        flow = torch.cat(chunks, dim=1).view(b, h_sub, w_sub, 2).permute(0, 3, 1, 2).contiguous()\n        if h_sub != h_pad or w_sub != w_pad:\n            flow = F.interpolate(flow, size=(h_pad, w_pad), mode=\'bilinear\', align_corners=True)\n        return flow * v_f.view(b, 1, 1, 1)\n\n    def _conv(i, o, k=3, s=1, p=1):\n        return nn.Sequential(nn.Conv2d(i, o, k, s, p, bias=True), nn.PReLU(o))\n\n    def _deconv(i, o):\n        return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1, bias=True), nn.PReLU(o))\n\n    class _Conv2(nn.Module):\n        def __init__(self, i, o, s=2):\n            super().__init__()\n            self.conv1, self.conv2 = _conv(i, o, 3, s, 1), _conv(o, o, 3, 1, 1)\n        def forward(self, x):\n            return self.conv2(self.conv1(x))\n\n    class _Contextnet(nn.Module):\n        def __init__(self, indim=1, c=16):\n            super().__init__()\n            self.conv1, self.conv2 = _Conv2(indim, c), _Conv2(c, 2 * c)\n            self.conv3, self.conv4 = _Conv2(2 * c, 4 * c), _Conv2(4 * c, 8 * c)\n        def forward(self, x, flow):\n            feats = []\n            for cv_blk in (self.conv1, self.conv2, self.conv3, self.conv4):\n                x = cv_blk(x)\n                flow = F.interpolate(flow, size=x.shape[2:], mode=\'bilinear\', align_corners=False) * 0.5\n                feats.append(warp_normalized_flow(x, flow))\n            return feats\n\n    class _IFBlock(nn.Module):\n        def __init__(self, in_planes, c=64):\n            super().__init__()\n            self.conv0 = nn.Sequential(_conv(in_planes, c // 2, 3, 2, 1), _conv(c // 2, c, 3, 2, 1))\n            self.convblock = nn.Sequential(*[_conv(c, c) for _ in range(8)])\n            self.lastconv = nn.ConvTranspose2d(c, 5, 4, 2, 1)\n        def forward(self, x, flow, scale):\n            oh, ow = x.shape[2], x.shape[3]\n            if scale != 1:\n                x = F.interpolate(x, scale_factor=1.0 / scale, mode=\'bilinear\', align_corners=False)\n            if flow is not None:\n                flow = F.interpolate(flow, size=x.shape[2:], mode=\'bilinear\', align_corners=False) * (1.0 / scale)\n                x = torch.cat((x, flow), 1)\n            x = self.conv0(x.contiguous())\n            x = self.convblock(x) + x\n            tmp = F.interpolate(self.lastconv(x.float().contiguous()), size=(oh, ow), mode=\'bilinear\', align_corners=False)\n            return tmp[:, :4] * scale * 2, tmp[:, 4:5]\n\n    class _IFNet(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.block0, self.block1, self.block2 = _IFBlock(6, 240), _IFBlock(17, 150), _IFBlock(17, 90)\n        def forward(self, i0, i1, scale=(4, 2, 1)):\n            r0, r1 = i0.repeat(1, 3, 1, 1), i1.repeat(1, 3, 1, 1)\n            w0, w1, flow, mask = i0, i1, None, None\n            for i, blk in enumerate((self.block0, self.block1, self.block2)):\n                if flow is not None:\n                    fd, md = blk(torch.cat((r0, r1, w0.repeat(1, 3, 1, 1), w1.repeat(1, 3, 1, 1), mask), 1), flow, scale[i])\n                    flow, mask = flow + fd, mask + md\n                else:\n                    flow, mask = blk(torch.cat((r0, r1), 1), None, scale[i])\n                if i < 2:\n                    w0, w1 = warp_pixel_flow(i0, flow[:, :2]), warp_pixel_flow(i1, flow[:, 2:4])\n            return flow, torch.sigmoid(mask)\n\n    class _OriginalUnet(nn.Module):\n        def __init__(self, indim=9, outdim=1, c=16):\n            super().__init__()\n            self.down0, self.down1 = _Conv2(indim, 2 * c), _Conv2(4 * c, 4 * c)\n            self.down2, self.down3 = _Conv2(8 * c, 8 * c), _Conv2(16 * c, 16 * c)\n            self.up0, self.up1 = _deconv(32 * c, 8 * c), _deconv(16 * c, 4 * c)\n            self.up2, self.up3 = _deconv(8 * c, 2 * c), _deconv(4 * c, c)\n            self.conv = nn.Conv2d(c, outdim, 3, 1, 1)\n        def forward(self, img0, img1, flow0, flow1, w0, w1, mask, c0, c1):\n            s0 = self.down0(torch.cat((img0, img1, w0, w1, flow0, flow1, mask), 1))\n            s1 = self.down1(torch.cat((s0, c0[0], c1[0]), 1))\n            s2 = self.down2(torch.cat((s1, c0[1], c1[1]), 1))\n            s3 = self.down3(torch.cat((s2, c0[2], c1[2]), 1))\n            x = self.up0(torch.cat((s3, c0[3], c1[3]), 1))\n            x = self.up1(torch.cat((x, s2), 1))\n            x = self.up2(torch.cat((x, s1), 1))\n            x = self.up3(torch.cat((x, s0), 1))\n            return torch.sigmoid(self.conv(x))\n\n    class TPSInbetweenOriginal(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.encoder = _Contextnet(1, 16)\n            self.IFNet = _IFNet()\n            self.unet = _OriginalUnet(9, 1, 16)\n        def forward(self, x0, x1, t_val, cf01, cf10, direct_t_flows=False):\n            t = t_val\n            if direct_t_flows:\n                cf0t, cf1t = cf01, cf10\n            else:\n                cf0t = cf01 * (t ** 2) - (1.0 - t) * t * cf10\n                cf1t = cf10 * ((1.0 - t) ** 2) - t * (1.0 - t) * cf01\n            ci0, ci1 = warp_normalized_flow(x0, cf0t), warp_normalized_flow(x1, cf1t)\n            coarse_w = torch.clamp((1.0 - t) * ci0 + t * ci1, 0.0, 1.0)\n            f0, f1 = self.encoder(x0, cf0t), self.encoder(x1, cf1t)\n            fbc, m = self.IFNet(ci0, ci1)\n            fi0, fi1 = warp_pixel_flow(ci0, fbc[:, :2]), warp_pixel_flow(ci1, fbc[:, 2:4])\n            res = self.unet(x0, x1, cf0t + fbc[:, :2], cf1t + fbc[:, 2:4], fi0, fi1, m, f0, f1)[:, 0:1] * 2.0 - 1.0\n            It = torch.clamp((1.0 - t) * fi0 + t * fi1 + res, 0.0, 1.0)\n            return It, coarse_w, fi0, fi1, fbc, m\n\n    def load_tps_weights(model, ckpt_path, device=\'cpu\', prefer_ema=False, is_rank0=True):\n        raw = torch.load(ckpt_path, map_location=device, weights_only=False)\n        if isinstance(raw, dict):\n            if prefer_ema and raw.get(\'ema_state\') is not None:\n                eo = raw[\'ema_state\']\n                sd = eo[\'shadow\'] if isinstance(eo, dict) and \'shadow\' in eo else eo\n            elif \'model_state\' in raw:\n                sd = raw[\'model_state\']\n            elif \'state_dict\' in raw:\n                sd = raw[\'state_dict\']\n            elif \'model\' in raw:\n                sd = raw[\'model\']\n            else:\n                sd = raw\n        else:\n            sd = raw\n        msd = model.state_dict()\n        matched = {k.replace(\'module.\', \'\').replace(\'base_model.\', \'\'): v for k, v in sd.items()\n                   if k.replace(\'module.\', \'\').replace(\'base_model.\', \'\') in msd and msd[k.replace(\'module.\', \'\').replace(\'base_model.\', \'\')].shape == v.shape}\n        missing, _ = model.load_state_dict(matched, strict=False)\n        if missing:\n            raise RuntimeError(f"Missing keys: {missing[:5]}")\n        if is_rank0:\n            print(f"[✓] Nạp {os.path.basename(ckpt_path)}: {len(matched)}/{len(msd)} active tensors!")\n        return model\n\n    @torch.no_grad()\n    def evaluate_metrics_dual_regime(pred_ink, gt_ink):\n        def _dil(x): return F.max_pool2d(x, (3, 3), (1, 1), (1, 1))\n        def _ero(x): return torch.min(-F.max_pool2d(-x, (3, 1), (1, 1), (1, 0)), -F.max_pool2d(-x, (1, 3), (1, 1), (0, 1)))\n        def _skel(img):\n            im = torch.clamp(img, 0.0, 1.0)\n            s = F.relu(im - _dil(_ero(im)))\n            im2 = _ero(im)\n            return torch.clamp(s + F.relu(im2 - _dil(_ero(im2))), 0.0, 1.0)\n        def _dt(b_m, steps=10):\n            c = b_m; acc = c\n            for _ in range(steps):\n                c = F.max_pool2d(c, 3, 1, 1); acc = acc + c\n            return torch.clamp(1.0 - acc / float(steps + 1), 0.0, 1.0)\n        def _blur(im, sig):\n            r = max(1, int(round(3 * sig))); x = torch.arange(-r, r + 1, device=im.device, dtype=torch.float32)\n            g = torch.exp(-(x ** 2) / (2 * sig * sig)); g = g / g.sum()\n            k = torch.outer(g, g).view(1, 1, 2 * r + 1, 2 * r + 1).to(im.dtype)\n            return F.conv2d(F.pad(im, (r, r, r, r), mode=\'replicate\'), k)\n        def _wcd(pb, gb):\n            dg, dp = _dt(gb), _dt(pb)\n            d1 = (pb * F.softplus(dg * 10)).sum((-2, -1)) / pb.sum((-2, -1)).clamp(min=1.0)\n            d2 = (gb * F.softplus(dp * 10)).sum((-2, -1)) / gb.sum((-2, -1)).clamp(min=1.0)\n            ap, ag = pb.sum((-2, -1)), gb.sum((-2, -1))\n            diff_rel = (ap - ag).abs() / torch.minimum(ap, ag).clamp(min=1.0)\n            h = torch.sigmoid(diff_rel)\n            h_leg = 0.5 * (1.0 + torch.minimum(ap, ag) / torch.maximum(ap, ag).clamp(min=1.0))\n            return h * (d1 + d2), h_leg * (d1 + d2)\n        p, g = pred_ink.float().clamp(0, 1), gt_ink.float().clamp(0, 1)\n        mse = ((p - g) ** 2).mean((-3, -2, -1))\n        psnr = 10.0 * torch.log10(1.0 / mse.clamp(min=1e-8))\n        mp, mg = F.avg_pool2d(p, 11, 1, 5), F.avg_pool2d(g, 11, 1, 5)\n        sp, sg = F.avg_pool2d(p * p, 11, 1, 5) - mp ** 2, F.avg_pool2d(g * g, 11, 1, 5) - mg ** 2\n        spg = F.avg_pool2d(p * g, 11, 1, 5) - mp * mg\n        ssim = (((2 * mp * mg + 1e-4) * (2 * spg + 9e-4)) / ((mp ** 2 + mg ** 2 + 1e-4) * (sp + sg + 9e-4))).mean((-3, -2, -1))\n        l1 = (p - g).abs().mean((-3, -2, -1))\n        p255, g255 = (1.0 - p) * 255.0, (1.0 - g) * 255.0\n        pwcr = (p255 >= 250.0).float().mean((-3, -2, -1))\n        rgd = ((p255 >= 180.0) & (p255 < 250.0)).float().mean((-3, -2, -1))\n        ghost = (((p255 >= 180.0) & (p255 <= 250.0)).float() * (g255 > 250.0).float()).sum((-3, -2, -1))\n        dark_g = ((p > 0.35).float() * (1.0 - _dil(_dil((g > 0.08).float())))).sum((-3, -2, -1))\n        pb, gb = (p > 0.15).float(), (g > 0.15).float()\n        pd_m = ((_blur(1 - p, 1.0) - _blur(1 - p, 1.6)) < -0.02).float()\n        gd_m = ((_blur(1 - g, 1.0) - _blur(1 - g, 1.6)) < -0.02).float()\n        wcd_val, wcd_leg = _wcd(pb, gb)\n        wcd_dog_val, wcd_dog_leg = _wcd(pd_m, gd_m)\n        pn = p / p.sum((-2, -1), keepdim=True).clamp(min=1e-6); gn = g / g.sum((-2, -1), keepdim=True).clamp(min=1e-6)\n        semd = 0.5 * ((pn.sum(-1).cumsum(-1) - gn.sum(-1).cumsum(-1)).abs().mean((-2, -1)) + (pn.sum(-2).cumsum(-1) - gn.sum(-2).cumsum(-1)).abs().mean((-2, -1)))\n        gdil, pdil = _dil(_dil(gb)), _dil(_dil(pb))\n        rec = (gb * pdil).sum((-3, -2, -1)) / gb.sum((-3, -2, -1)).clamp(min=1.0)\n        prec = (pb * gdil).sum((-3, -2, -1)) / pb.sum((-3, -2, -1)).clamp(min=1.0)\n        f1 = 2 * rec * prec / (rec + prec).clamp(min=1e-6)\n        sk_g = _skel(g); sk_m = (sk_g > 0.20).float(); pd1 = _dil(pb)\n        sk_r = (pd1 * sk_m).sum((-3, -2, -1)) / sk_m.sum((-3, -2, -1)).clamp(min=1.0)\n        nbr = F.avg_pool2d(sk_g, 3, 1, 1) * 9.0 - sk_g\n        ep = ((nbr <= 1.8) & (sk_g > 0.20)).float()\n        tip_r = (pd1 * ep).sum((-3, -2, -1)) / ep.sum((-3, -2, -1)).clamp(min=1.0)\n        dens = pb.sum((-3, -2, -1)) / gb.sum((-3, -2, -1)).clamp(min=1.0)\n        return {k: v.squeeze(-1) if v.ndim > 1 else v for k, v in {\n            \'psnr\': psnr, \'ssim\': ssim, \'val_l1\': l1, \'ie_mae\': l1 * 255.0,\n            \'pwcr\': pwcr, \'rgd\': rgd, \'ghost\': ghost, \'dark_ghost\': dark_g,\n            \'wcd\': wcd_val, \'wcd_legacy\': wcd_leg, \'wcd_dog\': wcd_dog_val, \'wcd_dog_legacy\': wcd_dog_leg, \'semd\': semd,\n            \'recall\': rec, \'precision\': prec, \'f1\': f1, \'density_ratio\': dens,\n            \'skel_recall\': sk_r, \'tip_recall\': tip_r,\n        }.items()}\n\n\n# ==============================================================================\n# CÁC HÀM ĐỌC ẢNH, TRÍCH ĐIỂM NEO GLUESTICK/SIFT & VẼ BÁO CÁO TRỰC QUAN\n# ==============================================================================\ndef load_ink_image(path):\n    with Image.open(path) as im:\n        gray = np.asarray(im.convert(\'L\'), dtype=np.float32)\n    return (255.0 - gray) / 255.0\n\n\ndef ink_to_u8_gray(ink_t):\n    arr = (1.0 - ink_t.squeeze().detach().cpu().numpy().clip(0.0, 1.0)) * 255.0\n    return np.clip(np.round(arr), 0, 255).astype(np.uint8)\n\n\ndef extract_sift_matches(im0_u8, im1_u8, max_pts=120):\n    if cv2 is None:\n        return np.zeros((2, 0, 2), dtype=np.float32)\n    h, w = im0_u8.shape\n    sift = cv2.SIFT_create(nfeatures=1000)\n    k0, d0 = sift.detectAndCompute(im0_u8, None)\n    k1, d1 = sift.detectAndCompute(im1_u8, None)\n    if d0 is None or d1 is None or len(k0) < 4 or len(k1) < 4:\n        return np.zeros((2, 0, 2), dtype=np.float32)\n    matches = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True).match(d0, d1)\n    if len(matches) < 4:\n        return np.zeros((2, 0, 2), dtype=np.float32)\n    p0 = np.float32([k0[m.queryIdx].pt for m in matches])\n    p1 = np.float32([k1[m.trainIdx].pt for m in matches])\n    _, mask = cv2.findHomography(p0, p1, cv2.RANSAC, 15.0)\n    if mask is not None:\n        p0, p1 = p0[mask.ravel() == 1], p1[mask.ravel() == 1]\n    p0[:, 0] /= float(w); p0[:, 1] /= float(h)\n    p1[:, 0] /= float(w); p1[:, 1] /= float(h)\n    if len(p0) > max_pts:\n        idx = np.linspace(0, len(p0) - 1, max_pts, dtype=int)\n        p0, p1 = p0[idx], p1[idx]\n    return np.stack([p0, p1], axis=0).astype(np.float32)\n\n\ndef try_init_gluestick(device):\n    tps_repo = os.path.join(REPO_ROOT, "models", "tps-inbetween", "tps-inbetween")\n    weights_p = os.path.join(tps_repo, "model", "resources", "weights", "checkpoint_GlueStick_MD.tar")\n    if not os.path.exists(weights_p):\n        return None\n    if cv2 is not None and \'pytlsd\' not in sys.modules:\n        def _cv2_lsd(img, scale=1.0):\n            if not isinstance(img, np.ndarray):\n                img = np.array(img)\n            if img.ndim == 3:\n                img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n            if img.dtype != np.uint8:\n                img = (img * 255.0).clip(0, 255).astype(np.uint8) if img.max() <= 1.01 and img.max() > 0 else np.clip(img, 0, 255).astype(np.uint8)\n            img_s = cv2.resize(img, (int(img.shape[1] * scale), int(img.shape[0] * scale)), interpolation=cv2.INTER_AREA) if scale != 1.0 else img\n            det = cv2.createLineSegmentDetector(cv2.LSD_REFINE_STD)\n            lines, _, _, _ = det.detect(img_s)\n            if lines is None or len(lines) == 0:\n                return np.zeros((0, 5), dtype=np.float32)\n            lines = lines.reshape(-1, 4).astype(np.float32)\n            if scale != 1.0:\n                lines /= float(scale)\n            return np.hstack([lines, np.ones((len(lines), 1), dtype=np.float32)])\n        fake = types.ModuleType(\'pytlsd\')\n        fake.lsd = _cv2_lsd\n        sys.modules[\'pytlsd\'] = fake\n    if tps_repo not in sys.path:\n        sys.path.insert(0, tps_repo)\n    try:\n        from model.gluestick.models.two_view_pipeline import TwoViewPipeline\n        conf = {\n            \'name\': \'two_view_pipeline\', \'use_lines\': True,\n            \'extractor\': {\n                \'name\': \'wireframe\',\n                \'sp_params\': {\'force_num_keypoints\': False, \'max_num_keypoints\': 1000},\n                \'wireframe_params\': {\'merge_points\': True, \'merge_line_endpoints\': True},\n                \'max_n_lines\': 300,\n            },\n            \'matcher\': {\'name\': \'gluestick\', \'weights\': weights_p, \'trainable\': False},\n            \'ground_truth\': {\'from_pose_depth\': False},\n        }\n        return TwoViewPipeline(conf).to(device).eval()\n    except Exception:\n        return None\n\n\ndef compute_wilcoxon_p(a_vals, b_vals):\n    if wilcoxon is None or len(a_vals) < 10:\n        return float(\'nan\')\n    diff = np.asarray(a_vals, dtype=np.float64) - np.asarray(b_vals, dtype=np.float64)\n    if np.allclose(diff, 0.0):\n        return 1.0\n    try:\n        _, p = wilcoxon(diff, alternative=\'two-sided\', zero_method=\'wilcox\')\n        return float(p)\n    except Exception:\n        return float(\'nan\')\n\n\ndef render_audit_row_panel(im0_u8, im1_u8, m0_u8, m1_u8, m3_u8, gt_u8, title_str, stats_m1, stats_m3):\n    """Vẽ dải 5 cột chuẩn: [Overlay I0/I1 | M0 Coarse | M1 Orig | M3 Finetune | Ground Truth]."""\n    h, w = gt_u8.shape\n    overlay = (0.5 * im0_u8.astype(np.float32) + 0.5 * im1_u8.astype(np.float32)).astype(np.uint8)\n    cols = [overlay, m0_u8, m1_u8, m3_u8, gt_u8]\n    titles = [\n        ("Overlay I0/I1", (210, 210, 210)),\n        ("M0 Coarse TPS", (180, 180, 180)),\n        (f"M1 Orig ({stats_m1})", (100, 150, 255)),\n        (f"M3 Fine ({stats_m3})", (90, 240, 120)),\n        ("Ground Truth", (245, 245, 245)),\n    ]\n    head_h, foot_h = 34, 28\n    canvas = np.full((head_h + h + foot_h, w * 5, 3), 26, dtype=np.uint8)\n    for i, (arr, (lbl, clr)) in enumerate(zip(cols, titles)):\n        bgr = cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR) if cv2 is not None else np.repeat(arr[:, :, None], 3, axis=-1)\n        canvas[head_h:head_h + h, i * w:(i + 1) * w] = bgr\n        if cv2 is not None:\n            cv2.putText(canvas, lbl, (i * w + 10, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.55, clr, 2, cv2.LINE_AA)\n            if i > 0:\n                cv2.line(canvas, (i * w, 0), (i * w, head_h + h + foot_h), (70, 70, 70), 2)\n    if cv2 is not None:\n        cv2.putText(canvas, title_str, (12, head_h + h + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (220, 220, 220), 1, cv2.LINE_AA)\n    return canvas\n\n\n# ==============================================================================\n# BENCHMARK #1: TẬP CHUỖI CÁCH LY `sakuga_golden_benchmark_sequences_v2`\n# ==============================================================================\ndef run_benchmark_isolated_sequences(bench_dir, models_dict, device, out_dir, downscale_factor=2, save_visuals=True, use_multilayer=True):\n    csv_path = os.path.join(bench_dir, "benchmark_sequences.csv")\n    clips_dir = os.path.join(bench_dir, "clips")\n    matches_dir = os.path.join(bench_dir, "matches")\n    df_bench = pd.read_csv(csv_path)\n    use_ml = bool(use_multilayer and (solve_multi_layer_tps is not None))\n\n    print("\\n" + "=" * 86)\n    print(f"🔬 BENCHMARK #1: TẬP CHUỖI CÁCH LY ({len(df_bench)} sequences / {df_bench[\'scene_id\'].nunique()} scenes) | MultiLayerTPS={use_ml}")\n    print(f"   Các mô hình đối chứng: {[\'M0_Coarse\'] + list(models_dict.keys())}")\n    print("=" * 86, flush=True)\n\n    per_sample_rows = []\n    # Lưu thông tin nhẹ để sau chọn Top-3 cải thiện & Top-3 tệ nhất (chỉ lưu path + metric)\n    sample_index = []\n\n    t_start = time.time()\n    for row_idx, row in df_bench.iterrows():\n        cid = str(row[\'clip_id\'])\n        sid = str(row[\'scene_id\'])\n        seq_len = int(row[\'seq_len\'])\n        disp = float(row.get(\'mean_disp\', 0.0))\n\n        im0 = load_ink_image(os.path.join(clips_dir, cid, "000001.png"))\n        im1 = load_ink_image(os.path.join(clips_dir, cid, f"{seq_len:06d}.png"))\n        h, w = im0.shape\n        h_pad = ((h + 31) // 32) * 32\n        w_pad = ((w + 31) // 32) * 32\n\n        x0 = F.pad(torch.from_numpy(im0)[None, None], (0, w_pad - w, 0, h_pad - h)).to(device)\n        x1 = F.pad(torch.from_numpy(im1)[None, None], (0, w_pad - w, 0, h_pad - h)).to(device)\n        orig_hw = torch.tensor([[h, w]], dtype=torch.int32, device=device)\n\n        # Đọc matches chuẩn Backward-Warp\n        m_cands = sorted(glob.glob(os.path.join(matches_dir, cid, "*.npy")))\n        if m_cands:\n            kps = np.load(m_cands[0]).astype(np.float32)\n            if use_ml:\n                cf01, cf10, _ = solve_multi_layer_tps(\n                    kps[0], kps[1], (h, w), h_pad, w_pad, device,\n                    direct_t_flows=False, max_pts=128, lambd=0.01,\n                    downscale_factor=downscale_factor, eps_static_px=1.5,\n                    use_dyn_anchors=True, transition_width_px=22.0, falloff_radius_px=105.0\n                )\n            else:\n                c0_np, th01_np, c1_np, th10_np, v_flag = solve_tps_backward_pair(kps[0], kps[1], max_pts=128, lambd=0.01)\n                c0_t = torch.from_numpy(c0_np)[None].to(device)\n                th01_t = torch.from_numpy(th01_np)[None].to(device)\n                c1_t = torch.from_numpy(c1_np)[None].to(device)\n                th10_t = torch.from_numpy(th10_np)[None].to(device)\n                v_t = torch.tensor([v_flag], dtype=torch.float32, device=device)\n                cf01 = tps_flow_from_theta_batch(th01_t, c0_t, v_t, orig_hw, h_pad, w_pad, device, downscale_factor=downscale_factor)\n                cf10 = tps_flow_from_theta_batch(th10_t, c1_t, v_t, orig_hw, h_pad, w_pad, device, downscale_factor=downscale_factor)\n        else:\n            cf01 = torch.zeros((1, 2, h_pad, w_pad), dtype=torch.float32, device=device)\n            cf10 = torch.zeros((1, 2, h_pad, w_pad), dtype=torch.float32, device=device)\n\n        for k in range(1, seq_len - 1):\n            fk = k + 1\n            t_val = float(k) / float(seq_len - 1)\n            t_tensor = torch.tensor([[[[t_val]]]], dtype=torch.float32, device=device)\n            gt_ink = torch.from_numpy(load_ink_image(os.path.join(clips_dir, cid, f"{fk:06d}.png")))[None, None].to(device)\n\n            sample_rec = {\n                \'clip_id\': cid, \'scene_id\': sid, \'seq_len\': seq_len, \'frame_k\': fk,\n                \'t_val\': round(t_val, 4), \'mean_disp\': disp, \'orig_w\': w, \'orig_h\': h\n            }\n            coarse_ref = None\n            for m_name, mdl in models_dict.items():\n                if device.type == \'cuda\':\n                    torch.cuda.synchronize(device)\n                ts = time.time()\n                with torch.no_grad(), torch.amp.autocast(\'cuda\', enabled=False):\n                    pred_p, cw_p, *_ = mdl(x0.float(), x1.float(), t_tensor.float(), cf01.float(), cf10.float())\n                if device.type == \'cuda\':\n                    torch.cuda.synchronize(device)\n                lat_ms = (time.time() - ts) * 1000.0\n\n                pred_crop = pred_p[:, :, :h, :w]\n                if coarse_ref is None:\n                    coarse_ref = cw_p[:, :, :h, :w]\n                    mm0 = evaluate_metrics_dual_regime(coarse_ref, gt_ink)\n                    for mk, mv in mm0.items():\n                        sample_rec[f"M0_{mk}"] = float(mv.item())\n\n                mm = evaluate_metrics_dual_regime(pred_crop, gt_ink)\n                for mk, mv in mm.items():\n                    sample_rec[f"{m_name}_{mk}"] = float(mv.item())\n                sample_rec[f"{m_name}_latency_ms"] = float(lat_ms)\n\n            per_sample_rows.append(sample_rec)\n            sample_index.append(sample_rec)\n\n        if (row_idx + 1) % 50 == 0 or (row_idx + 1) == len(df_bench):\n            print(f"   ... Hoàn thành {row_idx + 1}/{len(df_bench)} sequences ({time.time() - t_start:.1f}s)", flush=True)\n\n    df_res = pd.DataFrame(per_sample_rows)\n    os.makedirs(out_dir, exist_ok=True)\n    df_res.to_csv(os.path.join(out_dir, "benchmark_per_sample_comparison.csv"), index=False)\n\n    # Tổng hợp bảng so sánh toàn cục cho tất cả model tags\n    all_tags = [\'M0\'] + list(models_dict.keys())\n    metric_cols = [\'psnr\', \'ssim\', \'ie_mae\', \'pwcr\', \'rgd\', \'ghost\', \'wcd\', \'wcd_legacy\', \'wcd_dog\', \'wcd_dog_legacy\',\n                   \'semd\', \'skel_recall\', \'tip_recall\', \'recall\', \'precision\', \'f1\', \'density_ratio\']\n    summary_rows = []\n    for tag in all_tags:\n        srow = {\'model_tag\': tag, \'total_frames\': len(df_res), \'total_scenes\': df_res[\'scene_id\'].nunique()}\n        for mc in metric_cols:\n            col = f"{tag}_{mc}"\n            if col in df_res.columns:\n                srow[mc] = float(df_res[col].mean())\n        lat_col = f"{tag}_latency_ms"\n        srow[\'latency_ms\'] = float(df_res[lat_col].mean()) if lat_col in df_res.columns else 0.0\n        summary_rows.append(srow)\n    df_summary = pd.DataFrame(summary_rows)\n    df_summary.to_csv(os.path.join(out_dir, "benchmark_overall_summary.csv"), index=False)\n\n    # Phân tầng theo seq_len N={3,5,7} và Motion Tiers (<6, 6..14, >=14)\n    strat_rows = []\n    for sl in (3, 5, 7):\n        sub = df_res[df_res[\'seq_len\'] == sl]\n        if len(sub) == 0:\n            continue\n        for tag in all_tags:\n            r = {\'group_type\': \'seq_len\', \'group_val\': f"N={sl}", \'model_tag\': tag, \'frames\': len(sub)}\n            for mc in [\'psnr\', \'ssim\', \'wcd\', \'wcd_legacy\', \'wcd_dog\', \'wcd_dog_legacy\', \'semd\', \'skel_recall\', \'tip_recall\', \'f1\', \'ghost\', \'density_ratio\']:\n                r[mc] = float(sub[f"{tag}_{mc}"].mean())\n            strat_rows.append(r)\n\n    disp_bins = [(\'Small_<6px\', df_res[\'mean_disp\'] < 6.0),\n                 (\'Med_6_14px\', (df_res[\'mean_disp\'] >= 6.0) & (df_res[\'mean_disp\'] < 14.0)),\n                 (\'Large_>=14px\', df_res[\'mean_disp\'] >= 14.0)]\n    for dlabel, mask in disp_bins:\n        sub = df_res[mask]\n        if len(sub) == 0:\n            continue\n        for tag in all_tags:\n            r = {\'group_type\': \'motion_disp\', \'group_val\': dlabel, \'model_tag\': tag, \'frames\': len(sub)}\n            for mc in [\'psnr\', \'ssim\', \'wcd\', \'wcd_legacy\', \'wcd_dog\', \'wcd_dog_legacy\', \'semd\', \'skel_recall\', \'tip_recall\', \'f1\', \'ghost\']:\n                r[mc] = float(sub[f"{tag}_{mc}"].mean())\n            strat_rows.append(r)\n    pd.DataFrame(strat_rows).to_csv(os.path.join(out_dir, "benchmark_stratified_breakdown.csv"), index=False)\n\n    # Kiểm định thống kê không thiên vị M3_best (hoặc model thứ 2) vs M1_Original\n    stat_audit = {\'overall\': summary_rows}\n    if \'M1_Original\' in models_dict:\n        compare_tag = \'M3_Best\' if \'M3_Best\' in models_dict else next((k for k in models_dict if k != \'M1_Original\'), None)\n        if compare_tag is not None:\n            # Gom nhóm trung bình theo từng scene độc lập (149 scenes) trước Wilcoxon test!\n            scene_df = df_res.groupby(\'scene_id\').mean(numeric_only=True)\n            h2h = {}\n            for mc, better_higher in [(\'wcd\', False), (\'wcd_dog\', False), (\'semd\', False),\n                                      (\'skel_recall\', True), (\'tip_recall\', True), (\'f1\', True), (\'psnr\', True), (\'ssim\', True)]:\n                c_m1, c_m3 = f"M1_Original_{mc}", f"{compare_tag}_{mc}"\n                if better_higher:\n                    win_rate = float(np.mean(df_res[c_m3].values > df_res[c_m1].values))\n                else:\n                    win_rate = float(np.mean(df_res[c_m3].values < df_res[c_m1].values))\n                pval = compute_wilcoxon_p(scene_df[c_m3].values, scene_df[c_m1].values)\n                h2h[mc] = {\n                    \'m1_mean\': float(df_res[c_m1].mean()),\n                    \'m3_mean\': float(df_res[c_m3].mean()),\n                    \'delta\': float(df_res[c_m3].mean() - df_res[c_m1].mean()),\n                    \'frame_win_rate_pct\': round(win_rate * 100.0, 2),\n                    \'scene_wilcoxon_p\': pval,\n                    \'significant_p001\': (not math.isnan(pval)) and (pval < 0.01),\n                }\n            # Kết luận H0 vs H1 tự động dựa trên 3 trụ cột: Hình học + Tô-pô + Không dày nét (Anti-Thickening)\n            geom_pass = (h2h[\'wcd_dog\'][\'delta\'] < 0 and h2h[\'wcd_dog\'][\'significant_p001\']) or (h2h[\'wcd\'][\'delta\'] < 0 and h2h[\'wcd\'][\'significant_p001\'])\n            topo_pass = h2h[\'skel_recall\'][\'delta\'] > 0 and h2h[\'skel_recall\'][\'significant_p001\']\n            dens_m3 = float(df_res[f"{compare_tag}_density_ratio"].mean())\n            anti_thickening_pass = (0.95 <= dens_m3 <= 1.08)\n            stat_audit[\'head_to_head_M3_vs_M1\'] = h2h\n            if geom_pass and topo_pass and anti_thickening_pass:\n                stat_audit[\'hypothesis_verdict\'] = "H1_SUPPORTED (Cải thiện đồng thời cả Hình học + Tô-pô nét vẽ với p < 0.01 & Giữ mật độ nét chuẩn [0.95, 1.08])"\n            elif geom_pass and topo_pass and not anti_thickening_pass:\n                stat_audit[\'hypothesis_verdict\'] = f"PARTIAL_WITH_THICKENING (WCD/Skel cải thiện nhưng mật độ nét đạt {dens_m3:.3f} vượt ngưỡng kiểm soát)"\n            else:\n                stat_audit[\'hypothesis_verdict\'] = "MIXED_OR_H0 (Cần xem xét kỹ phân tách Raw vs DoG và chỉ số Skel/WCD)"\n\n            # Xuất Top-3 Cải Thiện & Top-3 Failure Cases theo delta WCD_DoG\n            if save_visuals and len(df_res) >= 6:\n                df_res[\'delta_wcd_dog\'] = df_res[f"{compare_tag}_wcd_dog"] - df_res["M1_Original_wcd_dog"]\n                best3 = df_res.sort_values(\'delta_wcd_dog\', ascending=True).head(3)\n                worst3 = df_res.sort_values(\'delta_wcd_dog\', ascending=False).head(3)\n                vis_dir = os.path.join(out_dir, "audit_cases_top3")\n                os.makedirs(vis_dir, exist_ok=True)\n                for tag_grp, sub_df in [("TOP3_IMPROVED", best3), ("TOP3_FAILURE_OR_HARD", worst3)]:\n                    panels = []\n                    for _, sr in sub_df.iterrows():\n                        cid, fk, sl = sr[\'clip_id\'], int(sr[\'frame_k\']), int(sr[\'seq_len\'])\n                        im0_l = load_ink_image(os.path.join(clips_dir, cid, "000001.png"))\n                        im1_l = load_ink_image(os.path.join(clips_dir, cid, f"{sl:06d}.png"))\n                        gt_l = load_ink_image(os.path.join(clips_dir, cid, f"{fk:06d}.png"))\n                        hh, ww = im0_l.shape\n                        hp, wp = ((hh + 31) // 32) * 32, ((ww + 31) // 32) * 32\n                        mc = sorted(glob.glob(os.path.join(matches_dir, cid, "*.npy")))\n                        if mc:\n                            kps = np.load(mc[0]).astype(np.float32)\n                            c0n, t01n, c1n, t10n, vf = solve_tps_backward_pair(kps[0], kps[1], 128, 0.01)\n                        else:\n                            c0n = c1n = np.zeros((128, 2), np.float32); t01n = t10n = np.zeros((131, 2), np.float32); vf = 0.0\n                        x0p = F.pad(torch.from_numpy(im0_l)[None, None], (0, wp - ww, 0, hp - hh)).to(device)\n                        x1p = F.pad(torch.from_numpy(im1_l)[None, None], (0, wp - ww, 0, hp - hh)).to(device)\n                        hw_t = torch.tensor([[hh, ww]], dtype=torch.int32, device=device)\n                        cf01_ = tps_flow_from_theta_batch(torch.from_numpy(t01n)[None].to(device), torch.from_numpy(c0n)[None].to(device), torch.tensor([vf], device=device), hw_t, hp, wp, device, downscale_factor)\n                        cf10_ = tps_flow_from_theta_batch(torch.from_numpy(t10n)[None].to(device), torch.from_numpy(c1n)[None].to(device), torch.tensor([vf], device=device), hw_t, hp, wp, device, downscale_factor)\n                        tt = torch.tensor([[[[float(sr[\'t_val\'])]]]], dtype=torch.float32, device=device)\n                        with torch.no_grad():\n                            pr1, cw_, *_ = models_dict[\'M1_Original\'](x0p, x1p, tt, cf01_, cf10_)\n                            pr3, *_ = models_dict[compare_tag](x0p, x1p, tt, cf01_, cf10_)\n                        panel = render_audit_row_panel(\n                            ink_to_u8_gray(x0p[:, :, :hh, :ww]), ink_to_u8_gray(x1p[:, :, :hh, :ww]),\n                            ink_to_u8_gray(cw_[:, :, :hh, :ww]), ink_to_u8_gray(pr1[:, :, :hh, :ww]),\n                            ink_to_u8_gray(pr3[:, :, :hh, :ww]), (255.0 - gt_l * 255.0).clip(0, 255).astype(np.uint8),\n                            f"[{tag_grp}] {cid} (N={sl}, k={fk}, t={sr[\'t_val\']:.2f}, disp={sr[\'mean_disp\']:.1f}px) | dWCD_DoG={sr[\'delta_wcd_dog\']:+.4f}",\n                            f"WCD={sr[\'M1_Original_wcd_dog\']:.3f}|Sk={sr[\'M1_Original_skel_recall\']*100:.0f}%",\n                            f"WCD={sr[f\'{compare_tag}_wcd_dog\']:.3f}|Sk={sr[f\'{compare_tag}_skel_recall\']*100:.0f}%"\n                        )\n                        panels.append(panel)\n                    # Ghép 3 hàng có thể hơi khác chiều rộng -> đệm ngang về max_w\n                    max_pw = max(p.shape[1] for p in panels)\n                    norm_panels = [np.pad(p, ((0, 0), (0, max_pw - p.shape[1]), (0, 0)), constant_values=26) for p in panels]\n                    stacked = np.vstack(norm_panels)\n                    Image.fromarray(stacked[:, :, ::-1] if cv2 is not None else stacked).save(os.path.join(vis_dir, f"{tag_grp}.png"))\n\n    with open(os.path.join(out_dir, "benchmark_statistical_audit.json"), \'w\', encoding=\'utf-8\') as f:\n        json.dump(stat_audit, f, indent=2, ensure_ascii=False)\n\n    print("\\n📊 KẾT QUẢ BENCHMARK #1 (ISOLATED 198 CLIPS / 412 FRAMES):")\n    print(df_summary.round(4).to_string(index=False))\n    if \'head_to_head_M3_vs_M1\' in stat_audit:\n        print(f"\\n⚖️ KIỂM ĐỊNH ĐỐI ĐẦU THEO SCENE (149 SCENES) | Kết luận: {stat_audit[\'hypothesis_verdict\']}")\n        for mc, st in stat_audit[\'head_to_head_M3_vs_M1\'].items():\n            print(f"   - {mc:12s}: M1={st[\'m1_mean\']:.4f} -> M3={st[\'m3_mean\']:.4f} (Delta={st[\'delta\']:+.4f} | WinRate={st[\'frame_win_rate_pct\']:.1f}% | p={st[\'scene_wilcoxon_p\']:.2e})")\n    return df_summary, stat_audit\n\n\n# ==============================================================================\n# BENCHMARK #2: TẠO FRAME THỰC TẾ LIÊN TỤC (`data/lineart_experiments/test`)\n# ==============================================================================\ndef run_benchmark_continuous_lineart_test(test_dir, models_dict, device, out_dir, gluestick_npz=None, downscale_factor=2, use_multilayer=True):\n    spec_path = os.path.join(test_dir, "continuous_keyframes_spec.json")\n    if not os.path.exists(spec_path):\n        print(f"[*] Không tìm thấy {spec_path}, bỏ qua Benchmark #2.")\n        return None\n\n    with open(spec_path, \'r\', encoding=\'utf-8\') as f:\n        spec = json.load(f)\n    frames_spec = spec[\'frames\']\n    key_indices = sorted(spec[\'key_indices\'])\n    use_ml = bool(use_multilayer and (solve_multi_layer_tps is not None))\n\n    print("\\n" + "=" * 86)\n    print(f"🎬 BENCHMARK #2: TẠO FRAME THỰC TẾ LIÊN TỤC ({len(frames_spec)} frames: {len(key_indices)} keyframes, {len(frames_spec)-len(key_indices)} inbetweens) | MultiLayerTPS={use_ml}")\n    print("=" * 86, flush=True)\n\n    # Nạp ảnh GT 960x540\n    gt_imgs_u8 = [np.asarray(Image.open(os.path.join(test_dir, item[\'filename\'])).convert(\'L\'), dtype=np.uint8) for item in frames_spec]\n    h, w = gt_imgs_u8[0].shape\n    h_pad, w_pad = ((h + 31) // 32) * 32, ((w + 31) // 32) * 32  # 540 -> 544, 960 -> 960\n\n    # Chuẩn bị cặp điểm neo GlueStick (ưu tiên precomputed .npz -> chạy GlueStick trực tiếp -> SIFT fallback)\n    gs_cache = dict(np.load(gluestick_npz)) if (gluestick_npz and os.path.exists(gluestick_npz)) else None\n    gs_model = None if gs_cache is not None else try_init_gluestick(device)\n    match_source = "GlueStick_MD_Precomputed" if gs_cache is not None else ("GlueStick_MD_Live" if gs_model is not None else "SIFT_Fallback")\n    print(f"[*] Bộ trích xuất điểm neo hình học: {match_source} | Pad: {w}x{h} -> {w_pad}x{h_pad}", flush=True)\n\n    seg_data = {}\n    for i in range(len(key_indices) - 1):\n        ka, kb = key_indices[i], key_indices[i + 1]\n        seg_key = f"seg_{ka}_{kb}"\n        ima_u8, imb_u8 = gt_imgs_u8[ka], gt_imgs_u8[kb]\n        if gs_cache is not None and seg_key in gs_cache:\n            kps = gs_cache[seg_key].astype(np.float32)\n        elif gs_model is not None:\n            t0_g = torch.from_numpy(ima_u8.astype(np.float32) / 255.0)[None, None].to(device)\n            t1_g = torch.from_numpy(imb_u8.astype(np.float32) / 255.0)[None, None].to(device)\n            with torch.no_grad():\n                pred_g = gs_model({\'image0\': t0_g, \'image1\': t1_g})\n            m0 = pred_g[\'matches0\'][0].cpu().numpy()\n            valid_g = m0 != -1\n            kp0 = pred_g[\'keypoints0\'][0].cpu().numpy()[valid_g].astype(np.float32)\n            kp1 = pred_g[\'keypoints1\'][0].cpu().numpy()[m0[valid_g]].astype(np.float32)\n            kp0[:, 0] /= float(w); kp0[:, 1] /= float(h)\n            kp1[:, 0] /= float(w); kp1[:, 1] /= float(h)\n            kps = np.stack([kp0, kp1], axis=0)\n        else:\n            kps = extract_sift_matches(ima_u8, imb_u8, max_pts=120)\n\n        x0_p = F.pad(torch.from_numpy((255.0 - ima_u8.astype(np.float32)) / 255.0)[None, None], (0, w_pad - w, 0, h_pad - h)).to(device)\n        x1_p = F.pad(torch.from_numpy((255.0 - imb_u8.astype(np.float32)) / 255.0)[None, None], (0, w_pad - w, 0, h_pad - h)).to(device)\n        hw_t = torch.tensor([[h, w]], dtype=torch.int32, device=device)\n        if use_ml:\n            cf01, cf10, _ = solve_multi_layer_tps(\n                kps[0], kps[1], (h, w), h_pad, w_pad, device,\n                direct_t_flows=False, max_pts=128, lambd=0.01,\n                downscale_factor=downscale_factor, eps_static_px=1.5,\n                use_dyn_anchors=True, transition_width_px=22.0, falloff_radius_px=105.0\n            )\n        else:\n            c0_np, t01_np, c1_np, t10_np, vf = solve_tps_backward_pair(kps[0], kps[1], max_pts=128, lambd=0.01)\n            cf01 = tps_flow_from_theta_batch(torch.from_numpy(t01_np)[None].to(device), torch.from_numpy(c0_np)[None].to(device), torch.tensor([vf], device=device), hw_t, h_pad, w_pad, device, downscale_factor)\n            cf10 = tps_flow_from_theta_batch(torch.from_numpy(t10_np)[None].to(device), torch.from_numpy(c1_np)[None].to(device), torch.tensor([vf], device=device), hw_t, h_pad, w_pad, device, downscale_factor)\n        seg_data[(ka, kb)] = {\'x0\': x0_p, \'x1\': x1_p, \'cf01\': cf01, \'cf10\': cf10, \'num_matches\': kps.shape[1]}\n\n    os.makedirs(out_dir, exist_ok=True)\n    reconstructed = {tag: [img.copy() for img in gt_imgs_u8] for tag in [\'M0\'] + list(models_dict.keys())}\n    per_frame_rows = []\n\n    for item in frames_spec:\n        f_idx = int(item[\'frame_idx\'])\n        if item[\'type\'] != \'INBETWEEN\':\n            continue\n        ka, kb, tv = int(item[\'ref_key_a\']), int(item[\'ref_key_b\']), float(item[\'t_val\'])\n        delta_f = kb - ka\n        seg = seg_data[(ka, kb)]\n        gt_ink = torch.from_numpy((255.0 - gt_imgs_u8[f_idx].astype(np.float32)) / 255.0)[None, None].to(device)\n        tt = torch.tensor([[[[tv]]]], dtype=torch.float32, device=device)\n\n        row_base = {\'frame_idx\': f_idx, \'time_sec\': item[\'time_sec\'], \'filename\': item[\'filename\'],\n                    \'segment\': f"{ka}->{kb}", \'delta_f\': delta_f, \'t_val\': round(tv, 4), \'matches\': seg[\'num_matches\']}\n        cw_done = False\n        for m_name, mdl in models_dict.items():\n            ts = time.time()\n            with torch.no_grad(), torch.amp.autocast(\'cuda\', enabled=False):\n                pr_p, cw_p, *_ = mdl(seg[\'x0\'].float(), seg[\'x1\'].float(), tt.float(), seg[\'cf01\'].float(), seg[\'cf10\'].float())\n            if device.type == \'cuda\':\n                torch.cuda.synchronize(device)\n            lat = (time.time() - ts) * 1000.0\n            pr_crop = pr_p[:, :, :h, :w]\n            reconstructed[m_name][f_idx] = ink_to_u8_gray(pr_crop)\n            mm = evaluate_metrics_dual_regime(pr_crop, gt_ink)\n\n            if not cw_done:\n                cw_crop = cw_p[:, :, :h, :w]\n                reconstructed[\'M0\'][f_idx] = ink_to_u8_gray(cw_crop)\n                mm0 = evaluate_metrics_dual_regime(cw_crop, gt_ink)\n                for mk, mv in mm0.items():\n                    row_base[f"M0_{mk}"] = float(mv.item())\n                cw_done = True\n\n            for mk, mv in mm.items():\n                row_base[f"{m_name}_{mk}"] = float(mv.item())\n            row_base[f"{m_name}_latency_ms"] = float(lat)\n\n        per_frame_rows.append(row_base)\n\n    # Tính chỉ số nhất quán thời gian (2C: Temporal Difference MAE & Background Flicker) trên toàn chuỗi liên tiếp\n    tags = [\'M0\'] + list(models_dict.keys())\n    for row in per_frame_rows:\n        fi = int(row[\'frame_idx\'])\n        g_cur = (255.0 - gt_imgs_u8[fi].astype(np.float32)) / 255.0\n        g_prev = (255.0 - gt_imgs_u8[fi - 1].astype(np.float32)) / 255.0\n        both_bg = (g_cur < 0.04) & (g_prev < 0.04)\n        dg = g_cur - g_prev\n        for t in tags:\n            p_cur = (255.0 - reconstructed[t][fi].astype(np.float32)) / 255.0\n            p_prev = (255.0 - reconstructed[t][fi - 1].astype(np.float32)) / 255.0\n            dp = p_cur - p_prev\n            row[f"{t}_temp_diff_mae"] = float(np.mean(np.abs(dp - dg)) * 255.0)\n            row[f"{t}_temp_bg_flicker"] = float(np.sum(np.abs(dp) * both_bg))\n\n    df_frames = pd.DataFrame(per_frame_rows)\n    df_frames.to_csv(os.path.join(out_dir, "lineart_test_per_frame_metrics.csv"), index=False)\n\n    # Tổng hợp theo toàn chuỗi & theo từng delta_f (3, 4, 5)\n    summary_list = []\n    for grp_name, sub_df in [("ALL_17_INBETWEENS", df_frames)] + [(f"delta_f={dfv}", df_frames[df_frames[\'delta_f\'] == dfv]) for dfv in sorted(df_frames[\'delta_f\'].unique())]:\n        for t in tags:\n            sr = {\'group\': grp_name, \'model_tag\': t, \'frames\': len(sub_df)}\n            for mc in [\'psnr\', \'ssim\', \'ie_mae\', \'wcd\', \'wcd_dog\', \'semd\', \'skel_recall\', \'tip_recall\', \'f1\',\n                       \'density_ratio\', \'ghost\', \'pwcr\', \'rgd\', \'temp_diff_mae\', \'temp_bg_flicker\']:\n                sr[mc] = float(sub_df[f"{t}_{mc}"].mean())\n            summary_list.append(sr)\n    df_sum2 = pd.DataFrame(summary_list)\n    df_sum2.to_csv(os.path.join(out_dir, "lineart_test_summary.csv"), index=False)\n\n    print("\\n📊 KẾT QUẢ BENCHMARK #2 (ALL 17 INBETWEEN FRAMES - GLUESTICK - FP32):")\n    print(df_sum2[df_sum2[\'group\'] == \'ALL_17_INBETWEENS\'].round(4).to_string(index=False))\n\n    # Xuất Video / GIF 24 frames, Error Heatmaps & Side-by-Side (nếu có cv2 hoặc imageio)\n    m1_tag = \'M1_Original\' if \'M1_Original\' in models_dict else None\n    m3_tag = \'M3_Best\' if \'M3_Best\' in models_dict else next((k for k in models_dict if k != \'M1_Original\'), None)\n    if m1_tag and m3_tag and cv2 is not None:\n        sbs_dir = os.path.join(out_dir, "frames_side_by_side")\n        heat_dir = os.path.join(out_dir, "error_heatmaps")\n        os.makedirs(sbs_dir, exist_ok=True)\n        os.makedirs(heat_dir, exist_ok=True)\n        gif_frames = []\n        montage_rows = []\n        # Vẽ thanh Side-by-Side 4 cột [Overlay Keyframes | Original M1 | Finetuned M3 | Ground Truth]\n        for item in frames_spec:\n            idx = int(item[\'frame_idx\'])\n            is_k = (item[\'type\'] == \'KEYFRAME\')\n            gt_u8 = gt_imgs_u8[idx]\n            m1_u8 = reconstructed[m1_tag][idx]\n            m3_u8 = reconstructed[m3_tag][idx]\n            if is_k:\n                ov_u8 = gt_u8\n                s1, s3 = "KEYFRAME", "KEYFRAME"\n            else:\n                ka, kb = int(item[\'ref_key_a\']), int(item[\'ref_key_b\'])\n                ov_u8 = (0.5 * gt_imgs_u8[ka].astype(np.float32) + 0.5 * gt_imgs_u8[kb].astype(np.float32)).astype(np.uint8)\n                fr_row = df_frames[df_frames[\'frame_idx\'] == idx].iloc[0]\n                s1 = f"PSNR:{fr_row[f\'{m1_tag}_psnr\']:.1f}|Sk:{fr_row[f\'{m1_tag}_skel_recall\']*100:.0f}%"\n                s3 = f"PSNR:{fr_row[f\'{m3_tag}_psnr\']:.1f}|Sk:{fr_row[f\'{m3_tag}_skel_recall\']*100:.0f}%"\n\n                # Bản đồ nhiệt sai số: [M1 Error Heatmap | M3 Error Heatmap]\n                d1 = np.clip(np.abs(m1_u8.astype(np.float32) - gt_u8.astype(np.float32)) * 2.5, 0, 255).astype(np.uint8)\n                d3 = np.clip(np.abs(m3_u8.astype(np.float32) - gt_u8.astype(np.float32)) * 2.5, 0, 255).astype(np.uint8)\n                hm_pair = np.hstack([cv2.applyColorMap(d1, cv2.COLORMAP_JET), cv2.applyColorMap(d3, cv2.COLORMAP_JET)])\n                hm_head = np.full((32, w * 2, 3), 24, dtype=np.uint8)\n                cv2.putText(hm_head, f"M1 ORIG ERROR [MAE={fr_row[f\'{m1_tag}_ie_mae\']:.2f}]", (12, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (100, 150, 255), 2, cv2.LINE_AA)\n                cv2.putText(hm_head, f"M3 FINE ERROR [MAE={fr_row[f\'{m3_tag}_ie_mae\']:.2f}]", (w + 12, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (80, 240, 120), 2, cv2.LINE_AA)\n                cv2.imwrite(os.path.join(heat_dir, f"heatmap_{idx:02d}_{item[\'filename\']}"), np.vstack([hm_head, hm_pair]))\n\n            head_h = 34\n            canvas = np.full((h + head_h, w * 4, 3), 24, dtype=np.uint8)\n            cols = [(ov_u8, f"INPUT [{item[\'filename\']}]", (210, 210, 210)),\n                    (m1_u8, f"M1 ORIG ({s1})", (100, 150, 255)),\n                    (m3_u8, f"M3 FINE ({s3})", (80, 240, 120)),\n                    (gt_u8, "GROUND TRUTH", (245, 245, 245))]\n            for ci, (arr, lbl, colr) in enumerate(cols):\n                canvas[head_h:, ci * w:(ci + 1) * w] = cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)\n                cv2.putText(canvas, lbl, (ci * w + 10, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.52, colr, 2, cv2.LINE_AA)\n                if ci > 0:\n                    cv2.line(canvas, (ci * w, 0), (ci * w, h + head_h), (70, 70, 70), 2)\n            out_png = os.path.join(sbs_dir, f"frame_{idx:02d}_{item[\'filename\']}")\n            cv2.imwrite(out_png, canvas)\n            half_frame = cv2.resize(canvas, (w * 2, (h + head_h) // 2))\n            gif_frames.append(half_frame[:, :, ::-1])\n            if (not is_k) and (idx in {2, 5, 9, 12, 17, 21}):\n                montage_rows.append(half_frame)\n\n        if montage_rows:\n            cv2.imwrite(os.path.join(out_dir, "montage_6segments_M1_vs_M3.png"), np.vstack(montage_rows))\n\n        if imageio is not None:\n            gif_path = os.path.join(out_dir, "lineart_test_comparison_M1_vs_M3.gif")\n            imageio.mimsave(gif_path, gif_frames, duration=0.14, loop=0)\n            print(f"[✓] Đã xuất ảnh động đối đầu GIF: {gif_path}")\n\n        mp4_path = os.path.join(out_dir, "lineart_test_comparison_M1_vs_M3.mp4")\n        fourcc = cv2.VideoWriter_fourcc(*\'mp4v\')\n        vw = cv2.VideoWriter(mp4_path, fourcc, 8.0, (gif_frames[0].shape[1], gif_frames[0].shape[0]))\n        for gf in gif_frames:\n            vw.write(gf[:, :, ::-1])\n        vw.release()\n        print(f"[✓] Đã xuất video đối đầu MP4: {mp4_path}")\n\n    return df_sum2\n\n\n# ==============================================================================\n# BENCHMARK #3: DEFENSE GEOMETRY SUITE (HÌNH TRÒN THU/PHÓNG, DỊCH XA, NỀN TĨNH)\n# ==============================================================================\ndef extract_hybrid_contour_matches(ink0, ink1, n_pts_per_comp=32):\n    """Ghép cặp đường bao ngoài theo cụm liên thông + lấy mẫu đều theo độ dài cung."""\n    if cv2 is None:\n        return np.zeros((0, 2), dtype=np.float32), np.zeros((0, 2), dtype=np.float32)\n    h, w = ink0.shape\n\n    def _extract(ink):\n        mask = (ink > 0.20).astype(np.uint8) * 255\n        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n        feats = []\n        for c in cnts:\n            peri = cv2.arcLength(c, True)\n            if peri < 15.0:\n                continue\n            pts_raw = c.reshape(-1, 2).astype(np.float64)\n            diffs = np.diff(np.vstack([pts_raw, pts_raw[0]]), axis=0)\n            seglen = np.linalg.norm(diffs, axis=1)\n            cum = np.concatenate([[0.0], np.cumsum(seglen)])\n            if cum[-1] < 1e-3:\n                continue\n            targets = np.linspace(0.0, cum[-1], n_pts_per_comp, endpoint=False)\n            pts = np.zeros((n_pts_per_comp, 2), dtype=np.float64)\n            for k, d in enumerate(targets):\n                idx = min(np.searchsorted(cum, d, side="right") - 1, len(pts_raw) - 1)\n                alpha = (d - cum[idx]) / max(seglen[idx], 1e-6)\n                pts[k] = (1.0 - alpha) * pts_raw[idx] + alpha * pts_raw[(idx + 1) % len(pts_raw)]\n            cx_m, cy_m = pts.mean(axis=0)\n            ang = np.arctan2(pts[:, 1] - cy_m, pts[:, 0] - cx_m)\n            pts = np.roll(pts, -int(np.argmin(np.abs(ang))), axis=0)\n            area_s = 0.5 * np.sum(pts[:, 0] * np.roll(pts[:, 1], -1) - np.roll(pts[:, 0], -1) * pts[:, 1])\n            if area_s < 0:\n                pts = np.vstack([pts[0:1], pts[1:][::-1]])\n            pts[:, 0] /= max(1.0, w - 1.0)\n            pts[:, 1] /= max(1.0, h - 1.0)\n            _, _, bw, bh = cv2.boundingRect(c)\n            feats.append({\'pts\': pts.astype(np.float32), \'cx\': cx_m / w, \'cy\': cy_m / h, \'aspect\': bw / max(1.0, bh)})\n        return feats\n\n    f0, f1 = _extract(ink0), _extract(ink1)\n    if not f0 or not f1:\n        return np.zeros((0, 2), dtype=np.float32), np.zeros((0, 2), dtype=np.float32)\n\n    cost = np.zeros((len(f0), len(f1)), dtype=np.float64)\n    for i, a in enumerate(f0):\n        for j, b in enumerate(f1):\n            cost[i, j] = 2.0 * abs(a[\'cy\'] - b[\'cy\']) + 0.3 * abs(a[\'cx\'] - b[\'cx\']) + 1.5 * abs(math.log(max(a[\'aspect\'], 0.1) / max(b[\'aspect\'], 0.1)))\n    try:\n        from scipy.optimize import linear_sum_assignment\n        ri, ci = linear_sum_assignment(cost)\n    except Exception:\n        ri = np.arange(min(len(f0), len(f1)))\n        ci = np.arange(min(len(f0), len(f1)))\n    return np.vstack([f0[i][\'pts\'] for i in ri]), np.vstack([f1[j][\'pts\'] for j in ci])\n\n\ndef run_benchmark_defense_geometry_suite(models_dict, device, out_dir, downscale_factor=2):\n    """Kiểm định 4 kịch bản hình học dịch chuyển lớn + co giãn kích thước bằng Hybrid Contour + Lagrangian TPS."""\n    if cv2 is None:\n        print("[*] Thiếu OpenCV, bỏ qua Benchmark #3 Defense Geometry Suite.")\n        return None\n    os.makedirs(out_dir, exist_ok=True)\n    H, W = 384, 640\n    t_list = [0.25, 0.50, 0.75]\n\n    def _draw_ellipse_case(cx0, cy0, rx0, ry0, th0, cx1, cy1, rx1, ry1, th1, static_line=False):\n        def _fn(t):\n            im = np.zeros((H, W), dtype=np.float32)\n            cx = int(round((1.0 - t) * cx0 + t * cx1))\n            cy = int(round((1.0 - t) * cy0 + t * cy1))\n            rx = int(round((1.0 - t) * rx0 + t * rx1))\n            ry = int(round((1.0 - t) * ry0 + t * ry1))\n            th = max(2, int(round((1.0 - t) * th0 + t * th1)))\n            cv2.ellipse(im, (cx, cy), (rx, ry), 0, 0, 360, 1.0, th, lineType=cv2.LINE_AA)\n            if static_line:\n                cv2.line(im, (80, 330), (560, 330), 1.0, 4, lineType=cv2.LINE_AA)\n            return im\n        return _fn\n\n    cases = [\n        ("ShrinkingCircle_75to32", _draw_ellipse_case(120, 192, 75, 75, 5, 520, 192, 32, 32, 4, False)),\n        ("ExpandingCircle_30to70", _draw_ellipse_case(120, 192, 30, 30, 4, 520, 192, 70, 70, 5, False)),\n        ("ShrinkingCircle_StaticGround", _draw_ellipse_case(120, 180, 75, 75, 5, 520, 180, 32, 32, 4, True)),\n        ("SquashOval90x40_to_Circle45", _draw_ellipse_case(140, 192, 90, 40, 4, 490, 192, 45, 45, 4, False)),\n    ]\n\n    hw_t = torch.tensor([[H, W]], dtype=torch.int32, device=device)\n    rows_per_case = []\n    tags = [\'M0\'] + list(models_dict.keys())\n\n    for cname, gt_fn in cases:\n        im0, im1 = gt_fn(0.0), gt_fn(1.0)\n        k0, k1 = extract_hybrid_contour_matches(im0, im1, n_pts_per_comp=32)\n        x0 = torch.from_numpy(im0)[None, None].to(device)\n        x1 = torch.from_numpy(im1)[None, None].to(device)\n\n        vis_rows = {t: [] for t in tags}\n        gt_imgs = []\n        for tv in t_list:\n            gt_im = gt_fn(tv)\n            gt_imgs.append(gt_im)\n            gt_t = torch.from_numpy(gt_im)[None, None].to(device)\n            if solve_multi_layer_tps is not None:\n                cf0t, cf1t, _ = solve_multi_layer_tps(\n                    k0, k1, (H, W), H, W, device,\n                    t_val=tv, direct_t_flows=True, max_pts=128, lambd=0.01,\n                    downscale_factor=downscale_factor, eps_static_px=1.5,\n                    use_dyn_anchors=False, transition_width_px=18.0, falloff_radius_px=160.0\n                )\n            else:\n                kt = (1.0 - tv) * k0 + tv * k1\n                c_t0, th_t0, v0 = solve_tps_direct(kt, k0, max_pts=128, lambd=0.01, use_anchors=False)\n                c_t1, th_t1, v1 = solve_tps_direct(kt, k1, max_pts=128, lambd=0.01, use_anchors=False)\n                cf0t = tps_flow_from_theta_batch(torch.from_numpy(th_t0)[None].to(device), torch.from_numpy(c_t0)[None].to(device), torch.tensor([v0], device=device), hw_t, H, W, device, downscale_factor)\n                cf1t = tps_flow_from_theta_batch(torch.from_numpy(th_t1)[None].to(device), torch.from_numpy(c_t1)[None].to(device), torch.tensor([v1], device=device), hw_t, H, W, device, downscale_factor)\n            tt = torch.tensor([[[[tv]]]], dtype=torch.float32, device=device)\n\n            rec = {\'case\': cname, \'t_val\': tv}\n            cw_done = False\n            for m_name, mdl in models_dict.items():\n                with torch.no_grad(), torch.amp.autocast(\'cuda\', enabled=False):\n                    pr, cw, *_ = mdl(x0.float(), x1.float(), tt, cf0t.float(), cf1t.float(), direct_t_flows=True)\n                if not cw_done:\n                    vis_rows[\'M0\'].append(ink_to_u8_gray(cw))\n                    mm0 = evaluate_metrics_dual_regime(cw, gt_t)\n                    for mk, mv in mm0.items():\n                        rec[f"M0_{mk}"] = float(mv.item())\n                    cw_done = True\n                vis_rows[m_name].append(ink_to_u8_gray(pr))\n                mm = evaluate_metrics_dual_regime(pr, gt_t)\n                for mk, mv in mm.items():\n                    rec[f"{m_name}_{mk}"] = float(mv.item())\n            rows_per_case.append(rec)\n\n        # Lưu ảnh lưới từng case\n        cols_all = [tags + [\'GT\']]\n        panel_rows = []\n        for tg in tags:\n            row_u8 = [(255.0 - im0 * 255.0).clip(0, 255).astype(np.uint8)] + vis_rows[tg] + [(255.0 - im1 * 255.0).clip(0, 255).astype(np.uint8)]\n            strip = np.hstack([cv2.cvtColor(u, cv2.COLOR_GRAY2BGR) for u in row_u8])\n            cv2.putText(strip, tg, (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (20, 110, 220), 2, cv2.LINE_AA)\n            panel_rows.append(strip)\n        gt_u8_row = [(255.0 - im0 * 255.0).clip(0, 255).astype(np.uint8)] + [(255.0 - g * 255.0).clip(0, 255).astype(np.uint8) for g in gt_imgs] + [(255.0 - im1 * 255.0).clip(0, 255).astype(np.uint8)]\n        gt_strip = np.hstack([cv2.cvtColor(u, cv2.COLOR_GRAY2BGR) for u in gt_u8_row])\n        cv2.putText(gt_strip, "GroundTruth", (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (20, 150, 40), 2, cv2.LINE_AA)\n        panel_rows.append(gt_strip)\n        cv2.imwrite(os.path.join(out_dir, f"{cname}_grid.png"), np.vstack(panel_rows))\n\n    df_cases = pd.DataFrame(rows_per_case)\n    df_cases.to_csv(os.path.join(out_dir, "defense_geometry_per_frame.csv"), index=False)\n\n    sum_rows = []\n    for tg in tags:\n        r = {\'model_tag\': tg, \'frames\': len(df_cases)}\n        for mc in [\'psnr\', \'wcd_dog\', \'f1\', \'density_ratio\', \'ghost\']:\n            r[mc] = float(df_cases[f"{tg}_{mc}"].mean())\n        sum_rows.append(r)\n    df_sum3 = pd.DataFrame(sum_rows)\n    df_sum3.to_csv(os.path.join(out_dir, "defense_geometry_summary.csv"), index=False)\n\n    print("\\n📐 KẾT QUẢ BENCHMARK #3 (DEFENSE GEOMETRY SUITE - LAGRANGIAN TPS):")\n    print(df_sum3.round(4).to_string(index=False))\n    return df_sum3\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Chạy Triple-Benchmark đối chứng khách quan (M0 vs M1 vs M3)")\n    parser.add_argument(\'--bench_dir\', type=str, default=SCRIPT_DIR)\n    parser.add_argument(\'--orig_ckpt\', type=str, default=DEFAULT_ORIG_CKPT)\n    parser.add_argument(\'--finetuned_ckpt\', type=str, default=None, help="Đường dẫn model_tps_finetuned_best.pt")\n    parser.add_argument(\'--last_ckpt\', type=str, default=None, help="Đường dẫn model_tps_finetuned_last.pt (tùy chọn)")\n    parser.add_argument(\'--lineart_test_dir\', type=str, default=DEFAULT_LINEART_TEST)\n    parser.add_argument(\'--gluestick_npz\', type=str, default=DEFAULT_GLUESTICK_CACHE)\n    parser.add_argument(\'--out_dir\', type=str, default=os.path.join(SCRIPT_DIR, "benchmark_results"))\n    parser.add_argument(\'--downscale\', type=int, default=2)\n    args = parser.parse_args()\n\n    device = torch.device(\'cuda:0\' if torch.cuda.is_available() else \'cpu\')\n    models_dict = {}\n\n    if os.path.exists(args.orig_ckpt):\n        m1 = TPSInbetweenOriginal().to(device)\n        load_tps_weights(m1, args.orig_ckpt, device=device, prefer_ema=False)\n        models_dict[\'M1_Original\'] = m1.eval()\n    else:\n        print(f"[!] Không tìm thấy Original Checkpoint: {args.orig_ckpt}")\n\n    if args.finetuned_ckpt and os.path.exists(args.finetuned_ckpt):\n        m3_b = TPSInbetweenOriginal().to(device)\n        load_tps_weights(m3_b, args.finetuned_ckpt, device=device, prefer_ema=True)\n        models_dict[\'M3_Best\'] = m3_b.eval()\n\n    if args.last_ckpt and os.path.exists(args.last_ckpt):\n        m3_l = TPSInbetweenOriginal().to(device)\n        load_tps_weights(m3_l, args.last_ckpt, device=device, prefer_ema=True)\n        models_dict[\'M3_Last\'] = m3_l.eval()\n\n    if not models_dict:\n        print("[!] Chưa nạp được mô hình nào để đối chuẩn!")\n        sys.exit(1)\n\n    run_benchmark_isolated_sequences(\n        args.bench_dir, models_dict, device,\n        os.path.join(args.out_dir, "bench1_isolated_sequences"),\n        downscale_factor=args.downscale, save_visuals=True\n    )\n    run_benchmark_continuous_lineart_test(\n        args.lineart_test_dir, models_dict, device,\n        os.path.join(args.out_dir, "bench2_continuous_lineart_test"),\n        gluestick_npz=args.gluestick_npz, downscale_factor=args.downscale\n    )\n    run_benchmark_defense_geometry_suite(\n        models_dict, device,\n        os.path.join(args.out_dir, "bench3_defense_geometry"),\n        downscale_factor=args.downscale\n    )\n    print(f"\\n🎉 HOÀN TẤT BÁO CÁO ĐỐI CHUẨN TẠI: {args.out_dir}")\n\n\nif __name__ == \'__main__\':\n    main()\n')

GLUESTICK_TEST_NPZ = os.path.join(WORKING_DIR, "lineart_test_gluestick_matches.npz")
with open(GLUESTICK_TEST_NPZ, "wb") as f:
    f.write(base64.b64decode('...'))

print(f"[✓] Đã ghi model_occlusion.py ({os.path.getsize(model_occ_path):,} B)")
print(f"[✓] Đã ghi ddp_worker_tps_occlusion.py ({os.path.getsize(worker_occ_path):,} B)")
print(f"[✓] Đã ghi multi_layer_tps.py ({os.path.getsize(ml_path):,} B)")
print(f"[✓] Đã ghi run_benchmark_suite.py ({os.path.getsize(bench_script_path):,} B)")
sys.stdout.flush()

In [ ]:
# ==============================================================================
# ⚡ [CELL 2]: ĐỊNH VỊ TOÀN BỘ INPUTS, WARM-START CHECKPOINT (PHA 3 REMAKE BEST)
# ==============================================================================
import os
import glob
import shutil
import zipfile
import pandas as pd

SSD_ROOT = "/tmp/tps_sequence_ssd" if os.path.exists("/tmp") else os.path.join(WORKING_DIR, "_ssd_cache")
TRAIN_STAGE_DIR = os.path.join(SSD_ROOT, "train_set")
BENCH_STAGE_DIR = os.path.join(SSD_ROOT, "bench_set")
LINEART_TEST_STAGE = os.path.join(SSD_ROOT, "lineart_test")
EXTRACT_CACHE_DIR = os.path.join(SSD_ROOT, "extracted_archives")
for d in [SSD_ROOT, TRAIN_STAGE_DIR, BENCH_STAGE_DIR, LINEART_TEST_STAGE, EXTRACT_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

SEARCH_ROOTS = [
    "/kaggle/input",
    "/kaggle/input/datasets",
    "/kaggle/input/tps-baseline-model",
    "/kaggle/input/datasets/dungbaoviec123/tps-baseline-model",
    SSD_ROOT,
] if os.path.exists("/kaggle/input") else [
    SSD_ROOT,
    "E:/CODE/PBL/PBL6/Dataset/sakuga_pipeline_sequence/final_dataset",
    "E:/CODE/PBL/PBL6/data/lineart_experiments/test",
    "E:/CODE/PBL/PBL6/checkpoints/Re-train/Sequence 2 pha 3 remake/checkpoints",
    "E:/CODE/PBL/PBL6/checkpoints/Re-train/Sequence 2 Blended/checkpoints",
    "E:/CODE/PBL/PBL6/checkpoints/pretrained_baselines/tps-original",
    os.path.abspath("."),
]

def find_file_in_roots(filename, roots=None):
    if roots is None:
        roots = SEARCH_ROOTS
    for root_base in roots:
        if not os.path.exists(root_base):
            continue
        hits = glob.glob(os.path.join(root_base, "**", filename), recursive=True)
        if hits:
            hits.sort(key=lambda p: len(p))
            return hits[0]
    return None

def safe_stage_dir_links(src_root, dst_root, subdirs):
    for sub in subdirs:
        src_p = os.path.join(src_root, sub)
        dst_p = os.path.join(dst_root, sub)
        if not os.path.exists(src_p) or os.path.abspath(src_p) == os.path.abspath(dst_p):
            continue
        if os.path.islink(dst_p) or (os.path.isdir(dst_p) and len(os.listdir(dst_p)) > 0):
            continue
        if os.path.isdir(dst_p):
            try:
                os.rmdir(dst_p)
            except Exception:
                pass
        try:
            os.symlink(src_p, dst_p, target_is_directory=True)
        except Exception:
            shutil.copytree(src_p, dst_p, dirs_exist_ok=True)

def safe_copy_file(src_p, dst_p):
    if os.path.abspath(src_p) != os.path.abspath(dst_p):
        shutil.copy2(src_p, dst_p)

print("=" * 84)
print("🔍 [CELL 2]: ĐANG TỰ ĐỘNG ĐỊNH VỊ CHECKPOINT & DATASET CHO PHA 4 OCCLUSION...")
print("=" * 84)

already_extracted_zips = set()
for rb in [r for r in SEARCH_ROOTS if r != SSD_ROOT and os.path.exists(r)]:
    candidate_zips = glob.glob(os.path.join(rb, "*.zip")) + glob.glob(os.path.join(rb, "*", "*.zip"))
    for zpath in candidate_zips:
        if zpath in already_extracted_zips:
            continue
        try:
            with zipfile.ZipFile(zpath, 'r') as zf:
                names = zf.namelist()
                if any(n.endswith(("train_sequences.csv", "benchmark_sequences.csv", "continuous_keyframes_spec.json")) for n in names):
                    z_stem = os.path.splitext(os.path.basename(zpath))[0]
                    dest_extract = os.path.join(EXTRACT_CACHE_DIR, z_stem)
                    if not os.path.exists(dest_extract) or len(os.listdir(dest_extract)) == 0:
                        os.makedirs(dest_extract, exist_ok=True)
                        print(f"📦 Giải nén {os.path.basename(zpath)} sang SSD...", flush=True)
                        zf.extractall(dest_extract)
                    already_extracted_zips.add(zpath)
        except Exception as e:
            print(f"   [!] Bỏ qua zip lỗi {zpath}: {e}")

# 1. Checkpoint Warm-Start: Ưu tiên model_tps_finetuned_best.pt (Phase 3 Remake Best)
P3_BEST_CHECKPOINT_PATH = find_file_in_roots("model_tps_finetuned_best.pt")
if not P3_BEST_CHECKPOINT_PATH:
    P3_BEST_CHECKPOINT_PATH = find_file_in_roots("model_tps_blended.pt")
if not P3_BEST_CHECKPOINT_PATH:
    P3_BEST_CHECKPOINT_PATH = find_file_in_roots("model_latest.pt")
assert P3_BEST_CHECKPOINT_PATH and os.path.exists(P3_BEST_CHECKPOINT_PATH), "❌ Thiếu checkpoint warm-start!"
print(f"[✓] 1. Warm-Start Checkpoint (Pha 3 Remake) : {P3_BEST_CHECKPOINT_PATH}")

BLENDED_CKPT_PATH = find_file_in_roots("model_tps_blended.pt")
M1_CHECKPOINT_PATH = find_file_in_roots("model_latest.pt")

# 2. Dataset Sequences
train_csv_found = find_file_in_roots("train_sequences.csv")
assert train_csv_found, "❌ Thiếu `train_sequences.csv`!"
TRAIN_SRC_ROOT = os.path.dirname(train_csv_found)
safe_stage_dir_links(TRAIN_SRC_ROOT, TRAIN_STAGE_DIR, ["train", "val", "matches"])
safe_copy_file(train_csv_found, os.path.join(TRAIN_STAGE_DIR, "train_sequences.csv"))
safe_copy_file(os.path.join(TRAIN_SRC_ROOT, "val_sequences.csv"), os.path.join(TRAIN_STAGE_DIR, "val_sequences.csv"))

PRECOMPUTED_ML_CACHE_SRC = find_file_in_roots("tps_cache_multilayer.npz")
TRAIN_CLIPS_DIR = os.path.join(TRAIN_STAGE_DIR, "train")
VAL_CLIPS_DIR = os.path.join(TRAIN_STAGE_DIR, "val")
print(f"[✓] 2. Dataset Train/Val Sequence          : {TRAIN_SRC_ROOT}")

# 3. Benchmark sets
bench_csv_found = find_file_in_roots("benchmark_sequences.csv")
BENCH_READY, BENCH_ROOT_PATH = False, None
if bench_csv_found and os.path.exists(bench_csv_found):
    b_src = os.path.dirname(bench_csv_found)
    safe_stage_dir_links(b_src, BENCH_STAGE_DIR, ["clips", "matches"])
    safe_copy_file(bench_csv_found, os.path.join(BENCH_STAGE_DIR, "benchmark_sequences.csv"))
    BENCH_ROOT_PATH, BENCH_READY = BENCH_STAGE_DIR, True
    print(f"[✓] 3. Benchmark #1 (Isolated 198 clips)   : {b_src}")

spec_found = find_file_in_roots("continuous_keyframes_spec.json")
LINEART_TEST_READY, LINEART_TEST_PATH = False, None
if spec_found and os.path.exists(spec_found):
    l_src = os.path.dirname(spec_found)
    for fn in os.listdir(l_src):
        src_f = os.path.join(l_src, fn)
        if os.path.isfile(src_f):
            safe_copy_file(src_f, os.path.join(LINEART_TEST_STAGE, fn))
    LINEART_TEST_PATH, LINEART_TEST_READY = LINEART_TEST_STAGE, True
    print(f"[✓] 4. Benchmark #2 (Lineart Test 24f)     : {l_src}")
print("=" * 84)

In [ ]:
# ==============================================================================
# 🚀 [CELL 3]: NẠP HOẶC TẠO BỘ CACHE ĐA LỚP `tps_cache_multilayer.npz`
# ==============================================================================
import os
import shutil
import pandas as pd
from build_tps_cache_multilayer import build_multilayer_cache

TPS_CACHE_MULTILAYER = os.path.join(SSD_ROOT, "tps_cache_multilayer.npz")
TRAIN_CSV_IDX_PATH = os.path.join(TRAIN_STAGE_DIR, "train_sequences_indexed.csv")
VAL_CSV_IDX_PATH = os.path.join(TRAIN_STAGE_DIR, "val_sequences_indexed.csv")

if PRECOMPUTED_ML_CACHE_SRC and os.path.exists(PRECOMPUTED_ML_CACHE_SRC):
    shutil.copy2(PRECOMPUTED_ML_CACHE_SRC, TPS_CACHE_MULTILAYER)
    df_tr = pd.read_csv(os.path.join(TRAIN_STAGE_DIR, "train_sequences.csv"))
    df_va = pd.read_csv(os.path.join(TRAIN_STAGE_DIR, "val_sequences.csv"))
    df_tr['cache_idx'] = range(len(df_tr))
    df_va['cache_idx'] = range(len(df_tr), len(df_tr) + len(df_va))
    df_tr.to_csv(TRAIN_CSV_IDX_PATH, index=False)
    df_va.to_csv(VAL_CSV_IDX_PATH, index=False)
    print(f"[♻️] Đã nạp trực tiếp `tps_cache_multilayer.npz` ({os.path.getsize(TPS_CACHE_MULTILAYER)/(1024**2):.2f} MB)!")
elif os.path.exists(TPS_CACHE_MULTILAYER) and os.path.exists(TRAIN_CSV_IDX_PATH) and os.path.exists(VAL_CSV_IDX_PATH):
    print(f"[♻️] Phát hiện cache sẵn có tại {TPS_CACHE_MULTILAYER}!")
else:
    build_multilayer_cache(
        TRAIN_STAGE_DIR,
        out_npz_path=TPS_CACHE_MULTILAYER,
        max_pts=128, max_clusters=4, max_mask_pts=64, max_static_pts=96,
        lambd=0.01, eps_static_px=1.5
    )

In [ ]:
# ==============================================================================
# 🔬 [CELL 4]: KIỂM TOÁN KIẾN TRÚC MÔ HÌNH PHA 4 & WARM-START TỪ PHA 3 REMAKE
# ==============================================================================
import gc
import torch
from model_occlusion import TPSInbetweenOcclusion, load_tps_occlusion_weights

m4 = TPSInbetweenOcclusion()
load_tps_occlusion_weights(m4, P3_BEST_CHECKPOINT_PATH, device='cpu', prefer_ema=True, is_rank0=True)
p_tot = sum(p.numel() for p in m4.parameters())
print(f"[✓] TỔNG ACTIVE PARAMS (Phase 4 Occlusion) = {p_tot:,}")
print(f"[✓] IFNet.block2.lastconv weight shape      = {tuple(m4.IFNet.block2.lastconv.weight.shape)} (6 channels)")
print(f"[✓] IFNet.block2.lastconv bias shape        = {tuple(m4.IFNet.block2.lastconv.bias.shape)} (6 channels)")
print(f"[✓] Khởi tạo bias Occlusion Head            = {m4.IFNet.block2.lastconv.bias[4:].detach().tolist()} (Sigmoid ≈ 0.88)")

del m4
gc.collect()

In [ ]:
# ==============================================================================
# 🚀 [CELL 5]: CHẠY HUẤN LUYỆN PHA 4 OCCLUSION HEAD (6 EPOCHS | 2x GPU DDP)
# ==============================================================================
import os
import sys
import subprocess
import torch

NUM_GPUS = max(1, torch.cuda.device_count())
TOTAL_EPOCHS = 6
GRAD_ACCUM = 2
MAX_CHUNK_FRAMES = 3
LR_BACKBONE = 6.0e-6
LR_UNET = 1.8e-5
EMA_DECAY = 0.996
W_PHOTO = 0.25
W_COV = 0.50
W_SMOOTH = 0.10
W_GHOST = 0.32
W_TEMP = 0.55

cmd = [
    "torchrun",
    f"--nproc_per_node={NUM_GPUS}",
    "--master_port=29519",
    worker_occ_path,
    "--train_csv", TRAIN_CSV_IDX_PATH,
    "--val_csv", VAL_CSV_IDX_PATH,
    "--train_clips_dir", TRAIN_CLIPS_DIR,
    "--val_clips_dir", VAL_CLIPS_DIR,
    "--tps_cache", TPS_CACHE_MULTILAYER,
    "--init_checkpoint", P3_BEST_CHECKPOINT_PATH,
    "--output_dir", OUTPUT_CKPT_DIR,
    "--epochs", str(TOTAL_EPOCHS),
    "--grad_accum", str(GRAD_ACCUM),
    "--max_chunk_frames", str(MAX_CHUNK_FRAMES),
    "--lr_backbone", str(LR_BACKBONE),
    "--lr_unet", str(LR_UNET),
    "--ema_decay", str(EMA_DECAY),
    "--w_photo", str(W_PHOTO),
    "--w_cov", str(W_COV),
    "--w_smooth", str(W_SMOOTH),
    "--w_ghost", str(W_GHOST),
    "--w_temp", str(W_TEMP),
    "--num_workers", "2",
]

print("=" * 84)
print(f"🚀 BẮT ĐẦU HUẤN LUYỆN PHA 4 OCCLUSION ({NUM_GPUS}x GPU | {TOTAL_EPOCHS} EPOCHS)")
print(f"   Warm-Start: {P3_BEST_CHECKPOINT_PATH}")
print(f"   Outputs   : {OUTPUT_CKPT_DIR}")
print("=" * 84, flush=True)

sub_env = os.environ.copy()
sub_env["PYTHONUNBUFFERED"] = "1"
sub_env["OMP_NUM_THREADS"] = "1"
sub_env["MKL_NUM_THREADS"] = "1"

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=sub_env)
for line in iter(proc.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()
proc.stdout.close()
ret = proc.wait()

if ret != 0:
    raise RuntimeError(f"❌ DDP Worker kết thúc bất thường với mã {ret}!")
else:
    print("🎉 HOÀN TẤT HUẤN LUYỆN PHA 4 OCCLUSION HEAD!")

In [ ]:
# ==============================================================================
# 🔄 [CELL 6 - KHẨN CẤP]: RESET KERNEL TRÁNH TRÀN RAM
# ==============================================================================
# import os
# os._exit(0)

In [ ]:
# ==============================================================================
# 🎬 [CELL 7]: BENCHMARK ĐỐI ĐẦU 4 PHIÊN BẢN (PHA 2 vs PHA 3 vs PHA 4 OCCLUSION)
# ==============================================================================
import os
import gc
import torch
import pandas as pd
from IPython.display import display, Image as IPyImage
from ddp_worker_tps_sequence_phase2 import TPSInbetweenOriginal, load_tps_weights
from model_occlusion import TPSInbetweenOcclusion, load_tps_occlusion_weights
from run_benchmark_suite import run_benchmark_continuous_lineart_test, run_benchmark_defense_geometry_suite

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
models_dict = {}

# 1. Pha 2 Blended
if BLENDED_CKPT_PATH and os.path.exists(BLENDED_CKPT_PATH):
    m_p2 = TPSInbetweenOriginal().to(device)
    load_tps_weights(m_p2, BLENDED_CKPT_PATH, device=device, prefer_ema=True)
    models_dict['Pha2_Blended'] = m_p2.eval()

# 2. Pha 3 Remake Raw
if P3_BEST_CHECKPOINT_PATH and os.path.exists(P3_BEST_CHECKPOINT_PATH):
    m_p3 = TPSInbetweenOriginal().to(device)
    load_tps_weights(m_p3, P3_BEST_CHECKPOINT_PATH, device=device, prefer_ema=True)
    models_dict['Pha3_Remake_Raw'] = m_p3.eval()

# 3. Pha 4 Occlusion Head
p4_best_ckpt = os.path.join(OUTPUT_CKPT_DIR, "model_tps_occlusion_best.pt")
p4_last_ckpt = os.path.join(OUTPUT_CKPT_DIR, "model_tps_occlusion_last.pt")

if os.path.exists(p4_best_ckpt):
    m_p4 = TPSInbetweenOcclusion().to(device)
    load_tps_occlusion_weights(m_p4, p4_best_ckpt, device=device, prefer_ema=True)
    models_dict['Pha4_Occlusion_Best'] = m_p4.eval()
elif os.path.exists(p4_last_ckpt):
    m_p4 = TPSInbetweenOcclusion().to(device)
    load_tps_occlusion_weights(m_p4, p4_last_ckpt, device=device, prefer_ema=True)
    models_dict['Pha4_Occlusion_Last'] = m_p4.eval()

print(f"📊 Đã nạp thành công {len(models_dict)} mô hình đối đầu: {list(models_dict.keys())}")

# Chạy Benchmark Case 1: Lineart Test 24 frames
BENCH2_OUT = os.path.join(BENCH_OUT_DIR, "bench_lineart_test_24f")
if LINEART_TEST_READY:
    df_sum2 = run_benchmark_continuous_lineart_test(
        LINEART_TEST_PATH, models_dict, device, BENCH2_OUT,
        gluestick_npz=GLUESTICK_TEST_NPZ, downscale_factor=2, use_multilayer=True
    )
    if df_sum2 is not None:
        print("📌 KẾT QUẢ CHUỖI 24 FRAMES THỰC TẾ:")
        display(df_sum2.round(4))

# Chạy Benchmark Case 2, 3, 4: Defense Geometry Suite
BENCH3_OUT = os.path.join(BENCH_OUT_DIR, "bench_defense_geometry")
df_sum3 = run_benchmark_defense_geometry_suite(models_dict, device, BENCH3_OUT, downscale_factor=2)
if df_sum3 is not None:
    print("📌 KẾT QUẢ BỘ DEFENSE GEOMETRY SUITE:")
    display(df_sum3.round(4))

del models_dict
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# 📈 [CELL 8]: DASHBOARD HỘI TỤ TRAINING PHA 4 OCCLUSION
# ==============================================================================
import os
import pandas as pd
from IPython.display import display

hist_csv = os.path.join(OUTPUT_CKPT_DIR, "training_history_occlusion.csv")
if os.path.exists(hist_csv):
    df_h = pd.read_csv(hist_csv)
    print("📈 LỊCH SỬ HUẤN LUYỆN PHA 4 OCCLUSION HEAD:")
    display(df_h.round(4))
else:
    print("[*] Chưa tìm thấy training_history_occlusion.csv.")

In [ ]:
# ==============================================================================
# 📦 [CELL 9]: ĐÓNG GÓI CHECKPOINT & KẾT QUẢ BENCHMARK PHA 4 (.ZIP)
# ==============================================================================
import os
import zipfile
import shutil
import torch

RELEASE_ZIP = os.path.join(WORKING_DIR, "tps_sequence_phase4_occlusion_release.zip")
pack_items = []

for src_p, zip_name in [
    (os.path.join(OUTPUT_CKPT_DIR, "model_tps_occlusion_best.pt"), "checkpoints/model_tps_occlusion_best.pt"),
    (os.path.join(OUTPUT_CKPT_DIR, "model_tps_occlusion_last.pt"), "checkpoints/model_tps_occlusion_last.pt"),
    (os.path.join(OUTPUT_CKPT_DIR, "training_history_occlusion.csv"), "checkpoints/training_history_occlusion.csv"),
]:
    if os.path.exists(src_p):
        pack_items.append((src_p, zip_name))

if os.path.exists(BENCH_OUT_DIR):
    for root, _, files in os.walk(BENCH_OUT_DIR):
        for f in files:
            full_p = os.path.join(root, f)
            pack_items.append((full_p, os.path.relpath(full_p, WORKING_DIR)))

with zipfile.ZipFile(RELEASE_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for src_f, arc_name in pack_items:
        zf.write(src_f, arcname=arc_name)

print(f"📦 ĐÃ ĐÓNG GÓI THÀNH CÔNG {len(pack_items)} TỆP VÀO: {RELEASE_ZIP} ({os.path.getsize(RELEASE_ZIP)/(1024**2):.2f} MB)")